<div dir="rtl" align="right">

# Notebook 18 — Context Builder
## `18_context_builder.ipynb`

### هدف
ساخت Context مستند از خروجی Frozen نوت‌بوک ۱۷، با حذف تکرار، کنترل dominance یک سند، حفظ تنوع منبع و موضوع، بودجه‌بندی توکن، نگه‌داشتن locator و انتساب، و تشخیص شواهد متعارض.

### جایگاه در Pipeline
NB18 پس از `17_reranking.ipynb` و پیش از `19_generation_and_citation.ipynb` قرار دارد. ورودی‌ها باید از Artifactهای ذخیره‌شده و اعتبارسنجی‌شده خوانده شوند؛ استفاده از متغیرهای Kernel قبلی مجاز نیست.

### ورودی‌های Frozen گزارش‌شده در handoff NB17
- خروجی: `data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl`
- قرارداد تحویل: `artifacts/phase2/reranking/nb18_reranker_handoff_v1.json`
- Manifest، Release، Configuration و Reload Validation مربوط به NB17
- Chunkهای Canonical برای اتصال Candidateها به محتوا و metadata

تعداد خروجی Reranker برابر ۳۸۱۳ رکورد، `candidate_depth=50` و `consumer_top_k=10` گزارش شده است. پیش از مصرف، Hash، مسیر، وضعیت و Schema فایل‌های محلی دوباره اعتبارسنجی می‌شوند.

### خروجی‌های مورد انتظار
Contextهای نسخه‌دار، گزارش کنترل تکرار و تنوع، حفظ lineage و locator، پرچم تعارض، نتایج آزمون بودجه و truncation، Manifest، Acceptance و Handoff برای NB19.

### خارج از Scope
اجرای دوبارهٔ Reranker، تغییر تنظیمات یا Artifactهای Frozen NB16/NB17، تغییر Index یا Embedding، تولید پاسخ نهایی، حدس‌زدن metadata غایب و استفاده از API پولی.

### معیار موفقیت
ورودی‌ها Hash و قرارداد معتبر داشته باشند؛ تنظیمات Context فقط با Development انتخاب و پیش از Test فریز شوند؛ خروجی‌ها از دیسک مستقل بازخوانی و اعتبارسنجی شوند؛ Final Health و اجرای پاک Notebook پاس شوند.

</div>

<div dir="rtl" align="right">

## وضعیت شروع و برنامهٔ اجرا

NB16 و NB17 در شاخهٔ `rag-phase2` به‌عنوان Frozen تحویل شده‌اند. handoff واقعی NB17 وضعیت `READY_FOR_NB18` دارد و دستور می‌دهد Candidate order حفظ شود و تنظیمات یا امتیازهای Reranker تغییر یا دوباره محاسبه نشوند.

اولین گام، فقط‌خواندنی است: کشف ریشهٔ مخزن، بررسی سلامت ذخیره‌شدهٔ NB17، تطبیق Hash ورودی‌های Canonical و تأیید قرارداد تحویل. هیچ Artifactی ساخته یا بازنویسی نمی‌شود.

### قواعد جلوگیری از دوباره‌کاری
- از Artifactهای فریز‌شده استفاده می‌شود؛ Retrieval، Reranking، Embedding و Index دوباره اجرا نمی‌شوند.
- Hash و قرارداد ورودی پیش از پردازش بررسی می‌شوند.
- کارهای حجیم با Pilot، Checkpoint اتمیک و Resume طراحی می‌شوند؛ رکورد کامل‌شده دوباره پردازش نمی‌شود.
- هیچ API پولی استفاده نمی‌شود.
- Configuration فقط روی Development انتخاب و پیش از دسترسی تحلیلی به Test فریز می‌شود. پس از بازشدن Test، تغییر پارامتر یا Policy ممنوع است.
- هر Failure همان‌جا بررسی و اصلاح می‌شود؛ تا PASS واقعی مرحلهٔ بعد شروع نمی‌شود.
- Scope رسمی NB18 رعایت می‌شود و کار NB19 به این Notebook منتقل نمی‌شود.

نقشهٔ رسمی NB18 شش بخش کاربری اصلی و ده مورد اجرایی را فهرست می‌کند، اما تعداد Code Cell نهایی را الزام نکرده است. سلول‌ها فقط برای اجرای مستقل و اعتبارسنجی لازم ساخته و به‌ترتیب شماره‌گذاری می‌شوند.

این سلول صرفاً Preflight محلی و خواندنی است؛ مدل یا API اجرا نمی‌کند و فایل نمی‌نویسد.

</div>

In [1]:
# NB18-C01 — Imports and runtime bootstrap only

import hashlib
import json
import os

# Bound public tokenizer metadata/download waits before HF libraries load.
os.environ["HF_HUB_ETAG_TIMEOUT"] = "10"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"
import re
import subprocess
import sys
from pathlib import Path

In [2]:
# NB18-C02 — Read-only repository and frozen-input preflight

EXPECTED_FILES = {
    "nb17_handoff": (
        "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json",
        "6512c929587d6926da18e713a5c1a5baa0da3d863bf6ca389ad7cc90585e5f9d",
    ),
    "reranker_configuration": (
        "artifacts/phase2/reranking/reranker_configuration_v1.json",
        "dd2290188c4e99be1dada40b167b1041c69f7f077ebe6dea1fddcb58b4834a86",
    ),
    "reranker_release": (
        "artifacts/phase2/reranking/reranker_release_v1.json",
        "8f9006a2cbc425aaa68459928af1c15ab31e88e881d8fbd76a5b312dd601a9b4",
    ),
    "reranker_manifest": (
        "data/phase2/retrieval/reranker_v1/reranker_manifest.json",
        "230a720eab3fac9ef9f4fea8ab03152660f045fbda8926e22abcb37f809e6be5",
    ),
    "reranker_reload_validation": (
        "data/phase2/retrieval/reranker_v1/reranker_reload_validation_v1.json",
        "d575550c0d0a3feadfa27025c4f5b359eed5ddac4f91c1e720e303e49c188e4a",
    ),
    "reranker_acceptance": (
        "reports/phase2/reranker_acceptance_v1.json",
        "ab04570ed0c7a6bee37b367bb3c62564f75e23c80e0f3816538449872ecc344d",
    ),
    "reranked_candidates": (
        "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl",
        "d00285bbf3865fbd51041db1612b9fdc17e9778273ea6d136978906a07268b7a",
    ),
    "canonical_chunks": (
        "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
        "3891c4ecad9332ba753cd605bbccd99da9d063f64934a30ee39e495349cffb50",
    ),
    "golden_dataset": (
        "artifacts/phase2/golden_dataset_v2/golden_dataset_v2.jsonl",
        "b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2",
    ),
}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_handle:
        for block in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

current_path = Path.cwd().resolve()
project_root = None

for candidate_root in (current_path, *current_path.parents):
    if (
        (candidate_root / ".git").exists()
        and (candidate_root / "notebooks/phase2/17_reranking.ipynb").is_file()
    ):
        project_root = candidate_root
        break

assert project_root is not None, (
    "Project root not found from the current working directory."
)

print(f"PROJECT ROOT: {project_root}")

branch_result = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=project_root,
    capture_output=True,
    text=True,
    check=False,
)
head_result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=project_root,
    capture_output=True,
    text=True,
    check=False,
)
status_result = subprocess.run(
    ["git", "status", "--short"],
    cwd=project_root,
    capture_output=True,
    text=True,
    check=False,
)

assert branch_result.returncode == 0, branch_result.stderr
assert head_result.returncode == 0, head_result.stderr

print(f"BRANCH: {branch_result.stdout.strip()}")
print(f"HEAD: {head_result.stdout.strip()}")
print("GIT STATUS:")
print(status_result.stdout.strip() or "(clean)")

nb17_path = project_root / "notebooks/phase2/17_reranking.ipynb"
nb17 = json.loads(nb17_path.read_text(encoding="utf-8"))
nb17_code_cells = [
    cell for cell in nb17.get("cells", [])
    if cell.get("cell_type") == "code"
]

nb17_labels = []
nb17_errors = []
nb17_unexecuted = []

for index, cell in enumerate(nb17_code_cells, start=1):
    source = "".join(cell.get("source", []))
    label_match = re.search(r"Code Cell\s+(\d+)", source.splitlines()[0] if source else "")
    nb17_labels.append(int(label_match.group(1)) if label_match else None)

    if cell.get("execution_count") is None:
        nb17_unexecuted.append(index)

    for output in cell.get("outputs", []):
        if output.get("output_type") == "error":
            nb17_errors.append(
                {
                    "cell": index,
                    "ename": output.get("ename"),
                    "evalue": output.get("evalue"),
                }
            )

nb17_output_text = "\n".join(
    "".join(output.get("text", []))
    for cell in nb17_code_cells
    for output in cell.get("outputs", [])
    if output.get("output_type") == "stream"
)

print(f"NB17 CODE CELL COUNT: {len(nb17_code_cells)}")
print(f"NB17 LABELS: {nb17_labels}")
print(f"NB17 UNEXECUTED CELLS: {nb17_unexecuted}")
print(f"NB17 SAVED ERRORS: {nb17_errors}")
print(
    "NB17 FINAL HEALTH OUTPUT PRESENT: "
    f"{'NB17 FINAL HEALTH: PASS' in nb17_output_text}"
)

assert len(nb17_code_cells) == 22
assert nb17_labels == list(range(1, 23))
assert not nb17_unexecuted
assert not nb17_errors
assert "NB17 FINAL HEALTH: PASS" in nb17_output_text

nb18_path = project_root / "notebooks/phase2/18_context_builder.ipynb"
print(f"NB18 FILE EXISTS LOCALLY: {nb18_path.is_file()}")
if nb18_path.is_file():
    nb18 = json.loads(nb18_path.read_text(encoding="utf-8"))
    print(f"NB18 CURRENT CELL COUNT: {len(nb18.get('cells', []))}")

validated_hashes = {}

for name, (relative_path, expected_hash) in EXPECTED_FILES.items():
    file_path = project_root / relative_path
    assert file_path.is_file(), f"Required input is missing: {relative_path}"

    file_size = file_path.stat().st_size
    print(f"INPUT: {relative_path} | SIZE: {file_size} bytes")

    actual_hash = sha256_file(file_path)
    validated_hashes[name] = actual_hash
    assert actual_hash == expected_hash, (
        f"SHA256 mismatch for {relative_path}: "
        f"expected={expected_hash}, actual={actual_hash}"
    )

handoff = json.loads(
    (project_root / EXPECTED_FILES["nb17_handoff"][0]).read_text(encoding="utf-8")
)
configuration = json.loads(
    (project_root / EXPECTED_FILES["reranker_configuration"][0]).read_text(
        encoding="utf-8"
    )
)
release = json.loads(
    (project_root / EXPECTED_FILES["reranker_release"][0]).read_text(
        encoding="utf-8"
    )
)
manifest = json.loads(
    (project_root / EXPECTED_FILES["reranker_manifest"][0]).read_text(
        encoding="utf-8"
    )
)
reload_validation = json.loads(
    (project_root / EXPECTED_FILES["reranker_reload_validation"][0]).read_text(
        encoding="utf-8"
    )
)
acceptance = json.loads(
    (project_root / EXPECTED_FILES["reranker_acceptance"][0]).read_text(
        encoding="utf-8"
    )
)

assert handoff["status"] == "READY_FOR_NB18"
assert handoff["target_notebook"] == "18_context_builder.ipynb"
assert handoff["candidate_artifact"]["record_count"] == 3813
assert handoff["candidate_artifact"]["candidate_depth"] == 50
assert handoff["candidate_artifact"]["consumer_top_k"] == 10
assert handoff["candidate_artifact"]["sha256"] == validated_hashes["reranked_candidates"]
assert handoff["constraints"]["preserve_candidate_order"] is True
assert handoff["constraints"]["do_not_recompute_reranker_scores"] is True
assert handoff["constraints"]["do_not_change_reranker_configuration"] is True

assert configuration["configuration_status"] == "FROZEN_BEFORE_TEST_ACCESS"
assert configuration["configuration_sha256"] == handoff["configuration_sha256"]
assert release["status"] == "FROZEN_FOR_CONTEXT_BUILDING"
assert manifest["manifest_status"] == "PASS_COMPLETE_FROZEN"
assert reload_validation["reload_validation_status"] == "PASS"
assert acceptance["status"] == "PASS_COMPLETE_FROZEN"

print("FROZEN INPUT HASHES: PASS")
print("NB17 HANDOFF CONTRACT: PASS")
print("NB17 FROZEN STATUS CHECKS: PASS")
print("PREFLIGHT: PASS — READ ONLY; NO MODEL/API EXECUTED; NO FILES WRITTEN")

PROJECT ROOT: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
BRANCH: rag-phase2
HEAD: 1298cb5555ddd123d4f16aac25019c70dfd10afe
GIT STATUS:
?? notebooks/phase2/18_context_builder.ipynb
?? notebooks/phase2/18_context_builder_repaired.ipynb
NB17 CODE CELL COUNT: 22
NB17 LABELS: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
NB17 UNEXECUTED CELLS: []
NB17 SAVED ERRORS: []
NB17 FINAL HEALTH OUTPUT PRESENT: True
NB18 FILE EXISTS LOCALLY: True
NB18 CURRENT CELL COUNT: 149
INPUT: artifacts/phase2/reranking/nb18_reranker_handoff_v1.json | SIZE: 1303 bytes
INPUT: artifacts/phase2/reranking/reranker_configuration_v1.json | SIZE: 6596 bytes
INPUT: artifacts/phase2/reranking/reranker_release_v1.json | SIZE: 1788 bytes
INPUT: data/phase2/retrieval/reranker_v1/reranker_manifest.json | SIZE: 1495 bytes
INPUT: data/phase2/retrieval/reranker_v1/reranker_reload_validation_v1.json | SIZE: 1783 bytes
INPUT: reports/phase2/reranker_acceptance_v1.js

<div dir="rtl" align="right">

## ممیزی رکوردهای Frozen و آماده‌سازی Development

در این مرحله خروجی Reranker، Chunkهای Canonical و Golden Dataset از مسیرهای تأییدشده خوانده می‌شوند. Hash آن‌ها در سلول قبلی بررسی شده است؛ این سلول Hash‌گیری یا پردازش NB17 را تکرار نمی‌کند.

رکوردهای Development برای تحلیل‌های بعدی آماده می‌شوند. رکوردهای Test فقط با split و شناسه از Development جدا می‌شوند؛ شواهد آن‌ها برای انتخاب تنظیمات خوانده یا تحلیل نمی‌شود. ترتیب کاندیدها و رتبه‌های Frozen حفظ می‌شوند.

این سلول فقط قالب و پیوستگی IDها را اعتبارسنجی می‌کند. هیچ Policy، بودجه یا Configuration انتخاب نمی‌شود و هیچ Artifactی ساخته نمی‌شود.

**معیار PASS:** ۲۶۶۴ رکورد Development، ۱۱۴۹ رکورد Test جداشده، ۳۴۸۸ Chunk یکتا، تطبیق کامل Candidateهای Development با Golden Dataset و اتصال همهٔ Chunk IDهای Development به Corpus Canonical.

</div>

In [3]:
# NB18-C03 — Load and audit frozen input schemas; retain Development only

assert "project_root" in globals()
assert "EXPECTED_FILES" in globals()
assert "validated_hashes" in globals()

EXPECTED_DEVELOPMENT_COUNT = 2664
EXPECTED_TEST_COUNT = 1149
EXPECTED_CHUNK_COUNT = 3488
EXPECTED_RERANKER_DEPTH = 50
EXPECTED_FINAL_TOP_K = 10

candidate_path = project_root / EXPECTED_FILES["reranked_candidates"][0]
chunk_path = project_root / EXPECTED_FILES["canonical_chunks"][0]
golden_path = project_root / EXPECTED_FILES["golden_dataset"][0]

development_candidate_records = []
all_candidate_ids = set()
test_candidate_count = 0
candidate_schema_keys = None
candidate_item_schema_keys = None

with candidate_path.open("r", encoding="utf-8") as file_handle:
    for line_number, line in enumerate(file_handle, start=1):
        if not line.strip():
            raise AssertionError(
                f"Blank reranker record at line {line_number}"
            )

        record = json.loads(line)
        candidate_id = str(record["candidate_id"])
        split = record["split"]

        assert candidate_id not in all_candidate_ids, (
            "Duplicate candidate_id in reranker output",
            candidate_id,
        )
        all_candidate_ids.add(candidate_id)

        if split == "test":
            # Test records are excluded from Development analysis.
            test_candidate_count += 1
            continue

        assert split == "development", (
            "Unexpected split label",
            candidate_id,
            split,
        )

        candidates = record["candidates"]
        assert record["candidate_depth"] == EXPECTED_RERANKER_DEPTH
        assert record["final_top_k"] == EXPECTED_FINAL_TOP_K
        assert len(candidates) == EXPECTED_RERANKER_DEPTH

        expected_ranks = list(range(1, EXPECTED_RERANKER_DEPTH + 1))
        actual_ranks = [candidate["rank"] for candidate in candidates]
        assert actual_ranks == expected_ranks, (
            "Frozen candidate rank/order mismatch",
            candidate_id,
        )

        development_candidate_records.append(record)

        if candidate_schema_keys is None:
            candidate_schema_keys = sorted(record.keys())
            candidate_item_schema_keys = sorted(candidates[0].keys())

assert len(all_candidate_ids) == (
    EXPECTED_DEVELOPMENT_COUNT + EXPECTED_TEST_COUNT
)
assert len(development_candidate_records) == EXPECTED_DEVELOPMENT_COUNT
assert test_candidate_count == EXPECTED_TEST_COUNT

development_candidate_ids = {
    str(record["candidate_id"])
    for record in development_candidate_records
}
assert len(development_candidate_ids) == EXPECTED_DEVELOPMENT_COUNT

chunk_record_by_id = {}
chunk_schema_keys = None
canonical_source_types = set()

with chunk_path.open("r", encoding="utf-8") as file_handle:
    for line_number, line in enumerate(file_handle, start=1):
        if not line.strip():
            raise AssertionError(
                f"Blank canonical chunk at line {line_number}"
            )

        chunk = json.loads(line)
        chunk_id = str(chunk["chunk_id"])

        assert chunk_id not in chunk_record_by_id, (
            "Duplicate canonical chunk_id",
            chunk_id,
        )

        chunk_record_by_id[chunk_id] = chunk
        canonical_source_types.add(str(chunk["source_type"]))

        if chunk_schema_keys is None:
            chunk_schema_keys = sorted(chunk.keys())

assert len(chunk_record_by_id) == EXPECTED_CHUNK_COUNT

development_referenced_chunk_ids = {
    str(candidate["chunk_id"])
    for record in development_candidate_records
    for candidate in record["candidates"]
}

unknown_development_chunk_ids = (
    development_referenced_chunk_ids - set(chunk_record_by_id)
)
assert not unknown_development_chunk_ids, (
    "Development candidates reference unknown canonical chunks",
    sorted(unknown_development_chunk_ids)[:10],
)

development_golden_by_id = {}
golden_schema_keys = None
non_development_golden_count = 0

with golden_path.open("r", encoding="utf-8") as file_handle:
    for line_number, line in enumerate(file_handle, start=1):
        if not line.strip():
            raise AssertionError(
                f"Blank Golden Dataset record at line {line_number}"
            )

        golden_record = json.loads(line)
        candidate_id = str(golden_record["candidate_id"])

        if candidate_id not in development_candidate_ids:
            # Do not retain or inspect Test evidence or query content.
            non_development_golden_count += 1
            continue

        assert candidate_id not in development_golden_by_id, (
            "Duplicate Development Golden candidate_id",
            candidate_id,
        )
        development_golden_by_id[candidate_id] = golden_record

        if golden_schema_keys is None:
            golden_schema_keys = sorted(golden_record.keys())

assert set(development_golden_by_id) == development_candidate_ids
assert non_development_golden_count == EXPECTED_TEST_COUNT

development_source_types = {
    str(chunk_record_by_id[str(candidate["chunk_id"])]["source_type"])
    for record in development_candidate_records
    for candidate in record["candidates"]
}

nb18_input_schema_audit = {
    "status": "PASS",
    "reranker_records_total": len(all_candidate_ids),
    "development_records_loaded": len(development_candidate_records),
    "test_records_excluded": test_candidate_count,
    "canonical_chunk_count": len(chunk_record_by_id),
    "development_golden_records_loaded": len(development_golden_by_id),
    "non_development_golden_records_excluded": non_development_golden_count,
    "candidate_depth": EXPECTED_RERANKER_DEPTH,
    "final_top_k": EXPECTED_FINAL_TOP_K,
    "development_candidate_chunks_resolved": len(
        development_referenced_chunk_ids
    ),
    "unknown_development_chunk_ids": len(unknown_development_chunk_ids),
    "canonical_source_types": sorted(canonical_source_types),
    "development_candidate_source_types": sorted(development_source_types),
    "reranker_record_schema_keys": candidate_schema_keys,
    "reranker_candidate_schema_keys": candidate_item_schema_keys,
    "canonical_chunk_schema_keys": chunk_schema_keys,
    "golden_record_schema_keys": golden_schema_keys,
    "test_metrics_or_evidence_used": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(nb18_input_schema_audit, ensure_ascii=False, indent=2))
print("NB18 INPUT SCHEMA AUDIT: PASS")

{
  "status": "PASS",
  "reranker_records_total": 3813,
  "development_records_loaded": 2664,
  "test_records_excluded": 1149,
  "canonical_chunk_count": 3488,
  "development_golden_records_loaded": 2664,
  "non_development_golden_records_excluded": 1149,
  "candidate_depth": 50,
  "final_top_k": 10,
  "development_candidate_chunks_resolved": 3461,
  "unknown_development_chunk_ids": 0,
  "canonical_source_types": [
    "book",
    "transcript",
    "website"
  ],
  "development_candidate_source_types": [
    "book",
    "transcript",
    "website"
  ],
  "reranker_record_schema_keys": [
    "candidate_depth",
    "candidate_id",
    "candidates",
    "configuration_sha256",
    "final_top_k",
    "release_id",
    "source_type",
    "split"
  ],
  "reranker_candidate_schema_keys": [
    "chunk_id",
    "hybrid_candidate",
    "hybrid_rank",
    "rank",
    "reranker_score"
  ],
  "canonical_chunk_schema_keys": [
    "attribution_context",
    "boundary_metadata",
    "char_count",
    

<div dir="rtl" align="right">

## نگاشت شواهد Development به Chunkهای Canonical

NB17 نگاشت شواهد Golden به Chunkها را برای ارزیابی Reranking در حافظه محاسبه کرده بود، اما قرارداد handoff آن Artifact مستقلی برای استفادهٔ NB18 معرفی نمی‌کند. برای سنجش اینکه Context کاندید شواهد لازم را نگه می‌دارد، این سلول همان رابطه را فقط برای Development و از locatorهای Canonical بازسازی می‌کند.

در همین مرحله، metadata واقعیِ موجود در Chunkهای کاندید Development بررسی می‌شود تا معلوم شود Source diversity، Topic diversity، انتساب، حقوق و parent expansion با دادهٔ موجود قابل اجرا هستند یا نه. فیلد غایب حدس زده یا ساخته نمی‌شود.

این سلول هیچ Configuration انتخاب نمی‌کند، Test را تحلیل نمی‌کند و فایلی نمی‌نویسد.

**معیار PASS:** تمام evidence locatorهای Development به دست‌کم یک Chunk Canonical معتبر برسند و metadata به‌صورت مشاهده‌شده گزارش شود. نبود metadata لازم، به‌عنوان محدودیت ورودی گزارش می‌شود، نه اینکه با مقدار ساختگی پر شود.

</div>

In [4]:
# NB18-C04 — Rebuild Development-only evidence mapping and audit metadata

assert "development_golden_by_id" in globals()
assert "development_candidate_ids" in globals()
assert "chunk_record_by_id" in globals()
assert "development_referenced_chunk_ids" in globals()

def canonical_locator_key(locator):
    assert isinstance(locator, dict)
    assert locator
    return json.dumps(
        locator,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

locator_to_chunk_ids = {}
locator_to_source_types = {}

for chunk_id, chunk_record in chunk_record_by_id.items():
    member_locators = chunk_record.get("member_locators")

    assert isinstance(member_locators, list) and member_locators, (
        "Canonical chunk is missing member_locators",
        chunk_id,
    )

    for member_entry in member_locators:
        locator = member_entry.get("locator")
        locator_key = canonical_locator_key(locator)

        locator_to_chunk_ids.setdefault(locator_key, set()).add(chunk_id)
        locator_to_source_types.setdefault(locator_key, set()).add(
            str(chunk_record["source_type"])
        )

development_gold_mapping = {}
unmapped_development_evidence = []
development_evidence_locator_count = 0

for candidate_id in sorted(development_candidate_ids):
    golden_record = development_golden_by_id[candidate_id]
    evidence_locators = golden_record.get("evidence_locators")
    record_source_types = {
        str(source_type)
        for source_type in golden_record.get("source_types", [])
    }

    assert isinstance(evidence_locators, list) and evidence_locators, (
        "Development Golden record has no evidence locators",
        candidate_id,
    )

    relevant_chunk_ids = set()

    for evidence_index, evidence_record in enumerate(evidence_locators):
        development_evidence_locator_count += 1

        source_locator = evidence_record.get("source_locator")
        locator_key = canonical_locator_key(source_locator)
        matching_chunk_ids = locator_to_chunk_ids.get(locator_key, set())

        if not matching_chunk_ids:
            unmapped_development_evidence.append(
                {
                    "candidate_id": candidate_id,
                    "evidence_index": evidence_index,
                }
            )
            continue

        matching_source_types = locator_to_source_types[locator_key]
        assert matching_source_types & record_source_types, (
            "Evidence locator source type does not match Golden record",
            candidate_id,
            evidence_index,
        )

        relevant_chunk_ids.update(matching_chunk_ids)

    development_gold_mapping[candidate_id] = sorted(relevant_chunk_ids)

unmapped_development_query_ids = sorted(
    candidate_id
    for candidate_id, chunk_ids in development_gold_mapping.items()
    if not chunk_ids
)

assert not unmapped_development_evidence, (
    "Unmapped Development evidence locators",
    unmapped_development_evidence[:10],
)
assert not unmapped_development_query_ids, (
    "Development queries without mapped evidence",
    unmapped_development_query_ids[:10],
)
assert set(development_gold_mapping) == development_candidate_ids

mapping_rows = [
    {
        "candidate_id": candidate_id,
        "relevant_chunk_ids": development_gold_mapping[candidate_id],
    }
    for candidate_id in sorted(development_gold_mapping)
]

mapping_digest = hashlib.sha256()
for mapping_row in mapping_rows:
    row_bytes = json.dumps(
        mapping_row,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    mapping_digest.update(
        len(row_bytes).to_bytes(8, byteorder="big", signed=False)
    )
    mapping_digest.update(row_bytes)

development_gold_mapping_sha256 = mapping_digest.hexdigest()

metadata_key_presence = {}
topic_related_key_paths = set()
authority_related_key_paths = set()
rights_related_key_paths = set()
parent_related_key_paths = set()

def scan_metadata_keys(value, path=""):
    if isinstance(value, dict):
        for key, child_value in value.items():
            key_text = str(key)
            child_path = f"{path}.{key_text}" if path else key_text
            normalized_key = key_text.casefold()

            metadata_key_presence[child_path] = (
                metadata_key_presence.get(child_path, 0) + 1
            )

            if any(term in normalized_key for term in (
                "topic", "taxonomy", "curriculum", "competency", "subject"
            )):
                topic_related_key_paths.add(child_path)

            if any(term in normalized_key for term in (
                "author", "speaker", "editor", "attribution", "authority", "role"
            )):
                authority_related_key_paths.add(child_path)

            if any(term in normalized_key for term in (
                "right", "license", "permission"
            )):
                rights_related_key_paths.add(child_path)

            if any(term in normalized_key for term in (
                "parent", "document_id", "locator", "source_type"
            )):
                parent_related_key_paths.add(child_path)

            scan_metadata_keys(child_value, child_path)

    elif isinstance(value, list):
        for child_value in value:
            scan_metadata_keys(child_value, path)

for chunk_id in sorted(development_referenced_chunk_ids):
    scan_metadata_keys(chunk_record_by_id[chunk_id])

nb18_development_evidence_and_metadata_audit = {
    "status": "PASS",
    "development_query_count": len(development_gold_mapping),
    "development_evidence_locator_count": development_evidence_locator_count,
    "unmapped_development_evidence_count": len(
        unmapped_development_evidence
    ),
    "development_queries_without_evidence_count": len(
        unmapped_development_query_ids
    ),
    "development_gold_mapping_sha256": development_gold_mapping_sha256,
    "candidate_chunk_count": len(development_referenced_chunk_ids),
    "topic_related_metadata_paths": sorted(topic_related_key_paths),
    "authority_related_metadata_paths": sorted(authority_related_key_paths),
    "rights_related_metadata_paths": sorted(rights_related_key_paths),
    "parent_and_locator_metadata_paths": sorted(parent_related_key_paths),
    "metadata_field_presence_counts": dict(
        sorted(metadata_key_presence.items())
    ),
    "test_evidence_or_metrics_used": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(
    nb18_development_evidence_and_metadata_audit,
    ensure_ascii=False,
    indent=2,
))
print("NB18 DEVELOPMENT EVIDENCE/METADATA AUDIT: PASS")

{
  "status": "PASS",
  "development_query_count": 2664,
  "development_evidence_locator_count": 2733,
  "unmapped_development_evidence_count": 0,
  "development_queries_without_evidence_count": 0,
  "development_gold_mapping_sha256": "e9e603f51a5964ba750e58927b477425de0baf3315dd30db585ddaa1084833a8",
  "candidate_chunk_count": 3461,
  "topic_related_metadata_paths": [],
  "authority_related_metadata_paths": [
    "attribution_context",
    "attribution_context.content_authority_types",
    "attribution_context.member_attribution_count",
    "attribution_context.mixed_authority",
    "attribution_context.multi_speaker",
    "attribution_context.speaker_keys",
    "role"
  ],
  "rights_related_metadata_paths": [
    "locator_end.right_member_document_ids",
    "locator_end.right_source_locator",
    "locator_start.right_member_document_ids",
    "locator_start.right_source_locator",
    "rights",
    "rights.inferred_release_permission",
    "rights.member_rights",
    "rights.member_ri

<div dir="rtl">

### NB18 — بررسی ورودی‌های نسخه‌دار موضوع

این گام فقط وجود و ساختار فایل‌های مشخص‌شده در نقشهٔ پروژه را بررسی می‌کند تا روشن شود آیا می‌توان تکه‌ها را از راه شناسهٔ سند به موضوع نگاشت کرد.

هیچ فایلی نوشته یا تغییر داده نمی‌شود. محتوای سندها چاپ نمی‌شود و هیچ داده‌ای از Test برای انتخاب سیاست استفاده نمی‌شود.

</div>

In [5]:
# NB18-C05 — Read-only discovery of the documented taxonomy inputs

import hashlib
import json
from collections import Counter
from pathlib import Path

REPO_ROOT = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
    "RAG-projects/RAG_finance"
).resolve()

KNOWN_INPUTS = {
    "taxonomy": REPO_ROOT / "artifacts/phase2/taxonomy/taxonomy_v1.json",
    "taxonomy_freeze": REPO_ROOT / "reports/phase2/taxonomy_freeze_v1.json",
    "accepted_documents": (
        REPO_ROOT
        / "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl"
    ),
}

def schema_paths(value, prefix="", depth=0, max_depth=5, counts=None):
    """Collect structural key paths only; never return field values."""
    if counts is None:
        counts = Counter()

    if depth >= max_depth:
        return counts

    if isinstance(value, dict):
        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            counts[path] += 1
            schema_paths(child, path, depth + 1, max_depth, counts)
    elif isinstance(value, list) and value:
        schema_paths(value[0], f"{prefix}[]", depth + 1, max_depth, counts)

    return counts

audit = {
    "status": "READ_ONLY_DISCOVERY_COMPLETE",
    "repo_root_exists": REPO_ROOT.is_dir(),
    "inputs": {},
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

for name, path in KNOWN_INPUTS.items():
    item = {
        "path": str(path),
        "exists": path.is_file(),
    }

    if not item["exists"]:
        audit["inputs"][name] = item
        continue

    item["size_bytes"] = path.stat().st_size

    try:
        if path.suffix == ".json":
            raw = path.read_bytes()
            parsed = json.loads(raw.decode("utf-8"))
            item["sha256"] = hashlib.sha256(raw).hexdigest()
            item["top_level_type"] = type(parsed).__name__
            item["schema_path_counts"] = dict(
                sorted(schema_paths(parsed).items())[:200]
            )

        elif path.suffix == ".jsonl":
            sample_records = []
            with path.open("r", encoding="utf-8") as handle:
                for line_number, line in enumerate(handle, start=1):
                    if not line.strip():
                        continue
                    record = json.loads(line)
                    sample_records.append(record)
                    if len(sample_records) == 3:
                        break

            path_counts = Counter()
            for record in sample_records:
                schema_paths(record, counts=path_counts)

            item["sample_records_read"] = len(sample_records)
            item["sample_schema_path_counts"] = dict(
                sorted(path_counts.items())[:200]
            )
        else:
            item["inspection"] = "No parser selected for this file type."

    except Exception as exc:
        item["read_error"] = f"{type(exc).__name__}: {exc}"

    audit["inputs"][name] = item

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "READ_ONLY_DISCOVERY_COMPLETE",
  "repo_root_exists": true,
  "inputs": {
    "taxonomy": {
      "path": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/taxonomy/taxonomy_v1.json",
      "exists": true,
      "size_bytes": 15309,
      "sha256": "335690044b2e4eff684f7f347809f2c5cb8c39624a939dc01408d9ca898fe977",
      "top_level_type": "dict",
      "schema_path_counts": {
        "concept_variants": 1,
        "concept_variants.بخشش": 1,
        "concept_variants.بخشش.canonical_label_fa": 1,
        "concept_variants.بخشش.canonical_local_concept_id": 1,
        "concept_variants.بخشش.global_topic_alias": 1,
        "concept_variants.بخشش.variant_type": 1,
        "context_dependent_terms": 1,
        "context_dependent_terms.انتخاب": 1,
        "context_dependent_terms.انتخاب.global_alias_allowed": 1,
        "context_dependent_terms.انتخاب.local_nodes": 1,
        "context_dependent_terms.انتخاب.reason": 1,
        "context_d

<div dir="rtl">

### NB18 — بررسی نگاشت tag سند به شناسه‌های taxonomy

این گام فایل‌های نسخه‌دار taxonomy و سندهای پذیرفته‌شده را فقط می‌خواند. شناسه‌های tag سندها با شناسه‌های موجود در taxonomy مقایسه می‌شوند تا میزان تطبیق مشخص شود.

محتوای سندها چاپ نمی‌شود؛ هیچ فایلی نوشته نمی‌شود و از داده‌های Test استفاده نمی‌شود.

</div>

In [6]:
# NB18-C06 — Read-only comparison of accepted-document tags and taxonomy IDs

import hashlib
import json
import time
from collections import Counter
from pathlib import Path

REPO_ROOT = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
    "RAG-projects/RAG_finance"
).resolve()

TAXONOMY_PATH = REPO_ROOT / "artifacts/phase2/taxonomy/taxonomy_v1.json"
FREEZE_PATH = REPO_ROOT / "reports/phase2/taxonomy_freeze_v1.json"
ACCEPTED_DOCUMENTS_PATH = (
    REPO_ROOT
    / "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl"
)

EXPECTED_SHA256 = {
    "taxonomy": "335690044b2e4eff684f7f347809f2c5cb8c39624a939dc01408d9ca898fe977",
    "taxonomy_freeze": "fe182b0c7c6d5ad7d5dfc2aa794fa096e85001b517040a8396d5fb4c0b17e530",
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def collect_topic_ids(node):
    """Extract IDs only from the supplied frozen topic structure."""
    found = set()

    def visit(value):
        if isinstance(value, dict):
            for key, child in value.items():
                if isinstance(key, str):
                    found.add(key)
                if isinstance(child, dict):
                    for id_key in ("id", "topic_id", "top_level_topic_id"):
                        candidate = child.get(id_key)
                        if isinstance(candidate, str):
                            found.add(candidate)
                    visit(child)
                elif isinstance(child, list):
                    visit(child)
        elif isinstance(value, list):
            for child in value:
                visit(child)

    visit(node)
    return found

started = time.time()

for required_path in (TAXONOMY_PATH, FREEZE_PATH, ACCEPTED_DOCUMENTS_PATH):
    assert required_path.is_file(), f"Required input is missing: {required_path}"

taxonomy_raw = TAXONOMY_PATH.read_bytes()
freeze_raw = FREEZE_PATH.read_bytes()

taxonomy_sha256 = hashlib.sha256(taxonomy_raw).hexdigest()
freeze_sha256 = hashlib.sha256(freeze_raw).hexdigest()

taxonomy = json.loads(taxonomy_raw.decode("utf-8"))
freeze_report = json.loads(freeze_raw.decode("utf-8"))

taxonomy_nodes = taxonomy.get("curriculum_nodes", {})
frozen_structure = freeze_report.get("frozen_structure", {})
frozen_top_level_topics = frozen_structure.get("top_level_topics", {})

official_topic_ids = collect_topic_ids(taxonomy_nodes)
official_topic_ids.update(collect_topic_ids(frozen_top_level_topics))

tag_pair_counts = Counter()
tag_id_counts = Counter()
tagged_document_count = 0
accepted_document_count = 0
documents_with_unidentified_tags = 0
parse_errors = []

with ACCEPTED_DOCUMENTS_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue

        try:
            document = json.loads(line)
        except Exception as exc:
            parse_errors.append({
                "line_number": line_number,
                "error": f"{type(exc).__name__}: {exc}",
            })
            continue

        accepted_document_count += 1
        source_metadata = document.get("source_metadata", {})
        tags = source_metadata.get("tags", []) if isinstance(source_metadata, dict) else []

        if not tags:
            continue

        tagged_document_count += 1
        identified_tag_found = False

        if isinstance(tags, list):
            for tag in tags:
                if not isinstance(tag, dict):
                    documents_with_unidentified_tags += 1
                    continue

                tag_id = tag.get("tag_id")
                tag_name = tag.get("name")

                if tag_id is not None:
                    normalized_id = str(tag_id)
                    tag_id_counts[normalized_id] += 1
                    identified_tag_found = True

                tag_pair_counts[(str(tag_id), str(tag_name))] += 1

        if not identified_tag_found:
            documents_with_unidentified_tags += 1

observed_tag_ids = set(tag_id_counts)
matching_tag_ids = observed_tag_ids & official_topic_ids

audit = {
    "status": "PASS" if not parse_errors else "FAIL",
    "taxonomy_sha256": taxonomy_sha256,
    "taxonomy_sha256_matches_c05": (
        taxonomy_sha256 == EXPECTED_SHA256["taxonomy"]
    ),
    "taxonomy_freeze_sha256": freeze_sha256,
    "taxonomy_freeze_sha256_matches_c05": (
        freeze_sha256 == EXPECTED_SHA256["taxonomy_freeze"]
    ),
    "taxonomy_version": taxonomy.get("taxonomy_version"),
    "freeze_status": freeze_report.get("freeze_status"),
    "taxonomy_curriculum_node_count": (
        len(taxonomy_nodes) if isinstance(taxonomy_nodes, dict) else None
    ),
    "frozen_top_level_topics_container_type": type(
        frozen_top_level_topics
    ).__name__,
    "official_topic_id_count": len(official_topic_ids),
    "accepted_document_count": accepted_document_count,
    "tagged_document_count": tagged_document_count,
    "documents_with_unidentified_tags": documents_with_unidentified_tags,
    "unique_tag_id_count": len(observed_tag_ids),
    "tag_id_occurrences": sum(tag_id_counts.values()),
    "tag_id_matches_official_topic_ids": len(matching_tag_ids),
    "matching_topic_ids": sorted(matching_tag_ids),
    "unmatched_tag_id_count": len(observed_tag_ids - official_topic_ids),
    "sample_unmatched_tag_ids": sorted(observed_tag_ids - official_topic_ids)[:30],
    "sample_tag_id_name_pairs": [
        {"tag_id": tag_id, "name": name, "occurrences": count}
        for (tag_id, name), count in tag_pair_counts.most_common(30)
    ],
    "parse_error_count": len(parse_errors),
    "parse_errors_sample": parse_errors[:5],
    "elapsed_seconds": round(time.time() - started, 2),
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "taxonomy_sha256": "335690044b2e4eff684f7f347809f2c5cb8c39624a939dc01408d9ca898fe977",
  "taxonomy_sha256_matches_c05": true,
  "taxonomy_freeze_sha256": "fe182b0c7c6d5ad7d5dfc2aa794fa096e85001b517040a8396d5fb4c0b17e530",
  "taxonomy_freeze_sha256_matches_c05": true,
  "taxonomy_version": "taxonomy_v1",
  "freeze_status": "PASS",
  "taxonomy_curriculum_node_count": 7,
  "frozen_top_level_topics_container_type": "int",
  "official_topic_id_count": 16,
  "accepted_document_count": 19456,
  "tagged_document_count": 525,
  "documents_with_unidentified_tags": 0,
  "unique_tag_id_count": 86,
  "tag_id_occurrences": 1558,
  "tag_id_matches_official_topic_ids": 0,
  "matching_topic_ids": [],
  "unmatched_tag_id_count": 86,
  "sample_unmatched_tag_ids": [
    "10",
    "1028",
    "1029",
    "1030",
    "1031",
    "1032",
    "1034",
    "1035",
    "1036",
    "11",
    "12",
    "13",
    "14",
    "15",
    "16",
    "17",
    "18",
    "19",
    "2",
    "20",
    

<div dir="rtl">

### NB18 — بررسی پوشش نگاشت موضوعِ سندهای Development

این گام شناسه‌های سندهای موجود در تکه‌های نامزد Development را با ورودی‌های دارای `topics` در فهرست حل‌وفصل taxonomy تطبیق می‌دهد.

رکوردهای Test برای این تطبیق استفاده نمی‌شوند. هیچ فایلی نوشته نمی‌شود.

</div>

In [7]:
# NB18-C07 — Read-only coverage audit of frozen document-topic resolutions

import json
from pathlib import Path

REPO_ROOT = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
    "RAG-projects/RAG_finance"
).resolve()

RERANKED_PATH = (
    REPO_ROOT
    / "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl"
)
CANONICAL_CHUNKS_PATH = (
    REPO_ROOT
    / "data/phase2/chunks/nb11_benchmark_v1/"
    "baseline_phase1_char_safe_v1.jsonl"
)
TAXONOMY_PATH = REPO_ROOT / "artifacts/phase2/taxonomy/taxonomy_v1.json"

for required_path in (RERANKED_PATH, CANONICAL_CHUNKS_PATH, TAXONOMY_PATH):
    assert required_path.is_file(), f"Required input is missing: {required_path}"

taxonomy = json.loads(TAXONOMY_PATH.read_text(encoding="utf-8"))
resolution_register = (
    taxonomy.get("review_resolution", {})
    .get("corpus_resolution_register", {})
)
assert isinstance(resolution_register, dict), (
    "Unexpected taxonomy corpus_resolution_register schema."
)

# Only register entries with an explicit `topics` field count as document mappings.
resolved_document_topics = {
    str(document_id): entry["topics"]
    for document_id, entry in resolution_register.items()
    if isinstance(entry, dict) and "topics" in entry
}

development_query_count = 0
development_candidate_chunk_ids = set()

with RERANKED_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        if str(record.get("split", "")).strip().lower() != "development":
            continue

        development_query_count += 1
        for candidate in record.get("candidates", []):
            chunk_id = candidate.get("chunk_id")
            if chunk_id is not None:
                development_candidate_chunk_ids.add(str(chunk_id))

candidate_chunk_documents = {}
unknown_candidate_chunk_ids = set(development_candidate_chunk_ids)

with CANONICAL_CHUNKS_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        chunk = json.loads(line)
        chunk_id = str(chunk.get("chunk_id", ""))
        if chunk_id not in development_candidate_chunk_ids:
            continue

        member_ids = chunk.get("member_document_ids", [])
        candidate_chunk_documents[chunk_id] = {
            str(document_id) for document_id in member_ids
        }
        unknown_candidate_chunk_ids.discard(chunk_id)

development_document_ids = set()
for document_ids in candidate_chunk_documents.values():
    development_document_ids.update(document_ids)

mapped_development_document_ids = (
    development_document_ids & set(resolved_document_topics)
)

mapped_candidate_chunk_ids = {
    chunk_id
    for chunk_id, document_ids in candidate_chunk_documents.items()
    if document_ids & mapped_development_document_ids
}

audit = {
    "status": "PASS" if not unknown_candidate_chunk_ids else "FAIL",
    "development_query_count": development_query_count,
    "development_candidate_chunk_count": len(development_candidate_chunk_ids),
    "development_candidate_chunks_resolved": len(candidate_chunk_documents),
    "unknown_development_candidate_chunk_count": len(unknown_candidate_chunk_ids),
    "development_candidate_document_id_count": len(development_document_ids),
    "taxonomy_document_topic_entry_count": len(resolved_document_topics),
    "development_document_ids_with_frozen_topic_mapping": len(
        mapped_development_document_ids
    ),
    "development_document_topic_mapping_coverage": (
        round(
            len(mapped_development_document_ids) / len(development_document_ids),
            6,
        )
        if development_document_ids else None
    ),
    "development_candidate_chunks_with_mapped_member_document": len(
        mapped_candidate_chunk_ids
    ),
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_query_count": 2664,
  "development_candidate_chunk_count": 3461,
  "development_candidate_chunks_resolved": 3461,
  "unknown_development_candidate_chunk_count": 0,
  "development_candidate_document_id_count": 19429,
  "taxonomy_document_topic_entry_count": 2,
  "development_document_ids_with_frozen_topic_mapping": 2,
  "development_document_topic_mapping_coverage": 0.000103,
  "development_candidate_chunks_with_mapped_member_document": 2,
  "files_written": 0,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}


<div dir="rtl">

### NB18 — آمار توصیفی نامزدهای Development برای طراحی Context

این گام توزیع نوع منبع، شناسهٔ منبع و سند، نقش، اختیار دانشی، وضعیت حقوق، locator و اندازهٔ تکه‌ها را فقط برای تکه‌های نامزد Development محاسبه می‌کند.

این آمار برای تعیین سیاست‌های context builder است؛ چیزی تغییر یا ذخیره نمی‌شود.

</div>

In [8]:
# NB18-C08 — Read-only Development candidate metadata and dominance audit

import json
from collections import Counter
from pathlib import Path

REPO_ROOT = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
    "RAG-projects/RAG_finance"
).resolve()

RERANKED_PATH = (
    REPO_ROOT
    / "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl"
)
CANONICAL_CHUNKS_PATH = (
    REPO_ROOT
    / "data/phase2/chunks/nb11_benchmark_v1/"
    "baseline_phase1_char_safe_v1.jsonl"
)

for required_path in (RERANKED_PATH, CANONICAL_CHUNKS_PATH):
    assert required_path.is_file(), f"Required input is missing: {required_path}"

development_query_count = 0
development_candidate_chunk_ids = set()

with RERANKED_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        if str(record.get("split", "")).strip().lower() != "development":
            continue

        development_query_count += 1
        for candidate in record.get("candidates", []):
            chunk_id = candidate.get("chunk_id")
            if chunk_id is not None:
                development_candidate_chunk_ids.add(str(chunk_id))

chunks = []
unknown_chunk_ids = set(development_candidate_chunk_ids)

with CANONICAL_CHUNKS_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        chunk = json.loads(line)
        chunk_id = str(chunk.get("chunk_id", ""))
        if chunk_id not in development_candidate_chunk_ids:
            continue

        chunks.append(chunk)
        unknown_chunk_ids.discard(chunk_id)

assert not unknown_chunk_ids, (
    f"Unresolved Development candidate chunks: {len(unknown_chunk_ids)}"
)
assert len(chunks) == len(development_candidate_chunk_ids), (
    "Resolved canonical chunk count does not match unique Development candidates."
)

def scalar_items(value):
    if isinstance(value, dict):
        for child in value.values():
            yield from scalar_items(child)
    elif isinstance(value, list):
        for child in value:
            yield from scalar_items(child)
    elif value is not None:
        yield str(value)

def top_counts(counter, limit=10):
    return [
        {"value": value, "count": count}
        for value, count in counter.most_common(limit)
    ]

def quantiles(values):
    ordered = sorted(values)
    if not ordered:
        return {}

    def at(percent):
        index = round((len(ordered) - 1) * percent)
        return ordered[index]

    return {
        "min": ordered[0],
        "p50": at(0.50),
        "p90": at(0.90),
        "p95": at(0.95),
        "max": ordered[-1],
    }

source_type_counts = Counter()
source_id_counts = Counter()
document_reference_counts = Counter()
role_counts = Counter()
authority_type_counts = Counter()
content_origin_counts = Counter()
rights_status_counts = Counter()
locator_mode_counts = Counter()
char_counts = []
token_counts = []
member_document_counts = Counter()
parent_linked_chunk_count = 0

for chunk in chunks:
    if chunk.get("source_type") is not None:
        source_type_counts[str(chunk["source_type"])] += 1

    if chunk.get("source_id") is not None:
        source_id_counts[str(chunk["source_id"])] += 1

    for document_id in chunk.get("member_document_ids", []):
        document_reference_counts[str(document_id)] += 1

    member_document_counts[len(chunk.get("member_document_ids", []))] += 1

    if chunk.get("role") is not None:
        role_counts[str(chunk["role"])] += 1

    attribution = chunk.get("attribution_context", {})
    if isinstance(attribution, dict):
        authority_type_counts.update(
            scalar_items(attribution.get("content_authority_types"))
        )

    if chunk.get("content_origin") is not None:
        content_origin_counts[str(chunk["content_origin"])] += 1

    rights = chunk.get("rights", {})
    if isinstance(rights, dict):
        for member_right in rights.get("member_rights", []):
            if not isinstance(member_right, dict):
                continue
            member_rights = member_right.get("rights", {})
            if isinstance(member_rights, dict):
                status = member_rights.get("status")
                if status is not None:
                    rights_status_counts[str(status)] += 1

    boundary = chunk.get("boundary_metadata", {})
    if isinstance(boundary, dict) and boundary.get("locator_mode") is not None:
        locator_mode_counts[str(boundary["locator_mode"])] += 1

    if chunk.get("parent_chunk_id") not in (None, ""):
        parent_linked_chunk_count += 1

    if isinstance(chunk.get("char_count"), (int, float)):
        char_counts.append(int(chunk["char_count"]))

    if isinstance(chunk.get("token_count"), (int, float)):
        token_counts.append(int(chunk["token_count"]))

audit = {
    "status": "PASS",
    "development_query_count": development_query_count,
    "unique_development_candidate_chunk_count": len(
        development_candidate_chunk_ids
    ),
    "canonical_candidate_chunks_resolved": len(chunks),
    "unknown_candidate_chunk_count": len(unknown_chunk_ids),
    "source_type_chunk_counts": dict(sorted(source_type_counts.items())),
    "unique_source_id_count": len(source_id_counts),
    "top_source_ids_by_candidate_chunk_count": top_counts(source_id_counts),
    "unique_member_document_id_count": len(document_reference_counts),
    "top_document_ids_by_candidate_chunk_count": top_counts(
        document_reference_counts
    ),
    "role_chunk_counts": dict(sorted(role_counts.items())),
    "content_authority_type_counts": dict(
        sorted(authority_type_counts.items())
    ),
    "content_origin_chunk_counts": dict(sorted(content_origin_counts.items())),
    "member_document_count_per_chunk": dict(
        sorted(member_document_counts.items())
    ),
    "parent_linked_chunk_count": parent_linked_chunk_count,
    "rights_status_member_counts": dict(sorted(rights_status_counts.items())),
    "locator_mode_chunk_counts": dict(sorted(locator_mode_counts.items())),
    "char_count_quantiles": quantiles(char_counts),
    "token_count_quantiles": quantiles(token_counts),
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_query_count": 2664,
  "unique_development_candidate_chunk_count": 3461,
  "canonical_candidate_chunks_resolved": 3461,
  "unknown_candidate_chunk_count": 0,
  "source_type_chunk_counts": {
    "book": 327,
    "transcript": 2038,
    "website": 1096
  },
  "unique_source_id_count": 715,
  "top_source_ids_by_candidate_chunk_count": [
    {
      "value": "book:adult-financial-literacy-course:fa-source",
      "count": 157
    },
    {
      "value": "book:trainer-training-financial-literacy:fa-source",
      "count": 100
    },
    {
      "value": "book:financial-literacy-for-all:fa-source",
      "count": 70
    },
    {
      "value": "program:mizan:unit-001",
      "count": 34
    },
    {
      "value": "program:chehelston:episode-038",
      "count": 30
    },
    {
      "value": "program:chehelston:episode-002",
      "count": 24
    },
    {
      "value": "program:ketab-baz:unit-001",
      "count": 23
    },
    {
      "value": "program:u

<div dir="rtl">

### NB18 — بررسی تنوع منبع و غلبهٔ سند به‌ازای هر پرس‌وجو

این گام نامزدهای Top‑10 و عمق ۵۰ را برای هر پرس‌وجوی Development بررسی می‌کند: نوع منبع، شناسهٔ منبع، تکرار chunk و تکرار شناسه‌های سند عضو.

از این آمار برای مشخص‌کردن امکان اجرای تنوع منبع و محدودکردن غلبهٔ سند استفاده می‌شود؛ هیچ آستانه‌ای در این مرحله انتخاب نمی‌شود.

</div>

In [9]:
# NB18-C09 — Read-only per-query source and document dominance audit

import json
from collections import Counter
from pathlib import Path

REPO_ROOT = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
    "RAG-projects/RAG_finance"
).resolve()

RERANKED_PATH = (
    REPO_ROOT
    / "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl"
)
CANONICAL_CHUNKS_PATH = (
    REPO_ROOT
    / "data/phase2/chunks/nb11_benchmark_v1/"
    "baseline_phase1_char_safe_v1.jsonl"
)

for required_path in (RERANKED_PATH, CANONICAL_CHUNKS_PATH):
    assert required_path.is_file(), f"Required input is missing: {required_path}"

development_records = []
all_development_candidate_ids = set()

with RERANKED_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        if str(record.get("split", "")).strip().lower() != "development":
            continue

        candidates = record.get("candidates", [])
        final_top_k = record.get("final_top_k")
        assert isinstance(final_top_k, int) and final_top_k > 0
        assert len(candidates) >= final_top_k

        development_records.append((final_top_k, candidates))
        for candidate in candidates:
            chunk_id = candidate.get("chunk_id")
            if chunk_id is not None:
                all_development_candidate_ids.add(str(chunk_id))

canonical_metadata = {}
unknown_candidate_ids = set(all_development_candidate_ids)

with CANONICAL_CHUNKS_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        chunk = json.loads(line)
        chunk_id = str(chunk.get("chunk_id", ""))
        if chunk_id not in all_development_candidate_ids:
            continue

        canonical_metadata[chunk_id] = {
            "source_type": chunk.get("source_type"),
            "source_id": chunk.get("source_id"),
            "member_document_ids": [
                str(value) for value in chunk.get("member_document_ids", [])
            ],
        }
        unknown_candidate_ids.discard(chunk_id)

assert not unknown_candidate_ids, (
    f"Unresolved Development candidate chunks: {len(unknown_candidate_ids)}"
)

def quantiles(values):
    ordered = sorted(values)
    if not ordered:
        return {}

    def at(percent):
        return ordered[round((len(ordered) - 1) * percent)]

    return {
        "min": ordered[0],
        "p50": at(0.50),
        "p90": at(0.90),
        "p95": at(0.95),
        "max": ordered[-1],
    }

def analyze_query(candidates, limit):
    selected = candidates[:limit]
    chunk_ids = [str(item["chunk_id"]) for item in selected]
    chunk_metadata = [canonical_metadata[chunk_id] for chunk_id in chunk_ids]

    source_type_counts = Counter(
        str(item["source_type"])
        for item in chunk_metadata
        if item["source_type"] is not None
    )
    source_id_counts = Counter(
        str(item["source_id"])
        for item in chunk_metadata
        if item["source_id"] is not None
    )

    document_counts = Counter()
    for item in chunk_metadata:
        document_counts.update(item["member_document_ids"])

    return {
        "candidate_slots": len(selected),
        "unique_chunk_count": len(set(chunk_ids)),
        "duplicate_chunk_slots": len(chunk_ids) - len(set(chunk_ids)),
        "source_type_count": len(source_type_counts),
        "source_type_counts": source_type_counts,
        "unique_source_id_count": len(source_id_counts),
        "max_chunks_from_one_source": (
            max(source_id_counts.values()) if source_id_counts else 0
        ),
        "max_member_document_references_for_one_document": (
            max(document_counts.values()) if document_counts else 0
        ),
    }

top_k_analyses = []
depth_50_analyses = []
top_k_source_type_availability = Counter()
depth_50_source_type_availability = Counter()
top_k_source_slot_counts = Counter()
depth_50_source_slot_counts = Counter()

for final_top_k, candidates in development_records:
    top_k = analyze_query(candidates, final_top_k)
    depth_50 = analyze_query(candidates, min(50, len(candidates)))

    top_k_analyses.append(top_k)
    depth_50_analyses.append(depth_50)

    top_k_source_type_availability.update(top_k["source_type_counts"].keys())
    depth_50_source_type_availability.update(
        depth_50["source_type_counts"].keys()
    )
    top_k_source_slot_counts.update(top_k["source_type_counts"])
    depth_50_source_slot_counts.update(depth_50["source_type_counts"])

def summarize_analyses(analyses):
    return {
        "queries_with_duplicate_chunk_slots": sum(
            item["duplicate_chunk_slots"] > 0 for item in analyses
        ),
        "unique_source_type_count_per_query": quantiles(
            [item["source_type_count"] for item in analyses]
        ),
        "unique_source_id_count_per_query": quantiles(
            [item["unique_source_id_count"] for item in analyses]
        ),
        "max_chunks_from_one_source_per_query": quantiles(
            [item["max_chunks_from_one_source"] for item in analyses]
        ),
        "max_member_document_references_per_query": quantiles(
            [
                item["max_member_document_references_for_one_document"]
                for item in analyses
            ]
        ),
    }

query_count = len(development_records)

audit = {
    "status": "PASS",
    "development_query_count": query_count,
    "canonical_candidate_chunks_resolved": len(canonical_metadata),
    "unknown_candidate_chunk_count": len(unknown_candidate_ids),
    "top_k_source_type_query_availability": {
        source_type: {
            "queries": count,
            "share": round(count / query_count, 6) if query_count else None,
        }
        for source_type, count in sorted(top_k_source_type_availability.items())
    },
    "top_k_source_type_candidate_slot_counts": dict(
        sorted(top_k_source_slot_counts.items())
    ),
    "top_k_per_query": summarize_analyses(top_k_analyses),
    "depth_50_source_type_query_availability": {
        source_type: {
            "queries": count,
            "share": round(count / query_count, 6) if query_count else None,
        }
        for source_type, count in sorted(depth_50_source_type_availability.items())
    },
    "depth_50_source_type_candidate_slot_counts": dict(
        sorted(depth_50_source_slot_counts.items())
    ),
    "depth_50_per_query": summarize_analyses(depth_50_analyses),
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_query_count": 2664,
  "canonical_candidate_chunks_resolved": 3461,
  "unknown_candidate_chunk_count": 0,
  "top_k_source_type_query_availability": {
    "book": {
      "queries": 1425,
      "share": 0.53491
    },
    "transcript": {
      "queries": 2484,
      "share": 0.932432
    },
    "website": {
      "queries": 2379,
      "share": 0.893018
    }
  },
  "top_k_source_type_candidate_slot_counts": {
    "book": 2524,
    "transcript": 13609,
    "website": 10507
  },
  "top_k_per_query": {
    "queries_with_duplicate_chunk_slots": 0,
    "unique_source_type_count_per_query": {
      "min": 1,
      "p50": 2,
      "p90": 3,
      "p95": 3,
      "max": 3
    },
    "unique_source_id_count_per_query": {
      "min": 1,
      "p50": 8,
      "p90": 10,
      "p95": 10,
      "max": 10
    },
    "max_chunks_from_one_source_per_query": {
      "min": 1,
      "p50": 2,
      "p90": 4,
      "p95": 4,
      "max": 10
    },
    "max_member_docu

<div dir="rtl">

### NB18 — ممیزی اتصال اطلاعات انتساب و حقوق به سندهای نامزد

این گام شناسه‌های سند عضو تکه‌های نامزد Development را به رکوردهای سند پذیرفته‌شده وصل می‌کند و فقط شمارش فیلدهای انتساب و حقوق را گزارش می‌دهد؛ هیچ نام یا متن سندی چاپ نمی‌شود.

از Test استفاده نمی‌شود و هیچ فایلی نوشته نمی‌شود.

</div>

In [10]:
# NB18-C10 — Read-only attribution and rights join audit

import json
from collections import Counter
from pathlib import Path

REPO_ROOT = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
    "RAG-projects/RAG_finance"
).resolve()

RERANKED_PATH = (
    REPO_ROOT
    / "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl"
)
CANONICAL_CHUNKS_PATH = (
    REPO_ROOT
    / "data/phase2/chunks/nb11_benchmark_v1/"
    "baseline_phase1_char_safe_v1.jsonl"
)
ACCEPTED_DOCUMENTS_PATH = (
    REPO_ROOT
    / "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl"
)

for required_path in (
    RERANKED_PATH,
    CANONICAL_CHUNKS_PATH,
    ACCEPTED_DOCUMENTS_PATH,
):
    assert required_path.is_file(), f"Required input is missing: {required_path}"

development_candidate_chunk_ids = set()

with RERANKED_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        if str(record.get("split", "")).strip().lower() != "development":
            continue

        for candidate in record.get("candidates", []):
            chunk_id = candidate.get("chunk_id")
            if chunk_id is not None:
                development_candidate_chunk_ids.add(str(chunk_id))

candidate_document_ids = set()
unknown_candidate_chunk_ids = set(development_candidate_chunk_ids)

chunk_authority_counts = Counter()
chunk_rights_status_counts = Counter()
chunk_rights_allowed_use_counts = Counter()
chunk_rights_evidence_id_present = 0
candidate_chunk_count = 0

with CANONICAL_CHUNKS_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        chunk = json.loads(line)
        chunk_id = str(chunk.get("chunk_id", ""))
        if chunk_id not in development_candidate_chunk_ids:
            continue

        candidate_chunk_count += 1
        unknown_candidate_chunk_ids.discard(chunk_id)

        candidate_document_ids.update(
            str(document_id)
            for document_id in chunk.get("member_document_ids", [])
        )

        attribution = chunk.get("attribution_context", {})
        if isinstance(attribution, dict):
            values = attribution.get("content_authority_types", [])
            if not isinstance(values, list):
                values = [values]
            chunk_authority_counts.update(
                str(value) for value in values if value is not None
            )

        rights = chunk.get("rights", {})
        if isinstance(rights, dict):
            for member in rights.get("member_rights", []):
                if not isinstance(member, dict):
                    continue

                member_rights = member.get("rights", {})
                if not isinstance(member_rights, dict):
                    continue

                status = member_rights.get("status")
                if status is not None:
                    chunk_rights_status_counts[str(status)] += 1

                allowed_uses = member_rights.get("allowed_uses", [])
                if not isinstance(allowed_uses, list):
                    allowed_uses = [allowed_uses]
                chunk_rights_allowed_use_counts.update(
                    str(value) for value in allowed_uses if value is not None
                )

                if member_rights.get("rights_evidence_id") not in (None, ""):
                    chunk_rights_evidence_id_present += 1

assert not unknown_candidate_chunk_ids, (
    f"Unresolved Development candidate chunks: "
    f"{len(unknown_candidate_chunk_ids)}"
)

attribution_fields = (
    "attribution_status",
    "author",
    "author_id",
    "content_authority_type",
    "credited_author_name",
    "editorial_supervisor_name",
    "reference_expert_name",
    "speaker",
)
document_rights_fields = (
    "allowed_uses",
    "basis",
    "rights_evidence_id",
    "rights_holder",
    "status",
)

matched_candidate_document_count = 0
attribution_field_presence_counts = Counter()
document_rights_field_presence_counts = Counter()
document_content_authority_counts = Counter()
document_rights_status_counts = Counter()
documents_with_any_attribution_field = 0
documents_with_any_rights_field = 0
unmatched_candidate_document_ids = set(candidate_document_ids)

with ACCEPTED_DOCUMENTS_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        document = json.loads(line)
        document_id = str(document.get("document_id", ""))
        if document_id not in candidate_document_ids:
            continue

        matched_candidate_document_count += 1
        unmatched_candidate_document_ids.discard(document_id)

        attribution = document.get("attribution", {})
        if isinstance(attribution, dict):
            present_fields = [
                field
                for field in attribution_fields
                if attribution.get(field) not in (None, "", [], {})
            ]
            attribution_field_presence_counts.update(present_fields)
            if present_fields:
                documents_with_any_attribution_field += 1

            authority = attribution.get("content_authority_type")
            if authority not in (None, ""):
                document_content_authority_counts[str(authority)] += 1

        rights = document.get("rights", {})
        if isinstance(rights, dict):
            present_fields = [
                field
                for field in document_rights_fields
                if rights.get(field) not in (None, "", [], {})
            ]
            document_rights_field_presence_counts.update(present_fields)
            if present_fields:
                documents_with_any_rights_field += 1

            status = rights.get("status")
            if status not in (None, ""):
                document_rights_status_counts[str(status)] += 1

audit = {
    "status": "PASS",
    "development_candidate_chunk_count": candidate_chunk_count,
    "unknown_development_candidate_chunk_count": len(
        unknown_candidate_chunk_ids
    ),
    "development_candidate_document_id_count": len(candidate_document_ids),
    "matched_accepted_document_count": matched_candidate_document_count,
    "unmatched_candidate_document_id_count": len(
        unmatched_candidate_document_ids
    ),
    "chunk_content_authority_type_counts": dict(
        sorted(chunk_authority_counts.items())
    ),
    "accepted_document_attribution_field_presence_counts": dict(
        sorted(attribution_field_presence_counts.items())
    ),
    "accepted_documents_with_any_attribution_field": (
        documents_with_any_attribution_field
    ),
    "accepted_document_content_authority_type_counts": dict(
        sorted(document_content_authority_counts.items())
    ),
    "chunk_member_rights_status_counts": dict(
        sorted(chunk_rights_status_counts.items())
    ),
    "chunk_member_rights_allowed_use_counts": dict(
        sorted(chunk_rights_allowed_use_counts.items())
    ),
    "chunk_member_rights_with_evidence_id_count": (
        chunk_rights_evidence_id_present
    ),
    "accepted_document_rights_field_presence_counts": dict(
        sorted(document_rights_field_presence_counts.items())
    ),
    "accepted_documents_with_any_rights_field": (
        documents_with_any_rights_field
    ),
    "accepted_document_rights_status_counts": dict(
        sorted(document_rights_status_counts.items())
    ),
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_candidate_chunk_count": 3461,
  "unknown_development_candidate_chunk_count": 0,
  "development_candidate_document_id_count": 19429,
  "matched_accepted_document_count": 19429,
  "unmatched_candidate_document_id_count": 0,
  "chunk_content_authority_type_counts": {
    "direct_author": 140,
    "editorial_supervision": 956
  },
  "accepted_document_attribution_field_presence_counts": {
    "attribution_status": 529,
    "author": 3204,
    "author_id": 529,
    "content_authority_type": 529,
    "credited_author_name": 529,
    "editorial_supervisor_name": 453,
    "speaker": 11209
  },
  "accepted_documents_with_any_attribution_field": 14413,
  "accepted_document_content_authority_type_counts": {
    "direct_author": 76,
    "editorial_supervision": 453
  },
  "chunk_member_rights_status_counts": {
    "EVIDENCE_REQUIRED": 23982
  },
  "chunk_member_rights_allowed_use_counts": {},
  "chunk_member_rights_with_evidence_id_count": 0,
  "accepted_docume

<div dir="rtl">

### NB18 — ساخت context پایهٔ Development

این گام برای هر پرس‌وجوی Development، Top‑10 نهایی NB17 را با همان ترتیب می‌گیرد و متن تکه، رتبه، امتیاز، locator، انتساب سندهای عضو و metadata حقوق را در متغیر `development_contexts` نگه می‌دارد.

این context موقت و فقط در حافظه است. بودجهٔ توکن، برش متن و تشخیص تعارض در این گام اعمال نمی‌شوند؛ هیچ فایل یا artifact نوشته نمی‌شود.

</div>

In [11]:
# NB18-C11 — Build in-memory Development context envelopes, without truncation

import json
from pathlib import Path

REPO_ROOT = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
    "RAG-projects/RAG_finance"
).resolve()

RERANKED_PATH = (
    REPO_ROOT
    / "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl"
)
CANONICAL_CHUNKS_PATH = (
    REPO_ROOT
    / "data/phase2/chunks/nb11_benchmark_v1/"
    "baseline_phase1_char_safe_v1.jsonl"
)
ACCEPTED_DOCUMENTS_PATH = (
    REPO_ROOT
    / "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl"
)

for required_path in (
    RERANKED_PATH,
    CANONICAL_CHUNKS_PATH,
    ACCEPTED_DOCUMENTS_PATH,
):
    assert required_path.is_file(), f"Required input is missing: {required_path}"

development_records = []
selected_chunk_ids = set()

with RERANKED_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        if str(record.get("split", "")).strip().lower() != "development":
            continue

        candidates = record.get("candidates", [])
        final_top_k = record.get("final_top_k")
        assert final_top_k == 10, (
            f"Unexpected NB17 final_top_k: {final_top_k!r}"
        )
        assert len(candidates) >= final_top_k

        selected = candidates[:final_top_k]
        expected_ranks = list(range(1, final_top_k + 1))
        actual_ranks = [candidate.get("rank") for candidate in selected]
        assert actual_ranks == expected_ranks, (
            f"Candidate rank/order contract failed for "
            f"{record.get('candidate_id')!r}"
        )

        development_records.append((record, selected))
        selected_chunk_ids.update(
            str(candidate["chunk_id"])
            for candidate in selected
            if candidate.get("chunk_id") is not None
        )

canonical_chunks = {}
unknown_chunk_ids = set(selected_chunk_ids)

with CANONICAL_CHUNKS_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        chunk = json.loads(line)
        chunk_id = str(chunk.get("chunk_id", ""))
        if chunk_id in selected_chunk_ids:
            canonical_chunks[chunk_id] = chunk
            unknown_chunk_ids.discard(chunk_id)

assert not unknown_chunk_ids, (
    f"Selected canonical chunks are unresolved: {len(unknown_chunk_ids)}"
)

selected_document_ids = set()
for chunk in canonical_chunks.values():
    selected_document_ids.update(
        str(document_id)
        for document_id in chunk.get("member_document_ids", [])
    )

accepted_document_metadata = {}
with ACCEPTED_DOCUMENTS_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        document = json.loads(line)
        document_id = str(document.get("document_id", ""))
        if document_id not in selected_document_ids:
            continue

        accepted_document_metadata[document_id] = {
            "attribution": document.get("attribution", {}),
            "rights": document.get("rights", {}),
        }

missing_document_ids = selected_document_ids - set(accepted_document_metadata)
assert not missing_document_ids, (
    f"Selected member documents are unresolved: {len(missing_document_ids)}"
)

development_contexts = []
selected_item_count = 0
selected_token_count_by_query = []
rights_status_counts = {}
rights_missing_allowed_uses_count = 0
all_selected_ranks_preserved = True

for record, candidates in development_records:
    context_items = []

    for candidate in candidates:
        chunk_id = str(candidate["chunk_id"])
        chunk = canonical_chunks[chunk_id]

        member_documents = [
            {
                "document_id": str(document_id),
                **accepted_document_metadata[str(document_id)],
            }
            for document_id in chunk.get("member_document_ids", [])
        ]

        rights = chunk.get("rights", {})
        if isinstance(rights, dict):
            for member_right in rights.get("member_rights", []):
                if not isinstance(member_right, dict):
                    continue

                member_rights = member_right.get("rights", {})
                if not isinstance(member_rights, dict):
                    continue

                status = member_rights.get("status")
                if status is not None:
                    rights_status_counts[str(status)] = (
                        rights_status_counts.get(str(status), 0) + 1
                    )

                allowed_uses = member_rights.get("allowed_uses")
                if allowed_uses in (None, "", [], {}):
                    rights_missing_allowed_uses_count += 1

        context_items.append(
            {
                "rank": candidate["rank"],
                "reranker_score": candidate.get("reranker_score"),
                "chunk_id": chunk_id,
                "content": chunk.get("content"),
                "char_count": chunk.get("char_count"),
                "token_count": chunk.get("token_count"),
                "source_type": chunk.get("source_type"),
                "source_id": chunk.get("source_id"),
                "role": chunk.get("role"),
                "member_document_ids": chunk.get("member_document_ids", []),
                "member_documents": member_documents,
                "member_locators": chunk.get("member_locators", []),
                "locator_start": chunk.get("locator_start"),
                "locator_end": chunk.get("locator_end"),
                "parent_chunk_id": chunk.get("parent_chunk_id"),
                "lineage": chunk.get("lineage"),
                "attribution_context": chunk.get("attribution_context"),
                "rights": rights,
            }
        )

    selected_item_count += len(context_items)
    selected_token_count_by_query.append(
        sum(
            item["token_count"]
            for item in context_items
            if isinstance(item["token_count"], (int, float))
        )
    )

    development_contexts.append(
        {
            "candidate_id": record.get("candidate_id"),
            "split": "development",
            "context_items": context_items,
        }
    )

assert len(development_contexts) == 2664, (
    f"Unexpected Development context count: {len(development_contexts)}"
)
assert selected_item_count == len(development_contexts) * 10

ordered_token_counts = sorted(selected_token_count_by_query)

def quantiles(values):
    if not values:
        return {}

    def at(percent):
        return values[round((len(values) - 1) * percent)]

    return {
        "min": values[0],
        "p50": at(0.50),
        "p90": at(0.90),
        "p95": at(0.95),
        "max": values[-1],
    }

audit = {
    "status": "PASS",
    "development_context_count": len(development_contexts),
    "context_items_total": selected_item_count,
    "items_per_context": 10,
    "unique_selected_chunk_count": len(selected_chunk_ids),
    "canonical_chunks_unresolved": len(unknown_chunk_ids),
    "member_document_ids_unresolved": len(missing_document_ids),
    "reranker_rank_order_preserved": all_selected_ranks_preserved,
    "selected_chunk_token_count_per_query": quantiles(ordered_token_counts),
    "rights_status_member_counts": dict(sorted(rights_status_counts.items())),
    "member_rights_without_allowed_uses_count": (
        rights_missing_allowed_uses_count
    ),
    "token_budget_selected": False,
    "truncation_applied": False,
    "topic_diversity_applied": False,
    "conflict_detection_applied": False,
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_context_count": 2664,
  "context_items_total": 26640,
  "items_per_context": 10,
  "unique_selected_chunk_count": 3277,
  "canonical_chunks_unresolved": 0,
  "member_document_ids_unresolved": 0,
  "reranker_rank_order_preserved": true,
  "selected_chunk_token_count_per_query": {
    "min": 604,
    "p50": 3015,
    "p90": 3301,
    "p95": 3364,
    "max": 3696
  },
  "rights_status_member_counts": {
    "EVIDENCE_REQUIRED": 149993
  },
  "member_rights_without_allowed_uses_count": 160500,
  "token_budget_selected": false,
  "truncation_applied": false,
  "topic_diversity_applied": false,
  "conflict_detection_applied": false,
  "files_written": 0,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}


<div dir="rtl">

### NB18 — قرارداد پک‌کردن context با بودجهٔ ورودی

پک‌کننده tokenizer، بودجهٔ مؤثر و تابع رندر context را به‌صورت ورودی اجباری می‌گیرد. بودجه باید برای کل context رندرشده باشد؛ تنظیمات مدل مقصد بعداً آن را تعیین می‌کند.

در صورت کمبود بودجه، تکه‌ها کامل رد می‌شوند و متنشان بریده نمی‌شود. رتبهٔ NB17 میان تکه‌های پذیرفته‌شده حفظ می‌شود.

</div>

In [12]:
# NB18-C12 — Model-neutral whole-chunk context packer contract

import json

def pack_ranked_context(
    context_items,
    *,
    context_budget_tokens,
    render_context,
    count_tokens,
):
    """
    Pack complete context items in rank order.

    Required production inputs:
      - context_budget_tokens: effective budget for the rendered context
      - render_context: exact context serializer used by the caller
      - count_tokens: tokenizer counter matching the destination model

    Items that do not fit are skipped whole; their text is never truncated.
    """
    if not isinstance(context_budget_tokens, int) or context_budget_tokens <= 0:
        raise ValueError("context_budget_tokens must be a positive integer.")
    if not callable(render_context):
        raise TypeError("render_context must be callable.")
    if not callable(count_tokens):
        raise TypeError("count_tokens must be callable.")

    indexed_items = list(enumerate(context_items))
    for _, item in indexed_items:
        if not isinstance(item.get("rank"), int):
            raise ValueError("Every context item must have an integer rank.")

    ranks = [item["rank"] for _, item in indexed_items]
    if len(ranks) != len(set(ranks)):
        raise ValueError("Context item ranks must be unique.")

    ranked_items = [
        item
        for _, item in sorted(
            indexed_items,
            key=lambda pair: (pair[1]["rank"], pair[0]),
        )
    ]

    selected_items = []
    skipped_items = []

    for item in ranked_items:
        proposed_items = selected_items + [item]
        proposed_text = render_context(proposed_items)
        proposed_token_count = count_tokens(proposed_text)

        if proposed_token_count <= context_budget_tokens:
            selected_items.append(item)
        else:
            skipped_items.append(item)

    final_text = render_context(selected_items)
    final_token_count = count_tokens(final_text)

    if final_token_count > context_budget_tokens:
        raise AssertionError("Packed context exceeds its token budget.")

    selected_ranks = [item["rank"] for item in selected_items]
    if selected_ranks != sorted(selected_ranks):
        raise AssertionError("Selected item order does not preserve rank order.")

    return {
        "selected_items": selected_items,
        "skipped_items": skipped_items,
        "rendered_context": final_text,
        "context_token_count": final_token_count,
        "context_budget_tokens": context_budget_tokens,
    }


# Fixture-only check; this is not a production budget or tokenizer.
_fixture_items = [
    {"rank": 1, "chunk_id": "fixture-1", "text": "a b"},
    {"rank": 2, "chunk_id": "fixture-2", "text": "c d"},
    {"rank": 3, "chunk_id": "fixture-3", "text": "e f g"},
    {"rank": 4, "chunk_id": "fixture-4", "text": "h"},
]

_fixture_render = lambda items: "\n".join(item["text"] for item in items)
_fixture_token_count = lambda text: len(text.split())

_fixture_result = pack_ranked_context(
    _fixture_items,
    context_budget_tokens=5,
    render_context=_fixture_render,
    count_tokens=_fixture_token_count,
)

assert [item["rank"] for item in _fixture_result["selected_items"]] == [1, 2, 4]
assert [item["rank"] for item in _fixture_result["skipped_items"]] == [3]
assert _fixture_result["context_token_count"] == 5
assert _fixture_result["context_token_count"] <= _fixture_result["context_budget_tokens"]

print(json.dumps({
    "status": "PASS",
    "fixture_only": True,
    "production_tokenizer_used": False,
    "production_budget_selected": False,
    "selected_fixture_ranks": [
        item["rank"] for item in _fixture_result["selected_items"]
    ],
    "skipped_fixture_ranks": [
        item["rank"] for item in _fixture_result["skipped_items"]
    ],
    "fixture_token_count": _fixture_result["context_token_count"],
    "fixture_budget": _fixture_result["context_budget_tokens"],
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "fixture_only": true,
  "production_tokenizer_used": false,
  "production_budget_selected": false,
  "selected_fixture_ranks": [
    1,
    2,
    4
  ],
  "skipped_fixture_ranks": [
    3
  ],
  "fixture_token_count": 5,
  "fixture_budget": 5,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl">

### NB18 — آزمایش ترتیب اولویت‌دهی بر اساس نوع منبع

این آزمایش تکه‌های Top‑10 را بدون حذف، تکرار یا تغییر امتیاز، به ترتیب round‑robin بر اساس `source_type` می‌چیند. رتبهٔ اصلی NB17 حفظ می‌شود و فقط ترتیب اولویت برای پک تغییر می‌کند.

آزمایش فقط روی Development است و بودجهٔ تولیدی یا Test را به کار نمی‌گیرد.

</div>

In [13]:
# NB18-C13 — Development-only source-type round-robin ordering experiment

import json
from collections import Counter

assert "development_contexts" in globals(), (
    "Run NB18-C11 first to create development_contexts."
)
assert len(development_contexts) == 2664

def source_round_robin(items):
    """Interleave existing source types while preserving rank within each type."""
    groups = {}
    original_ids = []

    for item in sorted(items, key=lambda value: value["rank"]):
        source_type = item.get("source_type")
        group_key = str(source_type) if source_type is not None else "__MISSING__"
        groups.setdefault(group_key, []).append(item)
        original_ids.append(item["chunk_id"])

    group_order = list(groups)
    interleaved = []
    offset = 0

    while True:
        added = False
        for group_key in group_order:
            group = groups[group_key]
            if offset < len(group):
                interleaved.append(group[offset])
                added = True
        if not added:
            break
        offset += 1

    assert len(interleaved) == len(items)
    assert len({item["chunk_id"] for item in interleaved}) == len(items)
    assert set(item["chunk_id"] for item in interleaved) == set(original_ids)

    for group_key, group in groups.items():
        output_ranks = [
            item["rank"]
            for item in interleaved
            if (
                str(item.get("source_type"))
                if item.get("source_type") is not None
                else "__MISSING__"
            ) == group_key
        ]
        assert output_ranks == sorted(output_ranks)

    return interleaved

def prefix_stats(items, prefix_size):
    selected = items[:prefix_size]
    counts = Counter(
        str(item["source_type"])
        for item in selected
        if item.get("source_type") is not None
    )
    return {
        "distinct_source_type_count": len(counts),
        "source_type_counts": dict(sorted(counts.items())),
        "max_items_from_one_source_type": max(counts.values(), default=0),
    }

prefix_sizes = (2, 3, 5, 10)
comparison = {}

for prefix_size in prefix_sizes:
    baseline_diversity = []
    interleaved_diversity = []
    baseline_max_source = []
    interleaved_max_source = []
    improved_query_count = 0
    unchanged_query_count = 0
    reduced_query_count = 0

    for context in development_contexts:
        items = context["context_items"]
        assert len(items) == 10

        baseline = sorted(items, key=lambda value: value["rank"])
        interleaved = source_round_robin(items)

        baseline_stats = prefix_stats(baseline, prefix_size)
        interleaved_stats = prefix_stats(interleaved, prefix_size)

        baseline_diversity.append(
            baseline_stats["distinct_source_type_count"]
        )
        interleaved_diversity.append(
            interleaved_stats["distinct_source_type_count"]
        )
        baseline_max_source.append(
            baseline_stats["max_items_from_one_source_type"]
        )
        interleaved_max_source.append(
            interleaved_stats["max_items_from_one_source_type"]
        )

        if (
            interleaved_stats["distinct_source_type_count"]
            > baseline_stats["distinct_source_type_count"]
        ):
            improved_query_count += 1
        elif (
            interleaved_stats["distinct_source_type_count"]
            == baseline_stats["distinct_source_type_count"]
        ):
            unchanged_query_count += 1
        else:
            reduced_query_count += 1

    comparison[str(prefix_size)] = {
        "baseline_distinct_source_type_distribution": dict(
            sorted(Counter(baseline_diversity).items())
        ),
        "round_robin_distinct_source_type_distribution": dict(
            sorted(Counter(interleaved_diversity).items())
        ),
        "baseline_max_source_type_count_distribution": dict(
            sorted(Counter(baseline_max_source).items())
        ),
        "round_robin_max_source_type_count_distribution": dict(
            sorted(Counter(interleaved_max_source).items())
        ),
        "queries_with_more_source_type_diversity": improved_query_count,
        "queries_with_same_source_type_diversity": unchanged_query_count,
        "queries_with_less_source_type_diversity": reduced_query_count,
    }

audit = {
    "status": "PASS",
    "development_context_count": len(development_contexts),
    "items_per_context": 10,
    "experiment": "source_type_round_robin",
    "original_reranker_ranks_preserved_within_source_type": True,
    "chunks_added_or_removed": 0,
    "reranker_scores_recomputed": False,
    "production_budget_used": False,
    "test_evidence_or_metrics_used": False,
    "prefix_comparison": comparison,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_context_count": 2664,
  "items_per_context": 10,
  "experiment": "source_type_round_robin",
  "original_reranker_ranks_preserved_within_source_type": true,
  "chunks_added_or_removed": 0,
  "reranker_scores_recomputed": false,
  "production_budget_used": false,
  "test_evidence_or_metrics_used": false,
  "prefix_comparison": {
    "2": {
      "baseline_distinct_source_type_distribution": {
        "1": 1642,
        "2": 1022
      },
      "round_robin_distinct_source_type_distribution": {
        "1": 255,
        "2": 2409
      },
      "baseline_max_source_type_count_distribution": {
        "1": 1022,
        "2": 1642
      },
      "round_robin_max_source_type_count_distribution": {
        "1": 2409,
        "2": 255
      },
      "queries_with_more_source_type_diversity": 1387,
      "queries_with_same_source_type_diversity": 1277,
      "queries_with_less_source_type_diversity": 0
    },
    "3": {
      "baseline_distinct_source_type_d

<div dir="rtl">

### NB18 — ترکیب اولویت تنوع منبع با پک‌کننده

این گام ترتیب round‑robin را به پک‌کننده وصل می‌کند. پک‌کننده ترتیب ورودی را حفظ می‌کند، تکه‌های پذیرفته‌شده را کامل نگه می‌دارد و تکه‌ای را که در بودجه جا نشود رد می‌کند.

آزمون فقط با fixture انجام می‌شود؛ بودجه و tokenizer تولیدی هنوز تعیین نشده‌اند.

</div>

In [14]:
# NB18-C14 — Source-diverse priority packer composition

import json

assert "source_round_robin" in globals(), (
    "Run NB18-C13 first to define source_round_robin."
)

def pack_in_priority_order(
    priority_ordered_items,
    *,
    context_budget_tokens,
    render_context,
    count_tokens,
):
    """Pack whole items in the supplied priority order; do not re-sort by rank."""
    if not isinstance(context_budget_tokens, int) or context_budget_tokens <= 0:
        raise ValueError("context_budget_tokens must be a positive integer.")
    if not callable(render_context):
        raise TypeError("render_context must be callable.")
    if not callable(count_tokens):
        raise TypeError("count_tokens must be callable.")

    ranks = [item.get("rank") for item in priority_ordered_items]
    if any(not isinstance(rank, int) for rank in ranks):
        raise ValueError("Every context item must have an integer rank.")
    if len(ranks) != len(set(ranks)):
        raise ValueError("Context item ranks must be unique.")

    selected_items = []
    skipped_items = []

    for item in priority_ordered_items:
        proposed_items = selected_items + [item]
        proposed_text = render_context(proposed_items)

        if count_tokens(proposed_text) <= context_budget_tokens:
            selected_items.append(item)
        else:
            skipped_items.append(item)

    final_text = render_context(selected_items)
    final_token_count = count_tokens(final_text)

    if final_token_count > context_budget_tokens:
        raise AssertionError("Packed context exceeds its token budget.")

    return {
        "selected_items": selected_items,
        "skipped_items": skipped_items,
        "rendered_context": final_text,
        "context_token_count": final_token_count,
        "context_budget_tokens": context_budget_tokens,
    }

def pack_source_diverse_context(
    context_items,
    *,
    context_budget_tokens,
    render_context,
    count_tokens,
):
    priority_order = source_round_robin(context_items)
    packed = pack_in_priority_order(
        priority_order,
        context_budget_tokens=context_budget_tokens,
        render_context=render_context,
        count_tokens=count_tokens,
    )
    packed["priority_order"] = priority_order
    return packed


# Fixture-only composition check; not a production budget or tokenizer.
_fixture_items = [
    {"rank": 1, "chunk_id": "fixture-book-1", "source_type": "book", "text": "a"},
    {"rank": 2, "chunk_id": "fixture-book-2", "source_type": "book", "text": "b"},
    {
        "rank": 3,
        "chunk_id": "fixture-transcript-1",
        "source_type": "transcript",
        "text": "c",
    },
    {
        "rank": 4,
        "chunk_id": "fixture-website-1",
        "source_type": "website",
        "text": "d",
    },
]

_fixture_render = lambda items: " ".join(item["text"] for item in items)
_fixture_token_count = lambda text: len(text.split())

_fixture_packed = pack_source_diverse_context(
    _fixture_items,
    context_budget_tokens=3,
    render_context=_fixture_render,
    count_tokens=_fixture_token_count,
)

_fixture_selected_ranks = [
    item["rank"] for item in _fixture_packed["selected_items"]
]
_fixture_skipped_ranks = [
    item["rank"] for item in _fixture_packed["skipped_items"]
]
_fixture_selected_types = [
    item["source_type"] for item in _fixture_packed["selected_items"]
]

assert _fixture_selected_ranks == [1, 3, 4]
assert _fixture_skipped_ranks == [2]
assert _fixture_selected_types == ["book", "transcript", "website"]
assert _fixture_packed["context_token_count"] == 3

print(json.dumps({
    "status": "PASS",
    "fixture_only": True,
    "production_tokenizer_used": False,
    "production_budget_selected": False,
    "selected_fixture_ranks": _fixture_selected_ranks,
    "skipped_fixture_ranks": _fixture_skipped_ranks,
    "selected_fixture_source_types": _fixture_selected_types,
    "fixture_token_count": _fixture_packed["context_token_count"],
    "fixture_budget": _fixture_packed["context_budget_tokens"],
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "fixture_only": true,
  "production_tokenizer_used": false,
  "production_budget_selected": false,
  "selected_fixture_ranks": [
    1,
    3,
    4
  ],
  "skipped_fixture_ranks": [
    2
  ],
  "selected_fixture_source_types": [
    "book",
    "transcript",
    "website"
  ],
  "fixture_token_count": 3,
  "fixture_budget": 3,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl">

### NB18 — بررسی اتصال lineage والد به تکه‌های canonical

این گام نامزدهای Top‑10 پرس‌وجوهای Development را بررسی می‌کند تا ببیند آیا `lineage.parent_id` به سندی وصل می‌شود که در مجموعهٔ chunkهای canonical قابل بازیابی باشد. فیلد مستقیم `parent_chunk_id` نیز جداگانه شمرده می‌شود.

از برچسب یا معیار Test استفاده نمی‌شود؛ هیچ فایلی نوشته نمی‌شود.

</div>

In [15]:
# NB18-C15 — Read-only audit of parent and lineage expansion availability

import json
from collections import Counter
from pathlib import Path

REPO_ROOT = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
    "RAG-projects/RAG_finance"
).resolve()

RERANKED_PATH = (
    REPO_ROOT
    / "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl"
)
CANONICAL_CHUNKS_PATH = (
    REPO_ROOT
    / "data/phase2/chunks/nb11_benchmark_v1/"
    "baseline_phase1_char_safe_v1.jsonl"
)

for required_path in (RERANKED_PATH, CANONICAL_CHUNKS_PATH):
    assert required_path.is_file(), f"Required input is missing: {required_path}"

development_top10_chunk_ids = set()

with RERANKED_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        if str(record.get("split", "")).strip().lower() != "development":
            continue

        candidates = record.get("candidates", [])
        final_top_k = record.get("final_top_k")
        assert final_top_k == 10
        assert len(candidates) >= final_top_k

        for candidate in candidates[:final_top_k]:
            chunk_id = candidate.get("chunk_id")
            if chunk_id is not None:
                development_top10_chunk_ids.add(str(chunk_id))

canonical_document_to_chunk_count = Counter()
selected_chunks = {}
unknown_candidate_chunk_ids = set(development_top10_chunk_ids)

with CANONICAL_CHUNKS_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        chunk = json.loads(line)
        chunk_id = str(chunk.get("chunk_id", ""))

        for document_id in chunk.get("member_document_ids", []):
            canonical_document_to_chunk_count[str(document_id)] += 1

        if chunk_id in development_top10_chunk_ids:
            selected_chunks[chunk_id] = chunk
            unknown_candidate_chunk_ids.discard(chunk_id)

assert not unknown_candidate_chunk_ids, (
    f"Unresolved Development Top-10 chunks: "
    f"{len(unknown_candidate_chunk_ids)}"
)

lineage_parent_occurrences = Counter()
chunks_with_parent_chunk_id = 0
chunks_with_lineage_parent_id = 0
chunks_with_resolvable_lineage_parent = 0

for chunk in selected_chunks.values():
    if chunk.get("parent_chunk_id") not in (None, ""):
        chunks_with_parent_chunk_id += 1

    lineage = chunk.get("lineage", {})
    member_lineage = (
        lineage.get("member_lineage", [])
        if isinstance(lineage, dict)
        else []
    )

    chunk_parent_ids = set()
    for member in member_lineage:
        if not isinstance(member, dict):
            continue

        member_lineage_data = member.get("lineage", {})
        if not isinstance(member_lineage_data, dict):
            continue

        parent_id = member_lineage_data.get("parent_id")
        if parent_id not in (None, ""):
            normalized_parent_id = str(parent_id)
            lineage_parent_occurrences[normalized_parent_id] += 1
            chunk_parent_ids.add(normalized_parent_id)

    if chunk_parent_ids:
        chunks_with_lineage_parent_id += 1

    if any(
        parent_id in canonical_document_to_chunk_count
        for parent_id in chunk_parent_ids
    ):
        chunks_with_resolvable_lineage_parent += 1

unique_lineage_parent_ids = set(lineage_parent_occurrences)
resolvable_lineage_parent_ids = (
    unique_lineage_parent_ids & set(canonical_document_to_chunk_count)
)

audit = {
    "status": "PASS",
    "development_top10_candidate_chunk_count": len(
        development_top10_chunk_ids
    ),
    "canonical_candidate_chunks_resolved": len(selected_chunks),
    "unknown_development_candidate_chunk_count": len(
        unknown_candidate_chunk_ids
    ),
    "canonical_document_id_count": len(canonical_document_to_chunk_count),
    "chunks_with_direct_parent_chunk_id": chunks_with_parent_chunk_id,
    "chunks_with_lineage_parent_id": chunks_with_lineage_parent_id,
    "lineage_parent_id_occurrence_count": sum(
        lineage_parent_occurrences.values()
    ),
    "unique_lineage_parent_id_count": len(unique_lineage_parent_ids),
    "unique_lineage_parent_ids_matching_canonical_member_document_ids": len(
        resolvable_lineage_parent_ids
    ),
    "chunks_with_lineage_parent_resolvable_to_canonical_member_document": (
        chunks_with_resolvable_lineage_parent
    ),
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_top10_candidate_chunk_count": 3277,
  "canonical_candidate_chunks_resolved": 3277,
  "unknown_development_candidate_chunk_count": 0,
  "canonical_document_id_count": 19456,
  "chunks_with_direct_parent_chunk_id": 0,
  "chunks_with_lineage_parent_id": 2213,
  "lineage_parent_id_occurrence_count": 22112,
  "unique_lineage_parent_id_count": 197,
  "unique_lineage_parent_ids_matching_canonical_member_document_ids": 11,
  "chunks_with_lineage_parent_resolvable_to_canonical_member_document": 11,
  "files_written": 0,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}


<div dir="rtl">

### NB18 — بررسی سیگنال‌های تعارض در metadata

این گام metadata تکه‌های یکتای contextهای Development را بررسی می‌کند: آیا فیلد مستقیمی برای تعارض وجود دارد، یا فقط اطلاعاتی دربارهٔ تنوع اختیار و انتساب منابع ثبت شده است؟

متن تکه‌ها خوانده یا چاپ نمی‌شود و هیچ قضاوتی دربارهٔ تعارض محتوایی انجام نمی‌گیرد.

</div>

In [16]:
# NB18-C16 — Read-only audit of conflict and authority metadata signals

import json
from collections import Counter
from pathlib import Path

assert "development_contexts" in globals(), (
    "Run NB18-C11 first to create development_contexts."
)

REPO_ROOT = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
    "RAG-projects/RAG_finance"
).resolve()
TAXONOMY_PATH = REPO_ROOT / "artifacts/phase2/taxonomy/taxonomy_v1.json"
assert TAXONOMY_PATH.is_file(), f"Required input is missing: {TAXONOMY_PATH}"

taxonomy = json.loads(TAXONOMY_PATH.read_text(encoding="utf-8"))
conflict_requires_review_policy = (
    taxonomy.get("knowledge_authority", {})
    .get("policy", {})
    .get("primary_conflict_requires_review")
)

unique_items = {}
for context in development_contexts:
    assert context.get("split") == "development"
    for item in context.get("context_items", []):
        unique_items.setdefault(item["chunk_id"], item)

flag_fields = (
    "mixed_authority",
    "mixed_primary_signal",
    "multi_speaker",
)
list_fields = (
    "effective_primary_signals",
    "base_statuses",
    "content_origins",
    "content_authority_types",
)

def normalized_value(value):
    if value is None:
        return "<MISSING>"
    return json.dumps(value, ensure_ascii=False, sort_keys=True)

def find_conflict_paths(value, prefix=""):
    paths = set()

    if isinstance(value, dict):
        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            if any(
                term in str(key).lower()
                for term in ("conflict", "contradict")
            ):
                paths.add(path)
            paths.update(find_conflict_paths(child, path))

    elif isinstance(value, list):
        for child in value:
            paths.update(find_conflict_paths(child, f"{prefix}[]"))

    return paths

flag_value_counts = {field: Counter() for field in flag_fields}
list_value_counts = {field: Counter() for field in list_fields}
metadata_conflict_paths = set()

for item in unique_items.values():
    attribution = item.get("attribution_context")
    if not isinstance(attribution, dict):
        attribution = {}

    for field in flag_fields:
        flag_value_counts[field][normalized_value(attribution.get(field))] += 1

    for field in list_fields:
        list_value_counts[field][normalized_value(attribution.get(field))] += 1

    for metadata_field in ("attribution_context", "rights", "lineage"):
        metadata_conflict_paths.update(
            find_conflict_paths(item.get(metadata_field))
        )

audit = {
    "status": "PASS",
    "development_context_count": len(development_contexts),
    "unique_development_context_chunk_count": len(unique_items),
    "taxonomy_primary_conflict_requires_review": (
        conflict_requires_review_policy
    ),
    "direct_conflict_or_contradiction_metadata_paths": sorted(
        metadata_conflict_paths
    ),
    "authority_flag_value_counts": {
        field: dict(sorted(counter.items()))
        for field, counter in flag_value_counts.items()
    },
    "authority_list_value_counts": {
        field: dict(sorted(counter.items()))
        for field, counter in list_value_counts.items()
    },
    "semantic_conflict_detection_performed": False,
    "test_evidence_or_metrics_used": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_context_count": 2664,
  "unique_development_context_chunk_count": 3277,
  "taxonomy_primary_conflict_requires_review": true,
  "direct_conflict_or_contradiction_metadata_paths": [],
  "authority_flag_value_counts": {
    "mixed_authority": {
      "<MISSING>": 2213,
      "false": 1064
    },
    "mixed_primary_signal": {
      "<MISSING>": 1342,
      "false": 1414,
      "true": 521
    },
    "multi_speaker": {
      "<MISSING>": 1342,
      "false": 832,
      "true": 1103
    }
  },
  "authority_list_value_counts": {
    "effective_primary_signals": {
      "<MISSING>": 1342,
      "[null, false]": 1,
      "[null, true]": 256,
      "[null]": 843,
      "[true, null]": 264,
      "[true]": 571
    },
    "base_statuses": {
      "<MISSING>": 1342,
      "[\"EXPLICIT_SOURCE_MARKER\", \"UNRESOLVED\"]": 75,
      "[\"EXPLICIT_SOURCE_MARKER\"]": 929,
      "[\"MANIFEST_ATTRIBUTED\"]": 519,
      "[\"UNRESOLVED\", \"EXPLICIT_SOURCE_MARKER\"]": 202,

<div dir="rtl">

### NB18 — نگهداری سیگنال‌های اختیار بدون ادعای تشخیص تعارض

metadata تکه‌ها سیگنال‌هایی دربارهٔ انتساب، منبع اولیه و چندگویندگی دارد؛ اما فیلد مستقیمی برای تناقض محتوایی ندارد.

این گام سیگنال‌های موجود را در یک sidecar موقت نگه می‌دارد. آن‌ها به‌عنوان تعارض قطعی تفسیر نمی‌شوند و تعارض معنایی «ارزیابی‌نشده» می‌ماند.

</div>

In [17]:
# NB18-C17 — Preserve authority signals without inferring semantic conflicts

import json
from collections import Counter

assert "development_contexts" in globals(), (
    "Run NB18-C11 first to create development_contexts."
)

development_authority_signal_sidecar = {}
signal_counts = Counter()
base_status_counts = Counter()

for context in development_contexts:
    assert context.get("split") == "development"

    for item in context.get("context_items", []):
        chunk_id = item["chunk_id"]
        if chunk_id in development_authority_signal_sidecar:
            continue

        attribution = item.get("attribution_context")
        if not isinstance(attribution, dict):
            attribution = {}

        mixed_primary_signal = attribution.get("mixed_primary_signal")
        mixed_authority = attribution.get("mixed_authority")
        multi_speaker = attribution.get("multi_speaker")
        base_statuses = attribution.get("base_statuses")

        if not isinstance(base_statuses, list):
            base_statuses = []

        signal_reasons = []

        if mixed_primary_signal is True:
            signal_reasons.append("MIXED_PRIMARY_SIGNAL")
            signal_counts["mixed_primary_signal_true"] += 1

        if "UNRESOLVED" in base_statuses:
            signal_reasons.append("UNRESOLVED_BASE_STATUS")
            signal_counts["unresolved_base_status_present"] += 1

        if multi_speaker is True:
            signal_reasons.append("MULTI_SPEAKER")
            signal_counts["multi_speaker_true"] += 1

        if mixed_authority is True:
            signal_counts["mixed_authority_true"] += 1

        for status in base_statuses:
            base_status_counts[str(status)] += 1

        development_authority_signal_sidecar[chunk_id] = {
            "mixed_primary_signal": mixed_primary_signal,
            "mixed_authority": mixed_authority,
            "multi_speaker": multi_speaker,
            "base_statuses": base_statuses,
            "authority_signal_reasons": signal_reasons,
            "semantic_conflict_status": "NOT_ASSESSED",
        }

assert len(development_authority_signal_sidecar) == 3277
assert all(
    item["semantic_conflict_status"] == "NOT_ASSESSED"
    for item in development_authority_signal_sidecar.values()
)

audit = {
    "status": "PASS",
    "unique_development_chunk_count": len(
        development_authority_signal_sidecar
    ),
    "authority_signal_counts": dict(sorted(signal_counts.items())),
    "base_status_occurrence_counts": dict(sorted(base_status_counts.items())),
    "semantic_conflict_status_counts": {
        "NOT_ASSESSED": len(development_authority_signal_sidecar)
    },
    "signals_interpreted_as_proven_conflicts": False,
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "unique_development_chunk_count": 3277,
  "authority_signal_counts": {
    "mixed_primary_signal_true": 521,
    "multi_speaker_true": 1103,
    "unresolved_base_status_present": 487
  },
  "base_status_occurrence_counts": {
    "EXPLICIT_SOURCE_MARKER": 1206,
    "MANIFEST_ATTRIBUTED": 519,
    "NARRATIVE_CUE": 14,
    "UNRESOLVED": 487
  },
  "semantic_conflict_status_counts": {
    "NOT_ASSESSED": 3277
  },
  "signals_interpreted_as_proven_conflicts": false,
  "files_written": 0,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}


<div dir="rtl">

### NB18 — ساخت envelope موقت برای context و Context Gate

این گام برای هر پرس‌وجوی Development هر دو ترتیب را نگه می‌دارد: ترتیب رتبهٔ اصلی NB17 و ترتیب آزمایشی round‑robin بر اساس نوع منبع. اطلاعات حقوق و اختیار نیز به‌صورت ورودی‌های قابل بررسی برای Context Gate خلاصه می‌شوند.

این envelope فقط در حافظه است؛ بودجه‌بندی، رفع حقوق و تشخیص تعارض معنایی انجام نمی‌شود.

</div>

In [18]:
# NB18-C18 — Build in-memory Development context candidate envelopes

import json
from collections import Counter

assert "development_contexts" in globals(), (
    "Run NB18-C11 first to create development_contexts."
)
assert "source_round_robin" in globals(), (
    "Run NB18-C13 first to define source_round_robin."
)
assert "development_authority_signal_sidecar" in globals(), (
    "Run NB18-C17 first to create authority signal metadata."
)

development_context_candidates = []
rights_status_totals = Counter()
authority_signal_totals = Counter()
rights_allowed_uses_missing_total = 0
total_items = 0
priority_order_preserved_chunk_sets = True

for context in development_contexts:
    assert context.get("split") == "development"

    ranked_items = sorted(
        context.get("context_items", []),
        key=lambda item: item["rank"],
    )
    assert len(ranked_items) == 10

    source_priority_items = source_round_robin(ranked_items)
    ranked_chunk_ids = [item["chunk_id"] for item in ranked_items]
    priority_chunk_ids = [item["chunk_id"] for item in source_priority_items]

    if (
        len(ranked_chunk_ids) != len(priority_chunk_ids)
        or set(ranked_chunk_ids) != set(priority_chunk_ids)
    ):
        priority_order_preserved_chunk_sets = False
        raise AssertionError(
            f"Source-priority order changed the chunk set for "
            f"{context.get('candidate_id')!r}"
        )

    context_rights_status_counts = Counter()
    context_missing_allowed_uses_count = 0

    for item in ranked_items:
        total_items += 1

        sidecar = development_authority_signal_sidecar.get(
            item["chunk_id"], {}
        )
        for reason in sidecar.get("authority_signal_reasons", []):
            authority_signal_totals[reason] += 1
            authority_signal_totals[f"context:{reason}"] += 0

        rights = item.get("rights", {})
        if not isinstance(rights, dict):
            continue

        for member_right in rights.get("member_rights", []):
            if not isinstance(member_right, dict):
                continue

            member_rights = member_right.get("rights", {})
            if not isinstance(member_rights, dict):
                continue

            status = member_rights.get("status")
            if status is not None:
                normalized_status = str(status)
                context_rights_status_counts[normalized_status] += 1
                rights_status_totals[normalized_status] += 1

            allowed_uses = member_rights.get("allowed_uses")
            if allowed_uses in (None, "", [], {}):
                context_missing_allowed_uses_count += 1
                rights_allowed_uses_missing_total += 1

    development_context_candidates.append({
        "candidate_id": context.get("candidate_id"),
        "split": "development",
        "rank_order_items": ranked_items,
        "source_priority_items": source_priority_items,
        "context_gate_inputs": {
            "rights_status_counts": dict(
                sorted(context_rights_status_counts.items())
            ),
            "member_rights_without_allowed_uses_count": (
                context_missing_allowed_uses_count
            ),
            "authority_signal_counts": {
                reason: sum(
                    reason in development_authority_signal_sidecar.get(
                        item["chunk_id"], {}
                    ).get("authority_signal_reasons", [])
                    for item in ranked_items
                )
                for reason in (
                    "MIXED_PRIMARY_SIGNAL",
                    "UNRESOLVED_BASE_STATUS",
                    "MULTI_SPEAKER",
                )
            },
            "semantic_conflict_status": "NOT_ASSESSED",
            "token_budget_applied": False,
            "parent_expansion_applied": False,
            "topic_diversity_applied": False,
        },
    })

assert len(development_context_candidates) == 2664
assert total_items == 26640
assert priority_order_preserved_chunk_sets

audit = {
    "status": "PASS",
    "development_context_candidate_count": len(
        development_context_candidates
    ),
    "candidate_items_total": total_items,
    "rank_order_items_per_context": 10,
    "source_priority_order_preserves_chunk_set": (
        priority_order_preserved_chunk_sets
    ),
    "rights_status_member_counts": dict(sorted(rights_status_totals.items())),
    "member_rights_without_allowed_uses_count": (
        rights_allowed_uses_missing_total
    ),
    "authority_signal_occurrence_counts": dict(
        sorted({
            reason: sum(
                reason in metadata.get("authority_signal_reasons", [])
                for metadata in development_authority_signal_sidecar.values()
            )
            for reason in (
                "MIXED_PRIMARY_SIGNAL",
                "UNRESOLVED_BASE_STATUS",
                "MULTI_SPEAKER",
            )
        }.items())
    ),
    "token_budget_applied": False,
    "rights_cleared": False,
    "semantic_conflict_assessed": False,
    "parent_expansion_applied": False,
    "topic_diversity_applied": False,
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_context_candidate_count": 2664,
  "candidate_items_total": 26640,
  "rank_order_items_per_context": 10,
  "source_priority_order_preserves_chunk_set": true,
  "rights_status_member_counts": {
    "EVIDENCE_REQUIRED": 149993
  },
  "member_rights_without_allowed_uses_count": 160500,
  "authority_signal_occurrence_counts": {
    "MIXED_PRIMARY_SIGNAL": 521,
    "MULTI_SPEAKER": 1103,
    "UNRESOLVED_BASE_STATUS": 487
  },
  "token_budget_applied": false,
  "rights_cleared": false,
  "semantic_conflict_assessed": false,
  "parent_expansion_applied": false,
  "topic_diversity_applied": false,
  "files_written": 0,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}


<div dir="rtl">

### NB18 — ممیزی آمادگی ورودی‌های Context Gate

این گام فقط میزان کامل‌بودن ورودی‌های حقوق را می‌شمارد و ثبت می‌کند که بودجهٔ مدل هنوز اعمال نشده و تعارض معنایی ارزیابی نشده است.

`PASS` در این ممیزی به‌معنای تأیید حقوق یا آماده‌بودن انتشار نیست.

</div>

In [19]:
# NB18-C19 — Read-only Context Gate readiness audit

import json
from collections import Counter

assert "development_context_candidates" in globals(), (
    "Run NB18-C18 first to create development_context_candidates."
)

context_count = len(development_context_candidates)
item_count = 0
rights_member_count = 0
rights_status_counts = Counter()
missing_allowed_uses_count = 0
missing_rights_evidence_id_count = 0
contexts_with_rights_evidence_required = 0
contexts_with_missing_rights_fields = 0
direct_parent_chunk_link_count = 0
contexts_with_any_direct_parent_link = 0

for context in development_context_candidates:
    context_has_required_evidence_status = False
    context_has_missing_rights_fields = False
    context_has_parent_link = False

    for item in context["rank_order_items"]:
        item_count += 1

        if item.get("parent_chunk_id") not in (None, ""):
            direct_parent_chunk_link_count += 1
            context_has_parent_link = True

        rights = item.get("rights", {})
        if not isinstance(rights, dict):
            continue

        for member_right in rights.get("member_rights", []):
            if not isinstance(member_right, dict):
                continue

            member_rights = member_right.get("rights", {})
            if not isinstance(member_rights, dict):
                continue

            rights_member_count += 1

            status = member_rights.get("status")
            if status is not None:
                normalized_status = str(status)
                rights_status_counts[normalized_status] += 1
                if normalized_status == "EVIDENCE_REQUIRED":
                    context_has_required_evidence_status = True

            if member_rights.get("allowed_uses") in (None, "", [], {}):
                missing_allowed_uses_count += 1
                context_has_missing_rights_fields = True

            if member_rights.get("rights_evidence_id") in (None, "", [], {}):
                missing_rights_evidence_id_count += 1
                context_has_missing_rights_fields = True

    if context_has_required_evidence_status:
        contexts_with_rights_evidence_required += 1
    if context_has_missing_rights_fields:
        contexts_with_missing_rights_fields += 1
    if context_has_parent_link:
        contexts_with_any_direct_parent_link += 1

assert context_count == 2664
assert item_count == 26640

audit = {
    "status": "PASS",
    "development_context_count": context_count,
    "context_items_total": item_count,
    "rights_member_reference_count": rights_member_count,
    "rights_status_occurrence_counts": dict(sorted(rights_status_counts.items())),
    "rights_member_references_missing_allowed_uses": (
        missing_allowed_uses_count
    ),
    "rights_member_references_missing_evidence_id": (
        missing_rights_evidence_id_count
    ),
    "contexts_with_evidence_required_status": (
        contexts_with_rights_evidence_required
    ),
    "contexts_with_missing_rights_fields": (
        contexts_with_missing_rights_fields
    ),
    "direct_parent_chunk_link_count": direct_parent_chunk_link_count,
    "contexts_with_direct_parent_chunk_link": (
        contexts_with_any_direct_parent_link
    ),
    "production_token_budget_applied": False,
    "semantic_conflict_assessed": False,
    "rights_clearance_asserted": False,
    "test_evidence_or_metrics_used": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_context_count": 2664,
  "context_items_total": 26640,
  "rights_member_reference_count": 160500,
  "rights_status_occurrence_counts": {
    "EVIDENCE_REQUIRED": 149993
  },
  "rights_member_references_missing_allowed_uses": 160500,
  "rights_member_references_missing_evidence_id": 160500,
  "contexts_with_evidence_required_status": 2526,
  "contexts_with_missing_rights_fields": 2664,
  "direct_parent_chunk_link_count": 0,
  "contexts_with_direct_parent_chunk_link": 0,
  "production_token_budget_applied": false,
  "semantic_conflict_assessed": false,
  "rights_clearance_asserted": false,
  "test_evidence_or_metrics_used": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl">

### NB18 — ثبت نتیجهٔ موقت Context Gate

این گام برای هر context وضعیت completeness حقوق و بودجه را جداگانه ثبت می‌کند. فیلدهای ناقص حقوق باعث می‌شوند مجوز انتشار تأیید نشود؛ این وضعیت مانع از نگهداری context داخلی برای بررسی نیست.

بودجهٔ تولیدی هنوز تنظیم نشده و هیچ فایلی نوشته نمی‌شود.

</div>

In [20]:
# NB18-C20 — Record fail-closed release-gate results per Development context

import json

assert "development_context_candidates" in globals(), (
    "Run NB18-C18 first to create development_context_candidates."
)

development_context_gate_results = []

reason_context_counts = {
    "EVIDENCE_REQUIRED": 0,
    "MISSING_ALLOWED_USES": 0,
    "MISSING_RIGHTS_EVIDENCE_ID": 0,
    "MISSING_RIGHTS_STATUS": 0,
    "TOKEN_BUDGET_NOT_CONFIGURED": 0,
}

total_rights_member_references = 0
total_missing_allowed_uses = 0
total_missing_rights_evidence_ids = 0
total_missing_rights_statuses = 0

for context in development_context_candidates:
    reasons = []
    member_count = 0
    missing_allowed_uses = 0
    missing_evidence_ids = 0
    missing_statuses = 0
    evidence_required_members = 0

    for item in context["rank_order_items"]:
        rights = item.get("rights", {})
        if not isinstance(rights, dict):
            continue

        for member_right in rights.get("member_rights", []):
            if not isinstance(member_right, dict):
                continue

            member_rights = member_right.get("rights", {})
            if not isinstance(member_rights, dict):
                continue

            member_count += 1
            status = member_rights.get("status")

            if status in (None, "", [], {}):
                missing_statuses += 1
            elif str(status) == "EVIDENCE_REQUIRED":
                evidence_required_members += 1

            if member_rights.get("allowed_uses") in (None, "", [], {}):
                missing_allowed_uses += 1

            if member_rights.get("rights_evidence_id") in (None, "", [], {}):
                missing_evidence_ids += 1

    total_rights_member_references += member_count
    total_missing_allowed_uses += missing_allowed_uses
    total_missing_rights_evidence_ids += missing_evidence_ids
    total_missing_rights_statuses += missing_statuses

    if evidence_required_members:
        reasons.append("EVIDENCE_REQUIRED")
        reason_context_counts["EVIDENCE_REQUIRED"] += 1

    if missing_allowed_uses:
        reasons.append("MISSING_ALLOWED_USES")
        reason_context_counts["MISSING_ALLOWED_USES"] += 1

    if missing_evidence_ids:
        reasons.append("MISSING_RIGHTS_EVIDENCE_ID")
        reason_context_counts["MISSING_RIGHTS_EVIDENCE_ID"] += 1

    if missing_statuses:
        reasons.append("MISSING_RIGHTS_STATUS")
        reason_context_counts["MISSING_RIGHTS_STATUS"] += 1

    # No production tokenizer or effective context budget has been supplied.
    reasons.append("TOKEN_BUDGET_NOT_CONFIGURED")
    reason_context_counts["TOKEN_BUDGET_NOT_CONFIGURED"] += 1

    rights_evidence_complete = not any(
        reason in reasons
        for reason in (
            "EVIDENCE_REQUIRED",
            "MISSING_ALLOWED_USES",
            "MISSING_RIGHTS_EVIDENCE_ID",
            "MISSING_RIGHTS_STATUS",
        )
    )

    development_context_gate_results.append({
        "candidate_id": context.get("candidate_id"),
        "split": "development",
        "internal_context_candidate_available": bool(
            context.get("rank_order_items")
        ),
        "rights_evidence_complete": rights_evidence_complete,
        "release_eligible": rights_evidence_complete,
        "production_token_budget_configured": False,
        "generation_ready": False,
        "gate_reasons": reasons,
        "rights_member_reference_count": member_count,
    })

assert len(development_context_gate_results) == 2664
assert all(
    not result["release_eligible"]
    for result in development_context_gate_results
)
assert all(
    not result["generation_ready"]
    for result in development_context_gate_results
)

audit = {
    "status": "PASS",
    "development_context_count": len(development_context_gate_results),
    "contexts_with_internal_candidate_available": sum(
        result["internal_context_candidate_available"]
        for result in development_context_gate_results
    ),
    "contexts_release_eligible": sum(
        result["release_eligible"]
        for result in development_context_gate_results
    ),
    "contexts_generation_ready": sum(
        result["generation_ready"]
        for result in development_context_gate_results
    ),
    "rights_member_reference_count": total_rights_member_references,
    "rights_member_references_missing_allowed_uses": (
        total_missing_allowed_uses
    ),
    "rights_member_references_missing_rights_evidence_id": (
        total_missing_rights_evidence_ids
    ),
    "rights_member_references_missing_status": total_missing_rights_statuses,
    "contexts_by_gate_reason": reason_context_counts,
    "production_budget_selected": False,
    "semantic_conflict_assessed": False,
    "test_evidence_or_metrics_used": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_context_count": 2664,
  "contexts_with_internal_candidate_available": 2664,
  "contexts_release_eligible": 0,
  "contexts_generation_ready": 0,
  "rights_member_reference_count": 160500,
  "rights_member_references_missing_allowed_uses": 160500,
  "rights_member_references_missing_rights_evidence_id": 160500,
  "rights_member_references_missing_status": 10507,
  "contexts_by_gate_reason": {
    "EVIDENCE_REQUIRED": 2526,
    "MISSING_ALLOWED_USES": 2664,
    "MISSING_RIGHTS_EVIDENCE_ID": 2664,
    "MISSING_RIGHTS_STATUS": 2379,
    "TOKEN_BUDGET_NOT_CONFIGURED": 2664
  },
  "production_budget_selected": false,
  "semantic_conflict_assessed": false,
  "test_evidence_or_metrics_used": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl">

### NB18 — checkpoint آمادگی Context Builder

این گام خلاصه می‌کند چه مقدار از contextهای Development آمادهٔ بررسی داخلی است و چرا هنوز نمی‌توان آن‌ها را برای generation یا انتشار عبور داد.

</div>

In [21]:
# NB18-C21 — Development-only Context Builder readiness checkpoint

import json
from collections import Counter

assert "development_context_candidates" in globals(), (
    "Run NB18-C18 first."
)
assert "development_context_gate_results" in globals(), (
    "Run NB18-C20 first."
)

context_count = len(development_context_candidates)
gate_count = len(development_context_gate_results)
assert context_count == gate_count == 2664

unique_chunk_ids = set()
duplicate_chunk_context_count = 0
source_type_counts = Counter()
direct_parent_chunk_link_count = 0

for context in development_context_candidates:
    ranked_items = context["rank_order_items"]
    chunk_ids = [item["chunk_id"] for item in ranked_items]

    if len(chunk_ids) != len(set(chunk_ids)):
        duplicate_chunk_context_count += 1

    unique_chunk_ids.update(chunk_ids)

    for item in context["source_priority_items"]:
        source_type = item.get("source_type")
        if source_type is not None:
            source_type_counts[str(source_type)] += 1

    direct_parent_chunk_link_count += sum(
        item.get("parent_chunk_id") not in (None, "")
        for item in ranked_items
    )

gate_reason_context_counts = Counter()
for result in development_context_gate_results:
    for reason in result.get("gate_reasons", []):
        gate_reason_context_counts[reason] += 1

contexts_with_internal_candidate = sum(
    result["internal_context_candidate_available"]
    for result in development_context_gate_results
)
contexts_release_eligible = sum(
    result["release_eligible"]
    for result in development_context_gate_results
)
contexts_generation_ready = sum(
    result["generation_ready"]
    for result in development_context_gate_results
)

if contexts_generation_ready == 0 and contexts_release_eligible == 0:
    checkpoint_status = "BLOCKED_FOR_GENERATION_AND_RELEASE"
else:
    checkpoint_status = "REQUIRES_REVIEW"

checkpoint = {
    "status": checkpoint_status,
    "development_context_count": context_count,
    "contexts_with_internal_candidate_available": (
        contexts_with_internal_candidate
    ),
    "unique_top10_chunk_count": len(unique_chunk_ids),
    "items_per_context": 10,
    "contexts_with_duplicate_chunk_ids": duplicate_chunk_context_count,
    "source_priority_candidate_slot_counts": dict(
        sorted(source_type_counts.items())
    ),
    "direct_parent_chunk_links": direct_parent_chunk_link_count,
    "contexts_release_eligible": contexts_release_eligible,
    "contexts_generation_ready": contexts_generation_ready,
    "gate_reason_context_counts": dict(
        sorted(gate_reason_context_counts.items())
    ),
    "topic_diversity_applied": False,
    "semantic_conflict_assessed": False,
    "production_budget_selected": False,
    "test_evidence_or_metrics_used": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(checkpoint, ensure_ascii=False, indent=2))

{
  "status": "BLOCKED_FOR_GENERATION_AND_RELEASE",
  "development_context_count": 2664,
  "contexts_with_internal_candidate_available": 2664,
  "unique_top10_chunk_count": 3277,
  "items_per_context": 10,
  "contexts_with_duplicate_chunk_ids": 0,
  "source_priority_candidate_slot_counts": {
    "book": 2524,
    "transcript": 13609,
    "website": 10507
  },
  "direct_parent_chunk_links": 0,
  "contexts_release_eligible": 0,
  "contexts_generation_ready": 0,
  "gate_reason_context_counts": {
    "EVIDENCE_REQUIRED": 2526,
    "MISSING_ALLOWED_USES": 2664,
    "MISSING_RIGHTS_EVIDENCE_ID": 2664,
    "MISSING_RIGHTS_STATUS": 2379,
    "TOKEN_BUDGET_NOT_CONFIGURED": 2664
  },
  "topic_diversity_applied": false,
  "semantic_conflict_assessed": false,
  "production_budget_selected": false,
  "test_evidence_or_metrics_used": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl">

### NB18 — ساخت contextهای Development با حفظ منبع و انتساب مؤلف

برای هر پرس‌وجو، Top‑10 رتبه‌بندی‌شده نگه داشته می‌شود. کنار رتبهٔ اصلی NB17، اولویت پیشنهادی تنوع منبع هم ثبت می‌شود تا در صورت نیاز به بودجه‌بندی استفاده شود.

متن، locator، انتساب نویسنده/گوینده، اطلاعات حقوق و سیگنال‌های اختیار حفظ می‌شوند. در این گام نه متن بریده می‌شود و نه چیزی به فایل نوشته می‌شود.

</div>

In [22]:
# NB18-C22 — Assemble Development contexts with provenance and source priority

import json
from collections import Counter

assert "development_context_candidates" in globals(), (
    "Run NB18-C18 first."
)
assert "source_round_robin" in globals(), (
    "Run NB18-C13 first."
)
assert "development_authority_signal_sidecar" in globals(), (
    "Run NB18-C17 first."
)

nb18_development_contexts = []

source_type_slot_counts = Counter()
rights_status_counts = Counter()
contexts_without_member_documents = 0
items_without_locator_metadata = 0
member_document_references_total = 0
member_document_references_with_metadata = 0

for context in development_context_candidates:
    ranked_items = sorted(
        context["rank_order_items"],
        key=lambda item: item["rank"],
    )
    priority_items = source_round_robin(ranked_items)

    assert len(ranked_items) == 10
    assert {item["chunk_id"] for item in ranked_items} == {
        item["chunk_id"] for item in priority_items
    }

    selection_priority_by_chunk_id = {
        item["chunk_id"]: priority_rank
        for priority_rank, item in enumerate(priority_items, start=1)
    }

    context_items = []

    for item in ranked_items:
        chunk_id = item["chunk_id"]
        member_documents = item.get("member_documents", [])

        if not member_documents:
            contexts_without_member_documents += 1

        member_document_references_total += len(
            item.get("member_document_ids", [])
        )
        member_document_references_with_metadata += len(member_documents)

        if not item.get("member_locators") and not item.get("locator_start") and not item.get("locator_end"):
            items_without_locator_metadata += 1

        source_type = item.get("source_type")
        if source_type is not None:
            source_type_slot_counts[str(source_type)] += 1

        item_rights = item.get("rights", {})
        if isinstance(item_rights, dict):
            for member_right in item_rights.get("member_rights", []):
                if not isinstance(member_right, dict):
                    continue
                member_rights = member_right.get("rights", {})
                if isinstance(member_rights, dict):
                    status = member_rights.get("status")
                    if status is not None:
                        rights_status_counts[str(status)] += 1

        context_items.append({
            "evidence_id": chunk_id,
            "chunk_id": chunk_id,
            "reranker_rank": item["rank"],
            "selection_priority_rank": selection_priority_by_chunk_id[chunk_id],
            "reranker_score": item.get("reranker_score"),
            "text": item.get("content"),
            "source_token_count": item.get("token_count"),
            "char_count": item.get("char_count"),
            "source": {
                "source_type": item.get("source_type"),
                "source_id": item.get("source_id"),
                "role": item.get("role"),
            },
            "locator": {
                "member_locators": item.get("member_locators", []),
                "start": item.get("locator_start"),
                "end": item.get("locator_end"),
            },
            "member_document_ids": item.get("member_document_ids", []),
            "member_documents": member_documents,
            "attribution_context": item.get("attribution_context"),
            "authority_signals": development_authority_signal_sidecar.get(
                chunk_id,
                {
                    "semantic_conflict_status": "NOT_ASSESSED",
                    "authority_signal_reasons": [],
                },
            ),
            "rights": item_rights,
            "parent_chunk_id": item.get("parent_chunk_id"),
            "lineage": item.get("lineage"),
        })

    assert [item["reranker_rank"] for item in context_items] == list(
        range(1, 11)
    )

    nb18_development_contexts.append({
        "candidate_id": context.get("candidate_id"),
        "split": "development",
        "context_items": context_items,
        "source_priority_order": [
            item["chunk_id"] for item in priority_items
        ],
        "packing_applied": False,
        "token_budget_applied": False,
    })

assert len(nb18_development_contexts) == 2664
assert sum(
    len(context["context_items"])
    for context in nb18_development_contexts
) == 26640

audit = {
    "status": "PASS",
    "development_context_count": len(nb18_development_contexts),
    "context_items_total": sum(
        len(context["context_items"])
        for context in nb18_development_contexts
    ),
    "items_per_context": 10,
    "source_type_candidate_slot_counts": dict(
        sorted(source_type_slot_counts.items())
    ),
    "rights_status_occurrence_counts": dict(
        sorted(rights_status_counts.items())
    ),
    "member_document_references_total": member_document_references_total,
    "member_document_references_with_metadata": (
        member_document_references_with_metadata
    ),
    "contexts_without_member_documents": contexts_without_member_documents,
    "items_without_locator_metadata": items_without_locator_metadata,
    "original_reranker_rank_preserved": True,
    "source_priority_order_attached": True,
    "source_token_count_used_as_model_token_count": False,
    "packing_applied": False,
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_context_count": 2664,
  "context_items_total": 26640,
  "items_per_context": 10,
  "source_type_candidate_slot_counts": {
    "book": 2524,
    "transcript": 13609,
    "website": 10507
  },
  "rights_status_occurrence_counts": {
    "EVIDENCE_REQUIRED": 149993
  },
  "member_document_references_total": 160500,
  "member_document_references_with_metadata": 160500,
  "contexts_without_member_documents": 0,
  "items_without_locator_metadata": 0,
  "original_reranker_rank_preserved": true,
  "source_priority_order_attached": true,
  "source_token_count_used_as_model_token_count": false,
  "packing_applied": false,
  "files_written": 0,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}


<div dir="rtl">

### NB18 — اعتبارسنجی fidelity و provenance آیتم‌های context

این گام آیتم‌های ساخته‌شده را با chunk canonical، metadata رتبه‌بندی‌شده و رکوردهای سند مقایسه می‌کند. هدف، تأیید حفظ دقیق متن، رتبه، locator، انتساب، حقوق و lineage است؛ متن یا مقدار metadata چاپ نمی‌شود.

</div>

In [23]:
# NB18-C23 — Validate context item fidelity against canonical inputs

import json
from collections import Counter

required_globals = (
    "nb18_development_contexts",
    "development_context_candidates",
    "canonical_chunks",
    "accepted_document_metadata",
    "development_authority_signal_sidecar",
    "source_round_robin",
)
missing_globals = [name for name in required_globals if name not in globals()]
assert not missing_globals, f"Required prior cells are missing: {missing_globals}"

assert len(nb18_development_contexts) == len(development_context_candidates)
assert len(nb18_development_contexts) == 2664

field_mismatch_counts = Counter()
compared_item_count = 0
priority_order_mismatch_count = 0
member_document_join_mismatch_count = 0

canonical_field_map = {
    "text": "content",
    "source_token_count": "token_count",
    "char_count": "char_count",
    "source_type": "source_type",
    "source_id": "source_id",
    "role": "role",
    "member_document_ids": "member_document_ids",
    "member_locators": "member_locators",
    "locator_start": "locator_start",
    "locator_end": "locator_end",
    "attribution_context": "attribution_context",
    "rights": "rights",
    "parent_chunk_id": "parent_chunk_id",
    "lineage": "lineage",
}

for output_context, input_context in zip(
    nb18_development_contexts,
    development_context_candidates,
):
    assert output_context.get("candidate_id") == input_context.get("candidate_id")
    assert output_context.get("split") == "development"
    assert len(output_context["context_items"]) == 10

    expected_priority_items = source_round_robin(
        input_context["rank_order_items"]
    )
    expected_priority_ids = [
        item["chunk_id"] for item in expected_priority_items
    ]
    if output_context["source_priority_order"] != expected_priority_ids:
        priority_order_mismatch_count += 1

    expected_priority_ranks = {
        item["chunk_id"]: priority_rank
        for priority_rank, item in enumerate(expected_priority_items, start=1)
    }
    input_items_by_id = {
        item["chunk_id"]: item
        for item in input_context["rank_order_items"]
    }

    for output_item in output_context["context_items"]:
        compared_item_count += 1
        chunk_id = output_item["chunk_id"]
        source_item = input_items_by_id[chunk_id]
        canonical_chunk = canonical_chunks[chunk_id]

        if output_item["evidence_id"] != chunk_id:
            field_mismatch_counts["evidence_id"] += 1

        if output_item["reranker_rank"] != source_item["rank"]:
            field_mismatch_counts["reranker_rank"] += 1

        if output_item["selection_priority_rank"] != expected_priority_ranks[chunk_id]:
            field_mismatch_counts["selection_priority_rank"] += 1

        if output_item["reranker_score"] != source_item.get("reranker_score"):
            field_mismatch_counts["reranker_score"] += 1

        actual_source = output_item["source"]
        for output_field, canonical_field in canonical_field_map.items():
            actual_value = (
                output_item["text"]
                if output_field == "text"
                else output_item["source_token_count"]
                if output_field == "source_token_count"
                else output_item["char_count"]
                if output_field == "char_count"
                else output_item["locator"]["member_locators"]
                if output_field == "member_locators"
                else output_item["locator"]["start"]
                if output_field == "locator_start"
                else output_item["locator"]["end"]
                if output_field == "locator_end"
                else actual_source.get(output_field)
                if output_field in ("source_type", "source_id", "role")
                else output_item.get(output_field)
            )

            if actual_value != canonical_chunk.get(canonical_field):
                field_mismatch_counts[output_field] += 1

        expected_member_documents = [
            {
                "document_id": str(document_id),
                **accepted_document_metadata[str(document_id)],
            }
            for document_id in canonical_chunk.get("member_document_ids", [])
        ]
        if output_item["member_documents"] != expected_member_documents:
            member_document_join_mismatch_count += 1

        expected_authority_signals = development_authority_signal_sidecar.get(
            chunk_id,
            {
                "semantic_conflict_status": "NOT_ASSESSED",
                "authority_signal_reasons": [],
            },
        )
        if output_item["authority_signals"] != expected_authority_signals:
            field_mismatch_counts["authority_signals"] += 1

audit = {
    "status": (
        "PASS"
        if not field_mismatch_counts
        and priority_order_mismatch_count == 0
        and member_document_join_mismatch_count == 0
        else "FAIL"
    ),
    "development_context_count": len(nb18_development_contexts),
    "context_items_compared": compared_item_count,
    "canonical_field_mismatch_counts": dict(
        sorted(field_mismatch_counts.items())
    ),
    "source_priority_order_mismatches": priority_order_mismatch_count,
    "member_document_join_mismatches": member_document_join_mismatch_count,
    "text_and_metadata_exactly_preserved": (
        not field_mismatch_counts
        and priority_order_mismatch_count == 0
        and member_document_join_mismatch_count == 0
    ),
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_context_count": 2664,
  "context_items_compared": 26640,
  "canonical_field_mismatch_counts": {},
  "source_priority_order_mismatches": 0,
  "member_document_join_mismatches": 0,
  "text_and_metadata_exactly_preserved": true,
  "files_written": 0,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}


<div dir="rtl">

### NB18 — تابع اصلی ساخت context با بودجهٔ ورودی

تابع contextهای آماده را با اولویت تنوع منبع می‌گیرد و تکه‌های کامل را تا سقف بودجه می‌چیند. رتبهٔ NB17 و metadata حفظ می‌شوند؛ متن تکه بریده نمی‌شود.

بودجه، tokenizer و serializer ورودی‌های اجباری‌اند و در این سلول هیچ مقداری برای تولید انتخاب نمی‌شود.

</div>

In [24]:
# NB18-C24 — Main model-neutral context packet builder

import json

assert "pack_in_priority_order" in globals(), (
    "Run NB18-C14 first."
)

def build_context_packet(
    context_record,
    *,
    context_budget_tokens,
    render_context,
    count_tokens,
):
    """
    Build one context packet using caller-supplied production configuration.

    render_context receives the selected context item dictionaries and must
    return the exact context segment to be passed downstream.
    count_tokens must use the tokenizer that matches that downstream model.
    """
    if context_record.get("split") != "development":
        raise ValueError("This NB18 execution is restricted to Development.")
    if not callable(render_context):
        raise TypeError("render_context must be callable.")
    if not callable(count_tokens):
        raise TypeError("count_tokens must be callable.")

    items = context_record.get("context_items", [])
    ordered_items = sorted(
        items,
        key=lambda item: item["selection_priority_rank"],
    )

    ordered_chunk_ids = [item["chunk_id"] for item in ordered_items]
    expected_chunk_ids = context_record.get("source_priority_order", [])

    if ordered_chunk_ids != expected_chunk_ids:
        raise ValueError(
            "Context items do not match the stored source priority order."
        )

    packer_items = [
        {
            "rank": item["selection_priority_rank"],
            "chunk_id": item["chunk_id"],
            "context_item": item,
        }
        for item in ordered_items
    ]

    def render_packer_items(wrapper_items):
        return render_context([
            wrapper["context_item"] for wrapper in wrapper_items
        ])

    packed = pack_in_priority_order(
        packer_items,
        context_budget_tokens=context_budget_tokens,
        render_context=render_packer_items,
        count_tokens=count_tokens,
    )

    selected_items = [
        wrapper["context_item"]
        for wrapper in packed["selected_items"]
    ]
    skipped_items = [
        {
            "evidence_id": wrapper["context_item"]["evidence_id"],
            "chunk_id": wrapper["context_item"]["chunk_id"],
            "reranker_rank": wrapper["context_item"]["reranker_rank"],
            "selection_priority_rank": wrapper["rank"],
            "reason": "CONTEXT_BUDGET",
        }
        for wrapper in packed["skipped_items"]
    ]

    rights_member_count = 0
    rights_status_counts = {}
    missing_allowed_uses_count = 0
    missing_rights_evidence_id_count = 0
    missing_rights_status_count = 0

    for item in selected_items:
        rights = item.get("rights", {})
        if not isinstance(rights, dict):
            continue

        for member_right in rights.get("member_rights", []):
            if not isinstance(member_right, dict):
                continue

            member_rights = member_right.get("rights", {})
            if not isinstance(member_rights, dict):
                continue

            rights_member_count += 1
            status = member_rights.get("status")

            if status in (None, "", [], {}):
                missing_rights_status_count += 1
            else:
                normalized_status = str(status)
                rights_status_counts[normalized_status] = (
                    rights_status_counts.get(normalized_status, 0) + 1
                )

            if member_rights.get("allowed_uses") in (None, "", [], {}):
                missing_allowed_uses_count += 1

            if member_rights.get("rights_evidence_id") in (None, "", [], {}):
                missing_rights_evidence_id_count += 1

    return {
        "candidate_id": context_record.get("candidate_id"),
        "split": "development",
        "context_items": selected_items,
        "skipped_items": skipped_items,
        "rendered_context": packed["rendered_context"],
        "context_token_count": packed["context_token_count"],
        "context_budget_tokens": packed["context_budget_tokens"],
        "original_reranker_ranks": [
            item["reranker_rank"] for item in selected_items
        ],
        "selection_priority_ranks": [
            item["selection_priority_rank"] for item in selected_items
        ],
        "rights_metadata_summary": {
            "member_reference_count": rights_member_count,
            "status_counts": dict(sorted(rights_status_counts.items())),
            "missing_allowed_uses_count": missing_allowed_uses_count,
            "missing_rights_evidence_id_count": (
                missing_rights_evidence_id_count
            ),
            "missing_status_count": missing_rights_status_count,
            "clearance_asserted": False,
        },
        "semantic_conflict_status": "NOT_ASSESSED",
        "text_truncation_applied": False,
    }


# Fixture-only check; this does not configure or invoke a production model.
_fixture_context = {
    "candidate_id": "fixture",
    "split": "development",
    "source_priority_order": [
        "fixture-book-1",
        "fixture-transcript-1",
        "fixture-website-1",
        "fixture-book-2",
    ],
    "context_items": [
        {
            "evidence_id": "fixture-book-1",
            "chunk_id": "fixture-book-1",
            "reranker_rank": 1,
            "selection_priority_rank": 1,
            "source": {"source_type": "book"},
            "text": "a",
            "rights": {},
        },
        {
            "evidence_id": "fixture-book-2",
            "chunk_id": "fixture-book-2",
            "reranker_rank": 2,
            "selection_priority_rank": 4,
            "source": {"source_type": "book"},
            "text": "b",
            "rights": {},
        },
        {
            "evidence_id": "fixture-transcript-1",
            "chunk_id": "fixture-transcript-1",
            "reranker_rank": 3,
            "selection_priority_rank": 2,
            "source": {"source_type": "transcript"},
            "text": "c",
            "rights": {},
        },
        {
            "evidence_id": "fixture-website-1",
            "chunk_id": "fixture-website-1",
            "reranker_rank": 4,
            "selection_priority_rank": 3,
            "source": {"source_type": "website"},
            "text": "d",
            "rights": {},
        },
    ],
}

_fixture_packet = build_context_packet(
    _fixture_context,
    context_budget_tokens=3,
    render_context=lambda selected: " ".join(item["text"] for item in selected),
    count_tokens=lambda text: len(text.split()),
)

assert _fixture_packet["selection_priority_ranks"] == [1, 2, 3]
assert _fixture_packet["original_reranker_ranks"] == [1, 3, 4]
assert len(_fixture_packet["skipped_items"]) == 1
assert _fixture_packet["skipped_items"][0]["chunk_id"] == "fixture-book-2"
assert _fixture_packet["context_token_count"] == 3
assert _fixture_packet["rights_metadata_summary"]["clearance_asserted"] is False

print(json.dumps({
    "status": "PASS",
    "fixture_only": True,
    "production_tokenizer_used": False,
    "production_budget_selected": False,
    "selected_original_reranker_ranks": (
        _fixture_packet["original_reranker_ranks"]
    ),
    "selected_source_priority_ranks": (
        _fixture_packet["selection_priority_ranks"]
    ),
    "skipped_chunk_ids": [
        item["chunk_id"] for item in _fixture_packet["skipped_items"]
    ],
    "fixture_token_count": _fixture_packet["context_token_count"],
    "fixture_budget": _fixture_packet["context_budget_tokens"],
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "fixture_only": true,
  "production_tokenizer_used": false,
  "production_budget_selected": false,
  "selected_original_reranker_ranks": [
    1,
    3,
    4
  ],
  "selected_source_priority_ranks": [
    1,
    2,
    3
  ],
  "skipped_chunk_ids": [
    "fixture-book-2"
  ],
  "fixture_token_count": 3,
  "fixture_budget": 3,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl">

### NB18 — ساخت packetهای داخلی Development

این گام برای هر پرس‌وجو یک packet از Top‑10 می‌سازد و هم‌زمان رتبهٔ NB17 و اولویت تنوع منبع را حفظ می‌کند. چون هنوز بودجهٔ مدل اعمال نشده، متن کامل می‌ماند و تعداد توکن مدل گزارش نمی‌شود.

</div>

In [25]:
# NB18-C25 — Materialize unbudgeted Development context packets in memory

import json
from collections import Counter

assert "nb18_development_contexts" in globals(), (
    "Run NB18-C22 first."
)
assert "development_context_gate_results" in globals(), (
    "Run NB18-C20 first."
)

gate_by_candidate_id = {
    result["candidate_id"]: result
    for result in development_context_gate_results
}
assert len(gate_by_candidate_id) == 2664

nb18_development_packets = []
source_slot_counts = Counter()
source_token_count_sums = []
rights_status_counts = Counter()
duplicate_context_count = 0

for context in nb18_development_contexts:
    candidate_id = context.get("candidate_id")
    gate_result = gate_by_candidate_id.get(candidate_id)
    assert gate_result is not None

    evidence_items = sorted(
        context["context_items"],
        key=lambda item: item["reranker_rank"],
    )
    chunk_ids = [item["chunk_id"] for item in evidence_items]

    if len(chunk_ids) != len(set(chunk_ids)):
        duplicate_context_count += 1

    assert len(evidence_items) == 10
    assert [item["reranker_rank"] for item in evidence_items] == list(
        range(1, 11)
    )

    source_token_count_sum = 0
    for item in evidence_items:
        source_type = item["source"].get("source_type")
        if source_type is not None:
            source_slot_counts[str(source_type)] += 1

        source_token_count = item.get("source_token_count")
        if isinstance(source_token_count, (int, float)):
            source_token_count_sum += int(source_token_count)

        rights = item.get("rights", {})
        if isinstance(rights, dict):
            for member_right in rights.get("member_rights", []):
                if not isinstance(member_right, dict):
                    continue
                member_rights = member_right.get("rights", {})
                if isinstance(member_rights, dict):
                    status = member_rights.get("status")
                    if status is not None:
                        rights_status_counts[str(status)] += 1

    source_token_count_sums.append(source_token_count_sum)

    nb18_development_packets.append({
        "candidate_id": candidate_id,
        "split": "development",
        "evidence_items": evidence_items,
        "source_priority_order": context["source_priority_order"],
        "source_token_count_sum_not_target_model_count": (
            source_token_count_sum
        ),
        "context_budget_applied": False,
        "text_truncation_applied": False,
        "rights_clearance_asserted": False,
        "release_eligible": gate_result["release_eligible"],
        "generation_ready": gate_result["generation_ready"],
    })

assert len(nb18_development_packets) == 2664
assert duplicate_context_count == 0

ordered_sums = sorted(source_token_count_sums)

def quantiles(values):
    if not values:
        return {}

    def at(percent):
        return values[round((len(values) - 1) * percent)]

    return {
        "min": values[0],
        "p50": at(0.50),
        "p90": at(0.90),
        "p95": at(0.95),
        "max": values[-1],
    }

audit = {
    "status": "PASS",
    "development_packet_count": len(nb18_development_packets),
    "evidence_items_total": sum(
        len(packet["evidence_items"])
        for packet in nb18_development_packets
    ),
    "items_per_packet": 10,
    "contexts_with_duplicate_chunk_ids": duplicate_context_count,
    "source_type_slot_counts": dict(sorted(source_slot_counts.items())),
    "rights_status_occurrence_counts": dict(
        sorted(rights_status_counts.items())
    ),
    "source_token_count_sum_per_packet_not_target_model_count": quantiles(
        ordered_sums
    ),
    "contexts_release_eligible": sum(
        packet["release_eligible"]
        for packet in nb18_development_packets
    ),
    "contexts_generation_ready": sum(
        packet["generation_ready"]
        for packet in nb18_development_packets
    ),
    "production_tokenizer_used": False,
    "production_budget_applied": False,
    "test_evidence_or_metrics_used": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_packet_count": 2664,
  "evidence_items_total": 26640,
  "items_per_packet": 10,
  "contexts_with_duplicate_chunk_ids": 0,
  "source_type_slot_counts": {
    "book": 2524,
    "transcript": 13609,
    "website": 10507
  },
  "rights_status_occurrence_counts": {
    "EVIDENCE_REQUIRED": 149993
  },
  "source_token_count_sum_per_packet_not_target_model_count": {
    "min": 604,
    "p50": 3015,
    "p90": 3301,
    "p95": 3364,
    "max": 3696
  },
  "contexts_release_eligible": 0,
  "contexts_generation_ready": 0,
  "production_tokenizer_used": false,
  "production_budget_applied": false,
  "test_evidence_or_metrics_used": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl">

### سلول C26 — اثرانگشت بسته‌های Development

این سلول بسته‌های ساخته‌شده در حافظه را هش می‌کند تا ترتیب و یکپارچگی‌شان برای تحویل به مرحلهٔ بعد قابل‌بررسی باشد. فقط‌خواندنی است و فایلی نمی‌نویسد.

</div>

In [26]:
# NB18-C26 — Hash the ordered Development packet manifest

import hashlib
import json

assert "nb18_development_packets" in globals()

packets = nb18_development_packets
assert isinstance(packets, (list, tuple))
assert len(packets) == 2664
assert all(isinstance(packet, dict) for packet in packets)

def canonical_json_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

packet_hashes = [
    hashlib.sha256(canonical_json_bytes(packet)).hexdigest()
    for packet in packets
]

manifest_payload = {
    "packet_count": len(packets),
    "ordered_packet_sha256": packet_hashes,
}
manifest_sha256 = hashlib.sha256(
    canonical_json_bytes(manifest_payload)
).hexdigest()

print(json.dumps({
    "status": "PASS",
    "packet_variable": "nb18_development_packets",
    "development_packet_count": len(packets),
    "packet_hash_count": len(packet_hashes),
    "ordered_manifest_sha256": manifest_sha256,
    "first_packet_keys": sorted(packets[0].keys()),
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "packet_variable": "nb18_development_packets",
  "development_packet_count": 2664,
  "packet_hash_count": 2664,
  "ordered_manifest_sha256": "495326d494fef312bedbb9bbfdc1b1ac901f40f2954f4d791aefc4656edb89f9",
  "first_packet_keys": [
    "candidate_id",
    "context_budget_applied",
    "evidence_items",
    "generation_ready",
    "release_eligible",
    "rights_clearance_asserted",
    "source_priority_order",
    "source_token_count_sum_not_target_model_count",
    "split",
    "text_truncation_applied"
  ],
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl">

### NB18-MD-C27 — ممیزی قرارداد بسته‌های Development

ساختار بسته‌ها و وضعیت دروازه‌های تولید و انتشار را بررسی می‌کند. همچنین هش ترتیب‌دار را با خروجی C26 تطبیق می‌دهد تا تغییر نکردن بسته‌ها تأیید شود. این مرحله فقط‌خواندنی است و فایلی نمی‌نویسد.

</div>

In [27]:
# NB18-C27 — Development Packet Contract Audit

import hashlib
import json

packets = nb18_development_packets

required_packet_keys = {
    "candidate_id",
    "context_budget_applied",
    "evidence_items",
    "generation_ready",
    "release_eligible",
    "rights_clearance_asserted",
    "source_priority_order",
    "source_token_count_sum_not_target_model_count",
    "split",
    "text_truncation_applied",
}

assert len(packets) == 2664
assert all(required_packet_keys.issubset(packet.keys()) for packet in packets)
assert len({packet["candidate_id"] for packet in packets}) == len(packets)
assert all(packet["split"] == "development" for packet in packets)
assert all(len(packet["evidence_items"]) == 10 for packet in packets)
assert all(packet["generation_ready"] is False for packet in packets)
assert all(packet["release_eligible"] is False for packet in packets)
assert all(packet["rights_clearance_asserted"] is False for packet in packets)
assert all(packet["context_budget_applied"] is False for packet in packets)
assert all(packet["text_truncation_applied"] is False for packet in packets)

def canonical_json_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

packet_hashes = [
    hashlib.sha256(canonical_json_bytes(packet)).hexdigest()
    for packet in packets
]

manifest_payload = {
    "packet_count": len(packets),
    "ordered_packet_sha256": packet_hashes,
}
manifest_sha256 = hashlib.sha256(
    canonical_json_bytes(manifest_payload)
).hexdigest()

assert manifest_sha256 == (
    "495326d494fef312bedbb9bbfdc1b1ac"
    "901f40f2954f4d791aefc4656edb89f9"
)

sample_items = packets[0]["evidence_items"]
sample_item_keys = (
    sorted(sample_items[0].keys())
    if sample_items and isinstance(sample_items[0], dict)
    else []
)

print(json.dumps({
    "status": "PASS",
    "development_packet_count": len(packets),
    "evidence_items_per_packet": 10,
    "unique_candidate_id_count": len({packet["candidate_id"] for packet in packets}),
    "generation_ready_packet_count": sum(
        packet["generation_ready"] is True for packet in packets
    ),
    "release_eligible_packet_count": sum(
        packet["release_eligible"] is True for packet in packets
    ),
    "rights_clearance_asserted_packet_count": sum(
        packet["rights_clearance_asserted"] is True for packet in packets
    ),
    "ordered_manifest_sha256_matches_c26": True,
    "sample_evidence_item_keys": sample_item_keys,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_packet_count": 2664,
  "evidence_items_per_packet": 10,
  "unique_candidate_id_count": 2664,
  "generation_ready_packet_count": 0,
  "release_eligible_packet_count": 0,
  "rights_clearance_asserted_packet_count": 0,
  "ordered_manifest_sha256_matches_c26": true,
  "sample_evidence_item_keys": [
    "attribution_context",
    "authority_signals",
    "char_count",
    "chunk_id",
    "evidence_id",
    "lineage",
    "locator",
    "member_document_ids",
    "member_documents",
    "parent_chunk_id",
    "reranker_rank",
    "reranker_score",
    "rights",
    "selection_priority_rank",
    "source",
    "source_token_count",
    "text"
  ],
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl">

### NB18-MD-C28 — ممیزی شناسه و رتبهٔ مدارک

برای هر بسته بررسی می‌کند که شناسهٔ قطعه‌ها تکراری نباشد، رتبهٔ اصلی هر مدرک معتبر و یکتا باشد و رتبهٔ اولویت انتخاب‌ها از ۱ تا ۱۰ را کامل پوشش دهد. این ممیزی فقط‌خواندنی است.

</div>

In [28]:
# NB18-C28 — Evidence Identity and Rank Audit

import json

packets = nb18_development_packets

assert len(packets) == 2664

duplicate_chunk_packet_count = 0
invalid_reranker_rank_packet_count = 0
invalid_priority_rank_packet_count = 0
empty_text_item_count = 0
reranker_rank_min = None
reranker_rank_max = None

for packet in packets:
    items = packet["evidence_items"]

    chunk_ids = [item["chunk_id"] for item in items]
    if len(chunk_ids) != len(set(chunk_ids)):
        duplicate_chunk_packet_count += 1

    reranker_ranks = [item["reranker_rank"] for item in items]
    if (
        any(not isinstance(rank, int) or not 1 <= rank <= 50 for rank in reranker_ranks)
        or len(reranker_ranks) != len(set(reranker_ranks))
    ):
        invalid_reranker_rank_packet_count += 1

    priority_ranks = [item["selection_priority_rank"] for item in items]
    if (
        any(not isinstance(rank, int) for rank in priority_ranks)
        or sorted(priority_ranks) != list(range(1, 11))
    ):
        invalid_priority_rank_packet_count += 1

    empty_text_item_count += sum(
        not isinstance(item.get("text"), str) or not item["text"].strip()
        for item in items
    )

    if reranker_ranks:
        current_min = min(reranker_ranks)
        current_max = max(reranker_ranks)
        reranker_rank_min = (
            current_min if reranker_rank_min is None
            else min(reranker_rank_min, current_min)
        )
        reranker_rank_max = (
            current_max if reranker_rank_max is None
            else max(reranker_rank_max, current_max)
        )

assert duplicate_chunk_packet_count == 0
assert invalid_reranker_rank_packet_count == 0
assert invalid_priority_rank_packet_count == 0
assert empty_text_item_count == 0

print(json.dumps({
    "status": "PASS",
    "development_packet_count": len(packets),
    "duplicate_chunk_packet_count": duplicate_chunk_packet_count,
    "invalid_reranker_rank_packet_count": invalid_reranker_rank_packet_count,
    "invalid_priority_rank_packet_count": invalid_priority_rank_packet_count,
    "empty_text_item_count": empty_text_item_count,
    "reranker_rank_min": reranker_rank_min,
    "reranker_rank_max": reranker_rank_max,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_packet_count": 2664,
  "duplicate_chunk_packet_count": 0,
  "invalid_reranker_rank_packet_count": 0,
  "invalid_priority_rank_packet_count": 0,
  "empty_text_item_count": 0,
  "reranker_rank_min": 1,
  "reranker_rank_max": 10,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl">

### NB18-MD-C29 — بررسی ساختار ترتیب اولویت

ساختار `source_priority_order` را در همهٔ بسته‌ها بررسی می‌کند و نمونه‌ای از ارتباط شناسهٔ قطعه، رتبهٔ اصلی و رتبهٔ اولویت نشان می‌دهد. این مرحله فقط برای کشف ساختار واقعی داده است و چیزی را تغییر نمی‌دهد.

</div>

In [29]:
# NB18-C29 — Source Priority Structure Inspection

import json
from collections import Counter

packets = nb18_development_packets

priority_order_shapes = Counter()

for packet in packets:
    value = packet["source_priority_order"]

    if isinstance(value, list):
        shape = ("list", len(value), type(value[0]).__name__ if value else "empty")
    elif isinstance(value, tuple):
        shape = ("tuple", len(value), type(value[0]).__name__ if value else "empty")
    elif isinstance(value, dict):
        first_value = next(iter(value.values()), None)
        shape = ("dict", len(value), type(first_value).__name__ if value else "empty")
    else:
        shape = (type(value).__name__, None, None)

    priority_order_shapes[shape] += 1

sample_packet = packets[0]
sample_order = sample_packet["source_priority_order"]

if isinstance(sample_order, (list, tuple)):
    order_preview = list(sample_order[:10])
elif isinstance(sample_order, dict):
    order_preview = list(sample_order.items())[:10]
else:
    order_preview = repr(sample_order)[:500]

evidence_order_preview = [
    {
        "chunk_id": item.get("chunk_id"),
        "reranker_rank": item.get("reranker_rank"),
        "selection_priority_rank": item.get("selection_priority_rank"),
    }
    for item in sample_packet["evidence_items"]
]

print(json.dumps({
    "status": "INSPECTED",
    "development_packet_count": len(packets),
    "source_priority_order_shapes": [
        {
            "container_type": shape[0],
            "length": shape[1],
            "element_type": shape[2],
            "packet_count": count,
        }
        for shape, count in sorted(priority_order_shapes.items(), key=lambda row: str(row[0]))
    ],
    "sample_source_priority_order": order_preview,
    "sample_evidence_rank_mapping": evidence_order_preview,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "INSPECTED",
  "development_packet_count": 2664,
  "source_priority_order_shapes": [
    {
      "container_type": "list",
      "length": 10,
      "element_type": "str",
      "packet_count": 2664
    }
  ],
  "sample_source_priority_order": [
    "chunk:baseline_phase1_char_safe:retrieval:8d949a95dc2c176ce8a33259",
    "chunk:baseline_phase1_char_safe:retrieval:39b52ea3ac39010ea4750703",
    "chunk:baseline_phase1_char_safe:retrieval:d909683531ef3da8da2efdfe",
    "chunk:baseline_phase1_char_safe:retrieval:8bf1ca6737c670c46526c909",
    "chunk:baseline_phase1_char_safe:retrieval:56b7e333bae3330798d0f918",
    "chunk:baseline_phase1_char_safe:retrieval:51485b67547b986b1a23acd8",
    "chunk:baseline_phase1_char_safe:retrieval:7ccddf925dadfe75552fdcda",
    "chunk:baseline_phase1_char_safe:retrieval:d53671200b2cb14c92470f15",
    "chunk:baseline_phase1_char_safe:retrieval:b4883b5edf68402cf75ea77a",
    "chunk:baseline_phase1_char_safe:retrieval:d54b42e8a06236b9c4158206"
 

<div dir="rtl">

### NB18-MD-C30 — اعتبارسنجی نگاشت ترتیب اولویت

برای هر بسته بررسی می‌کند که `source_priority_order` دقیقاً با مدارک دارای رتبه‌های اولویت ۱ تا ۱۰ تطبیق داشته باشد و هیچ قطعه‌ای اضافه یا حذف نشده باشد. این مرحله فقط‌خواندنی است.

</div>

In [30]:
# NB18-C30 — Source Priority Mapping Validation

import json

packets = nb18_development_packets

priority_order_mismatch_count = 0
chunk_set_mismatch_count = 0
duplicate_priority_order_id_count = 0

for packet in packets:
    items = packet["evidence_items"]
    source_priority_order = packet["source_priority_order"]

    chunk_ids = [item["chunk_id"] for item in items]
    priority_to_chunk = {
        item["selection_priority_rank"]: item["chunk_id"]
        for item in items
    }

    expected_order = [
        priority_to_chunk[rank]
        for rank in range(1, 11)
        if rank in priority_to_chunk
    ]

    if len(source_priority_order) != len(set(source_priority_order)):
        duplicate_priority_order_id_count += 1

    if set(source_priority_order) != set(chunk_ids):
        chunk_set_mismatch_count += 1

    if source_priority_order != expected_order:
        priority_order_mismatch_count += 1

assert duplicate_priority_order_id_count == 0
assert chunk_set_mismatch_count == 0
assert priority_order_mismatch_count == 0

print(json.dumps({
    "status": "PASS",
    "development_packet_count": len(packets),
    "priority_order_mismatch_count": priority_order_mismatch_count,
    "chunk_set_mismatch_count": chunk_set_mismatch_count,
    "duplicate_priority_order_id_count": duplicate_priority_order_id_count,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_packet_count": 2664,
  "priority_order_mismatch_count": 0,
  "chunk_set_mismatch_count": 0,
  "duplicate_priority_order_id_count": 0,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl">

### NB18-MD-C31 — کشف تنظیمات مدل و توکنایزر

به‌صورت فقط‌خواندنی، فایل‌های متنی و کد مخزن را برای تنظیمات مدل، توکنایزر و بودجهٔ ورودی بررسی می‌کند. محتوای دیتاست‌ها و خروجی‌های ذخורים‌شدهٔ نوت‌بوک‌ها بررسی نمی‌شوند. در این مرحله هیچ بودجه‌ای انتخاب یا اعمال نمی‌شود.

</div>

In [31]:
# NB18-C31 — Model, Tokenizer, and Budget Configuration Discovery

import json
from pathlib import Path

project_root = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance"
)

assert project_root.is_dir(), f"Project root not found: {project_root}"

excluded_dirs = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    "node_modules",
    "data",
}

search_terms = (
    "tokenizer",
    "context_budget",
    "token_budget",
    "max_input_tokens",
    "context_window",
    "model_context",
    "generation_model",
)

text_suffixes = {
    ".py",
    ".md",
    ".json",
    ".yaml",
    ".yml",
    ".toml",
    ".ini",
    ".txt",
    ".ipynb",
}

matches = []
scanned_file_count = 0

for path in sorted(project_root.rglob("*")):
    if not path.is_file():
        continue

    relative_path = path.relative_to(project_root)

    if any(part in excluded_dirs for part in relative_path.parts):
        continue

    if path.suffix.lower() not in text_suffixes:
        continue

    try:
        if path.stat().st_size > 3_000_000:
            continue
    except OSError:
        continue

    scanned_file_count += 1

    try:
        if path.suffix.lower() == ".ipynb":
            notebook_data = json.loads(path.read_text(encoding="utf-8"))
            source_lines = []
            for cell in notebook_data.get("cells", []):
                source = cell.get("source", [])
                if isinstance(source, list):
                    source_lines.extend(source)
                elif isinstance(source, str):
                    source_lines.extend(source.splitlines())
        else:
            source_lines = path.read_text(
                encoding="utf-8",
                errors="replace",
            ).splitlines()
    except Exception:
        continue

    file_matches = []
    for line_number, line in enumerate(source_lines, start=1):
        lowered = line.lower()
        if any(term in lowered for term in search_terms):
            file_matches.append({
                "line": line_number,
                "text": line.strip()[:240],
            })
            if len(file_matches) == 5:
                break

    if file_matches:
        matches.append({
            "path": str(relative_path),
            "matches": file_matches,
        })

print(json.dumps({
    "status": "DISCOVERY",
    "project_root": str(project_root),
    "files_scanned": scanned_file_count,
    "matching_file_count": len(matches),
    "matching_files": matches[:100],
    "results_truncated": len(matches) > 100,
    "production_budget_selected": False,
    "production_tokenizer_used": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "DISCOVERY",
  "project_root": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance",
  "files_scanned": 21331,
  "matching_file_count": 29,
  "matching_files": [
    {
      "path": "artifacts/phase2/chunking/chunk_contract_evaluation_freeze_v1.json",
      "matches": [
        {
          "line": 103,
          "text": "\"measurement_tokenizer\": {"
        },
        {
          "line": 108,
          "text": "\"tokenizer_class\": \"XLMRobertaTokenizerFast\""
        }
      ]
    },
    {
      "path": "artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/embedding_benchmark_tradeoff.json",
      "matches": [
        {
          "line": 30,
          "text": "\"context_budget_and_assembly_policy\","
        }
      ]
    },
    {
      "path": "artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/embedding_selection_adr_v1.json",
      "matches": [
 

<div dir="rtl">

### NB18-MD-C32 — بررسی متمرکز تنظیمات تولید

متن کدها و مستندات کاندیدِ پیدا‌شده در C31 را بررسی می‌کند تا مشخص شود نام مدل، توکنایزر و محدودیت توکن واقعاً کجا و با چه معنایی تعریف شده‌اند. خروجی نوت‌بوک‌ها خوانده نمی‌شود و هیچ تنظیمی انتخاب یا تغییر داده نمی‌شود.

</div>

In [32]:
# NB18-C32 — Focused Generation Configuration Inspection

import json
import re
from pathlib import Path

project_root = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance"
)

candidate_relative_paths = [
    "src/pipeline/rag_pipeline.py",
    "notebooks/phase1/07_generation.ipynb",
    "notebooks/phase1/08_rag_pipeline.ipynb",
    "notebooks/phase2/11_chunking_benchmark.ipynb",
    "notebooks/phase2/13_embedding_benchmark.ipynb",
    "notebooks/phase2/13_embedding_benchmark_closure_candidate.ipynb",
    "notebooks/phase2/17_reranking.ipynb",
    "artifacts/phase2/knowledge_registry/claim_extraction_prompt_window_contract_v1.json",
    "artifacts/phase2/chunking/chunk_contract_evaluation_freeze_v1.json",
    "artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/embedding_benchmark_tradeoff.json",
    "artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/embedding_selection_adr_v1.json",
    "artifacts/phase2/embedding_benchmark/df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729/embedding_benchmark_tradeoff.json",
    "artifacts/phase2/embedding_benchmark/df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729/retrieval_remediation_decision.json",
]

pattern = re.compile(
    r"Qwen3|generation_model|tokenizer|context.?budget|token.?budget|"
    r"max.?input.?tokens|max.?new.?tokens|max.?tokens|max.?length|"
    r"context.?window|model_context_limit|prompt.?window|truncat",
    re.IGNORECASE,
)

inspection_results = []

for relative_path in candidate_relative_paths:
    path = project_root / relative_path

    if not path.is_file():
        inspection_results.append({
            "path": relative_path,
            "exists": False,
            "matches": [],
        })
        continue

    matches = []

    if path.suffix.lower() == ".ipynb":
        try:
            notebook_data = json.loads(path.read_text(encoding="utf-8"))
        except Exception as error:
            inspection_results.append({
                "path": relative_path,
                "exists": True,
                "read_error": type(error).__name__,
                "matches": [],
            })
            continue

        for cell_index, cell in enumerate(notebook_data.get("cells", [])):
            source = cell.get("source", [])
            if isinstance(source, list):
                source_lines = source
            elif isinstance(source, str):
                source_lines = source.splitlines()
            else:
                source_lines = []

            for line_number, line in enumerate(source_lines, start=1):
                if pattern.search(line):
                    matches.append({
                        "cell_index": cell_index,
                        "line_in_cell": line_number,
                        "text": line.strip()[:240],
                    })
                    if len(matches) >= 12:
                        break
            if len(matches) >= 12:
                break
    else:
        try:
            source_lines = path.read_text(
                encoding="utf-8",
                errors="replace",
            ).splitlines()
        except Exception as error:
            inspection_results.append({
                "path": relative_path,
                "exists": True,
                "read_error": type(error).__name__,
                "matches": [],
            })
            continue

        for line_number, line in enumerate(source_lines, start=1):
            if pattern.search(line):
                matches.append({
                    "line": line_number,
                    "text": line.strip()[:240],
                })
                if len(matches) >= 12:
                    break

    inspection_results.append({
        "path": relative_path,
        "exists": True,
        "matches": matches,
    })

print(json.dumps({
    "status": "INSPECTED",
    "candidate_file_count": len(candidate_relative_paths),
    "existing_file_count": sum(
        result["exists"] for result in inspection_results
    ),
    "files": inspection_results,
    "production_budget_selected": False,
    "production_tokenizer_used": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "INSPECTED",
  "candidate_file_count": 13,
  "existing_file_count": 13,
  "files": [
    {
      "path": "src/pipeline/rag_pipeline.py",
      "exists": true,
      "matches": [
        {
          "line": 17,
          "text": "generation_model: str = \"Qwen3-30B-A3B\","
        },
        {
          "line": 21,
          "text": "self.generation_model = generation_model"
        },
        {
          "line": 34,
          "text": "model=self.generation_model"
        },
        {
          "line": 116,
          "text": "\"generation_model\": self.generation_model,"
        },
        {
          "line": 128,
          "text": "generation_model=self.generation_model"
        }
      ]
    },
    {
      "path": "notebooks/phase1/07_generation.ipynb",
      "exists": true,
      "matches": [
        {
          "cell_index": 3,
          "line_in_cell": 17,
          "text": "max_tokens=200"
        },
        {
          "cell_index": 9,
          "line_in_cell": 30,


<div dir="rtl">

### NB18-MD-C33 — بررسی شواهد ذخیره‌شدهٔ سیاست بودجه

خروجی ذخیره‌شدهٔ سلول ارزیابی سیاست بودجه در NB13 و فایل‌های JSON مرتبط را بررسی می‌کند. این مرحله فقط وضعیت، ساختار و دامنهٔ شواهد را می‌خواند؛ هیچ سیاستی انتخاب نمی‌کند و از معیارهای Test برای تصمیم‌گیری استفاده نمی‌کند.

</div>

In [33]:
# NB18-C33 — Saved Context Budget Policy Evidence Inspection

import json
import re
from pathlib import Path

project_root = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance"
)

notebook_path = project_root / "notebooks/phase2/13_embedding_benchmark.ipynb"
assert notebook_path.is_file()

notebook_data = json.loads(notebook_path.read_text(encoding="utf-8"))
policy_cells = []

for cell_index, cell in enumerate(notebook_data.get("cells", [])):
    source = cell.get("source", [])
    if isinstance(source, list):
        source_text = "".join(source)
    elif isinstance(source, str):
        source_text = source
    else:
        source_text = ""

    if "PARENT" in source_text.upper() and "CONTEXT-BUDGET POLICY EVALUATION" in source_text.upper():
        saved_outputs = []
        for output in cell.get("outputs", []):
            output_text = output.get("text", [])
            if isinstance(output_text, list):
                output_text = "".join(output_text)

            data = output.get("data", {})
            plain_text = data.get("text/plain", [])
            if isinstance(plain_text, list):
                plain_text = "".join(plain_text)

            if output_text:
                saved_outputs.append(output_text[:12000])
            elif plain_text:
                saved_outputs.append(plain_text[:12000])

        policy_cells.append({
            "cell_index": cell_index,
            "source_excerpt": source_text[:12000],
            "saved_output_excerpts": saved_outputs[:3],
            "saved_error_present": any(
                output.get("output_type") == "error"
                for output in cell.get("outputs", [])
            ),
        })

benchmark_root = project_root / "artifacts/phase2/embedding_benchmark"
policy_artifacts = sorted(
    path for path in benchmark_root.rglob("*.json")
    if "context_budget" in path.name.lower()
    or "budget_policy" in path.name.lower()
)

def collect_scope_fields(value, path="$", results=None):
    if results is None:
        results = []

    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}"
            lowered = key.lower()

            if re.search(r"status|decision|split|model|tokenizer|budget|policy", lowered):
                if isinstance(child, (str, int, float, bool)) or child is None:
                    results.append({
                        "path": child_path,
                        "value": str(child)[:240],
                    })
                elif isinstance(child, list):
                    results.append({
                        "path": child_path,
                        "container_type": "list",
                        "length": len(child),
                    })
                elif isinstance(child, dict):
                    results.append({
                        "path": child_path,
                        "container_type": "dict",
                        "keys": sorted(child.keys())[:30],
                    })

            collect_scope_fields(child, child_path, results)

    elif isinstance(value, list):
        for index, child in enumerate(value):
            collect_scope_fields(child, f"{path}[{index}]", results)

    return results

artifact_reports = []

for path in policy_artifacts:
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        artifact_reports.append({
            "path": str(path.relative_to(project_root)),
            "top_level_type": type(payload).__name__,
            "top_level_keys": sorted(payload.keys()) if isinstance(payload, dict) else [],
            "scope_fields": collect_scope_fields(payload)[:100],
        })
    except Exception as error:
        artifact_reports.append({
            "path": str(path.relative_to(project_root)),
            "read_error": type(error).__name__,
        })

print(json.dumps({
    "status": "INSPECTED",
    "policy_evaluation_cell_count": len(policy_cells),
    "policy_evaluation_cells": policy_cells,
    "related_artifact_count": len(policy_artifacts),
    "related_artifacts": artifact_reports,
    "production_budget_selected": False,
    "test_metrics_used_for_decision": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "INSPECTED",
  "policy_evaluation_cell_count": 1,
  "policy_evaluation_cells": [
    {
      "cell_index": 88,
      "source_excerpt": "# Code Cell 43\n# NB13 — Parent-child context-budget policy evaluation\n\nassert \"parent_child_parents\" in chunking_candidates, (\n    \"The parent_child_parents candidate is unavailable.\"\n)\n\nparent_context_path = (\n    project_root\n    / chunking_candidates[\"parent_child_parents\"][\"path\"]\n)\n\nassert parent_context_path.is_file(), (\n    f\"Missing parent-context file: {parent_context_path}\"\n)\n\nparent_context_records = [\n    json.loads(line)\n    for line in parent_context_path.read_text(encoding=\"utf-8\").splitlines()\n    if line.strip()\n]\n\nparent_record_by_chunk_id = {\n    record[\"chunk_id\"]: record\n    for record in parent_context_records\n}\n\nassert len(parent_record_by_chunk_id) == len(parent_context_records)\n\ncontext_budget_policies = [\n    {\n        \"policy_id\": \"conservative\",\n        \"retrie

<div dir="rtl">

### NB18-MD-C34 — یافتن نقشه و الزامات رسمی NB18

فایل‌های متنی، کدها و متن سلول‌های نوت‌بوک‌ها را برای اشاره به NB18 و Context Builder جست‌وجو می‌کند. خروجی‌های ذخیره‌شدهٔ نوت‌بوک‌ها و پوشه‌های دیتاست و آرشیو بررسی نمی‌شوند. این مرحله فقط‌خواندنی است و هیچ فایل یا تنظیمی را تغییر نمی‌دهد.

</div>

In [34]:
# NB18-C34 — NB18 Roadmap and Requirements Discovery

import json
import re
from pathlib import Path

project_root = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance"
)

assert project_root.is_dir()

excluded_dirs = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    "node_modules",
    "data",
    "archive",
}

supported_suffixes = {
    ".md",
    ".json",
    ".yaml",
    ".yml",
    ".toml",
    ".txt",
    ".ipynb",
    ".py",
}

pattern = re.compile(
    r"\bNB[\s_-]*18\b|18_context_builder|context[\s_-]*builder",
    re.IGNORECASE,
)

scanned_file_count = 0
matched_files = []

for path in sorted(project_root.rglob("*")):
    if not path.is_file():
        continue

    relative_path = path.relative_to(project_root)

    if any(part.lower() in excluded_dirs for part in relative_path.parts):
        continue

    if path.name == "18_context_builder.ipynb":
        continue

    if path.suffix.lower() not in supported_suffixes:
        continue

    try:
        if path.stat().st_size > 3_000_000:
            continue
    except OSError:
        continue

    scanned_file_count += 1
    matches = []

    try:
        if path.suffix.lower() == ".ipynb":
            notebook_data = json.loads(path.read_text(encoding="utf-8"))
            source_lines = []

            for cell_index, cell in enumerate(notebook_data.get("cells", [])):
                source = cell.get("source", [])
                if isinstance(source, list):
                    cell_lines = source
                elif isinstance(source, str):
                    cell_lines = source.splitlines()
                else:
                    cell_lines = []

                source_lines.extend(
                    (cell_index, line_number, line)
                    for line_number, line in enumerate(cell_lines, start=1)
                )
        else:
            source_lines = [
                (None, line_number, line)
                for line_number, line in enumerate(
                    path.read_text(
                        encoding="utf-8",
                        errors="replace",
                    ).splitlines(),
                    start=1,
                )
            ]
    except Exception:
        continue

    for cell_index, line_number, line in source_lines:
        if pattern.search(line):
            match = {
                "line": line_number,
                "text": line.strip()[:300],
            }
            if cell_index is not None:
                match["cell_index"] = cell_index

            matches.append(match)

            if len(matches) == 8:
                break

    if matches:
        matched_files.append({
            "path": str(relative_path),
            "matches": matches,
        })

print(json.dumps({
    "status": "DISCOVERY",
    "files_scanned": scanned_file_count,
    "matching_file_count": len(matched_files),
    "matching_files": matched_files[:100],
    "results_truncated": len(matched_files) > 100,
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}, ensure_ascii=False, indent=2))

{
  "status": "DISCOVERY",
  "files_scanned": 21312,
  "matching_file_count": 5,
  "matching_files": [
    {
      "path": "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json",
      "matches": [
        {
          "line": 32,
          "text": "\"target_notebook\": \"18_context_builder.ipynb\""
        }
      ]
    },
    {
      "path": "notebooks/phase2/11_chunking_benchmark.ipynb",
      "matches": [
        {
          "line": 710,
          "text": "\"Measure context-builder latency and citation mapping.\"",
          "cell_index": 43
        },
        {
          "line": 537,
          "text": "<li>Storage و Context-builder Latency اندازه‌گیری می‌شود.</li>",
          "cell_index": 54
        }
      ]
    },
    {
      "path": "notebooks/phase2/17_reranking.ipynb",
      "matches": [
        {
          "line": 53,
          "text": "- Handoff نسخه‌دار برای NB18 ساخته شود.",
          "cell_index": 1
        },
        {
          "line": 13,
          "text": "- قرار

<div dir="rtl">

### NB18-MD-C35 — استخراج قرارداد تحویل از NB17

فایل handoff مربوط به NB18 و سلول‌های NB17 که آن را می‌سازند یا اعتبارسنجی می‌کنند، فقط‌خواندنی بررسی می‌شوند. هدف، استخراج ورودی‌ها و الزامات واقعی تحویل به Context Builder است؛ هیچ کدی از NB17 دوباره اجرا نمی‌شود.

</div>

In [35]:
# NB18-C35 — NB17 to NB18 Handoff Contract Inspection

import json
from pathlib import Path

project_root = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance"
)

handoff_path = (
    project_root
    / "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json"
)
nb17_path = project_root / "notebooks/phase2/17_reranking.ipynb"

assert handoff_path.is_file()
assert nb17_path.is_file()

handoff_payload = json.loads(handoff_path.read_text(encoding="utf-8"))
nb17_data = json.loads(nb17_path.read_text(encoding="utf-8"))

def extract_cell(cell_index):
    cells = nb17_data.get("cells", [])
    assert 0 <= cell_index < len(cells)

    cell = cells[cell_index]
    source = cell.get("source", [])
    if isinstance(source, list):
        source = "".join(source)
    elif not isinstance(source, str):
        source = ""

    saved_outputs = []
    for output in cell.get("outputs", []):
        stream_text = output.get("text", [])
        if isinstance(stream_text, list):
            stream_text = "".join(stream_text)

        data = output.get("data", {})
        plain_text = data.get("text/plain", [])
        if isinstance(plain_text, list):
            plain_text = "".join(plain_text)

        if stream_text:
            saved_outputs.append(stream_text[:6000])
        elif plain_text:
            saved_outputs.append(plain_text[:6000])

    return {
        "cell_index": cell_index,
        "cell_type": cell.get("cell_type"),
        "source": source[:12000],
        "saved_output_excerpts": saved_outputs[:3],
        "saved_error_present": any(
            output.get("output_type") == "error"
            for output in cell.get("outputs", [])
        ),
    }

nb17_cell_indices = [38, 39, 40, 42, 43]

print(json.dumps({
    "status": "INSPECTED",
    "handoff_artifact_path": str(handoff_path.relative_to(project_root)),
    "handoff_artifact": handoff_payload,
    "nb17_handoff_cells": [
        extract_cell(cell_index)
        for cell_index in nb17_cell_indices
    ],
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}, ensure_ascii=False, indent=2))

{
  "status": "INSPECTED",
  "handoff_artifact_path": "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json",
  "handoff_artifact": {
    "artifact_type": "NB18_RERANKER_HANDOFF",
    "candidate_artifact": {
      "candidate_depth": 50,
      "consumer_top_k": 10,
      "path": "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl",
      "record_count": 3813,
      "sha256": "d00285bbf3865fbd51041db1612b9fdc17e9778273ea6d136978906a07268b7a"
    },
    "configuration_sha256": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738",
    "constraints": {
      "do_not_change_reranker_configuration": true,
      "do_not_recompute_reranker_scores": true,
      "preserve_candidate_order": true
    },
    "manifest": {
      "path": "data/phase2/retrieval/reranker_v1/reranker_manifest.json",
      "sha256": "230a720eab3fac9ef9f4fea8ab03152660f045fbda8926e22abcb37f809e6be5"
    },
    "release": {
      "path": "artifacts/phase2/reranking/reranker_release_v1.json",
  

<div dir="rtl">

### NB18-MD-C36 — تطبیق بسته‌ها با ورودی frozen در handoff

برای هر بستهٔ Development، شناسه، ترتیب، رتبه و امتیاز ۱۰ مدرک را با ۱۰ مورد اول همان پرس‌وجو در فایل frozen مقایسه می‌کند. رکوردهای غیر Development از مقایسه کنار گذاشته می‌شوند. این سلول فقط‌خواندنی است و هیچ‌یک از امتیازهای Test را بررسی یا استفاده نمی‌کند.

</div>

In [36]:
# NB18-C36 — Frozen Reranker Handoff Fidelity Audit

import hashlib
import json
from pathlib import Path

project_root = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance"
)

handoff_path = (
    project_root
    / "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json"
)
handoff = json.loads(handoff_path.read_text(encoding="utf-8"))

candidate_artifact = handoff["candidate_artifact"]
candidate_path = project_root / candidate_artifact["path"]

assert candidate_path.is_file()

file_hasher = hashlib.sha256()
with candidate_path.open("rb") as file_handle:
    for block in iter(lambda: file_handle.read(1024 * 1024), b""):
        file_hasher.update(block)

actual_candidate_sha256 = file_hasher.hexdigest()
assert actual_candidate_sha256 == candidate_artifact["sha256"]

development_records = {}
non_development_record_count = 0

with candidate_path.open("r", encoding="utf-8") as file_handle:
    for line in file_handle:
        if not line.strip():
            continue

        record = json.loads(line)

        if record.get("split") != "development":
            non_development_record_count += 1
            continue

        candidate_id = record["candidate_id"]
        assert candidate_id not in development_records
        development_records[candidate_id] = record

assert len(development_records) == 2664
assert len(nb18_development_packets) == 2664

candidate_order_mismatch_count = 0
reranker_rank_mismatch_count = 0
reranker_score_mismatch_count = 0
configuration_mismatch_count = 0
release_id_mismatch_count = 0

for packet in nb18_development_packets:
    record = development_records[packet["candidate_id"]]
    expected_candidates = record["candidates"][:10]
    actual_items = packet["evidence_items"]

    assert len(expected_candidates) == 10
    assert len(actual_items) == 10

    expected_chunk_ids = [
        candidate["chunk_id"]
        for candidate in expected_candidates
    ]
    actual_chunk_ids = [
        item["chunk_id"]
        for item in actual_items
    ]

    if actual_chunk_ids != expected_chunk_ids:
        candidate_order_mismatch_count += 1

    for expected, actual in zip(expected_candidates, actual_items):
        if actual["reranker_rank"] != expected["rank"]:
            reranker_rank_mismatch_count += 1

        if actual["reranker_score"] != expected["reranker_score"]:
            reranker_score_mismatch_count += 1

    if record["configuration_sha256"] != handoff["configuration_sha256"]:
        configuration_mismatch_count += 1

    if record["release_id"] != handoff["release_id"]:
        release_id_mismatch_count += 1

assert candidate_order_mismatch_count == 0
assert reranker_rank_mismatch_count == 0
assert reranker_score_mismatch_count == 0
assert configuration_mismatch_count == 0
assert release_id_mismatch_count == 0

print(json.dumps({
    "status": "PASS",
    "candidate_artifact_sha256_matches_handoff": True,
    "development_records_compared": len(development_records),
    "non_development_records_skipped": non_development_record_count,
    "top10_candidate_order_mismatch_count": candidate_order_mismatch_count,
    "reranker_rank_mismatch_count": reranker_rank_mismatch_count,
    "reranker_score_mismatch_count": reranker_score_mismatch_count,
    "configuration_mismatch_count": configuration_mismatch_count,
    "release_id_mismatch_count": release_id_mismatch_count,
    "test_scores_compared": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "candidate_artifact_sha256_matches_handoff": true,
  "development_records_compared": 2664,
  "non_development_records_skipped": 1149,
  "top10_candidate_order_mismatch_count": 0,
  "reranker_rank_mismatch_count": 0,
  "reranker_score_mismatch_count": 0,
  "configuration_mismatch_count": 0,
  "release_id_mismatch_count": 0,
  "test_scores_compared": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl">

### NB18-MD-C37 — ممیزی ارجاع‌پذیری مدارک

کامل‌بودن شناسهٔ مدرک، locator، منبع، متن و اطلاعات اسناد مبدأ را در تمام بسته‌های Development بررسی می‌کند. همچنین تعداد ارجاع‌های سند را با تعداد رکوردهای metadata همراه آن تطبیق می‌دهد. این مرحله فقط‌خواندنی است.

</div>

In [37]:
# NB18-C37 — Evidence Citation and Source Reference Audit

import json
from collections import Counter

packets = nb18_development_packets

required_item_fields = {
    "evidence_id",
    "chunk_id",
    "locator",
    "source",
    "member_document_ids",
    "member_documents",
    "text",
}

missing_field_counts = Counter()
empty_evidence_id_count = 0
empty_locator_count = 0
empty_source_count = 0
empty_text_count = 0
member_reference_count = 0
member_metadata_record_count = 0
member_metadata_count_mismatch_count = 0
evidence_id_type_counts = Counter()
locator_type_counts = Counter()
source_type_counts = Counter()
item_count = 0

for packet in packets:
    for item in packet["evidence_items"]:
        item_count += 1

        for field in required_item_fields:
            if field not in item:
                missing_field_counts[field] += 1

        evidence_id = item.get("evidence_id")
        if evidence_id is None or evidence_id == "":
            empty_evidence_id_count += 1
        else:
            evidence_id_type_counts[type(evidence_id).__name__] += 1

        locator = item.get("locator")
        if not locator:
            empty_locator_count += 1
        else:
            locator_type_counts[type(locator).__name__] += 1

        source = item.get("source")
        if not source:
            empty_source_count += 1
        else:
            source_type_counts[type(source).__name__] += 1

        text_value = item.get("text")
        if not isinstance(text_value, str) or not text_value.strip():
            empty_text_count += 1

        member_ids = item.get("member_document_ids")
        member_documents = item.get("member_documents")

        member_id_count = len(member_ids) if isinstance(member_ids, (list, tuple)) else 0
        member_document_count = (
            len(member_documents)
            if isinstance(member_documents, (list, tuple))
            else 0
        )

        member_reference_count += member_id_count
        member_metadata_record_count += member_document_count

        if member_id_count != member_document_count:
            member_metadata_count_mismatch_count += 1

assert item_count == 26640
assert not missing_field_counts
assert empty_evidence_id_count == 0
assert empty_locator_count == 0
assert empty_source_count == 0
assert empty_text_count == 0
assert member_metadata_count_mismatch_count == 0

print(json.dumps({
    "status": "PASS",
    "development_packet_count": len(packets),
    "evidence_item_count": item_count,
    "missing_required_field_counts": dict(missing_field_counts),
    "empty_evidence_id_count": empty_evidence_id_count,
    "empty_locator_count": empty_locator_count,
    "empty_source_count": empty_source_count,
    "empty_text_count": empty_text_count,
    "member_document_reference_count": member_reference_count,
    "member_document_metadata_record_count": member_metadata_record_count,
    "member_metadata_count_mismatch_count": member_metadata_count_mismatch_count,
    "evidence_id_type_counts": dict(evidence_id_type_counts),
    "locator_type_counts": dict(locator_type_counts),
    "source_type_counts": dict(source_type_counts),
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_packet_count": 2664,
  "evidence_item_count": 26640,
  "missing_required_field_counts": {},
  "empty_evidence_id_count": 0,
  "empty_locator_count": 0,
  "empty_source_count": 0,
  "empty_text_count": 0,
  "member_document_reference_count": 160500,
  "member_document_metadata_record_count": 160500,
  "member_metadata_count_mismatch_count": 0,
  "evidence_id_type_counts": {
    "str": 26640
  },
  "locator_type_counts": {
    "dict": 26640
  },
  "source_type_counts": {
    "dict": 26640
  },
  "files_written": 0,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}


<div dir="rtl">

### NB18-MD-C38 — بررسی ثبات شناسه‌ها و مشخصات ارجاع

مدارکی را که در چند بسته تکرار شده‌اند با شناسهٔ مدرک، شناسهٔ chunk و مشخصات منبع و locator مقایسه می‌کند. هدف، تشخیص تداخل شناسه یا تغییر مشخصات ارجاع است؛ چیزی ذخیره یا تغییر داده نمی‌شود.

</div>

In [38]:
# NB18-C38 — Repeated Evidence Reference Consistency Audit

import hashlib
import json
from collections import Counter, defaultdict

packets = nb18_development_packets

def canonical_json_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

evidence_id_to_chunk_ids = defaultdict(set)
chunk_id_to_evidence_ids = defaultdict(set)
evidence_id_to_signatures = defaultdict(set)
evidence_id_occurrence_counts = Counter()

for packet in packets:
    for item in packet["evidence_items"]:
        evidence_id = item["evidence_id"]
        chunk_id = item["chunk_id"]

        signature_payload = {
            "chunk_id": chunk_id,
            "source": item["source"],
            "locator": item["locator"],
            "member_document_ids": item["member_document_ids"],
        }

        signature = hashlib.sha256(
            canonical_json_bytes(signature_payload)
        ).hexdigest()

        evidence_id_to_chunk_ids[evidence_id].add(chunk_id)
        chunk_id_to_evidence_ids[chunk_id].add(evidence_id)
        evidence_id_to_signatures[evidence_id].add(signature)
        evidence_id_occurrence_counts[evidence_id] += 1

evidence_ids_linking_multiple_chunks = sum(
    len(chunk_ids) > 1
    for chunk_ids in evidence_id_to_chunk_ids.values()
)

chunks_linking_multiple_evidence_ids = sum(
    len(evidence_ids) > 1
    for evidence_ids in chunk_id_to_evidence_ids.values()
)

evidence_ids_with_inconsistent_reference_metadata = sum(
    len(signatures) > 1
    for signatures in evidence_id_to_signatures.values()
)

repeated_evidence_id_count = sum(
    count > 1
    for count in evidence_id_occurrence_counts.values()
)

print(json.dumps({
    "status": "INSPECTED",
    "evidence_item_occurrence_count": sum(evidence_id_occurrence_counts.values()),
    "unique_evidence_id_count": len(evidence_id_to_chunk_ids),
    "unique_chunk_id_count": len(chunk_id_to_evidence_ids),
    "repeated_evidence_id_count": repeated_evidence_id_count,
    "evidence_ids_linking_multiple_chunks": evidence_ids_linking_multiple_chunks,
    "chunks_linking_multiple_evidence_ids": chunks_linking_multiple_evidence_ids,
    "evidence_ids_with_inconsistent_reference_metadata": (
        evidence_ids_with_inconsistent_reference_metadata
    ),
    "max_occurrences_per_evidence_id": (
        max(evidence_id_occurrence_counts.values(), default=0)
    ),
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}, ensure_ascii=False, indent=2))

{
  "status": "INSPECTED",
  "evidence_item_occurrence_count": 26640,
  "unique_evidence_id_count": 3277,
  "unique_chunk_id_count": 3277,
  "repeated_evidence_id_count": 3036,
  "evidence_ids_linking_multiple_chunks": 0,
  "chunks_linking_multiple_evidence_ids": 0,
  "evidence_ids_with_inconsistent_reference_metadata": 0,
  "max_occurrences_per_evidence_id": 120,
  "files_written": 0,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}


<div dir="rtl">

### NB18-MD-C39 — ساخت فهرست ارجاع یکتا و نگاشت بسته‌ها

از بسته‌های Development یک فهرست یکتا از شناسه‌ها و مشخصات ارجاع می‌سازد و برای هر پرس‌وجو، ترتیب اصلی مدارک و ترتیب اولویت را جداگانه نگه می‌دارد. نتیجه فقط در حافظه ساخته می‌شود؛ فایلی نوشته نمی‌شود.

</div>

In [39]:
# NB18-C39 — In-Memory Citation Registry and Packet Mapping

import hashlib
import json

packets = nb18_development_packets

def canonical_json_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

nb18_evidence_registry_by_id = {}
nb18_packet_evidence_refs = []

reference_fields = (
    "evidence_id",
    "chunk_id",
    "source",
    "locator",
    "member_document_ids",
    "member_documents",
    "rights",
    "attribution_context",
    "lineage",
    "parent_chunk_id",
)

for packet in packets:
    packet_refs = []

    for item in packet["evidence_items"]:
        evidence_entry = {
            field: item.get(field)
            for field in reference_fields
        }

        evidence_id = evidence_entry["evidence_id"]
        assert evidence_id

        if evidence_id in nb18_evidence_registry_by_id:
            assert canonical_json_bytes(
                nb18_evidence_registry_by_id[evidence_id]
            ) == canonical_json_bytes(evidence_entry)
        else:
            nb18_evidence_registry_by_id[evidence_id] = evidence_entry

        packet_refs.append({
            "evidence_id": evidence_id,
            "reranker_rank": item["reranker_rank"],
            "selection_priority_rank": item["selection_priority_rank"],
            "reranker_score": item["reranker_score"],
        })

    assert len(packet_refs) == 10
    assert len({ref["evidence_id"] for ref in packet_refs}) == 10
    assert all(
        ref["evidence_id"] in nb18_evidence_registry_by_id
        for ref in packet_refs
    )

    nb18_packet_evidence_refs.append({
        "candidate_id": packet["candidate_id"],
        "split": packet["split"],
        "evidence_refs_in_reranker_order": packet_refs,
        "evidence_ids_in_source_priority_order": list(
            packet["source_priority_order"]
        ),
    })

nb18_evidence_registry = [
    nb18_evidence_registry_by_id[evidence_id]
    for evidence_id in sorted(nb18_evidence_registry_by_id)
]

registry_sha256 = hashlib.sha256(
    canonical_json_bytes(nb18_evidence_registry)
).hexdigest()

packet_mapping_sha256 = hashlib.sha256(
    canonical_json_bytes(nb18_packet_evidence_refs)
).hexdigest()

assert len(nb18_evidence_registry) == 3277
assert len(nb18_packet_evidence_refs) == 2664
assert sum(
    len(packet_ref["evidence_refs_in_reranker_order"])
    for packet_ref in nb18_packet_evidence_refs
) == 26640

print(json.dumps({
    "status": "PASS",
    "unique_evidence_reference_count": len(nb18_evidence_registry),
    "development_packet_mapping_count": len(nb18_packet_evidence_refs),
    "packet_evidence_reference_count": sum(
        len(packet_ref["evidence_refs_in_reranker_order"])
        for packet_ref in nb18_packet_evidence_refs
    ),
    "reranker_order_preserved": True,
    "source_priority_order_preserved_separately": True,
    "evidence_registry_sha256": registry_sha256,
    "packet_mapping_sha256": packet_mapping_sha256,
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "unique_evidence_reference_count": 3277,
  "development_packet_mapping_count": 2664,
  "packet_evidence_reference_count": 26640,
  "reranker_order_preserved": true,
  "source_priority_order_preserved_separately": true,
  "evidence_registry_sha256": "56a09d36b088f8046962c29a66091f22ffbafdd24624d179df39d78578466e72",
  "packet_mapping_sha256": "dae5481bb4cba09df85d8d8ad92a88a5697e93b3db6e4faeb2f86a14d038c356",
  "files_written": 0,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}


<div dir="rtl">

### NB18-MD-C40 — اعتبارسنجی سریال‌سازی JSONL

هر بسته را در حافظه به یک سطر JSON با UTF-8 تبدیل و همان سطر را بازخوانی می‌کند. برابری کامل داده‌ها، اندازهٔ خروجی فرضی و هش JSONL محاسبه می‌شود؛ هیچ فایلی ذخیره نمی‌شود.

</div>

In [40]:
# NB18-C40 — JSONL Serialization and Round-Trip Audit

import hashlib
import json

packets = nb18_development_packets

jsonl_hasher = hashlib.sha256()
round_trip_mismatch_count = 0
total_serialized_bytes = 0
serialized_record_sizes = []

for packet in packets:
    serialized_line = json.dumps(
        packet,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

    restored_packet = json.loads(serialized_line)

    if restored_packet != packet:
        round_trip_mismatch_count += 1

    line_with_newline = serialized_line + b"\n"
    jsonl_hasher.update(line_with_newline)
    total_serialized_bytes += len(line_with_newline)
    serialized_record_sizes.append(len(line_with_newline))

assert len(packets) == 2664
assert round_trip_mismatch_count == 0

print(json.dumps({
    "status": "PASS",
    "development_packet_count": len(packets),
    "round_trip_mismatch_count": round_trip_mismatch_count,
    "prospective_jsonl_total_bytes": total_serialized_bytes,
    "prospective_jsonl_sha256": jsonl_hasher.hexdigest(),
    "record_size_bytes": {
        "min": min(serialized_record_sizes),
        "max": max(serialized_record_sizes),
    },
    "file_written": False,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_packet_count": 2664,
  "round_trip_mismatch_count": 0,
  "prospective_jsonl_total_bytes": 332571983,
  "prospective_jsonl_sha256": "33bf34f8f99cf5bb3d8750fdea91f7cc31e0d1bf2cd84ed301d511cbf93e24a4",
  "record_size_bytes": {
    "min": 33773,
    "max": 443762
  },
  "file_written": false,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}


<div dir="rtl">

### NB18-MD-C41 — ارزیابی نمایش نرمال‌شدهٔ بسته‌ها

یک نمایش آزمایشی می‌سازد که اطلاعات هر مدرک را یک‌بار نگه می‌دارد و بسته‌ها به شناسه‌های آن ارجاع می‌دهند. اندازهٔ سریال‌شده با JSONL کامل مقایسه و بازسازی همهٔ بسته‌ها با دادهٔ اصلی تطبیق داده می‌شود. هیچ قالب یا مسیر ذخیره‌سازی نهایی در این مرحله انتخاب نمی‌شود.

</div>

In [41]:
# NB18-C41 — Normalized Representation Size and Fidelity Audit

import hashlib
import json

packets = nb18_development_packets

def canonical_json_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

rank_fields = {
    "reranker_rank",
    "selection_priority_rank",
    "reranker_score",
}

normalized_evidence_by_id = {}
normalized_evidence_signatures = {}
normalized_packet_stubs = []

for packet in packets:
    packet_refs = []

    for item in packet["evidence_items"]:
        evidence_id = item["evidence_id"]
        evidence_payload = {
            key: value
            for key, value in item.items()
            if key not in rank_fields
        }

        evidence_signature = hashlib.sha256(
            canonical_json_bytes(evidence_payload)
        ).hexdigest()

        if evidence_id in normalized_evidence_by_id:
            assert (
                normalized_evidence_signatures[evidence_id]
                == evidence_signature
            )
        else:
            normalized_evidence_by_id[evidence_id] = evidence_payload
            normalized_evidence_signatures[evidence_id] = evidence_signature

        packet_refs.append({
            "evidence_id": evidence_id,
            "reranker_rank": item["reranker_rank"],
            "selection_priority_rank": item["selection_priority_rank"],
            "reranker_score": item["reranker_score"],
        })

    packet_metadata = {
        key: value
        for key, value in packet.items()
        if key != "evidence_items"
    }

    normalized_packet_stubs.append({
        "packet_metadata": packet_metadata,
        "evidence_refs_in_reranker_order": packet_refs,
    })

normalized_evidence_hasher = hashlib.sha256()
normalized_evidence_bytes = 0
round_tripped_evidence_by_id = {}

for evidence_id in sorted(normalized_evidence_by_id):
    payload = normalized_evidence_by_id[evidence_id]
    line = canonical_json_bytes(payload) + b"\n"
    restored = json.loads(line)

    assert restored == payload
    assert restored["evidence_id"] == evidence_id

    normalized_evidence_hasher.update(line)
    normalized_evidence_bytes += len(line)
    round_tripped_evidence_by_id[evidence_id] = restored

normalized_packet_hasher = hashlib.sha256()
normalized_packet_bytes = 0
round_tripped_packet_stubs = []

for stub in normalized_packet_stubs:
    line = canonical_json_bytes(stub) + b"\n"
    restored = json.loads(line)

    assert restored == stub

    normalized_packet_hasher.update(line)
    normalized_packet_bytes += len(line)
    round_tripped_packet_stubs.append(restored)

reconstructed_packet_mismatch_count = 0

for original_packet, stub in zip(packets, round_tripped_packet_stubs):
    reconstructed_packet = dict(stub["packet_metadata"])
    reconstructed_items = []

    for reference in stub["evidence_refs_in_reranker_order"]:
        evidence_id = reference["evidence_id"]
        item = dict(round_tripped_evidence_by_id[evidence_id])

        for field in rank_fields:
            item[field] = reference[field]

        reconstructed_items.append(item)

    reconstructed_packet["evidence_items"] = reconstructed_items

    if canonical_json_bytes(reconstructed_packet) != canonical_json_bytes(original_packet):
        reconstructed_packet_mismatch_count += 1

normalized_total_bytes = normalized_evidence_bytes + normalized_packet_bytes
bytes_saved = total_serialized_bytes - normalized_total_bytes
size_reduction_percent = (
    100 * bytes_saved / total_serialized_bytes
    if total_serialized_bytes
    else 0.0
)

assert len(normalized_evidence_by_id) == 3277
assert len(normalized_packet_stubs) == 2664
assert reconstructed_packet_mismatch_count == 0
assert bytes_saved >= 0

print(json.dumps({
    "status": "PASS",
    "unique_evidence_record_count": len(normalized_evidence_by_id),
    "packet_stub_count": len(normalized_packet_stubs),
    "full_jsonl_bytes": total_serialized_bytes,
    "normalized_evidence_jsonl_bytes": normalized_evidence_bytes,
    "normalized_packet_map_jsonl_bytes": normalized_packet_bytes,
    "normalized_total_bytes": normalized_total_bytes,
    "bytes_saved": bytes_saved,
    "size_reduction_percent": round(size_reduction_percent, 2),
    "reconstructed_packet_mismatch_count": reconstructed_packet_mismatch_count,
    "normalized_evidence_sha256": normalized_evidence_hasher.hexdigest(),
    "normalized_packet_map_sha256": normalized_packet_hasher.hexdigest(),
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "unique_evidence_record_count": 3277,
  "packet_stub_count": 2664,
  "full_jsonl_bytes": 332571983,
  "normalized_evidence_jsonl_bytes": 45509955,
  "normalized_packet_map_jsonl_bytes": 7282352,
  "normalized_total_bytes": 52792307,
  "bytes_saved": 279779676,
  "size_reduction_percent": 84.13,
  "reconstructed_packet_mismatch_count": 0,
  "normalized_evidence_sha256": "0e6dccb9a234a517ea2560b7979c2f2ed96ee7465d0f82214364064751cf30fe",
  "normalized_packet_map_sha256": "d488fab2423700b81ed7226ba3df9b2d0f74d329d3a4e4c92dd171865f4755a1",
  "files_written": 0,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}


<div dir="rtl">

### NB18-MD-C42 — ثبت manifest نامزد خروجی NB18

مشخصات ورودی frozen، شمار بسته‌ها و مدارک، هش خروجی نرمال‌شده و وضعیت محدودیت‌های باز را در یک manifest درون‌حافظه‌ای جمع می‌کند. مسیر خروجی هنوز تعیین نشده و هیچ فایلی نوشته نمی‌شود.

</div>

In [42]:
# NB18-C42 — In-Memory NB18 Output Candidate Manifest

import hashlib
import json
from pathlib import Path

project_root = Path(
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance"
)

handoff_path = (
    project_root
    / "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json"
)
handoff = json.loads(handoff_path.read_text(encoding="utf-8"))

assert len(nb18_development_packets) == 2664
assert len(nb18_evidence_registry) == 3277
assert len(nb18_packet_evidence_refs) == 2664
assert reconstructed_packet_mismatch_count == 0

nb18_output_candidate_manifest = {
    "artifact_type": "NB18_CONTEXT_BUILDER_OUTPUT_CANDIDATE",
    "schema_version": "1.0",
    "status": "REVIEWABLE_IN_MEMORY_CANDIDATE",
    "source_handoff": {
        "path": str(handoff_path.relative_to(project_root)),
        "release_id": handoff["release_id"],
        "configuration_sha256": handoff["configuration_sha256"],
        "candidate_artifact_path": handoff["candidate_artifact"]["path"],
        "candidate_artifact_sha256": handoff["candidate_artifact"]["sha256"],
        "candidate_depth": handoff["candidate_artifact"]["candidate_depth"],
        "consumer_top_k": handoff["candidate_artifact"]["consumer_top_k"],
    },
    "development_scope": {
        "development_packet_count": len(nb18_development_packets),
        "evidence_item_occurrence_count": 26640,
        "unique_evidence_reference_count": len(nb18_evidence_registry),
        "non_development_records_used": 0,
    },
    "normalized_output_candidate": {
        "evidence_record_count": len(normalized_evidence_by_id),
        "evidence_jsonl_bytes": normalized_evidence_bytes,
        "evidence_jsonl_sha256": normalized_evidence_hasher.hexdigest(),
        "packet_map_count": len(normalized_packet_stubs),
        "packet_map_jsonl_bytes": normalized_packet_bytes,
        "packet_map_jsonl_sha256": normalized_packet_hasher.hexdigest(),
        "combined_bytes": normalized_total_bytes,
        "full_packet_jsonl_comparison_bytes": total_serialized_bytes,
        "size_reduction_percent": round(size_reduction_percent, 2),
        "round_trip_reconstruction_mismatches": (
            reconstructed_packet_mismatch_count
        ),
    },
    "constraints": {
        "reranker_order_preserved": True,
        "reranker_scores_preserved": True,
        "source_priority_order_separate": True,
        "production_tokenizer_used": False,
        "production_token_budget_applied": False,
        "rights_clearance_asserted": False,
        "generation_ready_packet_count": sum(
            packet["generation_ready"] is True
            for packet in nb18_development_packets
        ),
        "release_eligible_packet_count": sum(
            packet["release_eligible"] is True
            for packet in nb18_development_packets
        ),
    },
    "output_paths_assigned": False,
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

manifest_bytes = json.dumps(
    nb18_output_candidate_manifest,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8")

nb18_output_candidate_manifest_sha256 = hashlib.sha256(
    manifest_bytes
).hexdigest()

print(json.dumps({
    "status": "PASS",
    "manifest_sha256": nb18_output_candidate_manifest_sha256,
    "manifest": nb18_output_candidate_manifest,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "manifest_sha256": "f2320c3bc8d343c684478aa8e83a78cc3b362e3cf619fbf5b1c6356206fb3856",
  "manifest": {
    "artifact_type": "NB18_CONTEXT_BUILDER_OUTPUT_CANDIDATE",
    "schema_version": "1.0",
    "status": "REVIEWABLE_IN_MEMORY_CANDIDATE",
    "source_handoff": {
      "path": "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json",
      "release_id": "rag_finance_phase2_reranker_v1",
      "configuration_sha256": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738",
      "candidate_artifact_path": "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl",
      "candidate_artifact_sha256": "d00285bbf3865fbd51041db1612b9fdc17e9778273ea6d136978906a07268b7a",
      "candidate_depth": 50,
      "consumer_top_k": 10
    },
    "development_scope": {
      "development_packet_count": 2664,
      "evidence_item_occurrence_count": 26640,
      "unique_evidence_reference_count": 3277,
      "non_development_records_used": 0
    },
    "normal

<div dir="rtl">

### NB18-MD-C43 — ممیزی محل importها

سلول‌های کد نوت‌بوک را برای پیدا کردن importهای Bootstrap و importهای تکراری خارج از آن بررسی می‌کند. این مرحله فقط‌خواندنی است و تغییری در فایل ایجاد نمی‌کند.

</div>

In [43]:
# NB18-C43 — Notebook Import Placement Audit

notebook_path = (
    Path("/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance")
    / "notebooks"
    / "phase2"
    / "18_context_builder.ipynb"
)

assert notebook_path.is_file()

notebook_payload = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

code_cells = [
    cell
    for cell in notebook_payload.get("cells", [])
    if cell.get("cell_type") == "code"
]

import_pattern = re.compile(r"^\s*(?:import\s+|from\s+)")
cell_imports = []

for position, cell in enumerate(code_cells, start=1):
    source = cell.get("source", [])
    if isinstance(source, list):
        source = "".join(source)
    elif not isinstance(source, str):
        source = ""

    lines = source.splitlines()
    first_line = lines[0].strip() if lines else ""

    imports = [
        line.strip()
        for line in lines
        if import_pattern.match(line)
    ]

    cell_imports.append({
        "code_cell_position": position,
        "first_line": first_line,
        "import_lines": imports,
    })

bootstrap_candidates = [
    item
    for item in cell_imports
    if re.search(r"NB18-C01|Code Cell 01", item["first_line"])
]

bootstrap_position = (
    bootstrap_candidates[0]["code_cell_position"]
    if len(bootstrap_candidates) == 1
    else None
)

imports_outside_bootstrap = [
    item
    for item in cell_imports
    if item["import_lines"]
    and item["code_cell_position"] != bootstrap_position
]

print(json.dumps({
    "status": "INSPECTED",
    "code_cell_count": len(code_cells),
    "bootstrap_candidate_count": len(bootstrap_candidates),
    "bootstrap_code_cell_position": bootstrap_position,
    "bootstrap_import_lines": (
        bootstrap_candidates[0]["import_lines"]
        if len(bootstrap_candidates) == 1
        else []
    ),
    "imports_outside_bootstrap": imports_outside_bootstrap,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "INSPECTED",
  "code_cell_count": 73,
  "bootstrap_candidate_count": 1,
  "bootstrap_code_cell_position": 1,
  "bootstrap_import_lines": [
    "import hashlib",
    "import json",
    "import os",
    "import re",
    "import subprocess",
    "import sys",
    "from pathlib import Path"
  ],
  "imports_outside_bootstrap": [
    {
      "code_cell_position": 5,
      "first_line": "# NB18-C05 — Read-only discovery of the documented taxonomy inputs",
      "import_lines": [
        "import hashlib",
        "import json",
        "from collections import Counter",
        "from pathlib import Path"
      ]
    },
    {
      "code_cell_position": 6,
      "first_line": "# NB18-C06 — Read-only comparison of accepted-document tags and taxonomy IDs",
      "import_lines": [
        "import hashlib",
        "import json",
        "import time",
        "from collections import Counter",
        "from pathlib import Path"
      ]
    },
    {
      "code_cell_position": 7,
    

<div dir="rtl">

### بررسی نهایی سلامت نوت‌بوک

پس از `Restart Kernel → Run All`، تعداد و شماره‌گذاری Code Cellها، سلول‌های اجرا‌نشده، خطاهای ذخیره‌شده و هش manifest بررسی می‌شوند. تعداد Markdownها و importهای خارج از Bootstrap فقط گزارش می‌شوند.

</div>

In [44]:
# NB18-C44 — Final Notebook Health Check

notebook_path = (
    Path("/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance")
    / "notebooks"
    / "phase2"
    / "18_context_builder.ipynb"
)

assert notebook_path.is_file()

notebook_payload = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

all_cells = notebook_payload.get("cells", [])
code_cells = [
    cell for cell in all_cells
    if cell.get("cell_type") == "code"
]
markdown_cells = [
    cell for cell in all_cells
    if cell.get("cell_type") == "markdown"
]

def cell_source(cell):
    source = cell.get("source", [])
    if isinstance(source, list):
        return "".join(source)
    return source if isinstance(source, str) else ""

code_number_pattern = re.compile(
    r"^\s*#\s*(?:NB18-C|Code Cell\s+|C)?(\d{2})\b"
)

code_numbers = []
unnumbered_code_positions = []

for position, cell in enumerate(code_cells, start=1):
    lines = cell_source(cell).splitlines()
    first_line = lines[0] if lines else ""
    match = code_number_pattern.match(first_line)

    if match:
        code_numbers.append(int(match.group(1)))
    else:
        unnumbered_code_positions.append(position)

expected_code_numbers = list(range(1, len(code_cells) + 1))
duplicate_code_numbers = sorted(
    number
    for number in set(code_numbers)
    if code_numbers.count(number) > 1
)
missing_code_numbers = sorted(
    set(expected_code_numbers) - set(code_numbers)
)
nonsequential_code_numbers = code_numbers != expected_code_numbers

unexecuted_prior_code_positions = [
    position
    for position, (cell, number) in enumerate(
        zip(code_cells, code_numbers),
        start=1,
    )
    if number != len(code_cells)
    and cell.get("execution_count") is None
]

saved_errors = []
for position, cell in enumerate(code_cells, start=1):
    for output in cell.get("outputs", []):
        if output.get("output_type") == "error":
            saved_errors.append({
                "code_cell_position": position,
                "ename": output.get("ename"),
                "evalue": output.get("evalue"),
            })

bootstrap_positions = [
    position
    for position, number in enumerate(code_numbers, start=1)
    if number == 1
]

import_pattern = re.compile(r"^\s*(?:import\s+|from\s+)")
imports_outside_bootstrap = []

for position, (cell, number) in enumerate(
    zip(code_cells, code_numbers),
    start=1,
):
    if number == 1:
        continue

    imports = [
        line.strip()
        for line in cell_source(cell).splitlines()
        if import_pattern.match(line)
    ]

    if imports:
        imports_outside_bootstrap.append({
            "code_cell_position": position,
            "import_lines": imports,
        })

assert "nb18_output_candidate_manifest" in globals()

manifest_bytes = json.dumps(
    nb18_output_candidate_manifest,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8")

manifest_sha256_recomputed = hashlib.sha256(
    manifest_bytes
).hexdigest()

manifest_hash_matches = (
    manifest_sha256_recomputed
    == nb18_output_candidate_manifest_sha256
)

code_numbering_pass = (
    not unnumbered_code_positions
    and not duplicate_code_numbers
    and not missing_code_numbers
    and not nonsequential_code_numbers
)
execution_pass = not unexecuted_prior_code_positions
saved_error_pass = not saved_errors
bootstrap_pass = len(bootstrap_positions) == 1

notebook_health_pass = all([
    code_numbering_pass,
    execution_pass,
    saved_error_pass,
    bootstrap_pass,
    manifest_hash_matches,
])

print(json.dumps({
    "status": "PASS" if notebook_health_pass else "FAIL",
    "code_cell_count": len(code_cells),
    "code_cell_numbers": code_numbers,
    "unnumbered_code_positions": unnumbered_code_positions,
    "duplicate_code_numbers": duplicate_code_numbers,
    "missing_code_numbers": missing_code_numbers,
    "nonsequential_code_numbers": nonsequential_code_numbers,
    "unexecuted_prior_code_positions": unexecuted_prior_code_positions,
    "saved_error_count": len(saved_errors),
    "saved_errors": saved_errors,
    "markdown_cell_count": len(markdown_cells),
    "bootstrap_code_cell_positions": bootstrap_positions,
    "imports_outside_bootstrap_count": len(imports_outside_bootstrap),
    "manifest_sha256_matches": manifest_hash_matches,
    "output_artifacts_written": False,
    "production_token_budget_applied": False,
    "generation_ready_packet_count": sum(
        packet["generation_ready"] is True
        for packet in nb18_development_packets
    ),
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

print(
    "NOTEBOOK HEALTH: "
    + ("PASS" if notebook_health_pass else "FAIL")
)

{
  "status": "PASS",
  "code_cell_count": 73,
  "code_cell_numbers": [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23,
    24,
    25,
    26,
    27,
    28,
    29,
    30,
    31,
    32,
    33,
    34,
    35,
    36,
    37,
    38,
    39,
    40,
    41,
    42,
    43,
    44,
    45,
    46,
    47,
    48,
    49,
    50,
    51,
    52,
    53,
    54,
    55,
    56,
    57,
    58,
    59,
    60,
    61,
    62,
    63,
    64,
    65,
    66,
    67,
    68,
    69,
    70,
    71,
    72,
    73
  ],
  "unnumbered_code_positions": [],
  "duplicate_code_numbers": [],
  "missing_code_numbers": [],
  "nonsequential_code_numbers": false,
  "unexecuted_prior_code_positions": [],
  "saved_error_count": 0,
  "saved_errors": [],
  "markdown_cell_count": 76,
  "bootstrap_code_cell_positions": [
    1
  ],
  "imports_outside_bootstrap_count": 41,
  "man

<div dir="rtl" align="right">

## جمع‌بندی نهایی نوت‌بوک ۱۸

**هدف نوت‌بوک**

ساخت بسته‌های context برای پرس‌وجوهای Development بر پایهٔ خروجی frozen نوت‌بوک ۱۷، با حفظ ترتیب و امتیازهای reranker و اطلاعات منبع و ارجاع. این مرحله قرار نبود reranker را دوباره اجرا کند یا به مدل API وصل شود.

**کارهای انجام‌شده**

برای ۲۶۶۴ پرس‌وجوی Development، بسته‌هایی با ۱۰ مدرک ساخته شد؛ در مجموع ۲۶٬۶۴۰ مورد شواهد و ۳٬۲۷۷ ارجاع یکتا. تعداد ۱۱۴۹ رکورد غیر Development وارد بسته‌ها نشد.

ترتیب، رتبه و امتیاز ده مدرک هر بسته با ورودی frozen نوت‌بوک ۱۷ تطبیق داده شد. متن، locator، شناسه‌ها و metadata منبع حفظ شدند. نمایش نرمال‌شدهٔ بسته‌ها نیز بدون اختلاف، بسته‌های اصلی را بازسازی کرد و اندازهٔ سریال‌شده را ۸۴٫۱۳٪ کاهش داد.

**Artifactها و اعتبارسنجی**

خروجی‌های ارزیابی و هش‌ها در حافظه ساخته شدند، اما هیچ artifact پایداری در فایل ذخیره نشد؛ مسیر ذخیرهٔ خروجی NB18 هم تعیین نشده است. Final Health Check با ۴۴ Code Cell شماره‌گذاری‌شده، بدون خطای ذخیره‌شده و بدون سلول کد اجرا‌نشده، PASS شد.

در بررسی اولیهٔ schema، خطای `candidate_count` رخ داد؛ اصلاح شد تا از فیلدهای موجود در قرارداد ورودی استفاده شود. هیچ تغییری در تنظیمات یا امتیازهای reranker ندادیم.

**محدودیت‌های باقی‌مانده**

توکنایزر و بودجهٔ توکنِ مدل تولید انتخاب یا اعمال نشده‌اند. در نتیجه، بسته‌ها generation-ready نیستند. همچنین در metadata حقوقیِ منابع اطلاعات لازم برای release کامل نیست؛ این وضعیت نقص metadata در داده است و به‌تنهایی دربارهٔ قانونی‌بودن یا نبودن محتوا حکمی نمی‌دهد.

محتوای منابع در این نوت‌بوک بررسی یا بازنویسی معنایی نشد؛ متن و انتساب‌های موجود حفظ شدند. از امتیازهای Test برای تصمیم‌گیری استفاده نشد و هیچ فراخوانی API پولی انجام نشد.

**وضعیت نهایی**

NOTEBOOK OBJECTIVE: PARTIALLY ACHIEVED

</div>

<div dir="rtl" align="right">

## جمع‌بندی فاصله تا هدف نوت‌بوک ۱۸

این گام شواهد موجود در سلول‌های قبلی را کنار هم می‌گذارد تا مشخص شود کدام نیازهای Context Builder انجام شده‌اند و کدام هنوز باز هستند. رتبه و امتیاز Reranker تغییر نمی‌کند و داده‌های Test برای انتخاب سیاست استفاده نمی‌شوند.

**معیار موفقیت:** وضعیت کلی هدف به‌درستی `INCOMPLETE` بماند اگر بودجهٔ واقعی اعمال نشده یا Artifact نهایی ساخته نشده باشد. این سلول فقط وضعیت را در حافظه ثبت می‌کند و فایلی نمی‌نویسد.

</div>

In [45]:
# NB18-C45 — Consolidate the Existing Objective Gap

assert "nb18_development_packets" in globals()
assert "nb18_evidence_registry" in globals()
assert "nb18_packet_evidence_refs" in globals()
assert "reconstructed_packet_mismatch_count" in globals()
assert "nb18_output_candidate_manifest" in globals()

assert len(nb18_development_packets) == 2664
assert len(nb18_evidence_registry) == 3277
assert len(nb18_packet_evidence_refs) == 2664
assert reconstructed_packet_mismatch_count == 0

nb18_objective_gap_matrix = [
    {
        "requirement": "Preserve frozen reranker order and scores",
        "status": "PASS",
        "evidence": "C36 and C42; frozen handoff fidelity and manifest constraints",
    },
    {
        "requirement": "Build Development context candidates with provenance",
        "status": "PASS",
        "evidence": "C22-C25 and C37-C41; 2664 packets and round-trip audit",
    },
    {
        "requirement": "Deduplicate evidence representation",
        "status": "PASS",
        "evidence": "C38-C41; 3277 unique references and zero reconstruction mismatches",
    },
    {
        "requirement": "Control source and document dominance in final selection",
        "status": "PARTIAL",
        "evidence": "C09 and C13-C14 are audits or experiments; no frozen production policy",
    },
    {
        "requirement": "Apply useful topic diversity",
        "status": "BLOCKED",
        "evidence": "C07 found frozen topic mappings for only 2 of 19429 Development document IDs",
    },
    {
        "requirement": "Apply controlled parent expansion",
        "status": "BLOCKED",
        "evidence": "C15 and C19 report zero direct parent chunk links",
    },
    {
        "requirement": "Apply a production tokenizer and context budget",
        "status": "MISSING",
        "evidence": "C31-C33 and C42 report no selected production tokenizer or budget",
    },
    {
        "requirement": "Preserve locator, source, attribution, and rights metadata",
        "status": "PASS",
        "evidence": "C10, C23, C37, and C42; metadata preserved without asserting rights clearance",
    },
    {
        "requirement": "Assess semantic evidence conflicts",
        "status": "PARTIAL",
        "evidence": "C16-C17 preserve metadata signals; semantic conflicts remain NOT_ASSESSED",
    },
    {
        "requirement": "Write and validate versioned output artifacts",
        "status": "MISSING",
        "evidence": "C40-C42 are in-memory audits; output_paths_assigned is false",
    },
    {
        "requirement": "Produce generation-ready packets",
        "status": "BLOCKED",
        "evidence": "C19-C21 and C42 report zero generation-ready packets",
    },
]

allowed_gap_statuses = {"PASS", "PARTIAL", "MISSING", "BLOCKED"}
assert all(
    row["status"] in allowed_gap_statuses
    for row in nb18_objective_gap_matrix
)

nb18_objective_status = "INCOMPLETE"

print(json.dumps({
    "status": nb18_objective_status,
    "requirement_count": len(nb18_objective_gap_matrix),
    "requirements": nb18_objective_gap_matrix,
    "execution_health": "PASS",
    "execution_health_scope": "Structural and execution checks only",
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}, ensure_ascii=False, indent=2))

{
  "status": "INCOMPLETE",
  "requirement_count": 11,
  "requirements": [
    {
      "requirement": "Preserve frozen reranker order and scores",
      "status": "PASS",
      "evidence": "C36 and C42; frozen handoff fidelity and manifest constraints"
    },
    {
      "requirement": "Build Development context candidates with provenance",
      "status": "PASS",
      "evidence": "C22-C25 and C37-C41; 2664 packets and round-trip audit"
    },
    {
      "requirement": "Deduplicate evidence representation",
      "status": "PASS",
      "evidence": "C38-C41; 3277 unique references and zero reconstruction mismatches"
    },
    {
      "requirement": "Control source and document dominance in final selection",
      "status": "PARTIAL",
      "evidence": "C09 and C13-C14 are audits or experiments; no frozen production policy"
    },
    {
      "requirement": "Apply useful topic diversity",
      "status": "BLOCKED",
      "evidence": "C07 found frozen topic mappings for only 2 of 1942

<div dir="rtl" align="right">

## ثبت مواردی که هنوز در نقشه‌ها تعیین نشده‌اند

نقشهٔ ماژولار، نیازهای کلی NB18 را مشخص می‌کند؛ از جمله Token Budget، تنوع، Parent Expansion و Conflict Handling. اما عدد بودجه، Tokenizer، معیار انتخاب سیاست و قرارداد دقیق Artifactها را تعیین نمی‌کند.

این سلول موارد تعیین‌نشده را ثبت می‌کند تا در گام بعدی هیچ مقدار یا مسیری حدس زده نشود. این ثبت به معنی Freeze شدن Configuration نیست.

**معیار موفقیت:** همهٔ موارد فاقد تصمیم رسمی صریحاً `BLOCKED — USER DECISION REQUIRED` باقی بمانند.

</div>

In [46]:
# NB18-C46 — Record Unresolved Configuration Decisions

assert "nb18_objective_gap_matrix" in globals()
assert nb18_objective_status == "INCOMPLETE"

nb18_unresolved_decisions = [
    {
        "decision": "Production model and tokenizer",
        "status": "BLOCKED — USER DECISION REQUIRED",
        "reason": "The NB18 maps do not freeze a generation model or name an approved production tokenizer.",
    },
    {
        "decision": "Allowed context-budget values",
        "status": "BLOCKED — USER DECISION REQUIRED",
        "reason": "The NB18 maps require token budgeting but specify no approved budget values.",
    },
    {
        "decision": "Development policy-selection metric and threshold",
        "status": "BLOCKED — USER DECISION REQUIRED",
        "reason": "The NB18 maps require an experiment and ADR but specify no acceptance threshold.",
    },
    {
        "decision": "Parent-expansion policy and limit",
        "status": "BLOCKED — USER DECISION REQUIRED",
        "reason": "The modular plan requires controlled parent expansion but gives no exact selection rule or limit.",
    },
    {
        "decision": "Topic-diversity policy when frozen mapping coverage is insufficient",
        "status": "BLOCKED — USER DECISION REQUIRED",
        "reason": "Current frozen mapping covers only 2 of 19429 Development document IDs.",
    },
    {
        "decision": "Conflict-handling scope",
        "status": "BLOCKED — USER DECISION REQUIRED",
        "reason": "Conflict handling is required at a high level; semantic inference is not established by current metadata.",
    },
    {
        "decision": "Versioned output paths and exact output schemas",
        "status": "BLOCKED — USER DECISION REQUIRED",
        "reason": "The NB18 maps do not provide the final artifact paths or complete schemas.",
    },
]

assert all(
    item["status"] == "BLOCKED — USER DECISION REQUIRED"
    for item in nb18_unresolved_decisions
)

print(json.dumps({
    "status": "BLOCKED_BEFORE_CONFIGURATION_FREEZE",
    "unresolved_decision_count": len(nb18_unresolved_decisions),
    "decisions": nb18_unresolved_decisions,
    "test_unlocked": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "BLOCKED_BEFORE_CONFIGURATION_FREEZE",
  "unresolved_decision_count": 7,
  "decisions": [
    {
      "decision": "Production model and tokenizer",
      "status": "BLOCKED — USER DECISION REQUIRED",
      "reason": "The NB18 maps do not freeze a generation model or name an approved production tokenizer."
    },
    {
      "decision": "Allowed context-budget values",
      "status": "BLOCKED — USER DECISION REQUIRED",
      "reason": "The NB18 maps require token budgeting but specify no approved budget values."
    },
    {
      "decision": "Development policy-selection metric and threshold",
      "status": "BLOCKED — USER DECISION REQUIRED",
      "reason": "The NB18 maps require an experiment and ADR but specify no acceptance threshold."
    },
    {
      "decision": "Parent-expansion policy and limit",
      "status": "BLOCKED — USER DECISION REQUIRED",
      "reason": "The modular plan requires controlled parent expansion but gives no exact selection rule or limit

<div dir="rtl" align="right">

## آماده‌سازی قرارداد Freeze برای Development

پیش از ارزیابی سیاست‌ها باید Configuration شامل Tokenizer، بودجه، قواعد انتخاب، و تصمیم‌های لازم دیگر مشخص و ثبت شود. انتخاب‌ها فقط با Development انجام می‌شود. تا آن زمان Test قفل می‌ماند و هیچ نتیجه‌ای از آن برای تصمیم‌گیری خوانده نمی‌شود.

این گام فقط وضعیت آمادگی را کنترل می‌کند؛ Configuration را Freeze نمی‌کند.

**معیار موفقیت:** اگر هر تصمیم لازم باز باشد، وضعیت باید `BLOCKED_BEFORE_CONFIGURATION_FREEZE` باشد.

</div>

In [47]:
# NB18-C47 — Check Development-Only Configuration Freeze Readiness

assert "nb18_development_packets" in globals()
assert "nb18_unresolved_decisions" in globals()
assert "nb18_output_candidate_manifest" in globals()

packet_splits = {
    str(packet.get("split"))
    for packet in nb18_development_packets
}
assert packet_splits == {"development"}

manifest_constraints = nb18_output_candidate_manifest["constraints"]
assert manifest_constraints["production_tokenizer_used"] is False
assert manifest_constraints["production_token_budget_applied"] is False

open_decisions = [
    item
    for item in nb18_unresolved_decisions
    if item["status"] == "BLOCKED — USER DECISION REQUIRED"
]

nb18_configuration_freeze_readiness = {
    "status": (
        "BLOCKED_BEFORE_CONFIGURATION_FREEZE"
        if open_decisions
        else "READY_FOR_REVIEW"
    ),
    "development_packet_count": len(nb18_development_packets),
    "packet_splits": sorted(packet_splits),
    "open_decision_count": len(open_decisions),
    "configuration_frozen": False,
    "test_unlocked": False,
    "test_evidence_or_metrics_used": False,
    "frozen_reranker_modified": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

assert nb18_configuration_freeze_readiness["status"] == (
    "BLOCKED_BEFORE_CONFIGURATION_FREEZE"
)

print(json.dumps(
    nb18_configuration_freeze_readiness,
    ensure_ascii=False,
    indent=2,
))

{
  "status": "BLOCKED_BEFORE_CONFIGURATION_FREEZE",
  "development_packet_count": 2664,
  "packet_splits": [
    "development"
  ],
  "open_decision_count": 7,
  "configuration_frozen": false,
  "test_unlocked": false,
  "test_evidence_or_metrics_used": false,
  "frozen_reranker_modified": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

## ثبت وضعیت واقعی نوت‌بوک پیش از ادامه

سلول ۴۴ بررسی ساختاری و اجرایی انجام می‌دهد؛ PASS آن به معنی تکمیل هدف Context Builder نیست. این سلول دو وضعیت را جداگانه گزارش می‌کند: سلامت اجرای نوت‌بوک و تکمیل هدف NB18.

تا پیش از Freeze شدن Configuration، اجرای ارزیابی سیاست، بازکردن Test و تولید Artifact نهایی انجام نمی‌شود.

**معیار موفقیت:** سلامت اجرا با نتیجهٔ قبلی خلط نشود و هدف NB18 تا زمان تکمیل Gateهای واقعی `INCOMPLETE` بماند.

</div>

In [48]:
# NB18-C48 — Record the Interim Execution and Objective Status

assert "nb18_configuration_freeze_readiness" in globals()
assert "nb18_objective_gap_matrix" in globals()
assert nb18_configuration_freeze_readiness["test_unlocked"] is False

incomplete_requirements = [
    row
    for row in nb18_objective_gap_matrix
    if row["status"] != "PASS"
]

nb18_interim_completion_status = {
    "execution_health": "PASS",
    "execution_health_meaning": (
        "Existing structural checks passed; this does not establish NB18 completion."
    ),
    "objective_status": "INCOMPLETE",
    "incomplete_requirement_count": len(incomplete_requirements),
    "configuration_freeze_status": (
        nb18_configuration_freeze_readiness["status"]
    ),
    "test_unlocked": False,
    "output_artifacts_written": False,
    "generation_ready_packet_count": sum(
        packet.get("generation_ready") is True
        for packet in nb18_development_packets
    ),
    "ready_for_notebook_19": False,
    "files_written": 0,
    "paid_api_calls": 0,
    "test_evidence_or_metrics_used": False,
}

assert nb18_interim_completion_status["objective_status"] == "INCOMPLETE"
assert nb18_interim_completion_status["ready_for_notebook_19"] is False
assert nb18_interim_completion_status["generation_ready_packet_count"] == 0

print(json.dumps(
    nb18_interim_completion_status,
    ensure_ascii=False,
    indent=2,
))
print("NB18 OBJECTIVE: INCOMPLETE")

{
  "execution_health": "PASS",
  "execution_health_meaning": "Existing structural checks passed; this does not establish NB18 completion.",
  "objective_status": "INCOMPLETE",
  "incomplete_requirement_count": 7,
  "configuration_freeze_status": "BLOCKED_BEFORE_CONFIGURATION_FREEZE",
  "test_unlocked": false,
  "output_artifacts_written": false,
  "generation_ready_packet_count": 0,
  "ready_for_notebook_19": false,
  "files_written": 0,
  "paid_api_calls": 0,
  "test_evidence_or_metrics_used": false
}
NB18 OBJECTIVE: INCOMPLETE


<div dir="rtl" align="right">

## ثبت مدل هدف و قرارداد Tokenizer

مدل هدف از تنظیماتی که فرستادی مشخص است: `Qwen3-30B-A3B`. برای شمارش توکن، از Tokenizer متناظر با همین مدل استفاده می‌کنیم. این مرحله فقط قرارداد را ثبت می‌کند و به آدرس API وصل نمی‌شود.

حد ۴۰٬۹۶۰ توکن، سقف معماری ثبت‌شده برای مدل است؛ سقف Endpoint آروان یا بودجهٔ واقعی Context محسوب نمی‌شود.

**معیار موفقیت:** مدل و منبع Tokenizer مشخص باشند، ولی بودجه همچنان انتخاب‌نشده بماند.

</div>

In [49]:
# NB18-C49 — Register the Qwen Tokenizer Contract

NB18_TARGET_MODEL = "Qwen3-30B-A3B"
NB18_TOKENIZER_REPOSITORY = "Qwen/Qwen3-30B-A3B"
NB18_MODEL_ARCHITECTURAL_LIMIT = 40960

nb18_model_tokenizer_contract = {
    "status": "REGISTERED_FOR_LOCAL_MEASUREMENT",
    "target_model": NB18_TARGET_MODEL,
    "tokenizer_repository": NB18_TOKENIZER_REPOSITORY,
    "tokenizer_revision": "main",
    "model_architectural_limit_tokens": NB18_MODEL_ARCHITECTURAL_LIMIT,
    "architectural_limit_source": (
        "https://huggingface.co/Qwen/Qwen3-30B-A3B/blob/main/config.json"
    ),
    "tokenizer_usage_source": (
        "https://huggingface.co/Qwen/Qwen3-30B-A3B"
    ),
    "provider_endpoint_limit_tokens": None,
    "context_budget_selected": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

assert nb18_model_tokenizer_contract["target_model"] == "Qwen3-30B-A3B"
assert nb18_model_tokenizer_contract["context_budget_selected"] is False
assert nb18_model_tokenizer_contract["model_api_called"] is False

print(json.dumps(
    nb18_model_tokenizer_contract,
    ensure_ascii=False,
    indent=2,
))

{
  "status": "REGISTERED_FOR_LOCAL_MEASUREMENT",
  "target_model": "Qwen3-30B-A3B",
  "tokenizer_repository": "Qwen/Qwen3-30B-A3B",
  "tokenizer_revision": "main",
  "model_architectural_limit_tokens": 40960,
  "architectural_limit_source": "https://huggingface.co/Qwen/Qwen3-30B-A3B/blob/main/config.json",
  "tokenizer_usage_source": "https://huggingface.co/Qwen/Qwen3-30B-A3B",
  "provider_endpoint_limit_tokens": null,
  "context_budget_selected": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

## بررسی کتابخانهٔ Tokenizer در محیط

این سلول فقط بررسی می‌کند کتابخانهٔ `transformers` نصب است یا نه. چیزی نصب نمی‌کند و به سرویس مدل وصل نمی‌شود.

**معیار موفقیت:** موجودبودن کتابخانه مشخص شود؛ اگر نصب نباشد، وضعیت `BLOCKED` گزارش می‌شود.

</div>

In [50]:
# NB18-C50 — Check Local Tokenizer Library Availability

try:
    import transformers

    nb18_transformers_available = True
    nb18_transformers_version = transformers.__version__
    nb18_tokenizer_library_status = "PASS"
except ImportError:
    nb18_transformers_available = False
    nb18_transformers_version = None
    nb18_tokenizer_library_status = "BLOCKED"

nb18_tokenizer_library_audit = {
    "status": nb18_tokenizer_library_status,
    "transformers_available": nb18_transformers_available,
    "transformers_version": nb18_transformers_version,
    "installation_attempted": False,
    "network_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(
    nb18_tokenizer_library_audit,
    ensure_ascii=False,
    indent=2,
))

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "status": "PASS",
  "transformers_available": true,
  "transformers_version": "4.53.2",
  "installation_attempted": false,
  "network_used": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

## بارگذاری محلی Tokenizer بدون دسترسی شبکه

Tokenizer متناظر با مدل Qwen فقط از Cache محلی بارگذاری می‌شود. گزینهٔ `local_files_only=True` مانع دانلود خودکار می‌شود.

اگر فایل‌های Tokenizer در Cache نباشند، وضعیت `BLOCKED` گزارش می‌شود.

**معیار موفقیت:** Tokenizer بارگذاری شود و اثر انگشت آن ثبت شود تا شمارش توکن قابل‌ردیابی باشد.

</div>

<div dir="rtl" align="right">

## دریافت Tokenizer عمومی Qwen در Cache محلی

Tokenizer در Cache محلی موجود نبود. این ترمیم فقط فایل‌های لازم برای Tokenizer را از مخزن عمومی Qwen دریافت می‌کند. وزن‌های مدل دانلود نمی‌شوند و Endpoint آروان استفاده نمی‌شود.

فایل‌ها در Cache محلی Hugging Face ذخیره می‌شوند، خارج از مخزن پروژه.

**معیار موفقیت:** Tokenizer بارگذاری شود و اثر انگشت فایل‌های آن ثبت شود.

</div>

In [51]:
# NB18-C51 — Download and Load the Public Qwen Tokenizer

import hashlib
import json
import os

# These defaults also allow this cell to run after a kernel restart by itself.
NB18_TARGET_MODEL = globals().get("NB18_TARGET_MODEL", "Qwen3-30B-A3B")
NB18_TOKENIZER_REPOSITORY = globals().get(
    "NB18_TOKENIZER_REPOSITORY", "Qwen/Qwen3-30B-A3B"
)
assert NB18_TARGET_MODEL == "Qwen3-30B-A3B"
assert NB18_TOKENIZER_REPOSITORY == "Qwen/Qwen3-30B-A3B"

# Must be set before importing Transformers / huggingface_hub in a fresh kernel.
os.environ["HF_HUB_ETAG_TIMEOUT"] = "10"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"

nb18_tokenizer = None
nb18_tokenizer_fingerprint = None
nb18_tokenizer_load_error_type = None
nb18_tokenizer_revision = None

try:
    from transformers import AutoTokenizer

    nb18_tokenizer = AutoTokenizer.from_pretrained(
        NB18_TOKENIZER_REPOSITORY,
        revision="main",
        local_files_only=False,
        trust_remote_code=False,
        use_fast=True,
    )

    tokenizer_payload = nb18_tokenizer.backend_tokenizer.to_str().encode("utf-8")
    special_tokens_payload = json.dumps(
        nb18_tokenizer.special_tokens_map,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    nb18_tokenizer_revision = nb18_tokenizer.init_kwargs.get("_commit_hash")
    nb18_tokenizer_fingerprint = {
        "backend_sha256": hashlib.sha256(tokenizer_payload).hexdigest(),
        "special_tokens_sha256": hashlib.sha256(special_tokens_payload).hexdigest(),
        "tokenizer_class": type(nb18_tokenizer).__name__,
        "is_fast": bool(nb18_tokenizer.is_fast),
    }
    nb18_tokenizer_load_status = "PASS"
except Exception as error:
    nb18_tokenizer_load_status = "BLOCKED_TOKENIZER_DOWNLOAD_OR_LOAD_FAILED"
    nb18_tokenizer_load_error_type = type(error).__name__

nb18_tokenizer_load_audit = {
    "status": nb18_tokenizer_load_status,
    "model": NB18_TARGET_MODEL,
    "tokenizer_repository": NB18_TOKENIZER_REPOSITORY,
    "requested_revision": "main",
    "resolved_revision": nb18_tokenizer_revision,
    "tokenizer_loaded": nb18_tokenizer is not None,
    "tokenizer_fingerprint": nb18_tokenizer_fingerprint,
    "error_type": nb18_tokenizer_load_error_type,
    "public_tokenizer_download_attempted": True,
    "network_used_for_public_tokenizer": True,
    "model_api_called": False,
    "arvan_endpoint_called": False,
    "model_weights_downloaded": False,
    "project_files_written": 0,
    "huggingface_cache_may_be_written": True,
    "paid_api_calls": 0,
}

print(json.dumps(nb18_tokenizer_load_audit, ensure_ascii=False, indent=2))


{
  "status": "PASS",
  "model": "Qwen3-30B-A3B",
  "tokenizer_repository": "Qwen/Qwen3-30B-A3B",
  "requested_revision": "main",
  "resolved_revision": null,
  "tokenizer_loaded": true,
  "tokenizer_fingerprint": {
    "backend_sha256": "a63080b4b30dc861fe1b0e420c53a241304dc81aff3d9bb315e6e33e2a1ab794",
    "special_tokens_sha256": "94f02d4e8b7a6569bef78a7ff2c3b40744a0fd54e603a9c5a5bdba4d05256a10",
    "tokenizer_class": "Qwen2TokenizerFast",
    "is_fast": true
  },
  "error_type": null,
  "public_tokenizer_download_attempted": true,
  "network_used_for_public_tokenizer": true,
  "model_api_called": false,
  "arvan_endpoint_called": false,
  "model_weights_downloaded": false,
  "project_files_written": 0,
  "huggingface_cache_may_be_written": true,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

## شمارش توکن متن شواهد Development

این سلول شمارهٔ نسخهٔ موجود در Cache محلی را، در صورت دسترس‌بودن، ثبت می‌کند؛ سپس متن شواهد هر بستهٔ Development را با Tokenizer بارگذاری‌شده می‌شمارد.

شمارش فقط شامل متن Evidence است؛ قالب Prompt، پرسش، metadata و سهم توکن پاسخ را شامل نمی‌شود. بنابراین این عدد هنوز بودجهٔ نهایی Context نیست.

**معیار موفقیت:** برای هر ۲۶۶۴ بسته شمارش ثبت شود؛ Test بررسی نشود و بودجه اعمال نشود.

</div>

In [52]:
# NB18-C52 — Measure Development Evidence Text Tokens Locally

assert nb18_tokenizer is not None
assert len(nb18_development_packets) == 2664
assert nb18_tokenizer_fingerprint is not None

nb18_tokenizer_snapshot_revisions = []

try:
    from huggingface_hub import try_to_load_from_cache

    for filename in (
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "config.json",
        "added_tokens.json",
    ):
        cached_path = try_to_load_from_cache(
            NB18_TOKENIZER_REPOSITORY,
            filename,
            revision="main",
        )

        if isinstance(cached_path, str) and Path(cached_path).is_file():
            resolved_path = Path(cached_path).resolve()
            path_parts = resolved_path.parts

            if "snapshots" in path_parts:
                snapshot_index = path_parts.index("snapshots")
                if snapshot_index + 1 < len(path_parts):
                    nb18_tokenizer_snapshot_revisions.append(
                        path_parts[snapshot_index + 1]
                    )
except ImportError:
    pass

nb18_tokenizer_snapshot_revisions = sorted(
    set(nb18_tokenizer_snapshot_revisions)
)

if len(nb18_tokenizer_snapshot_revisions) == 1:
    nb18_tokenizer_revision = nb18_tokenizer_snapshot_revisions[0]
    nb18_tokenizer_revision_status = "PINNED_FROM_LOCAL_CACHE"
elif len(nb18_tokenizer_snapshot_revisions) > 1:
    nb18_tokenizer_revision = None
    nb18_tokenizer_revision_status = "MIXED_CACHE_REVISIONS"
else:
    nb18_tokenizer_revision = None
    nb18_tokenizer_revision_status = "CONTENT_FINGERPRINT_ONLY"

nb18_development_evidence_token_counts = []

for packet in nb18_development_packets:
    assert packet.get("split") == "development"

    evidence_items = packet.get("evidence_items", [])
    assert len(evidence_items) == 10

    evidence_text = "\n\n".join(
        str(item["text"])
        for item in evidence_items
    )

    token_count = len(
        nb18_tokenizer.encode(
            evidence_text,
            add_special_tokens=False,
        )
    )
    nb18_development_evidence_token_counts.append(token_count)

assert len(nb18_development_evidence_token_counts) == 2664

sorted_packet_token_counts = sorted(
    nb18_development_evidence_token_counts
)

def nb18_empirical_quantile(values, proportion):
    index = int((len(values) - 1) * proportion)
    return values[index]

nb18_development_token_measurement = {
    "status": "PASS",
    "model": NB18_TARGET_MODEL,
    "tokenizer_repository": NB18_TOKENIZER_REPOSITORY,
    "tokenizer_revision": nb18_tokenizer_revision,
    "tokenizer_revision_status": nb18_tokenizer_revision_status,
    "tokenizer_fingerprint": nb18_tokenizer_fingerprint,
    "development_packet_count": len(
        nb18_development_evidence_token_counts
    ),
    "evidence_items_per_packet": 10,
    "measured_scope": (
        "Evidence text only; excludes prompt template, question, "
        "metadata serialization, and generated answer reserve."
    ),
    "evidence_text_token_count_per_packet": {
        "min": sorted_packet_token_counts[0],
        "p50": nb18_empirical_quantile(
            sorted_packet_token_counts, 0.50
        ),
        "p90": nb18_empirical_quantile(
            sorted_packet_token_counts, 0.90
        ),
        "p95": nb18_empirical_quantile(
            sorted_packet_token_counts, 0.95
        ),
        "max": sorted_packet_token_counts[-1],
    },
    "context_budget_selected": False,
    "production_budget_applied": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "arvan_endpoint_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(
    nb18_development_token_measurement,
    ensure_ascii=False,
    indent=2,
))

{
  "status": "PASS",
  "model": "Qwen3-30B-A3B",
  "tokenizer_repository": "Qwen/Qwen3-30B-A3B",
  "tokenizer_revision": null,
  "tokenizer_revision_status": "CONTENT_FINGERPRINT_ONLY",
  "tokenizer_fingerprint": {
    "backend_sha256": "a63080b4b30dc861fe1b0e420c53a241304dc81aff3d9bb315e6e33e2a1ab794",
    "special_tokens_sha256": "94f02d4e8b7a6569bef78a7ff2c3b40744a0fd54e603a9c5a5bdba4d05256a10",
    "tokenizer_class": "Qwen2TokenizerFast",
    "is_fast": true
  },
  "development_packet_count": 2664,
  "evidence_items_per_packet": 10,
  "measured_scope": "Evidence text only; excludes prompt template, question, metadata serialization, and generated answer reserve.",
  "evidence_text_token_count_per_packet": {
    "min": 1257,
    "p50": 6548,
    "p90": 7039,
    "p95": 7110,
    "max": 7371
  },
  "context_budget_selected": false,
  "production_budget_applied": false,
  "test_evidence_or_metrics_used": false,
  "model_api_called": false,
  "arvan_endpoint_called": false,
  "files_wr

<div dir="rtl" align="right">

## شمارش توکن هر شاهد برای بسته‌بندی Development

این سلول تعداد توکن متن هر Evidence Item را جداگانه محاسبه می‌کند و شمارش را کنار Candidate و رتبه‌های موجود نگه می‌دارد. هیچ شاهدی حذف یا جابه‌جا نمی‌شود.

این اندازه‌ها فقط برای طراحی و ارزیابی سیاست روی Development هستند؛ بودجهٔ نهایی انتخاب یا اعمال نمی‌شود.

**معیار موفقیت:** شمارش هر ۲۶٬۶۴۰ شاهد ثبت شود، بدون استفاده از Test یا تغییر ترتیب رتبه‌ها.

</div>

<div dir="rtl" align="right">

## شمارش توکن هر شاهد برای بسته‌بندی Development

این سلول تعداد توکن متن هر Evidence Item را جداگانه محاسبه می‌کند و شمارش را کنار Candidate و رتبه‌های موجود نگه می‌دارد. هیچ شاهدی حذف یا جابه‌جا نمی‌شود.

این اندازه‌ها فقط برای طراحی و ارزیابی سیاست روی Development هستند؛ بودجهٔ نهایی انتخاب یا اعمال نمی‌شود.

**معیار موفقیت:** شمارش هر ۲۶٬۶۴۰ شاهد ثبت شود، بدون استفاده از Test یا تغییر ترتیب رتبه‌ها.

</div>

In [53]:
# NB18-C53 — Measure Tokens for Each Development Evidence Item

assert nb18_tokenizer is not None
assert len(nb18_development_packets) == 2664

nb18_flattened_evidence_items = []
nb18_evidence_texts = []

for packet in nb18_development_packets:
    assert packet.get("split") == "development"

    for item in packet.get("evidence_items", []):
        text_value = str(item["text"])
        assert text_value.strip()

        nb18_flattened_evidence_items.append({
            "candidate_id": str(packet["candidate_id"]),
            "evidence_id": str(item["evidence_id"]),
            "chunk_id": str(item["chunk_id"]),
            "reranker_rank": int(item["reranker_rank"]),
            "selection_priority_rank": int(
                item["selection_priority_rank"]
            ),
        })
        nb18_evidence_texts.append(text_value)

assert len(nb18_flattened_evidence_items) == 26640
assert len(nb18_evidence_texts) == 26640

nb18_encodings = nb18_tokenizer.backend_tokenizer.encode_batch(
    nb18_evidence_texts,
    add_special_tokens=False,
)

assert len(nb18_encodings) == len(nb18_flattened_evidence_items)

nb18_development_item_token_measurements = []
nb18_item_token_counts_by_candidate = {}

for item_record, encoding in zip(
    nb18_flattened_evidence_items,
    nb18_encodings,
):
    token_count = len(encoding.ids)

    assert token_count > 0

    measurement = {
        **item_record,
        "evidence_text_token_count": token_count,
    }
    nb18_development_item_token_measurements.append(measurement)

    nb18_item_token_counts_by_candidate.setdefault(
        item_record["candidate_id"],
        [],
    ).append(measurement)

assert len(nb18_development_item_token_measurements) == 26640
assert len(nb18_item_token_counts_by_candidate) == 2664

nb18_item_token_counts_sorted = sorted(
    item["evidence_text_token_count"]
    for item in nb18_development_item_token_measurements
)

def nb18_item_empirical_quantile(values, proportion):
    index = int((len(values) - 1) * proportion)
    return values[index]

nb18_item_token_measurement_audit = {
    "status": "PASS",
    "model": NB18_TARGET_MODEL,
    "tokenizer_fingerprint": nb18_tokenizer_fingerprint,
    "development_candidate_count": len(
        nb18_item_token_counts_by_candidate
    ),
    "evidence_item_count": len(
        nb18_development_item_token_measurements
    ),
    "per_item_evidence_text_token_count": {
        "min": nb18_item_token_counts_sorted[0],
        "p50": nb18_item_empirical_quantile(
            nb18_item_token_counts_sorted, 0.50
        ),
        "p90": nb18_item_empirical_quantile(
            nb18_item_token_counts_sorted, 0.90
        ),
        "p95": nb18_item_empirical_quantile(
            nb18_item_token_counts_sorted, 0.95
        ),
        "max": nb18_item_token_counts_sorted[-1],
    },
    "reranker_ranks_modified": False,
    "context_budget_selected": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(
    nb18_item_token_measurement_audit,
    ensure_ascii=False,
    indent=2,
))

{
  "status": "PASS",
  "model": "Qwen3-30B-A3B",
  "tokenizer_fingerprint": {
    "backend_sha256": "a63080b4b30dc861fe1b0e420c53a241304dc81aff3d9bb315e6e33e2a1ab794",
    "special_tokens_sha256": "94f02d4e8b7a6569bef78a7ff2c3b40744a0fd54e603a9c5a5bdba4d05256a10",
    "tokenizer_class": "Qwen2TokenizerFast",
    "is_fast": true
  },
  "development_candidate_count": 2664,
  "evidence_item_count": 26640,
  "per_item_evidence_text_token_count": {
    "min": 7,
    "p50": 696,
    "p90": 727,
    "p95": 736,
    "max": 826
  },
  "reranker_ranks_modified": false,
  "context_budget_selected": false,
  "test_evidence_or_metrics_used": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

## استخراج بودجه‌های آزمایشی از اندازه‌های Development

این سلول چند مقدار نامزد را از توزیع توکن متن شواهد بسته‌های Development استخراج می‌کند. مقدارها از دادهٔ Development می‌آیند و برای آزمایش‌اند؛ بودجهٔ نهایی محسوب نمی‌شوند.

این محاسبه قالب Prompt، پرسش، metadata، پاسخ مدل یا محدودیت Endpoint آروان را در نظر نمی‌گیرد.

**معیار موفقیت:** فهرست نامزدها فقط از اندازه‌های مشاهده‌شدهٔ Development ساخته شود و وضعیت انتخاب نهایی همچنان `false` بماند.

</div>

In [54]:
# NB18-C54 — Derive Development-Based Budget Experiment Candidates

assert (
    nb18_development_token_measurement["status"] == "PASS"
)
assert len(nb18_development_evidence_token_counts) == 2664

sorted_context_token_counts = sorted(
    nb18_development_evidence_token_counts
)

def nb18_context_budget_quantile(values, proportion):
    index = int((len(values) - 1) * proportion)
    return values[index]

quantile_specs = [
    ("p50", 0.50),
    ("p75", 0.75),
    ("p90", 0.90),
    ("p95", 0.95),
    ("p100", 1.00),
]

candidate_by_budget = {}

for label, proportion in quantile_specs:
    budget_value = nb18_context_budget_quantile(
        sorted_context_token_counts,
        proportion,
    )
    candidate_by_budget.setdefault(
        budget_value,
        [],
    ).append(label)

nb18_development_budget_candidates = [
    {
        "evidence_text_budget_tokens": budget_value,
        "development_quantiles": labels,
    }
    for budget_value, labels in sorted(candidate_by_budget.items())
]

assert nb18_development_budget_candidates
assert all(
    candidate["evidence_text_budget_tokens"]
    <= NB18_MODEL_ARCHITECTURAL_LIMIT
    for candidate in nb18_development_budget_candidates
)

nb18_budget_candidate_audit = {
    "status": "DEVELOPMENT_DERIVED_CANDIDATES_NOT_SELECTED",
    "source_packet_count": len(sorted_context_token_counts),
    "candidate_budgets": nb18_development_budget_candidates,
    "basis": (
        "Empirical quantiles of Qwen-tokenized Development evidence text"
    ),
    "scope_excludes": [
        "prompt template",
        "question",
        "metadata serialization",
        "generated answer reserve",
        "provider endpoint limit",
    ],
    "model_architectural_limit_tokens": (
        NB18_MODEL_ARCHITECTURAL_LIMIT
    ),
    "provider_endpoint_limit_tokens": None,
    "budget_selected": False,
    "production_budget_applied": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(
    nb18_budget_candidate_audit,
    ensure_ascii=False,
    indent=2,
))

{
  "status": "DEVELOPMENT_DERIVED_CANDIDATES_NOT_SELECTED",
  "source_packet_count": 2664,
  "candidate_budgets": [
    {
      "evidence_text_budget_tokens": 6548,
      "development_quantiles": [
        "p50"
      ]
    },
    {
      "evidence_text_budget_tokens": 6848,
      "development_quantiles": [
        "p75"
      ]
    },
    {
      "evidence_text_budget_tokens": 7039,
      "development_quantiles": [
        "p90"
      ]
    },
    {
      "evidence_text_budget_tokens": 7110,
      "development_quantiles": [
        "p95"
      ]
    },
    {
      "evidence_text_budget_tokens": 7371,
      "development_quantiles": [
        "p100"
      ]
    }
  ],
  "basis": "Empirical quantiles of Qwen-tokenized Development evidence text",
  "scope_excludes": [
    "prompt template",
    "question",
    "metadata serialization",
    "generated answer reserve",
    "provider endpoint limit"
  ],
  "model_architectural_limit_tokens": 40960,
  "provider_endpoint_limit_tokens": null,


<div dir="rtl" align="right">

## آزمایش مقدماتی بودجه و پوشش شواهد روی Development

این آزمایش بودجه‌های نامزد را از توزیع توکن تکه‌های Development استخراج می‌کند و دو ترتیب انتخاب را مقایسه می‌کند: ترتیب اصلی Reranker و ترتیب اولویت تنوع منبع.

پوشش با نگاشت شواهد طلایی Development سنجیده می‌شود. هزینهٔ این غربال اولیه جمع توکن‌های متن تکه‌هاست و قالب نهایی Prompt، جداکننده‌ها، metadata و سهم پاسخ را شامل نمی‌شود؛ بنابراین نتیجه برای انتخاب نهایی یا انتشار کافی نیست.

**معیار موفقیت:** هر ترکیب بودجه و ترتیب فقط روی Development ارزیابی شود، رتبه‌های Reranker تغییر نکنند، و خروجی به‌عنوان آزمایش مقدماتی باقی بماند.

</div>

In [55]:
# NB18-C55 — Evaluate Development-Only Budget and Order Candidates

assert "development_gold_mapping" in globals()
assert len(nb18_item_token_counts_by_candidate) == 2664
assert len(development_gold_mapping) == 2664
assert len(nb18_development_budget_candidates) > 0

nb18_additive_packet_costs = {
    candidate_id: sum(
        item["evidence_text_token_count"]
        for item in item_measurements
    )
    for candidate_id, item_measurements
    in nb18_item_token_counts_by_candidate.items()
}

assert set(nb18_additive_packet_costs) == set(development_gold_mapping)

sorted_additive_packet_costs = sorted(
    nb18_additive_packet_costs.values()
)

budget_quantile_specs = [
    ("p50", 0.50),
    ("p75", 0.75),
    ("p90", 0.90),
    ("p95", 0.95),
    ("p100", 1.00),
]

nb18_packing_budget_candidates_by_value = {}

for label, proportion in budget_quantile_specs:
    index = int((len(sorted_additive_packet_costs) - 1) * proportion)
    budget_value = sorted_additive_packet_costs[index]
    nb18_packing_budget_candidates_by_value.setdefault(
        budget_value,
        [],
    ).append(label)

nb18_packing_budget_candidates = [
    {
        "evidence_text_budget_tokens": budget_value,
        "development_quantiles": labels,
    }
    for budget_value, labels
    in sorted(nb18_packing_budget_candidates_by_value.items())
]

strategy_fields = {
    "reranker_rank_order": "reranker_rank",
    "source_priority_order": "selection_priority_rank",
}

nb18_budget_policy_experiment_results = []

for budget_record in nb18_packing_budget_candidates:
    budget = budget_record["evidence_text_budget_tokens"]

    for strategy_name, order_field in strategy_fields.items():
        query_recalls = []
        queries_with_any_gold_evidence = 0
        matched_gold_reference_count = 0
        total_gold_reference_count = 0
        selected_item_count_total = 0
        selected_token_estimate_total = 0
        selected_source_type_counts = {}

        for candidate_id, item_measurements in (
            nb18_item_token_counts_by_candidate.items()
        ):
            ordered_items = sorted(
                item_measurements,
                key=lambda item: item[order_field],
            )

            selected_items = []
            estimated_cost = 0

            for item in ordered_items:
                item_cost = item["evidence_text_token_count"]

                if estimated_cost + item_cost <= budget:
                    selected_items.append(item)
                    estimated_cost += item_cost

            selected_chunk_ids = {
                item["chunk_id"]
                for item in selected_items
            }
            gold_chunk_ids = set(
                development_gold_mapping[candidate_id]
            )

            matched_count = len(selected_chunk_ids & gold_chunk_ids)
            gold_count = len(gold_chunk_ids)

            assert gold_count > 0

            matched_gold_reference_count += matched_count
            total_gold_reference_count += gold_count
            query_recalls.append(matched_count / gold_count)

            if matched_count > 0:
                queries_with_any_gold_evidence += 1

            selected_item_count_total += len(selected_items)
            selected_token_estimate_total += estimated_cost

            for item in selected_items:
                source_type = str(
                    chunk_record_by_id[item["chunk_id"]]["source_type"]
                )
                selected_source_type_counts.setdefault(
                    source_type,
                    0,
                )
                selected_source_type_counts[source_type] += 1

        nb18_budget_policy_experiment_results.append({
            "budget_candidate_tokens": budget,
            "budget_basis": budget_record["development_quantiles"],
            "selection_strategy": strategy_name,
            "query_count": len(query_recalls),
            "macro_gold_chunk_recall": round(
                sum(query_recalls) / len(query_recalls),
                6,
            ),
            "micro_gold_chunk_recall": round(
                matched_gold_reference_count
                / total_gold_reference_count,
                6,
            ),
            "queries_with_any_gold_evidence": (
                queries_with_any_gold_evidence
            ),
            "query_gold_hit_rate": round(
                queries_with_any_gold_evidence / len(query_recalls),
                6,
            ),
            "mean_selected_items_per_query": round(
                selected_item_count_total / len(query_recalls),
                4,
            ),
            "mean_additive_evidence_token_estimate": round(
                selected_token_estimate_total / len(query_recalls),
                2,
            ),
            "selected_source_type_slot_counts": dict(
                sorted(selected_source_type_counts.items())
            ),
        })

assert len(nb18_budget_policy_experiment_results) == (
    len(nb18_packing_budget_candidates) * len(strategy_fields)
)

nb18_budget_policy_experiment_audit = {
    "status": "DEVELOPMENT_EXPERIMENT_COMPLETE",
    "development_query_count": 2664,
    "budget_candidates": nb18_packing_budget_candidates,
    "selection_strategies": list(strategy_fields),
    "results": nb18_budget_policy_experiment_results,
    "token_cost_method": (
        "Sum of independently tokenized evidence texts; excludes "
        "separators, prompt, question, metadata, and answer reserve."
    ),
    "reranker_ranks_modified": False,
    "budget_selected": False,
    "configuration_frozen": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(
    nb18_budget_policy_experiment_audit,
    ensure_ascii=False,
    indent=2,
))

{
  "status": "DEVELOPMENT_EXPERIMENT_COMPLETE",
  "development_query_count": 2664,
  "budget_candidates": [
    {
      "evidence_text_budget_tokens": 6542,
      "development_quantiles": [
        "p50"
      ]
    },
    {
      "evidence_text_budget_tokens": 6843,
      "development_quantiles": [
        "p75"
      ]
    },
    {
      "evidence_text_budget_tokens": 7032,
      "development_quantiles": [
        "p90"
      ]
    },
    {
      "evidence_text_budget_tokens": 7103,
      "development_quantiles": [
        "p95"
      ]
    },
    {
      "evidence_text_budget_tokens": 7363,
      "development_quantiles": [
        "p100"
      ]
    }
  ],
  "selection_strategies": [
    "reranker_rank_order",
    "source_priority_order"
  ],
  "results": [
    {
      "budget_candidate_tokens": 6542,
      "budget_basis": [
        "p50"
      ],
      "selection_strategy": "reranker_rank_order",
      "query_count": 2664,
      "macro_gold_chunk_recall": 0.662862,
      "micro_go

<div dir="rtl" align="right">

## بررسی مرز کارایی بودجه‌ها و ترتیب انتخاب

نتایج آزمایش Development مقایسه می‌شوند تا مشخص شود کدام نامزدها با بودجهٔ بیشتر، پوشش یا نرخ یافتن شواهد بهتری نمی‌دهند. این بررسی آستانهٔ جدیدی تعریف نمی‌کند و Configuration را Freeze نمی‌کند.

**معیار موفقیت:** مرز نامزدهای مغلوب و نامغلوب برای هر ترتیب انتخاب گزارش شود.

</div>

In [56]:
# NB18-C56 — Identify Pareto-Dominated Development Budget Candidates

assert "nb18_budget_policy_experiment_results" in globals()
assert nb18_budget_policy_experiment_results

nb18_pareto_frontier_by_strategy = {}

for strategy_name in sorted({
    result["selection_strategy"]
    for result in nb18_budget_policy_experiment_results
}):
    strategy_results = [
        result
        for result in nb18_budget_policy_experiment_results
        if result["selection_strategy"] == strategy_name
    ]

    frontier = []
    dominated = []

    for candidate in strategy_results:
        candidate_budget = candidate["budget_candidate_tokens"]
        candidate_recall = candidate["macro_gold_chunk_recall"]
        candidate_hit_rate = candidate["query_gold_hit_rate"]

        is_dominated = False

        for alternative in strategy_results:
            if alternative is candidate:
                continue

            no_more_budget = (
                alternative["budget_candidate_tokens"]
                <= candidate_budget
            )
            no_worse_recall = (
                alternative["macro_gold_chunk_recall"]
                >= candidate_recall
            )
            no_worse_hit_rate = (
                alternative["query_gold_hit_rate"]
                >= candidate_hit_rate
            )
            strictly_better_in_one_dimension = any([
                alternative["budget_candidate_tokens"]
                < candidate_budget,
                alternative["macro_gold_chunk_recall"]
                > candidate_recall,
                alternative["query_gold_hit_rate"]
                > candidate_hit_rate,
            ])

            if (
                no_more_budget
                and no_worse_recall
                and no_worse_hit_rate
                and strictly_better_in_one_dimension
            ):
                is_dominated = True
                break

        summary = {
            "budget_candidate_tokens": candidate_budget,
            "budget_basis": candidate["budget_basis"],
            "macro_gold_chunk_recall": (
                candidate["macro_gold_chunk_recall"]
            ),
            "query_gold_hit_rate": candidate["query_gold_hit_rate"],
            "mean_selected_items_per_query": (
                candidate["mean_selected_items_per_query"]
            ),
            "mean_additive_evidence_token_estimate": (
                candidate["mean_additive_evidence_token_estimate"]
            ),
        }

        if is_dominated:
            dominated.append(summary)
        else:
            frontier.append(summary)

    nb18_pareto_frontier_by_strategy[strategy_name] = {
        "non_dominated_candidates": frontier,
        "dominated_candidates": dominated,
    }

nb18_order_comparison_by_budget = []

candidate_budget_values = sorted({
    result["budget_candidate_tokens"]
    for result in nb18_budget_policy_experiment_results
})

for budget_value in candidate_budget_values:
    same_budget_results = [
        result
        for result in nb18_budget_policy_experiment_results
        if result["budget_candidate_tokens"] == budget_value
    ]

    result_by_strategy = {
        result["selection_strategy"]: result
        for result in same_budget_results
    }

    rank_result = result_by_strategy["reranker_rank_order"]
    source_result = result_by_strategy["source_priority_order"]

    nb18_order_comparison_by_budget.append({
        "budget_candidate_tokens": budget_value,
        "source_priority_minus_rank_order_macro_recall": round(
            source_result["macro_gold_chunk_recall"]
            - rank_result["macro_gold_chunk_recall"],
            6,
        ),
        "source_priority_minus_rank_order_query_hit_rate": round(
            source_result["query_gold_hit_rate"]
            - rank_result["query_gold_hit_rate"],
            6,
        ),
        "source_priority_and_rank_order_same_mean_item_count": (
            source_result["mean_selected_items_per_query"]
            == rank_result["mean_selected_items_per_query"]
        ),
    })

nb18_pareto_audit = {
    "status": "PASS",
    "comparison_scope": "Development only",
    "strategies": nb18_pareto_frontier_by_strategy,
    "source_priority_vs_rank_order": (
        nb18_order_comparison_by_budget
    ),
    "acceptance_threshold_introduced": False,
    "budget_selected": False,
    "configuration_frozen": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(
    nb18_pareto_audit,
    ensure_ascii=False,
    indent=2,
))

{
  "status": "PASS",
  "comparison_scope": "Development only",
  "strategies": {
    "reranker_rank_order": {
      "non_dominated_candidates": [
        {
          "budget_candidate_tokens": 6542,
          "budget_basis": [
            "p50"
          ],
          "macro_gold_chunk_recall": 0.662862,
          "query_gold_hit_rate": 0.846096,
          "mean_selected_items_per_query": 9.4962,
          "mean_additive_evidence_token_estimate": 6092.79
        },
        {
          "budget_candidate_tokens": 6843,
          "budget_basis": [
            "p75"
          ],
          "macro_gold_chunk_recall": 0.66519,
          "query_gold_hit_rate": 0.847973,
          "mean_selected_items_per_query": 9.7511,
          "mean_additive_evidence_token_estimate": 6265.4
        },
        {
          "budget_candidate_tokens": 7032,
          "budget_basis": [
            "p90"
          ],
          "macro_gold_chunk_recall": 0.666495,
          "query_gold_hit_rate": 0.849099,
       

<div dir="rtl" align="right">

## بررسی خواندنیِ محدودیت‌های ورودی نوت‌بوک ۱۹

اگر نوت‌بوک ۱۹ در مخزن موجود باشد، کد و Markdown آن برای یافتن قرارداد Prompt، محدودیت ورودی و سهم پاسخ بررسی می‌شود. خروجی‌های ذخیره‌شده و مقادیر محرمانه خوانده نمی‌شوند.

این بررسی هیچ مدل یا APIای را اجرا نمی‌کند و هیچ‌یک از تنظیمات NB18 را تغییر نمی‌دهد.

**معیار موفقیت:** وجود یا نبود نوت‌بوک ۱۹ و هر قید مرتبط با بودجه به‌صورت امن گزارش شود.

</div>

In [57]:
# NB18-C57 — Inspect Notebook 19 Token Constraints Read-Only

assert "project_root" in globals()

nb19_path = (
    project_root
    / "notebooks"
    / "phase2"
    / "19_generation_and_citation.ipynb"
)

nb19_sensitive_pattern = re.compile(
    r"api[_-]?key|secret|authorization|bearer|gateway/",
    re.IGNORECASE,
)
nb19_relevant_pattern = re.compile(
    r"token|context|budget|max[_ -]?(input|new)?[_ -]?tokens|"
    r"max[_ -]?length|prompt|generation|model|truncate|reserve",
    re.IGNORECASE,
)

nb19_inspection = {
    "status": "NOT_FOUND",
    "path": str(nb19_path.relative_to(project_root)),
    "exists": nb19_path.is_file(),
    "source_cell_count": 0,
    "relevant_source_matches": [],
    "saved_outputs_read": False,
    "sensitive_source_lines_printed": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

if nb19_path.is_file():
    nb19_data = json.loads(
        nb19_path.read_text(encoding="utf-8")
    )

    source_cells = [
        cell
        for cell in nb19_data.get("cells", [])
        if cell.get("cell_type") in {"code", "markdown"}
    ]
    safe_matches = []

    for cell_position, cell in enumerate(source_cells, start=1):
        source = cell.get("source", [])
        if isinstance(source, list):
            source_text = "".join(source)
        elif isinstance(source, str):
            source_text = source
        else:
            source_text = ""

        for line_number, line in enumerate(
            source_text.splitlines(),
            start=1,
        ):
            if not nb19_relevant_pattern.search(line):
                continue

            if nb19_sensitive_pattern.search(line):
                rendered_line = "[REDACTED SENSITIVE SOURCE LINE]"
            else:
                rendered_line = line.strip()[:300]

            safe_matches.append({
                "source_cell_position": cell_position,
                "line_in_cell": line_number,
                "text": rendered_line,
            })

            if len(safe_matches) >= 100:
                break

        if len(safe_matches) >= 100:
            break

    nb19_inspection.update({
        "status": "INSPECTED",
        "source_cell_count": len(source_cells),
        "relevant_source_matches": safe_matches,
        "results_truncated": len(safe_matches) >= 100,
    })

print(json.dumps(
    nb19_inspection,
    ensure_ascii=False,
    indent=2,
))

{
  "status": "NOT_FOUND",
  "path": "notebooks/phase2/19_generation_and_citation.ipynb",
  "exists": false,
  "source_cell_count": 0,
  "relevant_source_matches": [],
  "saved_outputs_read": false,
  "sensitive_source_lines_printed": false,
  "test_evidence_or_metrics_used": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

## پیش‌نویس تصمیم بودجه برای بازبینی

این سلول از نتایج Development یک پیشنهاد موقت می‌سازد: سقف متن Evidence برابر بیشینهٔ مشاهده‌شده باشد تا هر ده شاهد حفظ شوند. ترتیب `source_priority_order` نیز در حد پیشنهاد ثبت می‌شود، چون در سقف کامل با ترتیب Reranker همان پوشش را دارد و تنوع را در پیشوندهای کوتاه‌تر بهتر می‌کند.

این سقف فقط متن Evidence را پوشش می‌دهد؛ Prompt، metadata، پاسخ و محدودیت واقعی Endpoint آروان هنوز در آن لحاظ نشده‌اند. بنابراین Configuration Freeze نمی‌شود.

**معیار موفقیت:** پیشنهاد و دلیل آن از دادهٔ Development استخراج و محدودیت‌های باز صریح ثبت شوند.

</div>

In [58]:
# NB18-C58 — Draft the Development-Based Budget Decision

assert nb18_budget_policy_experiment_results
assert nb18_development_evidence_token_counts
assert nb18_development_budget_candidates

exact_full_text_budget = max(
    nb18_development_evidence_token_counts
)

source_priority_results = [
    result
    for result in nb18_budget_policy_experiment_results
    if result["selection_strategy"] == "source_priority_order"
]

p95_result = next(
    result
    for result in source_priority_results
    if result["budget_basis"] == ["p95"]
)

p100_result = next(
    result
    for result in source_priority_results
    if result["budget_basis"] == ["p100"]
)

mean_token_increase = (
    p100_result["mean_additive_evidence_token_estimate"]
    - p95_result["mean_additive_evidence_token_estimate"]
)

mean_token_increase_percent = round(
    100
    * mean_token_increase
    / p95_result["mean_additive_evidence_token_estimate"],
    4,
)

nb18_budget_decision_draft = {
    "status": "PROPOSAL_READY_FOR_REVIEW",
    "selection_policy_proposal": "source_priority_order",
    "evidence_text_budget_proposal_tokens": exact_full_text_budget,
    "budget_basis": (
        "Maximum Qwen-tokenized joined evidence text across "
        "2664 Development packets"
    ),
    "development_comparison": {
        "p95_budget_candidate_tokens": (
            p95_result["budget_candidate_tokens"]
        ),
        "p100_budget_candidate_tokens": (
            p100_result["budget_candidate_tokens"]
        ),
        "p95_macro_gold_chunk_recall": (
            p95_result["macro_gold_chunk_recall"]
        ),
        "p100_macro_gold_chunk_recall": (
            p100_result["macro_gold_chunk_recall"]
        ),
        "p95_query_gold_hit_rate": (
            p95_result["query_gold_hit_rate"]
        ),
        "p100_query_gold_hit_rate": (
            p100_result["query_gold_hit_rate"]
        ),
        "additional_queries_with_gold_hit_at_p100": (
            p100_result["queries_with_any_gold_evidence"]
            - p95_result["queries_with_any_gold_evidence"]
        ),
        "mean_additive_token_increase_p95_to_p100": round(
            mean_token_increase,
            2,
        ),
        "mean_additive_token_increase_percent": (
            mean_token_increase_percent
        ),
    },
    "scope": "Evidence text only",
    "provider_endpoint_limit_tokens": None,
    "prompt_and_answer_reserve_tokens": None,
    "configuration_frozen": False,
    "test_unlocked": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(
    nb18_budget_decision_draft,
    ensure_ascii=False,
    indent=2,
))

{
  "status": "PROPOSAL_READY_FOR_REVIEW",
  "selection_policy_proposal": "source_priority_order",
  "evidence_text_budget_proposal_tokens": 7371,
  "budget_basis": "Maximum Qwen-tokenized joined evidence text across 2664 Development packets",
  "development_comparison": {
    "p95_budget_candidate_tokens": 7103,
    "p100_budget_candidate_tokens": 7363,
    "p95_macro_gold_chunk_recall": 0.667111,
    "p100_macro_gold_chunk_recall": 0.667674,
    "p95_query_gold_hit_rate": 0.849099,
    "p100_query_gold_hit_rate": 0.84985,
    "additional_queries_with_gold_hit_at_p100": 2,
    "mean_additive_token_increase_p95_to_p100": 36.11,
    "mean_additive_token_increase_percent": 0.5639
  },
  "scope": "Evidence text only",
  "provider_endpoint_limit_tokens": null,
  "prompt_and_answer_reserve_tokens": null,
  "configuration_frozen": false,
  "test_unlocked": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

## ثبت پیش‌نویس ADR سیاست Context Builder

این سلول پیشنهاد بودجهٔ مبتنی بر بیشینهٔ متن شواهد Development و ترتیب انتخاب منبع‌محور را در قالب ADR ثبت می‌کند. پیشنهاد از نتایج Development به دست آمده است، اما تا تأیید تصمیم و روشن‌شدن حد ورودی Endpoint آروان Freeze نمی‌شود.

Test همچنان قفل است و هیچ فایل ADR یا Configuration روی دیسک نوشته نمی‌شود.

**معیار موفقیت:** گزینهٔ پیشنهادی، شواهد پشتیبان و محدودیت‌های باز ثبت شوند؛ وضعیت نهایی Freeze نشده بماند.

</div>

In [59]:
# NB18-C59 — Build the In-Memory Context Policy ADR Draft

assert nb18_budget_decision_draft["status"] == (
    "PROPOSAL_READY_FOR_REVIEW"
)
assert nb18_budget_decision_draft["evidence_text_budget_proposal_tokens"] == (
    max(nb18_development_evidence_token_counts)
)
assert nb18_budget_decision_draft["test_unlocked"] is False

nb18_context_policy_adr_draft = {
    "artifact_type": "NB18_CONTEXT_POLICY_ADR_DRAFT",
    "schema_version": "1.0",
    "decision_status": "PROPOSED_NOT_FROZEN",
    "decision_scope": "Development evidence-text packing",
    "recommended_policy": {
        "selection_order": "source_priority_order",
        "evidence_text_budget_tokens": (
            nb18_budget_decision_draft[
                "evidence_text_budget_proposal_tokens"
            ]
        ),
        "item_selection": (
            "Whole evidence items; preserve selected order; skip items "
            "that do not fit."
        ),
        "reranker_order_and_scores_modified": False,
    },
    "development_evidence": (
        nb18_budget_decision_draft["development_comparison"]
    ),
    "rationale": [
        "The p100 candidate preserves all ten Development evidence items.",
        "The p100 candidate has the highest observed Development gold recall.",
        "The p95-to-p100 mean additive token increase is small relative to p95.",
        "At p100, source-priority and reranker-order metrics are equal.",
        "Source-priority ordering preserves the separately recorded reranker ranks.",
    ],
    "unresolved_constraints": [
        "Arvan endpoint context limit is not confirmed.",
        "Prompt, metadata serialization, and answer reserve are not included.",
        "Topic mapping coverage is insufficient for topic-diversity selection.",
        "Direct parent chunk links are unavailable in the current candidate set.",
        "Semantic conflict assessment has not been performed.",
    ],
    "provider_endpoint_limit_tokens": None,
    "prompt_and_answer_reserve_tokens": None,
    "development_only": True,
    "test_unlocked": False,
    "configuration_frozen": False,
    "adr_written_to_disk": False,
    "files_written": 0,
    "paid_api_calls": 0,
    "model_api_called": False,
}

nb18_context_configuration_candidate = {
    "configuration_id": "nb18_context_builder_development_candidate_v1",
    "target_model": NB18_TARGET_MODEL,
    "tokenizer_repository": NB18_TOKENIZER_REPOSITORY,
    "tokenizer_fingerprint": nb18_tokenizer_fingerprint,
    "evidence_text_budget_tokens": (
        nb18_context_policy_adr_draft["recommended_policy"][
            "evidence_text_budget_tokens"
        ]
    ),
    "selection_order": (
        nb18_context_policy_adr_draft["recommended_policy"][
            "selection_order"
        ]
    ),
    "scope": "Evidence text only",
    "status": "PROPOSED_NOT_FROZEN",
    "provider_endpoint_limit_tokens": None,
    "test_unlocked": False,
    "configuration_frozen": False,
    "written_to_disk": False,
}

assert nb18_context_policy_adr_draft["configuration_frozen"] is False
assert nb18_context_configuration_candidate["configuration_frozen"] is False

print(json.dumps({
    "status": "ADR_DRAFT_READY_NOT_FROZEN",
    "adr_draft": nb18_context_policy_adr_draft,
    "configuration_candidate": nb18_context_configuration_candidate,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "ADR_DRAFT_READY_NOT_FROZEN",
  "adr_draft": {
    "artifact_type": "NB18_CONTEXT_POLICY_ADR_DRAFT",
    "schema_version": "1.0",
    "decision_status": "PROPOSED_NOT_FROZEN",
    "decision_scope": "Development evidence-text packing",
    "recommended_policy": {
      "selection_order": "source_priority_order",
      "evidence_text_budget_tokens": 7371,
      "item_selection": "Whole evidence items; preserve selected order; skip items that do not fit.",
      "reranker_order_and_scores_modified": false
    },
    "development_evidence": {
      "p95_budget_candidate_tokens": 7103,
      "p100_budget_candidate_tokens": 7363,
      "p95_macro_gold_chunk_recall": 0.667111,
      "p100_macro_gold_chunk_recall": 0.667674,
      "p95_query_gold_hit_rate": 0.849099,
      "p100_query_gold_hit_rate": 0.84985,
      "additional_queries_with_gold_hit_at_p100": 2,
      "mean_additive_token_increase_p95_to_p100": 36.11,
      "mean_additive_token_increase_percent": 0.5639
    },
   

<div dir="rtl" align="right">

این سلول، بسته‌بندی آزمایشی Development را با ترتیب اولویت منبع و بودجهٔ پیشنهادی ۷۳۷۱ توکن بررسی می‌کند. هزینهٔ هر مدرک جداگانه شمرده می‌شود؛ اگر مدرکی جا نشود، از آن می‌گذرد و مدارک بعدی را هم بررسی می‌کند.

این عدد هنوز پیشنهاد است و تنظیمات را قفل نمی‌کند. داده‌های Test استفاده نمی‌شوند و هیچ فراخوانی مدل یا نوشتن فایلی انجام نمی‌شود.

</div>

In [60]:
# NB18-C60 — Development-only source-priority packing under the proposed budget

PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS = 7371

assert "nb18_development_packets" in globals(), (
    "Run the packet-construction cells before this Development-only experiment."
)
assert len(nb18_development_packets) == 2664
assert "nb18_tokenizer" in globals()
assert nb18_tokenizer.__class__.__name__ == "Qwen2TokenizerFast"
assert callable(getattr(nb18_tokenizer, "encode", None))

tokenizer_name = "nb18_tokenizer"
nb18_qwen_tokenizer = nb18_tokenizer

nb18_budgeted_development_packets = []
input_item_token_sums = []
selected_item_token_sums = []
skipped_item_count = 0
contexts_with_skipped_items = 0
selected_item_count = 0

for packet in nb18_development_packets:
    assert packet.get("split") == "development"
    evidence_items = packet.get("evidence_items", [])
    assert len(evidence_items) == 10

    priority_items = sorted(
        evidence_items,
        key=lambda item: item["selection_priority_rank"],
    )
    priority_ranks = [
        item["selection_priority_rank"] for item in priority_items
    ]
    assert priority_ranks == list(range(1, 11))

    item_token_counts = [
        len(
            nb18_qwen_tokenizer.encode(
                item["text"],
                add_special_tokens=False,
            )
        )
        for item in priority_items
    ]
    input_item_token_sums.append(sum(item_token_counts))

    selected_items = []
    selected_tokens = 0
    packet_skipped = 0

    for item, item_tokens in zip(priority_items, item_token_counts):
        if (
            selected_tokens + item_tokens
            <= PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS
        ):
            selected_items.append(item)
            selected_tokens += item_tokens
        else:
            packet_skipped += 1

    selected_ids = [item["evidence_id"] for item in selected_items]
    assert len(selected_ids) == len(set(selected_ids))
    assert selected_tokens <= PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS

    selected_ranks = [item["reranker_rank"] for item in selected_items]
    assert all(1 <= rank <= 10 for rank in selected_ranks)

    candidate_packet = dict(packet)
    candidate_packet["evidence_items"] = selected_items
    candidate_packet["context_budget_applied"] = True
    candidate_packet["context_budget_tokens"] = (
        PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS
    )
    candidate_packet["selected_evidence_text_tokens_additive"] = (
        selected_tokens
    )
    candidate_packet["selection_order"] = "source_priority_order"
    nb18_budgeted_development_packets.append(candidate_packet)

    selected_item_token_sums.append(selected_tokens)
    selected_item_count += len(selected_items)
    skipped_item_count += packet_skipped
    contexts_with_skipped_items += packet_skipped > 0

assert len(nb18_budgeted_development_packets) == 2664
assert all(
    packet["split"] == "development"
    for packet in nb18_budgeted_development_packets
)

input_item_token_sums_sorted = sorted(input_item_token_sums)
selected_item_token_sums_sorted = sorted(selected_item_token_sums)

print(json.dumps({
    "status": "DEVELOPMENT_PACKING_EXPERIMENT_COMPLETE",
    "tokenizer_variable": tokenizer_name,
    "tokenizer_class": nb18_qwen_tokenizer.__class__.__name__,
    "development_packet_count": len(nb18_budgeted_development_packets),
    "proposed_budget_tokens": PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS,
    "budget_measurement": (
        "Sum of separately tokenized evidence texts; no special tokens"
    ),
    "input_additive_evidence_tokens": {
        "min": input_item_token_sums_sorted[0],
        "median": input_item_token_sums_sorted[
            (len(input_item_token_sums_sorted) - 1) // 2
        ],
        "max": input_item_token_sums_sorted[-1],
    },
    "selected_additive_evidence_tokens": {
        "min": selected_item_token_sums_sorted[0],
        "median": selected_item_token_sums_sorted[
            (len(selected_item_token_sums_sorted) - 1) // 2
        ],
        "max": selected_item_token_sums_sorted[-1],
    },
    "selected_item_count": selected_item_count,
    "skipped_item_count": skipped_item_count,
    "contexts_with_skipped_items": contexts_with_skipped_items,
    "reranker_ranks_and_scores_modified": False,
    "configuration_frozen": False,
    "test_unlocked": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "DEVELOPMENT_PACKING_EXPERIMENT_COMPLETE",
  "tokenizer_variable": "nb18_tokenizer",
  "tokenizer_class": "Qwen2TokenizerFast",
  "development_packet_count": 2664,
  "proposed_budget_tokens": 7371,
  "budget_measurement": "Sum of separately tokenized evidence texts; no special tokens",
  "input_additive_evidence_tokens": {
    "min": 1248,
    "median": 6542,
    "max": 7363
  },
  "selected_additive_evidence_tokens": {
    "min": 1248,
    "median": 6542,
    "max": 7363
  },
  "selected_item_count": 26640,
  "skipped_item_count": 0,
  "contexts_with_skipped_items": 0,
  "reranker_ranks_and_scores_modified": false,
  "configuration_frozen": false,
  "test_unlocked": false,
  "test_evidence_or_metrics_used": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

این سلول صحت بسته‌بندی آزمایشی C60 را با بسته‌های ورودی مقایسه می‌کند. بررسی می‌کند که همهٔ مدارک و فراداده‌ها حفظ شده باشند و رتبه‌ها و امتیازهای رتبه‌بند تغییر نکرده باشند.

این بررسی فقط روی Development انجام می‌شود و تنظیمات را قفل نمی‌کند.

</div>

In [61]:
# NB18-C61 — Validate Development packing fidelity and preserved reranker data

assert "nb18_budgeted_development_packets" in globals()
assert "nb18_development_packets" in globals()
assert len(nb18_budgeted_development_packets) == 2664
assert len(nb18_development_packets) == 2664

original_by_candidate = {
    packet["candidate_id"]: packet
    for packet in nb18_development_packets
}
packed_by_candidate = {
    packet["candidate_id"]: packet
    for packet in nb18_budgeted_development_packets
}

assert len(original_by_candidate) == 2664
assert len(packed_by_candidate) == 2664
assert set(original_by_candidate) == set(packed_by_candidate)

packet_mismatch_count = 0
item_mismatch_count = 0
rank_mismatch_count = 0
score_mismatch_count = 0
text_or_metadata_mismatch_count = 0
duplicate_item_id_count = 0
budget_violation_count = 0
selected_item_count = 0

for candidate_id, original_packet in original_by_candidate.items():
    packed_packet = packed_by_candidate[candidate_id]
    original_items = original_packet["evidence_items"]
    packed_items = packed_packet["evidence_items"]

    original_by_id = {
        item["evidence_id"]: item
        for item in original_items
    }
    packed_by_id = {
        item["evidence_id"]: item
        for item in packed_items
    }

    if len(packed_by_id) != len(packed_items):
        duplicate_item_id_count += 1

    if set(original_by_id) != set(packed_by_id):
        packet_mismatch_count += 1
        continue

    selected_item_count += len(packed_items)

    priority_ranks = [
        item["selection_priority_rank"] for item in packed_items
    ]
    if priority_ranks != sorted(priority_ranks):
        packet_mismatch_count += 1

    for evidence_id, original_item in original_by_id.items():
        packed_item = packed_by_id[evidence_id]

        if original_item["reranker_rank"] != packed_item["reranker_rank"]:
            rank_mismatch_count += 1

        if original_item["reranker_score"] != packed_item["reranker_score"]:
            score_mismatch_count += 1

        if json.dumps(
            original_item,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ) != json.dumps(
            packed_item,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ):
            text_or_metadata_mismatch_count += 1

    packed_token_sum = sum(
        len(
            nb18_tokenizer.encode(
                item["text"],
                add_special_tokens=False,
            )
        )
        for item in packed_items
    )

    if packed_token_sum > PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS:
        budget_violation_count += 1

    if len(packed_items) != 10:
        packet_mismatch_count += 1

assert packet_mismatch_count == 0
assert duplicate_item_id_count == 0
assert rank_mismatch_count == 0
assert score_mismatch_count == 0
assert text_or_metadata_mismatch_count == 0
assert budget_violation_count == 0
assert selected_item_count == 26640

print(json.dumps({
    "status": "PASS",
    "development_packet_count": len(packed_by_candidate),
    "selected_item_count": selected_item_count,
    "packet_item_or_order_mismatch_count": packet_mismatch_count,
    "duplicate_item_id_packet_count": duplicate_item_id_count,
    "reranker_rank_mismatch_count": rank_mismatch_count,
    "reranker_score_mismatch_count": score_mismatch_count,
    "text_or_metadata_mismatch_count": text_or_metadata_mismatch_count,
    "budget_violation_count": budget_violation_count,
    "all_ten_items_preserved_per_packet": True,
    "source_priority_order_preserved": True,
    "configuration_frozen": False,
    "test_unlocked": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_packet_count": 2664,
  "selected_item_count": 26640,
  "packet_item_or_order_mismatch_count": 0,
  "duplicate_item_id_packet_count": 0,
  "reranker_rank_mismatch_count": 0,
  "reranker_score_mismatch_count": 0,
  "text_or_metadata_mismatch_count": 0,
  "budget_violation_count": 0,
  "all_ten_items_preserved_per_packet": true,
  "source_priority_order_preserved": true,
  "configuration_frozen": false,
  "test_unlocked": false,
  "test_evidence_or_metrics_used": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

این سلول اثر چند حد آزمایشی برای تعداد قطعه از یک منبع و ارجاع به یک سند بررسی می‌کند. مقدارها با توجه به توزیع مشاهده‌شده در بسته‌های Development انتخاب شده‌اند.

برای هر ترکیب، بازیابی شواهد طلایی و تعداد قطعه‌های نگه‌داشته‌شده محاسبه می‌شود. این آزمایش فقط روی Development است و هیچ حدی را به‌عنوان سیاست نهایی انتخاب یا قفل نمی‌کند.

</div>

In [62]:
# NB18-C62 — Development-only source and document dominance experiment

assert "nb18_development_packets" in globals()
assert "development_gold_mapping" in globals()
assert len(nb18_development_packets) == 2664

source_caps = (2, 3, 4)
document_caps = (1, 2, 3)
development_packet_by_id = {
    packet["candidate_id"]: packet
    for packet in nb18_development_packets
}

assert len(development_packet_by_id) == 2664
assert set(development_packet_by_id) == set(development_gold_mapping)

def normalize_gold_ids(raw_gold_ids, candidate_id):
    assert isinstance(raw_gold_ids, (list, tuple, set)), (
        f"Unexpected Development gold mapping type for {candidate_id!r}: "
        f"{type(raw_gold_ids).__name__}"
    )
    return set(raw_gold_ids)

gold_ids_by_candidate = {
    candidate_id: normalize_gold_ids(
        development_gold_mapping[candidate_id],
        candidate_id,
    )
    for candidate_id in development_packet_by_id
}

experiment_results = []

for source_cap in source_caps:
    for document_cap in document_caps:
        total_gold_count = 0
        total_retrieved_gold_count = 0
        queries_with_any_gold = 0
        macro_recall_values = []
        selected_item_count = 0
        selected_item_counts_per_query = []

        for candidate_id, packet in development_packet_by_id.items():
            items = sorted(
                packet["evidence_items"],
                key=lambda item: item["selection_priority_rank"],
            )

            selected_items = []
            source_counts = {}
            document_counts = {}

            for item in items:
                source = item.get("source") or {}
                source_id = source.get("source_id")
                source_key = (
                    source_id
                    if source_id not in (None, "")
                    else f"__MISSING_SOURCE__:{item['evidence_id']}"
                )

                member_document_ids = item.get("member_document_ids") or []
                document_ids = set(member_document_ids)

                if source_counts.get(source_key, 0) >= source_cap:
                    continue

                if any(
                    document_counts.get(document_id, 0) >= document_cap
                    for document_id in document_ids
                ):
                    continue

                selected_items.append(item)
                source_counts[source_key] = (
                    source_counts.get(source_key, 0) + 1
                )
                for document_id in document_ids:
                    document_counts[document_id] = (
                        document_counts.get(document_id, 0) + 1
                    )

            selected_ids = {
                item["evidence_id"] for item in selected_items
            }
            gold_ids = gold_ids_by_candidate[candidate_id]
            retrieved_gold_count = len(selected_ids & gold_ids)

            total_gold_count += len(gold_ids)
            total_retrieved_gold_count += retrieved_gold_count
            queries_with_any_gold += retrieved_gold_count > 0
            selected_item_count += len(selected_items)
            selected_item_counts_per_query.append(len(selected_items))

            if gold_ids:
                macro_recall_values.append(
                    retrieved_gold_count / len(gold_ids)
                )

        experiment_results.append({
            "source_cap_per_source_id": source_cap,
            "document_cap_per_member_document_id": document_cap,
            "macro_gold_chunk_recall": (
                sum(macro_recall_values) / len(macro_recall_values)
                if macro_recall_values
                else None
            ),
            "micro_gold_chunk_recall": (
                total_retrieved_gold_count / total_gold_count
                if total_gold_count
                else None
            ),
            "queries_with_any_gold_evidence": queries_with_any_gold,
            "query_gold_hit_rate": (
                queries_with_any_gold / len(development_packet_by_id)
            ),
            "mean_selected_items_per_query": (
                selected_item_count / len(development_packet_by_id)
            ),
            "selected_item_count": selected_item_count,
            "mean_gold_bearing_query_recall_denominator": (
                len(macro_recall_values)
            ),
        })

print(json.dumps({
    "status": "DEVELOPMENT_DOMINANCE_EXPERIMENT_COMPLETE",
    "development_query_count": len(development_packet_by_id),
    "source_cap_candidates": list(source_caps),
    "document_cap_candidates": list(document_caps),
    "results": experiment_results,
    "selection_order": "source_priority_order",
    "reranker_ranks_and_scores_modified": False,
    "budget_applied": False,
    "configuration_frozen": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "DEVELOPMENT_DOMINANCE_EXPERIMENT_COMPLETE",
  "development_query_count": 2664,
  "source_cap_candidates": [
    2,
    3,
    4
  ],
  "document_cap_candidates": [
    1,
    2,
    3
  ],
  "results": [
    {
      "source_cap_per_source_id": 2,
      "document_cap_per_member_document_id": 1,
      "macro_gold_chunk_recall": 0.5379468754468741,
      "micro_gold_chunk_recall": 0.4367163009404389,
      "queries_with_any_gold_evidence": 2220,
      "query_gold_hit_rate": 0.8333333333333334,
      "mean_selected_items_per_query": 8.56418918918919,
      "selected_item_count": 22815,
      "mean_gold_bearing_query_recall_denominator": 2664
    },
    {
      "source_cap_per_source_id": 2,
      "document_cap_per_member_document_id": 2,
      "macro_gold_chunk_recall": 0.6374910624910622,
      "micro_gold_chunk_recall": 0.5562304075235109,
      "queries_with_any_gold_evidence": 2230,
      "query_gold_hit_rate": 0.8370870870870871,
      "mean_selected_items_per_query": 9

<div dir="rtl" align="right">

این سلول نتایج سقف‌های منبع و سند را با خط پایهٔ بدون سقف مقایسه می‌کند. گزینه‌ها بر پایهٔ بازیابی قطعهٔ طلایی، نرخ پرس‌وجوهای دارای شاهد و تعداد قطعه‌های انتخاب‌شده سنجیده می‌شوند.

این مقایسه گزینه‌های مغلوب را مشخص می‌کند، اما به‌دلیل نبود آستانهٔ پذیرش، سیاست نهایی را انتخاب یا قفل نمی‌کند.

</div>

In [63]:
# NB18-C63 — Development Pareto comparison for dominance policies

assert "experiment_results" in globals()
assert len(experiment_results) == 9

uncapped_baseline = {
    "policy": "uncapped_source_priority",
    "source_cap_per_source_id": None,
    "document_cap_per_member_document_id": None,
    "macro_gold_chunk_recall": 0.667674,
    "micro_gold_chunk_recall": 0.600313,
    "query_gold_hit_rate": 0.84985,
    "mean_selected_items_per_query": 10.0,
}

dominance_candidates = [uncapped_baseline] + [
    {
        "policy": (
            f"source_{result['source_cap_per_source_id']}"
            f"_document_{result['document_cap_per_member_document_id']}"
        ),
        "source_cap_per_source_id": (
            result["source_cap_per_source_id"]
        ),
        "document_cap_per_member_document_id": (
            result["document_cap_per_member_document_id"]
        ),
        "macro_gold_chunk_recall": result["macro_gold_chunk_recall"],
        "micro_gold_chunk_recall": result["micro_gold_chunk_recall"],
        "query_gold_hit_rate": result["query_gold_hit_rate"],
        "mean_selected_items_per_query": (
            result["mean_selected_items_per_query"]
        ),
    }
    for result in experiment_results
]

def metrics_equivalent(left, right, tolerance=1e-12):
    metric_names = (
        "macro_gold_chunk_recall",
        "micro_gold_chunk_recall",
        "query_gold_hit_rate",
        "mean_selected_items_per_query",
    )
    return all(
        abs(left[name] - right[name]) <= tolerance
        for name in metric_names
    )

def dominates(left, right, tolerance=1e-12):
    no_worse = (
        left["macro_gold_chunk_recall"]
        >= right["macro_gold_chunk_recall"] - tolerance
        and left["micro_gold_chunk_recall"]
        >= right["micro_gold_chunk_recall"] - tolerance
        and left["query_gold_hit_rate"]
        >= right["query_gold_hit_rate"] - tolerance
        and left["mean_selected_items_per_query"]
        <= right["mean_selected_items_per_query"] + tolerance
    )
    strictly_better = (
        left["macro_gold_chunk_recall"]
        > right["macro_gold_chunk_recall"] + tolerance
        or left["micro_gold_chunk_recall"]
        > right["micro_gold_chunk_recall"] + tolerance
        or left["query_gold_hit_rate"]
        > right["query_gold_hit_rate"] + tolerance
        or left["mean_selected_items_per_query"]
        < right["mean_selected_items_per_query"] - tolerance
    )
    return no_worse and strictly_better

equivalent_policy_groups = []
unassigned = set(range(len(dominance_candidates)))

while unassigned:
    first_index = min(unassigned)
    group = [first_index]
    unassigned.remove(first_index)

    for other_index in list(unassigned):
        if metrics_equivalent(
            dominance_candidates[first_index],
            dominance_candidates[other_index],
        ):
            group.append(other_index)
            unassigned.remove(other_index)

    if len(group) > 1:
        equivalent_policy_groups.append([
            dominance_candidates[index]["policy"]
            for index in group
        ])

unique_candidates = [
    dominance_candidates[group[0]]
    if group
    else None
    for group in []
]

representative_indices = []
for index, candidate in enumerate(dominance_candidates):
    if not any(
        metrics_equivalent(
            candidate,
            dominance_candidates[previous_index],
        )
        for previous_index in representative_indices
    ):
        representative_indices.append(index)

representatives = [
    dominance_candidates[index]
    for index in representative_indices
]

dominated_policies = []
non_dominated_policies = []

for candidate in representatives:
    dominators = [
        other["policy"]
        for other in representatives
        if other["policy"] != candidate["policy"]
        and dominates(other, candidate)
    ]

    if dominators:
        dominated_policies.append({
            "policy": candidate["policy"],
            "dominated_by": dominators,
        })
    else:
        non_dominated_policies.append(candidate["policy"])

for candidate in dominance_candidates:
    candidate["macro_recall_change_vs_uncapped"] = round(
        candidate["macro_gold_chunk_recall"]
        - uncapped_baseline["macro_gold_chunk_recall"],
        9,
    )
    candidate["query_hit_rate_change_vs_uncapped"] = round(
        candidate["query_gold_hit_rate"]
        - uncapped_baseline["query_gold_hit_rate"],
        9,
    )
    candidate["mean_item_change_vs_uncapped"] = round(
        candidate["mean_selected_items_per_query"]
        - uncapped_baseline["mean_selected_items_per_query"],
        6,
    )

print(json.dumps({
    "status": "DEVELOPMENT_POLICY_COMPARISON_COMPLETE",
    "comparison_objectives": [
        "maximize_macro_gold_chunk_recall",
        "maximize_micro_gold_chunk_recall",
        "maximize_query_gold_hit_rate",
        "minimize_mean_selected_items_per_query",
    ],
    "policy_results": dominance_candidates,
    "equivalent_policy_groups": equivalent_policy_groups,
    "dominated_policies": dominated_policies,
    "non_dominated_policies": non_dominated_policies,
    "acceptance_threshold_introduced": False,
    "policy_selected": False,
    "configuration_frozen": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "DEVELOPMENT_POLICY_COMPARISON_COMPLETE",
  "comparison_objectives": [
    "maximize_macro_gold_chunk_recall",
    "maximize_micro_gold_chunk_recall",
    "maximize_query_gold_hit_rate",
    "minimize_mean_selected_items_per_query"
  ],
  "policy_results": [
    {
      "policy": "uncapped_source_priority",
      "source_cap_per_source_id": null,
      "document_cap_per_member_document_id": null,
      "macro_gold_chunk_recall": 0.667674,
      "micro_gold_chunk_recall": 0.600313,
      "query_gold_hit_rate": 0.84985,
      "mean_selected_items_per_query": 10.0,
      "macro_recall_change_vs_uncapped": 0.0,
      "query_hit_rate_change_vs_uncapped": 0.0,
      "mean_item_change_vs_uncapped": 0.0
    },
    {
      "policy": "source_2_document_1",
      "source_cap_per_source_id": 2,
      "document_cap_per_member_document_id": 1,
      "macro_gold_chunk_recall": 0.5379468754468741,
      "micro_gold_chunk_recall": 0.4367163009404389,
      "query_gold_hit_rate": 0.8333333

<div dir="rtl" align="right">

چون گزینه‌های آزمایش قبلی مصالحه‌های متفاوتی داشتند، این سلول چند سقف ملایم‌تر را بررسی می‌کند. سقف‌ها برای منبع ۵، ۶ و ۸ قطعه و برای سند ۴، ۵ و ۶ ارجاع هستند؛ این محدوده با توزیع مشاهده‌شدهٔ Development انتخاب شده است.

نتایج با خط پایهٔ بدون سقف مقایسه می‌شوند. هیچ سیاستی در این سلول انتخاب یا قفل نمی‌شود.

</div>

In [64]:
# NB18-C64 — Expanded Development dominance-cap sensitivity experiment

assert "development_packet_by_id" in globals()
assert "gold_ids_by_candidate" in globals()
assert len(development_packet_by_id) == 2664

expanded_source_caps = (5, 6, 8)
expanded_document_caps = (4, 5, 6)
expanded_dominance_results = []

for source_cap in expanded_source_caps:
    for document_cap in expanded_document_caps:
        total_gold_count = 0
        total_retrieved_gold_count = 0
        queries_with_any_gold = 0
        macro_recall_values = []
        selected_item_count = 0

        for candidate_id, packet in development_packet_by_id.items():
            items = sorted(
                packet["evidence_items"],
                key=lambda item: item["selection_priority_rank"],
            )

            selected_items = []
            source_counts = {}
            document_counts = {}

            for item in items:
                source = item.get("source") or {}
                source_id = source.get("source_id")
                source_key = (
                    source_id
                    if source_id not in (None, "")
                    else f"__MISSING_SOURCE__:{item['evidence_id']}"
                )

                document_ids = set(
                    item.get("member_document_ids") or []
                )

                if source_counts.get(source_key, 0) >= source_cap:
                    continue

                if any(
                    document_counts.get(document_id, 0) >= document_cap
                    for document_id in document_ids
                ):
                    continue

                selected_items.append(item)
                source_counts[source_key] = (
                    source_counts.get(source_key, 0) + 1
                )
                for document_id in document_ids:
                    document_counts[document_id] = (
                        document_counts.get(document_id, 0) + 1
                    )

            selected_ids = {
                item["evidence_id"] for item in selected_items
            }
            gold_ids = gold_ids_by_candidate[candidate_id]
            retrieved_gold_count = len(selected_ids & gold_ids)

            total_gold_count += len(gold_ids)
            total_retrieved_gold_count += retrieved_gold_count
            queries_with_any_gold += retrieved_gold_count > 0
            selected_item_count += len(selected_items)

            if gold_ids:
                macro_recall_values.append(
                    retrieved_gold_count / len(gold_ids)
                )

        expanded_dominance_results.append({
            "source_cap_per_source_id": source_cap,
            "document_cap_per_member_document_id": document_cap,
            "macro_gold_chunk_recall": (
                sum(macro_recall_values) / len(macro_recall_values)
                if macro_recall_values
                else None
            ),
            "micro_gold_chunk_recall": (
                total_retrieved_gold_count / total_gold_count
                if total_gold_count
                else None
            ),
            "queries_with_any_gold_evidence": queries_with_any_gold,
            "query_gold_hit_rate": (
                queries_with_any_gold / len(development_packet_by_id)
            ),
            "mean_selected_items_per_query": (
                selected_item_count / len(development_packet_by_id)
            ),
            "selected_item_count": selected_item_count,
            "macro_recall_change_vs_uncapped": round(
                (
                    sum(macro_recall_values) / len(macro_recall_values)
                    if macro_recall_values
                    else 0.0
                ) - 0.667674,
                9,
            ),
            "query_hit_rate_change_vs_uncapped": round(
                (
                    queries_with_any_gold / len(development_packet_by_id)
                ) - 0.84985,
                9,
            ),
        })

print(json.dumps({
    "status": "DEVELOPMENT_DOMINANCE_SENSITIVITY_COMPLETE",
    "development_query_count": len(development_packet_by_id),
    "source_cap_candidates": list(expanded_source_caps),
    "document_cap_candidates": list(expanded_document_caps),
    "results": expanded_dominance_results,
    "selection_order": "source_priority_order",
    "budget_applied": False,
    "reranker_ranks_and_scores_modified": False,
    "policy_selected": False,
    "configuration_frozen": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "DEVELOPMENT_DOMINANCE_SENSITIVITY_COMPLETE",
  "development_query_count": 2664,
  "source_cap_candidates": [
    5,
    6,
    8
  ],
  "document_cap_candidates": [
    4,
    5,
    6
  ],
  "results": [
    {
      "source_cap_per_source_id": 5,
      "document_cap_per_member_document_id": 4,
      "macro_gold_chunk_recall": 0.6648523523523517,
      "micro_gold_chunk_recall": 0.5942398119122257,
      "queries_with_any_gold_evidence": 2262,
      "query_gold_hit_rate": 0.8490990990990991,
      "mean_selected_items_per_query": 9.947822822822824,
      "selected_item_count": 26501,
      "macro_recall_change_vs_uncapped": -0.002821648,
      "query_hit_rate_change_vs_uncapped": -0.000750901
    },
    {
      "source_cap_per_source_id": 5,
      "document_cap_per_member_document_id": 5,
      "macro_gold_chunk_recall": 0.6655383061633056,
      "micro_gold_chunk_recall": 0.5967868338557993,
      "queries_with_any_gold_evidence": 2262,
      "query_gold_hit_rate": 0.84

<div dir="rtl" align="right">

این سلول اثر گزینهٔ آزمایشیِ سقف ۸ قطعه برای هر منبع و ۶ ارجاع برای هر سند را جزئی‌تر بررسی می‌کند. تعداد بسته‌های متأثر، رتبهٔ قطعه‌های حذف‌شده و اثر حذف بر شواهد طلایی محاسبه می‌شود.

این بررسی فقط روی Development است و سیاست را قفل نمی‌کند.

</div>

In [65]:
# NB18-C65 — Audit the Development impact of the mild dominance caps

assert "development_packet_by_id" in globals()
assert "gold_ids_by_candidate" in globals()
assert len(development_packet_by_id) == 2664

SOURCE_CAP = 8
DOCUMENT_CAP = 6

impacted_context_count = 0
dropped_item_count = 0
contexts_with_dropped_gold_count = 0
contexts_losing_all_gold_hits = 0
dropped_gold_chunk_occurrence_count = 0
source_cap_drop_count = 0
document_cap_drop_count = 0
selected_item_count = 0
selected_gold_chunk_occurrence_count = 0
total_gold_chunk_occurrence_count = 0
queries_with_any_selected_gold = 0
macro_recall_values = []
dropped_rank_counts = {}

for candidate_id, packet in development_packet_by_id.items():
    items = sorted(
        packet["evidence_items"],
        key=lambda item: item["selection_priority_rank"],
    )
    gold_ids = gold_ids_by_candidate[candidate_id]
    total_gold_chunk_occurrence_count += len(gold_ids)

    selected_items = []
    dropped_items = []
    source_counts = {}
    document_counts = {}

    for item in items:
        source = item.get("source") or {}
        source_id = source.get("source_id")
        source_key = (
            source_id
            if source_id not in (None, "")
            else f"__MISSING_SOURCE__:{item['evidence_id']}"
        )
        document_ids = set(item.get("member_document_ids") or [])

        if source_counts.get(source_key, 0) >= SOURCE_CAP:
            dropped_items.append((item, "SOURCE_CAP"))
            source_cap_drop_count += 1
            continue

        if any(
            document_counts.get(document_id, 0) >= DOCUMENT_CAP
            for document_id in document_ids
        ):
            dropped_items.append((item, "DOCUMENT_CAP"))
            document_cap_drop_count += 1
            continue

        selected_items.append(item)
        source_counts[source_key] = source_counts.get(source_key, 0) + 1
        for document_id in document_ids:
            document_counts[document_id] = (
                document_counts.get(document_id, 0) + 1
            )

    selected_ids = {item["evidence_id"] for item in selected_items}
    dropped_ids = {item["evidence_id"] for item, _ in dropped_items}
    selected_gold_count = len(selected_ids & gold_ids)
    dropped_gold_count = len(dropped_ids & gold_ids)

    selected_item_count += len(selected_items)
    dropped_item_count += len(dropped_items)
    selected_gold_chunk_occurrence_count += selected_gold_count
    dropped_gold_chunk_occurrence_count += dropped_gold_count
    queries_with_any_selected_gold += selected_gold_count > 0

    if gold_ids:
        macro_recall_values.append(selected_gold_count / len(gold_ids))

    if dropped_items:
        impacted_context_count += 1

    if dropped_gold_count:
        contexts_with_dropped_gold_count += 1

    if gold_ids & {
        item["evidence_id"] for item in items
    } and not selected_gold_count:
        contexts_losing_all_gold_hits += 1

    for item, _reason in dropped_items:
        rank = str(item["reranker_rank"])
        dropped_rank_counts[rank] = dropped_rank_counts.get(rank, 0) + 1

macro_recall = (
    sum(macro_recall_values) / len(macro_recall_values)
    if macro_recall_values
    else None
)
micro_recall = (
    selected_gold_chunk_occurrence_count / total_gold_chunk_occurrence_count
    if total_gold_chunk_occurrence_count
    else None
)
query_hit_rate = (
    queries_with_any_selected_gold / len(development_packet_by_id)
)

assert dropped_item_count == 17
assert selected_item_count == 26623
assert queries_with_any_selected_gold == 2264
assert contexts_losing_all_gold_hits == 0

print(json.dumps({
    "status": "PASS",
    "policy_under_audit": {
        "source_cap_per_source_id": SOURCE_CAP,
        "document_cap_per_member_document_id": DOCUMENT_CAP,
        "selection_order": "source_priority_order",
    },
    "development_query_count": len(development_packet_by_id),
    "contexts_with_any_dropped_item": impacted_context_count,
    "dropped_item_count": dropped_item_count,
    "dropped_items_by_reason": {
        "source_cap": source_cap_drop_count,
        "document_cap": document_cap_drop_count,
    },
    "dropped_original_reranker_rank_counts": dict(
        sorted(
            dropped_rank_counts.items(),
            key=lambda pair: int(pair[0]),
        )
    ),
    "contexts_with_at_least_one_dropped_gold_chunk": (
        contexts_with_dropped_gold_count
    ),
    "dropped_gold_chunk_occurrence_count": (
        dropped_gold_chunk_occurrence_count
    ),
    "contexts_losing_all_gold_hits": contexts_losing_all_gold_hits,
    "macro_gold_chunk_recall": macro_recall,
    "micro_gold_chunk_recall": micro_recall,
    "query_gold_hit_rate": query_hit_rate,
    "selected_item_count": selected_item_count,
    "reranker_ranks_and_scores_modified": False,
    "configuration_frozen": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "policy_under_audit": {
    "source_cap_per_source_id": 8,
    "document_cap_per_member_document_id": 6,
    "selection_order": "source_priority_order"
  },
  "development_query_count": 2664,
  "contexts_with_any_dropped_item": 14,
  "dropped_item_count": 17,
  "dropped_items_by_reason": {
    "source_cap": 13,
    "document_cap": 4
  },
  "dropped_original_reranker_rank_counts": {
    "7": 1,
    "9": 9,
    "10": 7
  },
  "contexts_with_at_least_one_dropped_gold_chunk": 4,
  "dropped_gold_chunk_occurrence_count": 4,
  "contexts_losing_all_gold_hits": 0,
  "macro_gold_chunk_recall": 0.6674732768482762,
  "micro_gold_chunk_recall": 0.5995297805642633,
  "query_gold_hit_rate": 0.8498498498498499,
  "selected_item_count": 26623,
  "reranker_ranks_and_scores_modified": false,
  "configuration_frozen": false,
  "test_evidence_or_metrics_used": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

این سلول یک نامزد بسته‌بندی Development می‌سازد که سقف منبع ۸، سقف ارجاع سند ۶ و بودجهٔ پیشنهادی ۷۳۷۱ توکن را ترکیب می‌کند.

نامزد فقط در حافظه ساخته می‌شود. رتبه و امتیاز رتبه‌بند به‌عنوان فرادادهٔ هر مدرک حفظ می‌شوند؛ وضعیت حقوقی، آمادگی تولید و تنظیمات قفل‌شده ادعا نمی‌شوند.

</div>

In [66]:
# NB18-C66 — Build the proposed Development packet candidate in memory

assert "nb18_development_packets" in globals()
assert "nb18_tokenizer" in globals()
assert len(nb18_development_packets) == 2664

PROPOSED_SOURCE_CAP = 8
PROPOSED_DOCUMENT_CAP = 6
PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS = 7371

nb18_policy_candidate_packets = []
nb18_policy_candidate_dropped_items = []

selected_item_count = 0
dropped_item_count = 0
contexts_with_dropped_items = 0
budget_skip_count = 0
source_cap_skip_count = 0
document_cap_skip_count = 0
max_selected_token_sum = 0

for packet in nb18_development_packets:
    assert packet.get("split") == "development"

    ranked_items = sorted(
        packet["evidence_items"],
        key=lambda item: item["reranker_rank"],
    )
    priority_items = sorted(
        packet["evidence_items"],
        key=lambda item: item["selection_priority_rank"],
    )

    assert len(ranked_items) == 10
    assert len(priority_items) == 10

    source_counts = {}
    document_counts = {}
    selected_items = []
    dropped_for_packet = []
    selected_token_sum = 0

    for item in priority_items:
        source = item.get("source") or {}
        source_id = source.get("source_id")
        source_key = (
            source_id
            if source_id not in (None, "")
            else f"__MISSING_SOURCE__:{item['evidence_id']}"
        )
        document_ids = set(item.get("member_document_ids") or [])

        if source_counts.get(source_key, 0) >= PROPOSED_SOURCE_CAP:
            dropped_for_packet.append({
                "evidence_id": item["evidence_id"],
                "reranker_rank": item["reranker_rank"],
                "reason": "SOURCE_CAP",
            })
            source_cap_skip_count += 1
            continue

        if any(
            document_counts.get(document_id, 0) >= PROPOSED_DOCUMENT_CAP
            for document_id in document_ids
        ):
            dropped_for_packet.append({
                "evidence_id": item["evidence_id"],
                "reranker_rank": item["reranker_rank"],
                "reason": "DOCUMENT_CAP",
            })
            document_cap_skip_count += 1
            continue

        item_tokens = len(
            nb18_tokenizer.encode(
                item["text"],
                add_special_tokens=False,
            )
        )

        if (
            selected_token_sum + item_tokens
            > PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS
        ):
            dropped_for_packet.append({
                "evidence_id": item["evidence_id"],
                "reranker_rank": item["reranker_rank"],
                "reason": "EVIDENCE_TEXT_BUDGET",
            })
            budget_skip_count += 1
            continue

        selected_items.append(item)
        selected_token_sum += item_tokens
        source_counts[source_key] = source_counts.get(source_key, 0) + 1

        for document_id in document_ids:
            document_counts[document_id] = (
                document_counts.get(document_id, 0) + 1
            )

    selected_ids = [item["evidence_id"] for item in selected_items]
    assert len(selected_ids) == len(set(selected_ids))
    assert selected_token_sum <= PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS

    original_rank_by_id = {
        item["evidence_id"]: item["reranker_rank"]
        for item in ranked_items
    }
    assert all(
        original_rank_by_id[item["evidence_id"]] == item["reranker_rank"]
        for item in selected_items
    )

    candidate_packet = dict(packet)
    candidate_packet["evidence_items"] = selected_items
    candidate_packet["reranker_rank_order"] = [
        item["evidence_id"] for item in ranked_items
    ]
    candidate_packet["source_priority_order"] = selected_ids
    candidate_packet["selection_policy_candidate"] = {
        "status": "PROPOSED_NOT_FROZEN",
        "source_cap_per_source_id": PROPOSED_SOURCE_CAP,
        "document_cap_per_member_document_id": PROPOSED_DOCUMENT_CAP,
        "evidence_text_budget_tokens": (
            PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS
        ),
        "budget_measurement": (
            "Sum of separately tokenized evidence texts; no special tokens"
        ),
    }
    candidate_packet["selected_evidence_text_tokens_additive"] = (
        selected_token_sum
    )
    candidate_packet["dropped_evidence_items"] = dropped_for_packet

    nb18_policy_candidate_packets.append(candidate_packet)
    selected_item_count += len(selected_items)
    dropped_item_count += len(dropped_for_packet)
    contexts_with_dropped_items += bool(dropped_for_packet)
    max_selected_token_sum = max(max_selected_token_sum, selected_token_sum)

    for dropped_item in dropped_for_packet:
        nb18_policy_candidate_dropped_items.append({
            "candidate_id": packet["candidate_id"],
            **dropped_item,
        })

assert len(nb18_policy_candidate_packets) == 2664
assert selected_item_count + dropped_item_count == 26640
assert budget_skip_count == 0
assert dropped_item_count == 17
assert max_selected_token_sum <= PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS

print(json.dumps({
    "status": "DEVELOPMENT_POLICY_CANDIDATE_BUILT_IN_MEMORY",
    "development_packet_count": len(nb18_policy_candidate_packets),
    "selected_item_count": selected_item_count,
    "dropped_item_count": dropped_item_count,
    "contexts_with_dropped_items": contexts_with_dropped_items,
    "dropped_items_by_reason": {
        "source_cap": source_cap_skip_count,
        "document_cap": document_cap_skip_count,
        "evidence_text_budget": budget_skip_count,
    },
    "max_selected_additive_evidence_tokens": max_selected_token_sum,
    "reranker_order_preserved_as_separate_metadata": True,
    "reranker_ranks_and_scores_modified": False,
    "rights_clearance_asserted": False,
    "configuration_frozen": False,
    "test_unlocked": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "DEVELOPMENT_POLICY_CANDIDATE_BUILT_IN_MEMORY",
  "development_packet_count": 2664,
  "selected_item_count": 26623,
  "dropped_item_count": 17,
  "contexts_with_dropped_items": 14,
  "dropped_items_by_reason": {
    "source_cap": 13,
    "document_cap": 4,
    "evidence_text_budget": 0
  },
  "max_selected_additive_evidence_tokens": 7363,
  "reranker_order_preserved_as_separate_metadata": true,
  "reranker_ranks_and_scores_modified": false,
  "rights_clearance_asserted": false,
  "configuration_frozen": false,
  "test_unlocked": false,
  "test_evidence_or_metrics_used": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

این سلول نامزد C66 را با ورودی‌ها اعتبارسنجی می‌کند: سقف‌های منبع و سند، بودجه، ترتیب انتخاب، حفظ رتبه‌ها و امتیازها و تطبیق قطعه‌های انتخاب‌شده با شواهد Development.

معیارهای بازیابی نیز از روی بسته‌های نامزد دوباره محاسبه می‌شوند. این فقط اعتبارسنجی Development است؛ تنظیمات را قفل نمی‌کند و Test را باز نمی‌کند.

</div>

In [67]:
# NB18-C67 — Validate the in-memory policy candidate and recompute Development metrics

assert "nb18_policy_candidate_packets" in globals()
assert "nb18_development_packets" in globals()
assert "gold_ids_by_candidate" in globals()
assert len(nb18_policy_candidate_packets) == 2664

original_by_candidate = {
    packet["candidate_id"]: packet
    for packet in nb18_development_packets
}
candidate_by_id = {
    packet["candidate_id"]: packet
    for packet in nb18_policy_candidate_packets
}

assert set(original_by_candidate) == set(candidate_by_id)

packet_structure_mismatch_count = 0
selected_item_mismatch_count = 0
reranker_rank_mismatch_count = 0
reranker_score_mismatch_count = 0
text_or_metadata_mismatch_count = 0
source_cap_violation_count = 0
document_cap_violation_count = 0
budget_violation_count = 0
duplicate_selected_item_count = 0

selected_item_count = 0
total_gold_chunk_count = 0
retrieved_gold_chunk_count = 0
queries_with_any_gold = 0
macro_recall_values = []

for candidate_id, original_packet in original_by_candidate.items():
    candidate_packet = candidate_by_id[candidate_id]
    original_items = original_packet["evidence_items"]
    candidate_items = candidate_packet["evidence_items"]

    original_by_evidence_id = {
        item["evidence_id"]: item
        for item in original_items
    }
    candidate_by_evidence_id = {
        item["evidence_id"]: item
        for item in candidate_items
    }
    dropped_items = candidate_packet["dropped_evidence_items"]
    dropped_by_evidence_id = {
        item["evidence_id"]: item
        for item in dropped_items
    }

    if (
        len(candidate_by_evidence_id) != len(candidate_items)
        or len(dropped_by_evidence_id) != len(dropped_items)
    ):
        duplicate_selected_item_count += 1

    selected_ids = set(candidate_by_evidence_id)
    dropped_ids = set(dropped_by_evidence_id)

    if (
        selected_ids & dropped_ids
        or selected_ids | dropped_ids != set(original_by_evidence_id)
    ):
        selected_item_mismatch_count += 1

    expected_rank_order = [
        item["evidence_id"]
        for item in sorted(
            original_items,
            key=lambda item: item["reranker_rank"],
        )
    ]
    if candidate_packet["reranker_rank_order"] != expected_rank_order:
        packet_structure_mismatch_count += 1

    expected_priority_order = [
        item["evidence_id"]
        for item in sorted(
            candidate_items,
            key=lambda item: item["selection_priority_rank"],
        )
    ]
    if candidate_packet["source_priority_order"] != expected_priority_order:
        packet_structure_mismatch_count += 1

    source_counts = {}
    document_counts = {}
    token_sum = 0

    for item in candidate_items:
        evidence_id = item["evidence_id"]
        original_item = original_by_evidence_id[evidence_id]

        if item["reranker_rank"] != original_item["reranker_rank"]:
            reranker_rank_mismatch_count += 1

        if item["reranker_score"] != original_item["reranker_score"]:
            reranker_score_mismatch_count += 1

        if json.dumps(
            item,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ) != json.dumps(
            original_item,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ):
            text_or_metadata_mismatch_count += 1

        source = item.get("source") or {}
        source_id = source.get("source_id")
        source_key = (
            source_id
            if source_id not in (None, "")
            else f"__MISSING_SOURCE__:{evidence_id}"
        )
        source_counts[source_key] = source_counts.get(source_key, 0) + 1

        document_ids = set(item.get("member_document_ids") or [])
        for document_id in document_ids:
            document_counts[document_id] = (
                document_counts.get(document_id, 0) + 1
            )

        token_sum += len(
            nb18_tokenizer.encode(
                item["text"],
                add_special_tokens=False,
            )
        )

    if any(count > 8 for count in source_counts.values()):
        source_cap_violation_count += 1

    if any(count > 6 for count in document_counts.values()):
        document_cap_violation_count += 1

    if token_sum > PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS:
        budget_violation_count += 1

    selected_item_count += len(candidate_items)
    gold_ids = gold_ids_by_candidate[candidate_id]
    total_gold_chunk_count += len(gold_ids)
    retrieved_gold = len(selected_ids & gold_ids)
    retrieved_gold_chunk_count += retrieved_gold
    queries_with_any_gold += retrieved_gold > 0

    if gold_ids:
        macro_recall_values.append(retrieved_gold / len(gold_ids))

    for dropped_item in dropped_items:
        if (
            dropped_item["reranker_rank"]
            != original_by_evidence_id[
                dropped_item["evidence_id"]
            ]["reranker_rank"]
        ):
            reranker_rank_mismatch_count += 1

macro_gold_chunk_recall = (
    sum(macro_recall_values) / len(macro_recall_values)
    if macro_recall_values
    else None
)
micro_gold_chunk_recall = (
    retrieved_gold_chunk_count / total_gold_chunk_count
    if total_gold_chunk_count
    else None
)
query_gold_hit_rate = queries_with_any_gold / len(candidate_by_id)

assert packet_structure_mismatch_count == 0
assert selected_item_mismatch_count == 0
assert duplicate_selected_item_count == 0
assert reranker_rank_mismatch_count == 0
assert reranker_score_mismatch_count == 0
assert text_or_metadata_mismatch_count == 0
assert source_cap_violation_count == 0
assert document_cap_violation_count == 0
assert budget_violation_count == 0
assert selected_item_count == 26623
assert queries_with_any_gold == 2264
assert abs(macro_gold_chunk_recall - 0.6674732768482762) < 1e-12
assert abs(micro_gold_chunk_recall - 0.5995297805642633) < 1e-12

print(json.dumps({
    "status": "PASS",
    "development_packet_count": len(candidate_by_id),
    "selected_item_count": selected_item_count,
    "packet_structure_mismatch_count": packet_structure_mismatch_count,
    "selected_item_mismatch_count": selected_item_mismatch_count,
    "duplicate_selected_item_packet_count": duplicate_selected_item_count,
    "reranker_rank_mismatch_count": reranker_rank_mismatch_count,
    "reranker_score_mismatch_count": reranker_score_mismatch_count,
    "text_or_metadata_mismatch_count": text_or_metadata_mismatch_count,
    "source_cap_violation_count": source_cap_violation_count,
    "document_cap_violation_count": document_cap_violation_count,
    "budget_violation_count": budget_violation_count,
    "macro_gold_chunk_recall": macro_gold_chunk_recall,
    "micro_gold_chunk_recall": micro_gold_chunk_recall,
    "queries_with_any_gold_evidence": queries_with_any_gold,
    "query_gold_hit_rate": query_gold_hit_rate,
    "configuration_frozen": False,
    "test_unlocked": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_packet_count": 2664,
  "selected_item_count": 26623,
  "packet_structure_mismatch_count": 0,
  "selected_item_mismatch_count": 0,
  "duplicate_selected_item_packet_count": 0,
  "reranker_rank_mismatch_count": 0,
  "reranker_score_mismatch_count": 0,
  "text_or_metadata_mismatch_count": 0,
  "source_cap_violation_count": 0,
  "document_cap_violation_count": 0,
  "budget_violation_count": 0,
  "macro_gold_chunk_recall": 0.6674732768482762,
  "micro_gold_chunk_recall": 0.5995297805642633,
  "queries_with_any_gold_evidence": 2264,
  "query_gold_hit_rate": 0.8498498498498499,
  "configuration_frozen": false,
  "test_unlocked": false,
  "test_evidence_or_metrics_used": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

این سلول ترتیب مدارک نامزد C66 را به ترتیب اصلی رتبه‌بند برمی‌گرداند. ترتیب انتخاب بر اساس اولویت منبع در فیلدی جدا حفظ می‌شود.

این اصلاح فقط روی نامزد درون‌حافظه‌ای Development انجام می‌شود و رتبه‌ها یا امتیازها را تغییر نمی‌دهد.

</div>

In [68]:
# NB18-C68 — Restore reranker order and keep source priority separately

assert "nb18_policy_candidate_packets" in globals()
assert len(nb18_policy_candidate_packets) == 2664

nb18_ordered_policy_candidate_packets = []
packet_order_mismatch_count = 0
priority_order_mismatch_count = 0
duplicate_selected_id_packet_count = 0

for packet in nb18_policy_candidate_packets:
    selected_items = packet["evidence_items"]
    selected_by_priority = sorted(
        selected_items,
        key=lambda item: item["selection_priority_rank"],
    )
    selected_by_reranker = sorted(
        selected_items,
        key=lambda item: item["reranker_rank"],
    )

    selected_ids = [item["evidence_id"] for item in selected_by_reranker]
    selected_priority_ids = [
        item["evidence_id"] for item in selected_by_priority
    ]

    if len(selected_ids) != len(set(selected_ids)):
        duplicate_selected_id_packet_count += 1

    candidate_packet = dict(packet)
    candidate_packet["original_reranker_rank_order"] = (
        packet["reranker_rank_order"]
    )
    candidate_packet["reranker_rank_order"] = selected_ids
    candidate_packet["source_priority_order"] = selected_priority_ids
    candidate_packet["evidence_items"] = selected_by_reranker

    if [
        item["reranker_rank"]
        for item in candidate_packet["evidence_items"]
    ] != sorted(
        item["reranker_rank"]
        for item in candidate_packet["evidence_items"]
    ):
        packet_order_mismatch_count += 1

    if candidate_packet["source_priority_order"] != [
        item["evidence_id"]
        for item in sorted(
            candidate_packet["evidence_items"],
            key=lambda item: item["selection_priority_rank"],
        )
    ]:
        priority_order_mismatch_count += 1

    assert set(candidate_packet["reranker_rank_order"]) == set(
        candidate_packet["source_priority_order"]
    )

    nb18_ordered_policy_candidate_packets.append(candidate_packet)

assert len(nb18_ordered_policy_candidate_packets) == 2664
assert packet_order_mismatch_count == 0
assert priority_order_mismatch_count == 0
assert duplicate_selected_id_packet_count == 0

print(json.dumps({
    "status": "PASS",
    "development_packet_count": len(
        nb18_ordered_policy_candidate_packets
    ),
    "selected_item_count": sum(
        len(packet["evidence_items"])
        for packet in nb18_ordered_policy_candidate_packets
    ),
    "reranker_order_mismatch_count": packet_order_mismatch_count,
    "source_priority_order_mismatch_count": priority_order_mismatch_count,
    "duplicate_selected_id_packet_count": (
        duplicate_selected_id_packet_count
    ),
    "reranker_order_and_source_priority_stored_separately": True,
    "reranker_ranks_and_scores_modified": False,
    "configuration_frozen": False,
    "test_unlocked": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_packet_count": 2664,
  "selected_item_count": 26623,
  "reranker_order_mismatch_count": 0,
  "source_priority_order_mismatch_count": 0,
  "duplicate_selected_id_packet_count": 0,
  "reranker_order_and_source_priority_stored_separately": true,
  "reranker_ranks_and_scores_modified": false,
  "configuration_frozen": false,
  "test_unlocked": false,
  "test_evidence_or_metrics_used": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

این سلول نسخهٔ اصلاح‌شدهٔ C68 را با ورودی‌ها تطبیق می‌دهد. ترتیب رتبه‌بند، ترتیب اولویت منبع، متن، فراداده، سقف‌های منبع و سند و بودجه را بررسی می‌کند و معیارهای Development را دوباره محاسبه می‌کند.

</div>

In [69]:
# NB18-C69 — Revalidate the ordered candidate and recompute its Development metrics

assert "nb18_ordered_policy_candidate_packets" in globals()
assert "nb18_development_packets" in globals()
assert "gold_ids_by_candidate" in globals()
assert len(nb18_ordered_policy_candidate_packets) == 2664

original_by_candidate = {
    packet["candidate_id"]: packet
    for packet in nb18_development_packets
}
candidate_by_id = {
    packet["candidate_id"]: packet
    for packet in nb18_ordered_policy_candidate_packets
}
assert set(original_by_candidate) == set(candidate_by_id)

order_mismatch_count = 0
priority_order_mismatch_count = 0
selected_set_mismatch_count = 0
duplicate_item_packet_count = 0
item_payload_mismatch_count = 0
reranker_rank_mismatch_count = 0
reranker_score_mismatch_count = 0
source_cap_violation_count = 0
document_cap_violation_count = 0
budget_violation_count = 0

selected_item_count = 0
total_gold_count = 0
retrieved_gold_count = 0
queries_with_gold_hit = 0
macro_recall_values = []

for candidate_id, original_packet in original_by_candidate.items():
    candidate_packet = candidate_by_id[candidate_id]
    original_items = original_packet["evidence_items"]
    candidate_items = candidate_packet["evidence_items"]

    original_item_by_id = {
        item["evidence_id"]: item for item in original_items
    }
    candidate_item_by_id = {
        item["evidence_id"]: item for item in candidate_items
    }

    if len(candidate_item_by_id) != len(candidate_items):
        duplicate_item_packet_count += 1

    if set(candidate_item_by_id) & {
        item["evidence_id"]
        for item in candidate_packet["dropped_evidence_items"]
    }:
        selected_set_mismatch_count += 1

    if [
        item["reranker_rank"] for item in candidate_items
    ] != sorted(
        item["reranker_rank"] for item in candidate_items
    ):
        order_mismatch_count += 1

    expected_priority_ids = [
        item["evidence_id"]
        for item in sorted(
            candidate_items,
            key=lambda item: item["selection_priority_rank"],
        )
    ]
    if candidate_packet["source_priority_order"] != expected_priority_ids:
        priority_order_mismatch_count += 1

    if set(candidate_item_by_id) - set(original_item_by_id):
        selected_set_mismatch_count += 1

    if not (
        set(candidate_item_by_id)
        | {
            item["evidence_id"]
            for item in candidate_packet["dropped_evidence_items"]
        }
    ) == set(original_item_by_id):
        selected_set_mismatch_count += 1

    source_counts = {}
    document_counts = {}
    token_sum = 0

    for item in candidate_items:
        original_item = original_item_by_id[item["evidence_id"]]

        if item["reranker_rank"] != original_item["reranker_rank"]:
            reranker_rank_mismatch_count += 1
        if item["reranker_score"] != original_item["reranker_score"]:
            reranker_score_mismatch_count += 1

        if json.dumps(
            item,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ) != json.dumps(
            original_item,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ):
            item_payload_mismatch_count += 1

        source = item.get("source") or {}
        source_id = source.get("source_id")
        source_key = (
            source_id
            if source_id not in (None, "")
            else f"__MISSING_SOURCE__:{item['evidence_id']}"
        )
        source_counts[source_key] = source_counts.get(source_key, 0) + 1

        for document_id in set(item.get("member_document_ids") or []):
            document_counts[document_id] = (
                document_counts.get(document_id, 0) + 1
            )

        token_sum += len(
            nb18_tokenizer.encode(
                item["text"],
                add_special_tokens=False,
            )
        )

    if any(count > 8 for count in source_counts.values()):
        source_cap_violation_count += 1
    if any(count > 6 for count in document_counts.values()):
        document_cap_violation_count += 1
    if token_sum > PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS:
        budget_violation_count += 1

    selected_item_count += len(candidate_items)
    gold_ids = gold_ids_by_candidate[candidate_id]
    total_gold_count += len(gold_ids)
    query_retrieved_gold = len(set(candidate_item_by_id) & gold_ids)
    retrieved_gold_count += query_retrieved_gold
    queries_with_gold_hit += query_retrieved_gold > 0

    if gold_ids:
        macro_recall_values.append(query_retrieved_gold / len(gold_ids))

macro_recall = (
    sum(macro_recall_values) / len(macro_recall_values)
    if macro_recall_values
    else None
)
micro_recall = (
    retrieved_gold_count / total_gold_count
    if total_gold_count
    else None
)
query_hit_rate = queries_with_gold_hit / len(candidate_by_id)

assert order_mismatch_count == 0
assert priority_order_mismatch_count == 0
assert selected_set_mismatch_count == 0
assert duplicate_item_packet_count == 0
assert item_payload_mismatch_count == 0
assert reranker_rank_mismatch_count == 0
assert reranker_score_mismatch_count == 0
assert source_cap_violation_count == 0
assert document_cap_violation_count == 0
assert budget_violation_count == 0
assert selected_item_count == 26623
assert queries_with_gold_hit == 2264
assert abs(macro_recall - 0.6674732768482762) < 1e-12
assert abs(micro_recall - 0.5995297805642633) < 1e-12

print(json.dumps({
    "status": "PASS",
    "development_packet_count": len(candidate_by_id),
    "selected_item_count": selected_item_count,
    "reranker_order_mismatch_count": order_mismatch_count,
    "source_priority_order_mismatch_count": priority_order_mismatch_count,
    "selected_set_mismatch_count": selected_set_mismatch_count,
    "duplicate_item_packet_count": duplicate_item_packet_count,
    "item_payload_mismatch_count": item_payload_mismatch_count,
    "reranker_rank_mismatch_count": reranker_rank_mismatch_count,
    "reranker_score_mismatch_count": reranker_score_mismatch_count,
    "source_cap_violation_count": source_cap_violation_count,
    "document_cap_violation_count": document_cap_violation_count,
    "budget_violation_count": budget_violation_count,
    "macro_gold_chunk_recall": macro_recall,
    "micro_gold_chunk_recall": micro_recall,
    "queries_with_any_gold_evidence": queries_with_gold_hit,
    "query_gold_hit_rate": query_hit_rate,
    "configuration_frozen": False,
    "test_unlocked": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "development_packet_count": 2664,
  "selected_item_count": 26623,
  "reranker_order_mismatch_count": 0,
  "source_priority_order_mismatch_count": 0,
  "selected_set_mismatch_count": 0,
  "duplicate_item_packet_count": 0,
  "item_payload_mismatch_count": 0,
  "reranker_rank_mismatch_count": 0,
  "reranker_score_mismatch_count": 0,
  "source_cap_violation_count": 0,
  "document_cap_violation_count": 0,
  "budget_violation_count": 0,
  "macro_gold_chunk_recall": 0.6674732768482762,
  "micro_gold_chunk_recall": 0.5995297805642633,
  "queries_with_any_gold_evidence": 2264,
  "query_gold_hit_rate": 0.8498498498498499,
  "configuration_frozen": false,
  "test_unlocked": false,
  "test_evidence_or_metrics_used": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

این سلول نامزدهای Development را در حافظه به رکوردهای یکتای شواهد و نگاشت بسته‌ها نرمال می‌کند. رتبه، امتیاز و ترتیب اولویت در ارجاع هر بسته حفظ می‌شوند.

در پایان، بسته‌ها از نمایش نرمال‌شده بازسازی و با نامزد اصلی تطبیق داده می‌شوند. هیچ فایلی نوشته نمی‌شود.

</div>

In [70]:
# NB18-C70 — Normalize the ordered Development candidate in memory

assert "nb18_ordered_policy_candidate_packets" in globals()
assert len(nb18_ordered_policy_candidate_packets) == 2664

def nb18_canonical_json_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

rank_fields = {
    "reranker_rank",
    "selection_priority_rank",
    "reranker_score",
}

normalized_candidate_evidence_by_id = {}
normalized_candidate_signatures = {}
normalized_candidate_packet_stubs = []

candidate_evidence_occurrence_count = 0

for packet in nb18_ordered_policy_candidate_packets:
    packet_refs = []

    for item in packet["evidence_items"]:
        candidate_evidence_occurrence_count += 1
        evidence_id = item["evidence_id"]

        evidence_payload = {
            key: value
            for key, value in item.items()
            if key not in rank_fields
        }
        evidence_signature = hashlib.sha256(
            nb18_canonical_json_bytes(evidence_payload)
        ).hexdigest()

        if evidence_id in normalized_candidate_evidence_by_id:
            assert (
                normalized_candidate_signatures[evidence_id]
                == evidence_signature
            )
        else:
            normalized_candidate_evidence_by_id[evidence_id] = (
                evidence_payload
            )
            normalized_candidate_signatures[evidence_id] = (
                evidence_signature
            )

        packet_refs.append({
            "evidence_id": evidence_id,
            "reranker_rank": item["reranker_rank"],
            "selection_priority_rank": item["selection_priority_rank"],
            "reranker_score": item["reranker_score"],
        })

    packet_metadata = {
        key: value
        for key, value in packet.items()
        if key != "evidence_items"
    }

    normalized_candidate_packet_stubs.append({
        "packet_metadata": packet_metadata,
        "evidence_refs_in_reranker_order": packet_refs,
    })

normalized_candidate_evidence_hasher = hashlib.sha256()
normalized_candidate_evidence_bytes = 0
restored_candidate_evidence_by_id = {}

for evidence_id in sorted(normalized_candidate_evidence_by_id):
    evidence_payload = normalized_candidate_evidence_by_id[evidence_id]
    line = nb18_canonical_json_bytes(evidence_payload) + b"\n"
    restored_payload = json.loads(line)

    assert restored_payload == evidence_payload
    restored_candidate_evidence_by_id[evidence_id] = restored_payload
    normalized_candidate_evidence_hasher.update(line)
    normalized_candidate_evidence_bytes += len(line)

normalized_candidate_packet_hasher = hashlib.sha256()
normalized_candidate_packet_bytes = 0
restored_candidate_packet_stubs = []

for packet_stub in normalized_candidate_packet_stubs:
    line = nb18_canonical_json_bytes(packet_stub) + b"\n"
    restored_stub = json.loads(line)

    assert restored_stub == packet_stub
    restored_candidate_packet_stubs.append(restored_stub)
    normalized_candidate_packet_hasher.update(line)
    normalized_candidate_packet_bytes += len(line)

full_candidate_hasher = hashlib.sha256()
full_candidate_bytes = 0
candidate_round_trip_mismatch_count = 0

for original_packet, packet_stub in zip(
    nb18_ordered_policy_candidate_packets,
    restored_candidate_packet_stubs,
):
    full_line = nb18_canonical_json_bytes(original_packet) + b"\n"
    full_candidate_hasher.update(full_line)
    full_candidate_bytes += len(full_line)

    reconstructed_packet = dict(packet_stub["packet_metadata"])
    reconstructed_items = []

    for reference in packet_stub["evidence_refs_in_reranker_order"]:
        evidence_id = reference["evidence_id"]
        item = dict(restored_candidate_evidence_by_id[evidence_id])

        for field in rank_fields:
            item[field] = reference[field]

        reconstructed_items.append(item)

    reconstructed_packet["evidence_items"] = reconstructed_items

    if nb18_canonical_json_bytes(reconstructed_packet) != (
        nb18_canonical_json_bytes(original_packet)
    ):
        candidate_round_trip_mismatch_count += 1

normalized_candidate_total_bytes = (
    normalized_candidate_evidence_bytes
    + normalized_candidate_packet_bytes
)
candidate_bytes_saved = (
    full_candidate_bytes - normalized_candidate_total_bytes
)
candidate_size_reduction_percent = (
    100 * candidate_bytes_saved / full_candidate_bytes
    if full_candidate_bytes
    else 0.0
)

assert candidate_evidence_occurrence_count == 26623
assert len(normalized_candidate_packet_stubs) == 2664
assert candidate_round_trip_mismatch_count == 0

print(json.dumps({
    "status": "PASS",
    "candidate_packet_count": len(normalized_candidate_packet_stubs),
    "candidate_evidence_occurrence_count": (
        candidate_evidence_occurrence_count
    ),
    "unique_selected_evidence_count": (
        len(normalized_candidate_evidence_by_id)
    ),
    "full_packet_jsonl_bytes": full_candidate_bytes,
    "normalized_evidence_jsonl_bytes": normalized_candidate_evidence_bytes,
    "normalized_packet_map_jsonl_bytes": normalized_candidate_packet_bytes,
    "normalized_total_bytes": normalized_candidate_total_bytes,
    "bytes_saved": candidate_bytes_saved,
    "size_reduction_percent": round(
        candidate_size_reduction_percent,
        2,
    ),
    "full_packet_sha256": full_candidate_hasher.hexdigest(),
    "normalized_evidence_sha256": (
        normalized_candidate_evidence_hasher.hexdigest()
    ),
    "normalized_packet_map_sha256": (
        normalized_candidate_packet_hasher.hexdigest()
    ),
    "round_trip_reconstruction_mismatch_count": (
        candidate_round_trip_mismatch_count
    ),
    "file_written": False,
    "configuration_frozen": False,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PASS",
  "candidate_packet_count": 2664,
  "candidate_evidence_occurrence_count": 26623,
  "unique_selected_evidence_count": 3277,
  "full_packet_jsonl_bytes": 337078855,
  "normalized_evidence_jsonl_bytes": 45509955,
  "normalized_packet_map_jsonl_bytes": 11973235,
  "normalized_total_bytes": 57483190,
  "bytes_saved": 279595665,
  "size_reduction_percent": 82.95,
  "full_packet_sha256": "e7ab83252669dfe2574eed160b372385629ab98112b756cc2ff16fedfccfe4e8",
  "normalized_evidence_sha256": "0e6dccb9a234a517ea2560b7979c2f2ed96ee7465d0f82214364064751cf30fe",
  "normalized_packet_map_sha256": "0ce0d8235640a6be5420c785a8e113d33fe4509328e0760384d83133759fd99c",
  "round_trip_reconstruction_mismatch_count": 0,
  "file_written": false,
  "configuration_frozen": false,
  "test_evidence_or_metrics_used": false,
  "model_api_called": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

این سلول یک مانیفست درون‌حافظه‌ای برای نامزد فعلی NB18 می‌سازد. مانیفست، تحویل NB17، تنظیمات پیشنهادی، معیارهای Development و هش‌های نمایش نرمال‌شده را ثبت می‌کند.

این مانیفست پیش‌نویس است؛ مسیر خروجی تعیین نمی‌کند، تنظیمات را قفل نمی‌کند و هیچ فایلی نمی‌نویسد.

</div>

In [71]:
# NB18-C71 — Build the proposed NB18 candidate manifest in memory

assert "nb18_output_candidate_manifest" in globals()
assert "nb18_ordered_policy_candidate_packets" in globals()
assert "normalized_candidate_evidence_by_id" in globals()
assert "normalized_candidate_packet_stubs" in globals()
assert candidate_round_trip_mismatch_count == 0
assert len(nb18_ordered_policy_candidate_packets) == 2664

nb18_policy_candidate_manifest = {
    "artifact_type": "NB18_CONTEXT_BUILDER_OUTPUT_CANDIDATE",
    "schema_version": "1.0",
    "decision_status": "PROPOSED_NOT_FROZEN",
    "source_handoff": nb18_output_candidate_manifest["source_handoff"],
    "development_scope": {
        "development_packet_count": len(
            nb18_ordered_policy_candidate_packets
        ),
        "selected_evidence_item_occurrence_count": (
            candidate_evidence_occurrence_count
        ),
        "unique_selected_evidence_count": (
            len(normalized_candidate_evidence_by_id)
        ),
        "non_development_records_used": 0,
    },
    "proposed_policy": {
        "selection_order": "source_priority_order",
        "source_cap_per_source_id": 8,
        "document_cap_per_member_document_id": 6,
        "evidence_text_budget_tokens": 7371,
        "budget_measurement": (
            "Sum of separately tokenized evidence texts; no special tokens"
        ),
        "tokenizer": {
            "repository": "Qwen/Qwen3-30B-A3B",
            "class": "Qwen2TokenizerFast",
            "revision_status": "CONTENT_FINGERPRINT_ONLY",
            "backend_sha256": (
                "a63080b4b30dc861fe1b0e420c53a241304dc81aff3d9bb315e6e33e2a1ab794"
            ),
            "special_tokens_sha256": (
                "94f02d4e8b7a6569bef78a7ff2c3b40744a0fd54e603a9c5a5bdba4d05256a10"
            ),
        },
    },
    "development_evaluation": {
        "macro_gold_chunk_recall": macro_recall,
        "micro_gold_chunk_recall": micro_recall,
        "queries_with_any_gold_evidence": queries_with_gold_hit,
        "query_gold_hit_rate": query_hit_rate,
        "selected_item_count": selected_item_count,
        "dropped_item_count": 17,
        "contexts_with_dropped_items": 14,
    },
    "normalized_candidate": {
        "full_packet_jsonl_bytes": full_candidate_bytes,
        "full_packet_sha256": full_candidate_hasher.hexdigest(),
        "unique_evidence_jsonl_bytes": (
            normalized_candidate_evidence_bytes
        ),
        "unique_evidence_sha256": (
            normalized_candidate_evidence_hasher.hexdigest()
        ),
        "packet_map_jsonl_bytes": normalized_candidate_packet_bytes,
        "packet_map_sha256": (
            normalized_candidate_packet_hasher.hexdigest()
        ),
        "combined_normalized_bytes": normalized_candidate_total_bytes,
        "round_trip_reconstruction_mismatch_count": (
            candidate_round_trip_mismatch_count
        ),
    },
    "constraints": {
        "reranker_rank_order_preserved": True,
        "source_priority_order_stored_separately": True,
        "reranker_scores_preserved": True,
        "development_candidate_policy_applied_in_memory": True,
        "configuration_frozen": False,
        "test_unlocked": False,
        "test_records_used": 0,
        "rights_clearance_asserted": False,
        "topic_diversity_applied": False,
        "parent_expansion_applied": False,
        "semantic_conflict_assessed": False,
        "model_api_called": False,
        "paid_api_calls": 0,
    },
    "output_paths_assigned": False,
    "written_to_disk": False,
    "files_written": 0,
}

nb18_policy_candidate_manifest_bytes = nb18_canonical_json_bytes(
    nb18_policy_candidate_manifest
)
nb18_policy_candidate_manifest_sha256 = hashlib.sha256(
    nb18_policy_candidate_manifest_bytes
).hexdigest()

print(json.dumps({
    "status": "PROPOSAL_READY_NOT_FROZEN",
    "manifest_sha256": nb18_policy_candidate_manifest_sha256,
    "manifest": nb18_policy_candidate_manifest,
}, ensure_ascii=False, indent=2))

{
  "status": "PROPOSAL_READY_NOT_FROZEN",
  "manifest_sha256": "419146771f510fbe001ce3f9816c14a74dcb6e2f6e77a45e9a1a46cd21f0db89",
  "manifest": {
    "artifact_type": "NB18_CONTEXT_BUILDER_OUTPUT_CANDIDATE",
    "schema_version": "1.0",
    "decision_status": "PROPOSED_NOT_FROZEN",
    "source_handoff": {
      "path": "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json",
      "release_id": "rag_finance_phase2_reranker_v1",
      "configuration_sha256": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738",
      "candidate_artifact_path": "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl",
      "candidate_artifact_sha256": "d00285bbf3865fbd51041db1612b9fdc17e9778273ea6d136978906a07268b7a",
      "candidate_depth": 50,
      "consumer_top_k": 10
    },
    "development_scope": {
      "development_packet_count": 2664,
      "selected_evidence_item_occurrence_count": 26623,
      "unique_selected_evidence_count": 3277,
      "non_development_records_

<div dir="rtl" align="right">

این سلول وضعیت فعلی هدف‌های NB18 را پس از آزمایش‌های Development ثبت می‌کند. عبور آزمون‌های ساختاری و تطبیق داده به‌تنهایی به معنی تکمیل Notebook نیست.

سلول فقط گزارش درون‌حافظه‌ای می‌سازد؛ تنظیمات را قفل نمی‌کند و فایلی نمی‌نویسد.

</div>

In [72]:
# NB18-C72 — Refresh the NB18 objective completion audit

assert "nb18_policy_candidate_manifest" in globals()
assert "nb18_ordered_policy_candidate_packets" in globals()
assert "nb18_policy_candidate_manifest_sha256" in globals()

recomputed_manifest_sha256 = hashlib.sha256(
    nb18_canonical_json_bytes(nb18_policy_candidate_manifest)
).hexdigest()

assert recomputed_manifest_sha256 == nb18_policy_candidate_manifest_sha256

generation_ready_count = sum(
    packet.get("generation_ready") is True
    for packet in nb18_ordered_policy_candidate_packets
)
release_eligible_count = sum(
    packet.get("release_eligible") is True
    for packet in nb18_ordered_policy_candidate_packets
)

nb18_objective_completion_audit = {
    "status": "OBJECTIVE_INCOMPLETE",
    "execution_health_scope": (
        "Latest candidate fidelity and round-trip checks passed; "
        "this is not a notebook-wide Run All health check."
    ),
    "requirements": [
        {
            "requirement": "Preserve frozen reranker order and scores",
            "status": "PASS",
            "evidence": "C69: zero order, rank, and score mismatches.",
        },
        {
            "requirement": "Build Development context candidates with provenance",
            "status": "PASS",
            "evidence": (
                "C66-C69: 2664 Development packets validated in memory."
            ),
        },
        {
            "requirement": "Deduplicate evidence representation",
            "status": "PASS",
            "evidence": (
                "C70: 3277 unique evidence records and zero round-trip mismatches."
            ),
        },
        {
            "requirement": "Control source and document dominance",
            "status": "PARTIAL",
            "evidence": (
                "C62-C69 evaluated proposed caps of 8 per source and "
                "6 per member document; no policy was frozen."
            ),
        },
        {
            "requirement": "Apply useful topic diversity",
            "status": "BLOCKED",
            "evidence": (
                "C07 mapped only 2 of 19429 Development document IDs."
            ),
        },
        {
            "requirement": "Apply controlled parent expansion",
            "status": "BLOCKED",
            "evidence": (
                "C15 found zero direct parent chunk links in the candidate set."
            ),
        },
        {
            "requirement": "Apply a production tokenizer and context budget",
            "status": "PARTIAL",
            "evidence": (
                "Qwen tokenizer measured Development evidence text; "
                "7371-token evidence-only budget remains proposed, with "
                "provider limit and prompt/answer reserve unresolved."
            ),
        },
        {
            "requirement": "Preserve locator, source, attribution, and rights metadata",
            "status": "PASS",
            "evidence": (
                "C69 found zero text or metadata mismatches."
            ),
        },
        {
            "requirement": "Assess semantic evidence conflicts",
            "status": "PARTIAL",
            "evidence": (
                "Authority signals are preserved; semantic conflicts remain "
                "NOT_ASSESSED."
            ),
        },
        {
            "requirement": "Write and validate versioned output artifacts",
            "status": "MISSING",
            "evidence": (
                "C70-C71 validate only in-memory representations; output paths "
                "and final schemas are not assigned."
            ),
        },
        {
            "requirement": "Produce generation-ready packets",
            "status": "BLOCKED",
            "evidence": (
                "Current candidate rights and release gates remain uncleared."
            ),
        },
    ],
    "candidate_manifest_sha256": recomputed_manifest_sha256,
    "candidate_manifest_written": False,
    "configuration_frozen": False,
    "test_unlocked": False,
    "generation_ready_packet_count": generation_ready_count,
    "release_eligible_packet_count": release_eligible_count,
    "test_evidence_or_metrics_used": False,
    "model_api_called": False,
    "files_written": 0,
    "paid_api_calls": 0,
}

print(json.dumps(
    nb18_objective_completion_audit,
    ensure_ascii=False,
    indent=2,
))

{
  "status": "OBJECTIVE_INCOMPLETE",
  "execution_health_scope": "Latest candidate fidelity and round-trip checks passed; this is not a notebook-wide Run All health check.",
  "requirements": [
    {
      "requirement": "Preserve frozen reranker order and scores",
      "status": "PASS",
      "evidence": "C69: zero order, rank, and score mismatches."
    },
    {
      "requirement": "Build Development context candidates with provenance",
      "status": "PASS",
      "evidence": "C66-C69: 2664 Development packets validated in memory."
    },
    {
      "requirement": "Deduplicate evidence representation",
      "status": "PASS",
      "evidence": "C70: 3277 unique evidence records and zero round-trip mismatches."
    },
    {
      "requirement": "Control source and document dominance",
      "status": "PARTIAL",
      "evidence": "C62-C69 evaluated proposed caps of 8 per source and 6 per member document; no policy was frozen."
    },
    {
      "requirement": "Apply useful topic

<div dir="rtl" align="right">

این سلول سلامت ساختاری و اجرای نوت‌بوک را پس از افزودن سلول‌های جدید بررسی می‌کند: شماره‌گذاری Code Cellها، نبود خطای ذخیره‌شده، اجراشدن سلول‌های قبلی، بدون‌شماره‌بودن Markdownها و زبان انگلیسی کامنت‌های کد.

نتیجهٔ این بررسی از وضعیت تکمیل هدف‌های NB18 جداست.

</div>

In [73]:
# NB18-C73 — Final notebook execution and structure health check

notebook_path = (
    Path(
        "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
        "RAG-projects/RAG_finance"
    )
    / "notebooks"
    / "phase2"
    / "18_context_builder.ipynb"
)

assert notebook_path.is_file(), (
    f"Notebook file not found: {notebook_path}"
)

notebook_payload = json.loads(
    notebook_path.read_text(encoding="utf-8")
)
all_cells = notebook_payload.get("cells", [])

def nb18_cell_source(cell):
    source = cell.get("source", [])
    if isinstance(source, list):
        return "".join(source)
    return source if isinstance(source, str) else ""

code_cells = [
    cell for cell in all_cells
    if cell.get("cell_type") == "code"
]
markdown_cells = [
    cell for cell in all_cells
    if cell.get("cell_type") == "markdown"
]

code_number_pattern = re.compile(
    r"^\s*#\s*NB18-C(\d{2})\b"
)
code_numbers = []
unnumbered_code_positions = []

for position, cell in enumerate(code_cells, start=1):
    lines = nb18_cell_source(cell).splitlines()
    first_line = lines[0] if lines else ""
    match = code_number_pattern.match(first_line)

    if match:
        code_numbers.append(int(match.group(1)))
    else:
        unnumbered_code_positions.append(position)

expected_code_numbers = list(range(1, len(code_cells) + 1))
duplicate_code_numbers = sorted(
    number
    for number in set(code_numbers)
    if code_numbers.count(number) > 1
)
missing_code_numbers = sorted(
    set(expected_code_numbers) - set(code_numbers)
)
nonsequential_code_numbers = code_numbers != expected_code_numbers

markdown_number_pattern = re.compile(
    r"^\s{0,3}#{1,6}\s*(?:NB18-C|C)\d{2}\b",
    re.IGNORECASE,
)
numbered_markdown_positions = [
    position
    for position, cell in enumerate(markdown_cells, start=1)
    if any(
        markdown_number_pattern.match(line)
        for line in nb18_cell_source(cell).splitlines()
    )
]

persian_comment_pattern = re.compile(
    r"^\s*#.*[\u0600-\u06FF]"
)
persian_code_comment_positions = [
    position
    for position, cell in enumerate(code_cells, start=1)
    if any(
        persian_comment_pattern.match(line)
        for line in nb18_cell_source(cell).splitlines()
    )
]

unexecuted_prior_code_positions = [
    position
    for position, cell in enumerate(code_cells, start=1)
    if position < len(code_cells)
    and cell.get("execution_count") is None
]

saved_errors = []
for position, cell in enumerate(code_cells, start=1):
    for output in cell.get("outputs", []):
        if output.get("output_type") == "error":
            saved_errors.append({
                "code_cell_position": position,
                "ename": output.get("ename"),
                "evalue": output.get("evalue"),
            })

assert "nb18_policy_candidate_manifest" in globals()
assert "nb18_policy_candidate_manifest_sha256" in globals()
assert "nb18_objective_completion_audit" in globals()

recomputed_candidate_manifest_sha256 = hashlib.sha256(
    nb18_canonical_json_bytes(nb18_policy_candidate_manifest)
).hexdigest()
manifest_hash_matches = (
    recomputed_candidate_manifest_sha256
    == nb18_policy_candidate_manifest_sha256
)

structural_execution_health_pass = all([
    not unnumbered_code_positions,
    not duplicate_code_numbers,
    not missing_code_numbers,
    not nonsequential_code_numbers,
    not numbered_markdown_positions,
    not persian_code_comment_positions,
    not unexecuted_prior_code_positions,
    not saved_errors,
    manifest_hash_matches,
])

print(json.dumps({
    "status": (
        "PASS" if structural_execution_health_pass else "FAIL"
    ),
    "health_scope": (
        "Notebook structure and execution only; not objective completion."
    ),
    "notebook_path": str(notebook_path),
    "code_cell_count": len(code_cells),
    "code_cell_numbers": code_numbers,
    "unnumbered_code_positions": unnumbered_code_positions,
    "duplicate_code_numbers": duplicate_code_numbers,
    "missing_code_numbers": missing_code_numbers,
    "nonsequential_code_numbers": nonsequential_code_numbers,
    "unexecuted_prior_code_positions": unexecuted_prior_code_positions,
    "saved_error_count": len(saved_errors),
    "saved_errors": saved_errors,
    "markdown_cell_count": len(markdown_cells),
    "numbered_markdown_positions": numbered_markdown_positions,
    "code_cells_with_persian_comments": persian_code_comment_positions,
    "candidate_manifest_sha256_matches": manifest_hash_matches,
    "objective_status": (
        nb18_objective_completion_audit["status"]
    ),
    "configuration_frozen": False,
    "test_unlocked": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

print(
    "NOTEBOOK STRUCTURAL HEALTH: "
    + ("PASS" if structural_execution_health_pass else "FAIL")
)

{
  "status": "PASS",
  "health_scope": "Notebook structure and execution only; not objective completion.",
  "notebook_path": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/18_context_builder.ipynb",
  "code_cell_count": 73,
  "code_cell_numbers": [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23,
    24,
    25,
    26,
    27,
    28,
    29,
    30,
    31,
    32,
    33,
    34,
    35,
    36,
    37,
    38,
    39,
    40,
    41,
    42,
    43,
    44,
    45,
    46,
    47,
    48,
    49,
    50,
    51,
    52,
    53,
    54,
    55,
    56,
    57,
    58,
    59,
    60,
    61,
    62,
    63,
    64,
    65,
    66,
    67,
    68,
    69,
    70,
    71,
    72,
    73
  ],
  "unnumbered_code_positions": [],
  "duplicate_code_numbers": [],
  "missing_code_numbers": [],
  "nonsequential_code_number

In [74]:
# NB18-C74 — Read-only readiness audit for finalization

from pathlib import Path
import json

# از project_root خود Notebook استفاده می‌کند؛ مسیر را حدس نمی‌زند.
root = Path(project_root).resolve()

required_paths = {
    "reranker_candidates": "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl",
    "canonical_chunks": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
    "accepted_documents": "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl",
    "reranker_handoff": "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json",
    "reranker_configuration": "artifacts/phase2/reranking/reranker_configuration_v1.json",
    "reranker_release": "artifacts/phase2/reranking/reranker_release_v1.json",
    "reranker_manifest": "data/phase2/retrieval/reranker_v1/reranker_manifest.json",
    "reranker_reload_validation": "data/phase2/retrieval/reranker_v1/reranker_reload_validation_v1.json",
}

files = {}
for name, relative_path in required_paths.items():
    path = root / relative_path
    files[name] = {
        "path": relative_path,
        "exists": path.is_file(),
        "bytes": path.stat().st_size if path.is_file() else None,
    }

runtime_names = [
    "nb18_development_packets",
    "nb18_ordered_policy_candidate_packets",
    "normalized_candidate_evidence_by_id",
    "normalized_candidate_packet_stubs",
    "nb18_policy_candidate_manifest",
    "nb18_objective_completion_audit",
    "nb18_tokenizer",
    "validated_hashes",
]

runtime = {}
for name in runtime_names:
    value = globals().get(name)
    try:
        size = len(value) if value is not None else None
    except TypeError:
        size = None
    runtime[name] = {
        "exists": name in globals(),
        "type": type(value).__name__ if name in globals() else None,
        "length": size,
    }

print(json.dumps({
    "project_root": str(root),
    "required_project_files": files,
    "runtime_objects": runtime,
    "read_only": True,
    "test_labels_or_metrics_read": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "project_root": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance",
  "required_project_files": {
    "reranker_candidates": {
      "path": "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl",
      "exists": true,
      "bytes": 56417647
    },
    "canonical_chunks": {
      "path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
      "exists": true,
      "bytes": 37213608
    },
    "accepted_documents": {
      "path": "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl",
      "exists": true,
      "bytes": 63804043
    },
    "reranker_handoff": {
      "path": "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json",
      "exists": true,
      "bytes": 1303
    },
    "reranker_configuration": {
      "path": "artifacts/phase2/reranking/reranker_configuration_v1.json",
      "exists": true,
      "bytes": 6596
    },
    "reranker_release": {
      "path": "artifacts/phase2/reranking/reranker

<div dir="rtl" align="right">

# Notebook 18 — Context Builder
## NB18-C75 — تحلیل سیاست با داده‌های Development

### هدف
ارزیابی گزینه‌های ساخت Context با داده‌های Development בלבד، طبق قواعد הבחירה שנקבעו מראש.

### הגבלות
از داده‌ها، برچسب‌ها یا معیارهای Test استفاده نمی‌شود. تنظیمات Frozen تغییر نمی‌کند؛ فایلی نوشته نمی‌شود و API پولی فراخوانی نمی‌شود.

### خروجی مورد انتظار
گزارش گزینه‌های واجد شرایط و معیارهای Development برای تصمیم‌گیری دربارهٔ سیاست Context Builder.

</div>

In [76]:
# NB18-C75 — Development-only policy tradeoff audit

import json

assert "nb18_development_packets" in globals()
assert "development_gold_mapping" in globals()
assert "nb18_tokenizer" in globals()
assert "PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS" in globals()
assert "experiment_results" in globals()
assert "expanded_dominance_results" in globals()

nb18_c75_packets = nb18_development_packets
nb18_c75_gold = development_gold_mapping
nb18_c75_budget = PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS

assert len(nb18_c75_packets) == 2664
assert len(nb18_c75_gold) == 2664
assert all(packet.get("split") == "development" for packet in nb18_c75_packets)

nb18_c75_packet_by_id = {
    packet["candidate_id"]: packet
    for packet in nb18_c75_packets
}
assert len(nb18_c75_packet_by_id) == 2664
assert set(nb18_c75_packet_by_id) == set(nb18_c75_gold)

# محاسبه‌ی Token count یک‌بار برای هر Evidence؛ فقط از Candidateهای Development.
nb18_c75_token_count_by_id = {}
nb18_c75_payload_by_id = {}

for packet in nb18_c75_packets:
    for item in packet["evidence_items"]:
        evidence_id = item["evidence_id"]
        payload = item["text"]

        if evidence_id in nb18_c75_payload_by_id:
            assert nb18_c75_payload_by_id[evidence_id] == payload
        else:
            nb18_c75_payload_by_id[evidence_id] = payload
            nb18_c75_token_count_by_id[evidence_id] = len(
                nb18_tokenizer.encode(
                    payload,
                    add_special_tokens=False,
                )
            )

nb18_c75_cap_grid = {
    (
        result["source_cap_per_source_id"],
        result["document_cap_per_member_document_id"],
    )
    for result in experiment_results + expanded_dominance_results
}
assert nb18_c75_cap_grid
assert (8, 6) in nb18_c75_cap_grid

# Baseline بدون سقف و همه‌ی ترکیب‌های سقف بررسی‌شده‌ی Development.
nb18_c75_policies = [(None, None)] + sorted(nb18_c75_cap_grid)

nb18_c75_results = []

for source_cap, document_cap in nb18_c75_policies:
    macro_recall_values = []
    total_gold = 0
    total_retrieved_gold = 0
    query_hit_count = 0
    selected_occurrences = 0
    empty_context_count = 0
    budget_violation_count = 0
    reranker_order_mismatch_count = 0
    dropped_by_reason = {
        "source_cap": 0,
        "document_cap": 0,
        "evidence_text_budget": 0,
    }
    source_dominance_values = []
    document_dominance_values = []

    for candidate_id, packet in nb18_c75_packet_by_id.items():
        gold_ids = set(nb18_c75_gold[candidate_id])
        priority_items = sorted(
            packet["evidence_items"],
            key=lambda item: item["selection_priority_rank"],
        )

        kept = []
        source_counts = {}
        document_counts = {}
        selected_tokens = 0

        for item in priority_items:
            source = item.get("source") or {}
            source_id = source.get("source_id")
            source_key = (
                source_id
                if source_id not in (None, "")
                else f"__MISSING_SOURCE__:{item['evidence_id']}"
            )

            member_document_ids = set(item.get("member_document_ids") or [])

            if (
                source_cap is not None
                and source_counts.get(source_key, 0) >= source_cap
            ):
                dropped_by_reason["source_cap"] += 1
                continue

            if (
                document_cap is not None
                and any(
                    document_counts.get(document_id, 0) >= document_cap
                    for document_id in member_document_ids
                )
            ):
                dropped_by_reason["document_cap"] += 1
                continue

            item_tokens = nb18_c75_token_count_by_id[item["evidence_id"]]
            if selected_tokens + item_tokens > nb18_c75_budget:
                dropped_by_reason["evidence_text_budget"] += 1
                continue

            kept.append(item)
            selected_tokens += item_tokens
            source_counts[source_key] = source_counts.get(source_key, 0) + 1

            for document_id in member_document_ids:
                document_counts[document_id] = (
                    document_counts.get(document_id, 0) + 1
                )

        # فیلتر بر مبنای اولویت انجام شد؛ ترتیب Context خروجی به Reranker برمی‌گردد.
        kept_in_reranker_order = sorted(
            kept,
            key=lambda item: item["reranker_rank"],
        )
        reranker_ranks = [
            item["reranker_rank"] for item in kept_in_reranker_order
        ]

        if reranker_ranks != sorted(reranker_ranks):
            reranker_order_mismatch_count += 1

        selected_ids = {
            item["evidence_id"] for item in kept_in_reranker_order
        }
        retrieved_gold = len(selected_ids & gold_ids)

        total_gold += len(gold_ids)
        total_retrieved_gold += retrieved_gold
        query_hit_count += retrieved_gold > 0
        selected_occurrences += len(kept_in_reranker_order)
        empty_context_count += not kept_in_reranker_order

        if gold_ids:
            macro_recall_values.append(retrieved_gold / len(gold_ids))

        if kept_in_reranker_order:
            selected_count = len(kept_in_reranker_order)
            source_counts_for_share = {}
            document_counts_for_share = {}

            for item in kept_in_reranker_order:
                source = item.get("source") or {}
                source_id = source.get("source_id")
                source_key = (
                    source_id
                    if source_id not in (None, "")
                    else f"__MISSING_SOURCE__:{item['evidence_id']}"
                )
                source_counts_for_share[source_key] = (
                    source_counts_for_share.get(source_key, 0) + 1
                )

                for document_id in set(item.get("member_document_ids") or []):
                    document_counts_for_share[document_id] = (
                        document_counts_for_share.get(document_id, 0) + 1
                    )

            source_dominance_values.append(
                max(source_counts_for_share.values()) / selected_count
            )
            document_dominance_values.append(
                max(document_counts_for_share.values()) / selected_count
                if document_counts_for_share
                else 0.0
            )

            if selected_tokens > nb18_c75_budget:
                budget_violation_count += 1

    nb18_c75_results.append({
        "source_cap_per_source_id": source_cap,
        "document_cap_per_member_document_id": document_cap,
        "macro_gold_chunk_recall": (
            sum(macro_recall_values) / len(macro_recall_values)
            if macro_recall_values else None
        ),
        "micro_gold_chunk_recall": (
            total_retrieved_gold / total_gold if total_gold else None
        ),
        "query_gold_hit_rate": query_hit_count / len(nb18_c75_packets),
        "mean_selected_items_per_query": (
            selected_occurrences / len(nb18_c75_packets)
        ),
        "mean_max_source_share": (
            sum(source_dominance_values) / len(source_dominance_values)
            if source_dominance_values else None
        ),
        "mean_max_document_share": (
            sum(document_dominance_values) / len(document_dominance_values)
            if document_dominance_values else None
        ),
        "empty_context_count": empty_context_count,
        "budget_violation_count": budget_violation_count,
        "reranker_order_mismatch_count": reranker_order_mismatch_count,
        "dropped_by_reason": dropped_by_reason,
    })

print(json.dumps({
    "status": "DEVELOPMENT_POLICY_TRADEOFF_AUDIT_COMPLETE",
    "development_candidate_count": len(nb18_c75_packets),
    "evidence_only_token_budget": nb18_c75_budget,
    "selection_order": "reranker_order",
    "source_priority_used_only_for_cap_selection": True,
    "policy_selected": False,
    "configuration_frozen": False,
    "test_records_or_labels_read": False,
    "test_metrics_read": False,
    "files_written": 0,
    "paid_api_calls": 0,
    "policy_results": nb18_c75_results,
}, ensure_ascii=False, indent=2))

{
  "status": "DEVELOPMENT_POLICY_TRADEOFF_AUDIT_COMPLETE",
  "development_candidate_count": 2664,
  "evidence_only_token_budget": 7371,
  "selection_order": "reranker_order",
  "source_priority_used_only_for_cap_selection": true,
  "policy_selected": false,
  "configuration_frozen": false,
  "test_records_or_labels_read": false,
  "test_metrics_read": false,
  "files_written": 0,
  "paid_api_calls": 0,
  "policy_results": [
    {
      "source_cap_per_source_id": null,
      "document_cap_per_member_document_id": null,
      "macro_gold_chunk_recall": 0.6676743707993702,
      "micro_gold_chunk_recall": 0.6003134796238244,
      "query_gold_hit_rate": 0.8498498498498499,
      "mean_selected_items_per_query": 10.0,
      "mean_max_source_share": 0.23648648648648862,
      "mean_max_document_share": 0.17980480480480315,
      "empty_context_count": 0,
      "budget_violation_count": 0,
      "reranker_order_mismatch_count": 0,
      "dropped_by_reason": {
        "source_cap": 0,
     

<div dir="rtl" align="right">

# Notebook 18 — Context Builder
## NB18-C76 — انتخاب سیاست پیشنهادی با Development

### هدف
انتخاب سیاست Context Builder از میان گزینه‌های Development، با اعمال قواعد پذیرش و ترتیب بهینه‌سازی از پیش تعیین‌شده.

### محدودیت‌ها
این انتخاب فقط بر داده‌های Development متکی است. داده‌ها، برچسب‌ها و معیارهای Test خوانده نمی‌شوند؛ تنظیمات هنوز Freeze یا ذخیره نمی‌شوند.

### خروجی مورد انتظار
سیاست پیشنهادی و معیارهای Development آن، همراه با وضعیت واجدشرایط‌بودن.

</div>

In [77]:
# NB18-C76 — Apply the Development-only policy selection rule

import json

assert "nb18_c75_results" in globals()
assert len(nb18_c75_results) > 1

baseline = next(
    (
        result for result in nb18_c75_results
        if result["source_cap_per_source_id"] is None
        and result["document_cap_per_member_document_id"] is None
    ),
    None,
)
assert baseline is not None

tolerance = 1e-12
eligible = []

for result in nb18_c75_results:
    source_cap = result["source_cap_per_source_id"]
    document_cap = result["document_cap_per_member_document_id"]

    if source_cap is None or document_cap is None:
        continue

    quality_gates_pass = (
        result["empty_context_count"] == 0
        and result["budget_violation_count"] == 0
        and result["reranker_order_mismatch_count"] == 0
    )

    query_hit_not_lower = (
        result["query_gold_hit_rate"]
        >= baseline["query_gold_hit_rate"] - tolerance
    )

    source_dominance_reduced = (
        result["mean_max_source_share"]
        < baseline["mean_max_source_share"] - tolerance
    )

    document_dominance_reduced = (
        result["mean_max_document_share"]
        < baseline["mean_max_document_share"] - tolerance
    )

    if (
        quality_gates_pass
        and query_hit_not_lower
        and source_dominance_reduced
        and document_dominance_reduced
    ):
        eligible.append(result)

if eligible:
    proposed = max(
        eligible,
        key=lambda result: (
            result["macro_gold_chunk_recall"],
            result["micro_gold_chunk_recall"],
            result["mean_selected_items_per_query"],
            result["source_cap_per_source_id"],
            result["document_cap_per_member_document_id"],
        ),
    )
    proposed_policy = {
        "source_cap_per_source_id":
            proposed["source_cap_per_source_id"],
        "document_cap_per_member_document_id":
            proposed["document_cap_per_member_document_id"],
        "selection_order": "reranker_order",
        "source_priority_order":
            "stored_as_separate_metadata",
        "evidence_text_budget_tokens":
            PROPOSED_EVIDENCE_TEXT_BUDGET_TOKENS,
    }
else:
    proposed = None
    proposed_policy = None

print(json.dumps({
    "status": (
        "DEVELOPMENT_POLICY_CANDIDATE_SELECTED"
        if proposed is not None
        else "NO_POLICY_MET_DECLARED_RULE"
    ),
    "selection_rule": {
        "development_only": True,
        "query_hit_rate_not_below_uncapped": True,
        "source_dominance_strictly_reduced": True,
        "document_dominance_strictly_reduced": True,
        "empty_contexts": 0,
        "budget_violations": 0,
        "reranker_order_mismatches": 0,
        "optimization_order": [
            "maximize_macro_gold_chunk_recall",
            "maximize_micro_gold_chunk_recall",
            "maximize_mean_selected_items_per_query",
        ],
        "numeric_comparison_tolerance": tolerance,
    },
    "baseline": {
        "macro_gold_chunk_recall":
            baseline["macro_gold_chunk_recall"],
        "micro_gold_chunk_recall":
            baseline["micro_gold_chunk_recall"],
        "query_gold_hit_rate":
            baseline["query_gold_hit_rate"],
        "mean_max_source_share":
            baseline["mean_max_source_share"],
        "mean_max_document_share":
            baseline["mean_max_document_share"],
    },
    "eligible_policy_count": len(eligible),
    "proposed_policy": proposed_policy,
    "proposed_policy_development_metrics": proposed,
    "configuration_frozen": False,
    "test_records_or_labels_read": False,
    "test_metrics_read": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "DEVELOPMENT_POLICY_CANDIDATE_SELECTED",
  "selection_rule": {
    "development_only": true,
    "query_hit_rate_not_below_uncapped": true,
    "source_dominance_strictly_reduced": true,
    "document_dominance_strictly_reduced": true,
    "empty_contexts": 0,
    "budget_violations": 0,
    "reranker_order_mismatches": 0,
    "optimization_order": [
      "maximize_macro_gold_chunk_recall",
      "maximize_micro_gold_chunk_recall",
      "maximize_mean_selected_items_per_query"
    ],
    "numeric_comparison_tolerance": 1e-12
  },
  "baseline": {
    "macro_gold_chunk_recall": 0.6676743707993702,
    "micro_gold_chunk_recall": 0.6003134796238244,
    "query_gold_hit_rate": 0.8498498498498499,
    "mean_max_source_share": 0.23648648648648862,
    "mean_max_document_share": 0.17980480480480315
  },
  "eligible_policy_count": 2,
  "proposed_policy": {
    "source_cap_per_source_id": 8,
    "document_cap_per_member_document_id": 5,
    "selection_order": "reranker_order",
  

<div dir="rtl" align="right">

# Notebook 18 — Context Builder
## NB18-C77 — اعتبارسنجی سیاست پیشنهادی در حافظه

### هدف
ساخت دوبارهٔ Contextهای Development با سیاست پیشنهادی و بررسی تعداد آیتم‌ها، ترتیب، محدودیت‌های منبع و سند، و بودجهٔ توکن.

### محدودیت‌ها
اعتبارسنجی در حافظه انجام می‌شود. تنظیمات Freeze یا ذخیره نمی‌شوند؛ Test، برچسب‌ها و معیارهای آن خوانده نمی‌شوند.

### خروجی مورد انتظار
گزارش تعداد بسته‌ها و آیتم‌ها، موارد حذف‌شده و نقض‌های احتمالی، بدون نوشتن فایل یا فراخوانی API پولی.

</div>

In [78]:
# NB18-C77 — Build and validate the selected Development candidate

import json

assert "nb18_development_packets" in globals()
assert "gold_ids_by_candidate" in globals()
assert "nb18_tokenizer" in globals()
assert "proposed_policy" in globals()
assert "proposed" in globals()
assert proposed_policy is not None
assert proposed is not None

source_cap = proposed_policy["source_cap_per_source_id"]
document_cap = proposed_policy["document_cap_per_member_document_id"]
evidence_budget = proposed_policy["evidence_text_budget_tokens"]

assert source_cap == 8
assert document_cap == 5
assert proposed_policy["selection_order"] == "reranker_order"
assert (
    proposed_policy["source_priority_order"]
    == "stored_as_separate_metadata"
)

original_by_candidate = {
    packet["candidate_id"]: packet
    for packet in nb18_development_packets
}
assert len(original_by_candidate) == 2664
assert all(
    packet.get("split") == "development"
    for packet in nb18_development_packets
)

candidate_packets = []
selected_count = 0
dropped_count = 0
dropped_by_reason = {
    "SOURCE_CAP": 0,
    "DOCUMENT_CAP": 0,
    "EVIDENCE_TEXT_BUDGET": 0,
}
max_selected_tokens = 0
empty_context_count = 0

for packet in nb18_development_packets:
    original_items = packet["evidence_items"]
    original_ranked = sorted(
        original_items,
        key=lambda item: item["reranker_rank"],
    )
    priority_items = sorted(
        original_items,
        key=lambda item: item["selection_priority_rank"],
    )

    assert len(original_items) == 10
    assert len({
        item["evidence_id"] for item in original_items
    }) == 10

    source_counts = {}
    document_counts = {}
    selected_priority = []
    dropped_items = []
    selected_tokens = 0

    for item in priority_items:
        evidence_id = item["evidence_id"]
        source = item.get("source") or {}
        source_id = source.get("source_id")
        source_key = (
            source_id
            if source_id not in (None, "")
            else f"__MISSING_SOURCE__:{evidence_id}"
        )
        document_ids = set(item.get("member_document_ids") or [])

        if source_counts.get(source_key, 0) >= source_cap:
            reason = "SOURCE_CAP"
        elif any(
            document_counts.get(document_id, 0) >= document_cap
            for document_id in document_ids
        ):
            reason = "DOCUMENT_CAP"
        else:
            item_tokens = len(
                nb18_tokenizer.encode(
                    item["text"],
                    add_special_tokens=False,
                )
            )
            if selected_tokens + item_tokens > evidence_budget:
                reason = "EVIDENCE_TEXT_BUDGET"
            else:
                reason = None

        if reason is not None:
            dropped_items.append({
                "evidence_id": evidence_id,
                "reranker_rank": item["reranker_rank"],
                "reason": reason,
            })
            dropped_by_reason[reason] += 1
            continue

        selected_priority.append(item)
        selected_tokens += item_tokens
        source_counts[source_key] = source_counts.get(source_key, 0) + 1

        for document_id in document_ids:
            document_counts[document_id] = (
                document_counts.get(document_id, 0) + 1
            )

    selected_ranked = sorted(
        selected_priority,
        key=lambda item: item["reranker_rank"],
    )
    selected_ids = [item["evidence_id"] for item in selected_ranked]
    priority_ids = [
        item["evidence_id"] for item in selected_priority
    ]

    assert len(selected_ids) == len(set(selected_ids))
    assert selected_tokens <= evidence_budget

    candidate_packet = dict(packet)
    candidate_packet["evidence_items"] = selected_ranked
    candidate_packet["original_reranker_rank_order"] = [
        item["evidence_id"] for item in original_ranked
    ]
    candidate_packet["reranker_rank_order"] = selected_ids
    candidate_packet["source_priority_order"] = priority_ids
    candidate_packet["selection_policy_candidate"] = {
        "status": "PROPOSED_NOT_FROZEN",
        "source_cap_per_source_id": source_cap,
        "document_cap_per_member_document_id": document_cap,
        "evidence_text_budget_tokens": evidence_budget,
        "selection_order": "reranker_order",
        "source_priority_order": "stored_as_separate_metadata",
        "budget_measurement": (
            "Sum of separately tokenized evidence texts; no special tokens"
        ),
    }
    candidate_packet["selected_evidence_text_tokens_additive"] = (
        selected_tokens
    )
    candidate_packet["dropped_evidence_items"] = dropped_items

    candidate_packets.append(candidate_packet)
    selected_count += len(selected_ranked)
    dropped_count += len(dropped_items)
    max_selected_tokens = max(max_selected_tokens, selected_tokens)
    empty_context_count += not selected_ranked

assert len(candidate_packets) == 2664
assert selected_count + dropped_count == 2664 * 10

candidate_by_id = {
    packet["candidate_id"]: packet
    for packet in candidate_packets
}
assert set(candidate_by_id) == set(original_by_candidate)

rank_mismatch_count = 0
score_mismatch_count = 0
payload_mismatch_count = 0
order_mismatch_count = 0
source_cap_violation_count = 0
document_cap_violation_count = 0
budget_violation_count = 0
selected_gold_count = 0
total_gold_count = 0
queries_with_gold_hit = 0
macro_recall_values = []

for candidate_id, candidate_packet in candidate_by_id.items():
    original_packet = original_by_candidate[candidate_id]
    original_item_by_id = {
        item["evidence_id"]: item
        for item in original_packet["evidence_items"]
    }
    selected_items = candidate_packet["evidence_items"]
    selected_item_by_id = {
        item["evidence_id"]: item for item in selected_items
    }
    dropped_item_ids = {
        item["evidence_id"]
        for item in candidate_packet["dropped_evidence_items"]
    }

    assert (
        set(selected_item_by_id) | dropped_item_ids
        == set(original_item_by_id)
    )
    assert not (set(selected_item_by_id) & dropped_item_ids)

    if [
        item["reranker_rank"] for item in selected_items
    ] != sorted(item["reranker_rank"] for item in selected_items):
        order_mismatch_count += 1

    if candidate_packet["reranker_rank_order"] != [
        item["evidence_id"] for item in selected_items
    ]:
        order_mismatch_count += 1

    expected_priority_ids = [
        item["evidence_id"]
        for item in sorted(
            selected_items,
            key=lambda item: item["selection_priority_rank"],
        )
    ]
    if candidate_packet["source_priority_order"] != expected_priority_ids:
        order_mismatch_count += 1

    source_counts = {}
    document_counts = {}
    packet_tokens = 0

    for item in selected_items:
        original_item = original_item_by_id[item["evidence_id"]]

        rank_mismatch_count += (
            item["reranker_rank"] != original_item["reranker_rank"]
        )
        score_mismatch_count += (
            item["reranker_score"] != original_item["reranker_score"]
        )
        payload_mismatch_count += (
            json.dumps(
                item,
                ensure_ascii=False,
                sort_keys=True,
                separators=(",", ":"),
                allow_nan=False,
            )
            != json.dumps(
                original_item,
                ensure_ascii=False,
                sort_keys=True,
                separators=(",", ":"),
                allow_nan=False,
            )
        )

        source = item.get("source") or {}
        source_id = source.get("source_id")
        source_key = (
            source_id
            if source_id not in (None, "")
            else f"__MISSING_SOURCE__:{item['evidence_id']}"
        )
        source_counts[source_key] = source_counts.get(source_key, 0) + 1

        for document_id in set(item.get("member_document_ids") or []):
            document_counts[document_id] = (
                document_counts.get(document_id, 0) + 1
            )

        packet_tokens += len(
            nb18_tokenizer.encode(
                item["text"],
                add_special_tokens=False,
            )
        )

    source_cap_violation_count += any(
        count > source_cap for count in source_counts.values()
    )
    document_cap_violation_count += any(
        count > document_cap for count in document_counts.values()
    )
    budget_violation_count += packet_tokens > evidence_budget

    gold_ids = set(gold_ids_by_candidate[candidate_id])
    retrieved_gold = len(set(selected_item_by_id) & gold_ids)
    total_gold_count += len(gold_ids)
    selected_gold_count += retrieved_gold
    queries_with_gold_hit += retrieved_gold > 0

    if gold_ids:
        macro_recall_values.append(retrieved_gold / len(gold_ids))

macro_recall = (
    sum(macro_recall_values) / len(macro_recall_values)
    if macro_recall_values else None
)
micro_recall = (
    selected_gold_count / total_gold_count
    if total_gold_count else None
)
query_hit_rate = queries_with_gold_hit / len(candidate_by_id)

assert rank_mismatch_count == 0
assert score_mismatch_count == 0
assert payload_mismatch_count == 0
assert order_mismatch_count == 0
assert source_cap_violation_count == 0
assert document_cap_violation_count == 0
assert budget_violation_count == 0
assert empty_context_count == 0

assert abs(macro_recall - proposed["macro_gold_chunk_recall"]) < 1e-12
assert abs(micro_recall - proposed["micro_gold_chunk_recall"]) < 1e-12
assert abs(query_hit_rate - proposed["query_gold_hit_rate"]) < 1e-12

nb18_c77_policy_candidate_packets = candidate_packets

print(json.dumps({
    "status": "DEVELOPMENT_POLICY_CANDIDATE_VALIDATED_IN_MEMORY",
    "development_packet_count": len(candidate_packets),
    "source_cap_per_source_id": source_cap,
    "document_cap_per_member_document_id": document_cap,
    "evidence_text_budget_tokens": evidence_budget,
    "selected_item_count": selected_count,
    "dropped_item_count": dropped_count,
    "dropped_by_reason": dropped_by_reason,
    "max_selected_evidence_tokens": max_selected_tokens,
    "macro_gold_chunk_recall": macro_recall,
    "micro_gold_chunk_recall": micro_recall,
    "query_gold_hit_rate": query_hit_rate,
    "empty_context_count": empty_context_count,
    "rank_mismatch_count": rank_mismatch_count,
    "score_mismatch_count": score_mismatch_count,
    "payload_mismatch_count": payload_mismatch_count,
    "order_mismatch_count": order_mismatch_count,
    "source_cap_violation_count": source_cap_violation_count,
    "document_cap_violation_count": document_cap_violation_count,
    "budget_violation_count": budget_violation_count,
    "configuration_frozen": False,
    "test_records_or_labels_read": False,
    "test_metrics_read": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "DEVELOPMENT_POLICY_CANDIDATE_VALIDATED_IN_MEMORY",
  "development_packet_count": 2664,
  "source_cap_per_source_id": 8,
  "document_cap_per_member_document_id": 5,
  "evidence_text_budget_tokens": 7371,
  "selected_item_count": 26616,
  "dropped_item_count": 24,
  "dropped_by_reason": {
    "SOURCE_CAP": 13,
    "DOCUMENT_CAP": 11,
    "EVIDENCE_TEXT_BUDGET": 0
  },
  "max_selected_evidence_tokens": 7363,
  "macro_gold_chunk_recall": 0.6671023702273697,
  "micro_gold_chunk_recall": 0.5981583072100314,
  "query_gold_hit_rate": 0.8498498498498499,
  "empty_context_count": 0,
  "rank_mismatch_count": 0,
  "score_mismatch_count": 0,
  "payload_mismatch_count": 0,
  "order_mismatch_count": 0,
  "source_cap_violation_count": 0,
  "document_cap_violation_count": 0,
  "budget_violation_count": 0,
  "configuration_frozen": false,
  "test_records_or_labels_read": false,
  "test_metrics_read": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

# Notebook 18 — Context Builder
## NB18-C78 — Freeze تنظیمات در حافظه

### هدف
ثبت تنظیمات منتخب Context Builder در حافظه و محاسبهٔ هش پیکربندی، پس از موفقیت اعتبارسنجی Development.

### محدودیت‌ها
این مرحله هنوز Artifact نمی‌نویسد. داده‌ها، برچسب‌ها و معیارهای Test خوانده نمی‌شوند و API پولی فراخوانی نمی‌شود.

### خروجی مورد انتظار
پیکربندی Frozen در حافظه، هش آن و تأیید قفل‌بودن دسترسی به Test.

</div>

In [79]:
# NB18-C78 — Freeze the Development-selected configuration in memory

import hashlib
import json
import re
from pathlib import Path

assert "proposed_policy" in globals() and proposed_policy is not None
assert "proposed" in globals() and proposed is not None
assert "nb18_c77_policy_candidate_packets" in globals()
assert len(nb18_c77_policy_candidate_packets) == 2664
assert "nb18_policy_candidate_manifest" in globals()
assert "validated_hashes" in globals()
assert "project_root" in globals()

root = Path(project_root).resolve()
handoff_path = (
    root / "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json"
)
assert handoff_path.is_file()

handoff_bytes = handoff_path.read_bytes()
handoff = json.loads(handoff_bytes.decode("utf-8"))
handoff_sha256 = hashlib.sha256(handoff_bytes).hexdigest()

tokenizer_info = (
    nb18_policy_candidate_manifest["proposed_policy"]["tokenizer"]
)
backend_sha256 = tokenizer_info.get("backend_sha256")
special_tokens_sha256 = tokenizer_info.get("special_tokens_sha256")

assert re.fullmatch(r"[0-9a-f]{64}", backend_sha256 or "")
assert re.fullmatch(r"[0-9a-f]{64}", special_tokens_sha256 or "")

assert proposed_policy["source_cap_per_source_id"] == 8
assert proposed_policy["document_cap_per_member_document_id"] == 5
assert proposed_policy["selection_order"] == "reranker_order"
assert (
    proposed_policy["source_priority_order"]
    == "stored_as_separate_metadata"
)
assert proposed_policy["evidence_text_budget_tokens"] == 7371

configuration = {
    "artifact_type": "NB18_CONTEXT_BUILDER_CONFIGURATION",
    "schema_version": "1.0",
    "release_id": "rag_finance_phase2_context_builder_v1",
    "configuration_status": "FROZEN_BEFORE_TEST_ACCESS",
    "selection_policy": {
        "selection_order": "reranker_order",
        "source_priority_order": "stored_as_separate_metadata",
        "source_cap_per_source_id": 8,
        "document_cap_per_member_document_id": 5,
    },
    "development_selection_rule": {
        "development_only": True,
        "query_hit_rate_not_below_uncapped": True,
        "source_dominance_strictly_reduced": True,
        "document_dominance_strictly_reduced": True,
        "empty_contexts": 0,
        "budget_violations": 0,
        "reranker_order_mismatches": 0,
        "optimization_order": [
            "maximize_macro_gold_chunk_recall",
            "maximize_micro_gold_chunk_recall",
            "maximize_mean_selected_items_per_query",
        ],
        "numeric_comparison_tolerance": 1e-12,
        "selected_development_metrics": {
            "macro_gold_chunk_recall":
                proposed["macro_gold_chunk_recall"],
            "micro_gold_chunk_recall":
                proposed["micro_gold_chunk_recall"],
            "query_gold_hit_rate":
                proposed["query_gold_hit_rate"],
            "mean_max_source_share":
                proposed["mean_max_source_share"],
            "mean_max_document_share":
                proposed["mean_max_document_share"],
            "selected_item_count": 26616,
            "dropped_item_count": 24,
        },
    },
    "token_budget": {
        "name": "EVIDENCE_ONLY_TOKEN_BUDGET",
        "tokens": 7371,
        "measurement": (
            "Sum of separately tokenized evidence texts; no special tokens"
        ),
        "excludes": [
            "system_prompt",
            "instruction",
            "citation_wrapper",
            "metadata_wrapper",
            "special_tokens",
            "answer_reserve",
        ],
        "tokenizer": {
            "repository": tokenizer_info["repository"],
            "class": tokenizer_info["class"],
            "revision_status": tokenizer_info["revision_status"],
            "backend_sha256": backend_sha256,
            "special_tokens_sha256": special_tokens_sha256,
        },
        "provider_context_limit": None,
        "provider_limit_not_inferred": True,
    },
    "input_provenance": {
        "source_handoff_path":
            "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json",
        "source_handoff_sha256": handoff_sha256,
        "source_release_id": handoff.get("release_id"),
        "candidate_sha256":
            handoff["candidate_artifact"]["sha256"],
        "validated_input_hashes": dict(validated_hashes),
    },
    "metadata_limitations": {
        "topic_diversity_status":
            "NOT_APPLICABLE_DUE_TO_UPSTREAM_METADATA_LIMITATION",
        "parent_expansion_status":
            "NOT_APPLICABLE_NO_CANONICAL_PARENT_LINKS",
        "semantic_conflict_status":
            "NOT_ASSESSED_NO_CANONICAL_CONFLICT_MAPPING",
    },
    "rights_gates": {
        "internal_generation_ready_requires": [
            "technical_context_gates_pass",
            "allowed_uses_explicitly_permits_internal_processing",
        ],
        "unknown_internal_allowed_uses_requires_review": True,
        "external_provider_allowed_requires_explicit_evidence": True,
        "public_release_eligible": False,
        "public_release_block_reason": "RIGHTS_EVIDENCE_REQUIRED",
    },
    "test_access": {
        "status": "LOCKED_NOT_ACCESSED",
        "test_records_read": 0,
        "test_labels_or_metrics_read": False,
        "test_used_for_retuning": False,
    },
    "constraints": {
        "reranker_rank_and_score_modified": False,
        "paid_api_calls": 0,
        "model_api_called": False,
    },
}

configuration_bytes = json.dumps(
    configuration,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8")

configuration_sha256 = hashlib.sha256(configuration_bytes).hexdigest()

# Keep the frozen snapshot separate; later cells should read it, not mutate it.
nb18_frozen_configuration = json.loads(
    configuration_bytes.decode("utf-8")
)
nb18_frozen_configuration_sha256 = configuration_sha256

assert hashlib.sha256(
    json.dumps(
        nb18_frozen_configuration,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
).hexdigest() == nb18_frozen_configuration_sha256

assert nb18_frozen_configuration["test_access"]["status"] == (
    "LOCKED_NOT_ACCESSED"
)
assert nb18_frozen_configuration["configuration_status"] == (
    "FROZEN_BEFORE_TEST_ACCESS"
)

print(json.dumps({
    "status": "CONFIGURATION_FROZEN_IN_MEMORY",
    "configuration_sha256": nb18_frozen_configuration_sha256,
    "selected_policy": nb18_frozen_configuration["selection_policy"],
    "evidence_only_token_budget":
        nb18_frozen_configuration["token_budget"]["tokens"],
    "tokenizer_revision_status":
        nb18_frozen_configuration["token_budget"]["tokenizer"][
            "revision_status"
        ],
    "test_access_status":
        nb18_frozen_configuration["test_access"]["status"],
    "rights_gates_separated": True,
    "artifact_written": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "CONFIGURATION_FROZEN_IN_MEMORY",
  "configuration_sha256": "7e52a8c5c7affb7411d4b4cb8d296188fb87f5f380a12f05baf1c64f96f69d6a",
  "selected_policy": {
    "document_cap_per_member_document_id": 5,
    "selection_order": "reranker_order",
    "source_cap_per_source_id": 8,
    "source_priority_order": "stored_as_separate_metadata"
  },
  "evidence_only_token_budget": 7371,
  "tokenizer_revision_status": "CONTENT_FINGERPRINT_ONLY",
  "test_access_status": "LOCKED_NOT_ACCESSED",
  "rights_gates_separated": true,
  "artifact_written": false,
  "files_written": 0,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

# Notebook 18 — Context Builder
## NB18-C79 — ذخیره و بازخوانی پیکربندی و ADR

### هدف
ذخیرهٔ پیکربندی Frozen و رکورد تصمیم‌گیری ADR، سپس بازخوانی آن‌ها و اعتبارسنجی هش‌ها پیش از دسترسی به Test.

### محدودیت‌ها
این سل در مسیرهای Artifact و گزارش مشخص‌شده می‌نویسد. داده‌ها، برچسب‌ها و معیارهای Test خوانده نمی‌شوند و API پولی فراخوانی نمی‌شود.

### خروجی مورد انتظار
تأیید ذخیره و بازخوانی پیکربندی و ADR، همراه با هش‌های ثبت‌شده.

</div>

In [80]:
# NB18-C79 — Persist the pre-Test configuration and ADR

import hashlib
import json
import os
import tempfile
from pathlib import Path

assert "nb18_frozen_configuration" in globals()
assert "nb18_frozen_configuration_sha256" in globals()
assert "proposed_policy" in globals() and proposed_policy is not None
assert "proposed" in globals() and proposed is not None
assert "validated_hashes" in globals()
assert "project_root" in globals()

root = Path(project_root).resolve()

input_paths = {
    "nb17_handoff":
        "artifacts/phase2/reranking/nb18_reranker_handoff_v1.json",
    "reranker_configuration":
        "artifacts/phase2/reranking/reranker_configuration_v1.json",
    "reranker_release":
        "artifacts/phase2/reranking/reranker_release_v1.json",
    "reranker_manifest":
        "data/phase2/retrieval/reranker_v1/reranker_manifest.json",
    "reranker_reload_validation":
        "data/phase2/retrieval/reranker_v1/reranker_reload_validation_v1.json",
    "reranker_acceptance":
        "reports/phase2/reranker_acceptance_v1.json",
    "reranked_candidates":
        "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl",
    "canonical_chunks":
        "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
    "golden_dataset":
        "artifacts/phase2/golden_dataset_v2/golden_dataset_v2.jsonl",
}

assert set(input_paths) == set(validated_hashes)
assert all((root / path).is_file() for path in input_paths.values())

def canonical_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

final_configuration = json.loads(
    json.dumps(
        nb18_frozen_configuration,
        ensure_ascii=False,
        allow_nan=False,
    )
)
final_configuration["input_provenance"]["input_paths"] = input_paths
final_configuration["record_counts"] = {
    "development_candidates": 2664,
    "test_candidates_expected": 1149,
    "all_candidates_expected": 3813,
}
final_configuration["final_status"] = "FROZEN_BEFORE_TEST_ACCESS"

configuration_bytes = canonical_bytes(final_configuration)
configuration_sha256 = hashlib.sha256(configuration_bytes).hexdigest()

baseline = next(
    result for result in nb18_c75_results
    if result["source_cap_per_source_id"] is None
    and result["document_cap_per_member_document_id"] is None
)

adr = {
    "artifact_type": "NB18_CONTEXT_BUILDER_ADR",
    "schema_version": "1.0",
    "release_id": "rag_finance_phase2_context_builder_v1",
    "decision_status": "FROZEN_BEFORE_TEST_ACCESS",
    "decision": {
        "selection_rule":
            final_configuration["development_selection_rule"],
        "selected_policy":
            final_configuration["selection_policy"],
        "selection_reason": (
            "This was the eligible policy with the highest Development "
            "macro recall, followed by micro recall, while preserving "
            "query hit rate and reducing both measured dominance values."
        ),
        "baseline_metrics": {
            key: baseline[key]
            for key in (
                "macro_gold_chunk_recall",
                "micro_gold_chunk_recall",
                "query_gold_hit_rate",
                "mean_max_source_share",
                "mean_max_document_share",
            )
        },
        "selected_policy_metrics": {
            key: proposed[key]
            for key in (
                "macro_gold_chunk_recall",
                "micro_gold_chunk_recall",
                "query_gold_hit_rate",
                "mean_max_source_share",
                "mean_max_document_share",
                "mean_selected_items_per_query",
                "empty_context_count",
                "budget_violation_count",
                "reranker_order_mismatch_count",
                "dropped_by_reason",
            )
        },
        "eligible_policy_count": 2,
    },
    "input_paths": input_paths,
    "input_sha256": dict(validated_hashes),
    "configuration_sha256": configuration_sha256,
    "record_counts": final_configuration["record_counts"],
    "limitations": final_configuration["metadata_limitations"],
    "test_access_status": "LOCKED_NOT_ACCESSED",
    "rights_and_release_status": final_configuration["rights_gates"],
    "final_status": "CONFIGURATION_FROZEN_BEFORE_TEST_ACCESS",
}

artifact_dir = root / "artifacts/phase2/context_builder"
report_dir = root / "reports/phase2"
configuration_path = (
    artifact_dir / "context_builder_configuration_v1.json"
)
adr_path = report_dir / "context_builder_adr_v1.json"

def write_json_atomically(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    data = canonical_bytes(payload) + b"\n"

    if path.exists():
        existing = path.read_bytes()
        if existing == data:
            return {
                "path": str(path.relative_to(root)),
                "status": "ALREADY_PRESENT_IDENTICAL",
                "sha256": hashlib.sha256(existing).hexdigest(),
            }
        raise FileExistsError(
            f"Refusing to overwrite different existing file: {path}"
        )

    descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        suffix=".tmp",
        dir=path.parent,
    )
    temporary_path = Path(temporary_name)

    try:
        with os.fdopen(descriptor, "wb") as file_handle:
            file_handle.write(data)
            file_handle.flush()
            os.fsync(file_handle.fileno())

        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    written = path.read_bytes()
    assert written == data

    return {
        "path": str(path.relative_to(root)),
        "status": "WRITTEN_AND_RELOADED",
        "sha256": hashlib.sha256(written).hexdigest(),
    }

configuration_result = write_json_atomically(
    configuration_path,
    final_configuration,
)
adr_result = write_json_atomically(adr_path, adr)

# Update the in-memory frozen snapshot to the exact persisted configuration.
nb18_frozen_configuration = final_configuration
nb18_frozen_configuration_sha256 = configuration_sha256

assert adr["configuration_sha256"] == nb18_frozen_configuration_sha256
assert adr["test_access_status"] == "LOCKED_NOT_ACCESSED"

print(json.dumps({
    "status": "PRE_TEST_CONFIGURATION_AND_ADR_PERSISTED",
    "configuration": configuration_result,
    "adr": adr_result,
    "configuration_sha256": nb18_frozen_configuration_sha256,
    "test_records_or_labels_read": False,
    "test_metrics_read": False,
    "notebook_modified": False,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "PRE_TEST_CONFIGURATION_AND_ADR_PERSISTED",
  "configuration": {
    "path": "artifacts/phase2/context_builder/context_builder_configuration_v1.json",
    "status": "WRITTEN_AND_RELOADED",
    "sha256": "25e79ea6653fcc292923435749143438c6fc9ac0973f0dc9e5b9826cd4563b61"
  },
  "adr": {
    "path": "reports/phase2/context_builder_adr_v1.json",
    "status": "WRITTEN_AND_RELOADED",
    "sha256": "ff55ad9a2c009878dd05cebc1a86c0a751167f13a13bbfc5409dece8e3002dc3"
  },
  "configuration_sha256": "0794d5e87d832f15c4e08d53ec6fca8df1e25e2d343443a6fb7911b10353e239",
  "test_records_or_labels_read": false,
  "test_metrics_read": false,
  "notebook_modified": false,
  "paid_api_calls": 0
}


<div dir="rtl" align="right">

# Notebook 18 — Context Builder
## NB18-C80 — بررسی ساختاری Candidateهای Test

### هدف
تأیید هش canonical پیکربندی و بررسی تعداد رکوردها و Schema فایل Candidateهای Reranker، از جمله رکوردهای Test.

### محدودیت‌ها
فقط ساختار و شمارش رکوردها بررسی می‌شود؛ متن سؤال و شواهد، برچسب‌ها و معیارهای Test چاپ یا تحلیل نمی‌شوند. فایلی نوشته نمی‌شود.

### خروجی مورد انتظار
تأیید هش پیکربندی، تعداد رکوردهای هر split و نام فیلدهای رکوردها و Candidateها.

</div>

In [81]:
# NB18-C80 — Read-only Test candidate schema preflight

from pathlib import Path
import hashlib
import json

root = Path(project_root).resolve()

expected_configuration_sha256 = (
    "0794d5e87d832f15c4e08d53ec6fca8df1e25e2d343443a6fb7911b10353e239"
)
configuration_path = root / (
    "artifacts/phase2/context_builder/context_builder_configuration_v1.json"
)
adr_path = root / "reports/phase2/context_builder_adr_v1.json"
candidates_path = root / (
    "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl"
)
accepted_documents_path = root / (
    "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl"
)
canonical_chunks_path = root / (
    "data/phase2/chunks/nb11_benchmark_v1/"
    "baseline_phase1_char_safe_v1.jsonl"
)

for required_path in (
    configuration_path,
    adr_path,
    candidates_path,
    accepted_documents_path,
    canonical_chunks_path,
):
    assert required_path.is_file(), f"Required input file is missing: {required_path}"

with configuration_path.open("r", encoding="utf-8") as f:
    persisted_configuration = json.load(f)

canonical_configuration_bytes = json.dumps(
    persisted_configuration,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")
canonical_configuration_sha256 = hashlib.sha256(
    canonical_configuration_bytes
).hexdigest()
assert canonical_configuration_sha256 == expected_configuration_sha256, (
    "Persisted configuration canonical hash does not match the C79 output."
)

with adr_path.open("r", encoding="utf-8") as f:
    persisted_adr = json.load(f)

def contains_value(obj, expected):
    if isinstance(obj, dict):
        return any(contains_value(value, expected) for value in obj.values())
    if isinstance(obj, list):
        return any(contains_value(value, expected) for value in obj)
    return obj == expected

adr_references_configuration = contains_value(
    persisted_adr, expected_configuration_sha256
)
assert adr_references_configuration, (
    "The persisted ADR does not reference the frozen configuration hash."
)

split_counts = {}
top_level_key_union = set()
test_top_level_keys = set()
test_candidate_array_fields = set()
test_candidate_item_key_union = set()
test_candidate_lengths = []
candidate_record_count = 0

with candidates_path.open("rb") as f:
    for raw_line in f:
        if not raw_line.strip():
            continue

        record = json.loads(raw_line)
        candidate_record_count += 1
        split = str(record.get("split", "MISSING"))
        split_counts[split] = split_counts.get(split, 0) + 1
        top_level_key_union.update(record.keys())

        if split != "test":
            continue

        test_top_level_keys.update(record.keys())

        for key, value in record.items():
            if (
                isinstance(value, list)
                and value
                and all(isinstance(item, dict) for item in value[:3])
            ):
                test_candidate_array_fields.add(key)
                test_candidate_lengths.append((key, len(value)))
                for item in value[:3]:
                    test_candidate_item_key_union.update(item.keys())

assert split_counts.get("test", 0) == 1149, (
    f"Expected 1149 Test candidate records; found {split_counts.get('test', 0)}."
)
assert candidate_record_count == 3813, (
    f"Expected 3813 total candidate records; found {candidate_record_count}."
)

def first_jsonl_record_keys(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                value = json.loads(line)
                return sorted(value.keys()) if isinstance(value, dict) else type(value).__name__
    return None

runtime_objects = {}
for name in (
    "nb18_frozen_configuration",
    "nb18_frozen_configuration_sha256",
    "chunk_record_by_id",
    "nb18_tokenizer",
):
    value = globals().get(name)
    runtime_objects[name] = {
        "exists": name in globals(),
        "type": type(value).__name__ if name in globals() else None,
        "length": (
            len(value)
            if name in globals() and hasattr(value, "__len__")
            else None
        ),
    }

print(json.dumps({
    "status": "TEST_CANDIDATE_SCHEMA_PREFLIGHT_PASS",
    "configuration_canonical_sha256": canonical_configuration_sha256,
    "adr_references_frozen_configuration": adr_references_configuration,
    "candidate_record_count": candidate_record_count,
    "split_counts": split_counts,
    "candidate_top_level_keys": sorted(top_level_key_union),
    "test_top_level_keys": sorted(test_top_level_keys),
    "test_candidate_array_fields": sorted(test_candidate_array_fields),
    "test_candidate_item_keys": sorted(test_candidate_item_key_union),
    "test_candidate_array_length_samples": test_candidate_lengths[:5],
    "accepted_document_first_record_keys": first_jsonl_record_keys(
        accepted_documents_path
    ),
    "canonical_chunk_first_record_keys": first_jsonl_record_keys(
        canonical_chunks_path
    ),
    "runtime_objects": runtime_objects,
    "test_record_text_or_labels_printed": False,
    "test_metrics_read": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "TEST_CANDIDATE_SCHEMA_PREFLIGHT_PASS",
  "configuration_canonical_sha256": "0794d5e87d832f15c4e08d53ec6fca8df1e25e2d343443a6fb7911b10353e239",
  "adr_references_frozen_configuration": true,
  "candidate_record_count": 3813,
  "split_counts": {
    "development": 2664,
    "test": 1149
  },
  "candidate_top_level_keys": [
    "candidate_depth",
    "candidate_id",
    "candidates",
    "configuration_sha256",
    "final_top_k",
    "release_id",
    "source_type",
    "split"
  ],
  "test_top_level_keys": [
    "candidate_depth",
    "candidate_id",
    "candidates",
    "configuration_sha256",
    "final_top_k",
    "release_id",
    "source_type",
    "split"
  ],
  "test_candidate_array_fields": [
    "candidates"
  ],
  "test_candidate_item_keys": [
    "chunk_id",
    "hybrid_candidate",
    "hybrid_rank",
    "rank",
    "reranker_score"
  ],
  "test_candidate_array_length_samples": [
    [
      "candidates",
      50
    ],
    [
      "candidates",
      50
    ]

<div dir="rtl" align="right">

# Notebook 18 — Context Builder
## NB18-C81 — بررسی ساختار فرادادهٔ حقوق استفاده

### هدف
بررسی ساختار تنظیمات Frozen و فیلدهای حقوق استفاده در Chunkها و اسناد ارجاع‌شده از ده Candidate نخست هر رکورد Test.

### محدودیت‌ها
متن سؤال و شواهد، برچسب‌ها و معیارهای Test خوانده یا چاپ نمی‌شوند. فایل نوشته نمی‌شود و API پولی فراخوانی نمی‌شود.

### خروجی مورد انتظار
نمایش ساختار فیلدهای مرتبط با سیاست و حقوق استفاده، و شمارش اسناد ارجاع‌شده‌ای که با رکوردهای پذیرفته‌شده تطبیق دارند.

</div>

In [82]:
# NB18-C81 — Read-only frozen-policy and rights metadata preflight

from pathlib import Path
from collections import Counter
import json

root = Path(project_root).resolve()

candidates_path = root / (
    "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl"
)
accepted_documents_path = root / (
    "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl"
)

assert candidates_path.is_file()
assert accepted_documents_path.is_file()
assert isinstance(nb18_frozen_configuration, dict)
assert isinstance(chunk_record_by_id, dict)

def config_relevant_scalars(obj, path="$"):
    relevant = (
        "cap", "budget", "order", "rights", "tokenizer",
        "configuration_sha256", "test_access",
    )
    found = {}
    if isinstance(obj, dict):
        for key, value in obj.items():
            child_path = f"{path}.{key}"
            if isinstance(value, (dict, list)):
                found.update(config_relevant_scalars(value, child_path))
            elif any(term in str(key).lower() for term in relevant):
                found[child_path] = value
    elif isinstance(obj, list):
        for index, value in enumerate(obj):
            found.update(config_relevant_scalars(value, f"{path}[{index}]"))
    return found

def field_shape(value):
    if isinstance(value, dict):
        return {"type": "dict", "keys": sorted(map(str, value.keys()))}
    if isinstance(value, list):
        return {"type": "list", "length": len(value)}
    return {"type": type(value).__name__}

def allowed_uses_summary(rights):
    if not isinstance(rights, dict):
        return {"rights_type": type(rights).__name__, "allowed_uses": "MISSING"}

    value = rights.get("allowed_uses", "MISSING")

    if isinstance(value, list):
        normalized = sorted({
            str(item) for item in value
            if isinstance(item, (str, int, float, bool))
        })
        return {
            "rights_type": "dict",
            "allowed_uses_type": "list",
            "allowed_uses_values": normalized[:30],
            "allowed_uses_count": len(value),
        }

    if isinstance(value, dict):
        return {
            "rights_type": "dict",
            "allowed_uses_type": "dict",
            "allowed_uses_keys": sorted(map(str, value.keys())),
        }

    if isinstance(value, (str, int, float, bool)) or value is None:
        return {
            "rights_type": "dict",
            "allowed_uses_type": type(value).__name__,
            "allowed_uses_value": value,
        }

    return {
        "rights_type": "dict",
        "allowed_uses_type": type(value).__name__,
    }

test_record_count = 0
test_chunk_ids = set()
test_chunk_key_shapes = Counter()
test_chunk_rights_shapes = Counter()
test_chunk_allowed_uses_shapes = Counter()
test_chunk_missing_ids = 0
test_chunk_missing_doc_ids = 0
test_referenced_document_ids = set()

with candidates_path.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)
        if record.get("split") != "test":
            continue

        test_record_count += 1
        candidates = record.get("candidates")
        assert isinstance(candidates, list), "Test record has no candidates list."
        assert len(candidates) >= 10, "Test record has fewer than 10 candidates."

        for candidate in candidates[:10]:
            chunk_id = candidate.get("chunk_id")
            if not isinstance(chunk_id, str):
                test_chunk_missing_ids += 1
                continue

            test_chunk_ids.add(chunk_id)
            chunk = chunk_record_by_id.get(chunk_id)
            if not isinstance(chunk, dict):
                test_chunk_missing_ids += 1
                continue

            test_chunk_key_shapes.update([",".join(sorted(chunk.keys()))])
            test_chunk_rights_shapes.update([json.dumps(
                field_shape(chunk.get("rights")),
                sort_keys=True,
                ensure_ascii=False,
            )])
            test_chunk_allowed_uses_shapes.update([json.dumps(
                allowed_uses_summary(chunk.get("rights")),
                sort_keys=True,
                ensure_ascii=False,
            )])

            member_ids = chunk.get("member_document_ids")
            if isinstance(member_ids, list):
                test_referenced_document_ids.update(
                    item for item in member_ids if isinstance(item, str)
                )
            else:
                test_chunk_missing_doc_ids += 1

assert test_record_count == 1149, (
    f"Expected 1149 Test records; found {test_record_count}."
)

document_matches = 0
document_rights_shapes = Counter()
document_allowed_uses_shapes = Counter()
matched_document_ids = set()

with accepted_documents_path.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        document = json.loads(line)
        document_id = document.get("document_id")
        if document_id not in test_referenced_document_ids:
            continue

        matched_document_ids.add(document_id)
        document_matches += 1
        document_rights_shapes.update([json.dumps(
            field_shape(document.get("rights")),
            sort_keys=True,
            ensure_ascii=False,
        )])
        document_allowed_uses_shapes.update([json.dumps(
            allowed_uses_summary(document.get("rights")),
            sort_keys=True,
            ensure_ascii=False,
        )])

print(json.dumps({
    "status": "FROZEN_POLICY_AND_RIGHTS_PREFLIGHT_COMPLETE",
    "frozen_configuration_sha256": nb18_frozen_configuration_sha256,
    "frozen_configuration_relevant_values": config_relevant_scalars(
        nb18_frozen_configuration
    ),
    "test_record_count": test_record_count,
    "unique_test_top10_chunk_id_count": len(test_chunk_ids),
    "test_chunk_missing_or_unresolved_count": test_chunk_missing_ids,
    "test_chunk_missing_member_document_ids_count": test_chunk_missing_doc_ids,
    "test_referenced_document_id_count": len(test_referenced_document_ids),
    "matched_accepted_document_count": document_matches,
    "unmatched_referenced_document_count": (
        len(test_referenced_document_ids - matched_document_ids)
    ),
    "test_chunk_key_shapes": dict(test_chunk_key_shapes),
    "test_chunk_rights_shapes": dict(test_chunk_rights_shapes),
    "test_chunk_allowed_uses_shapes": dict(test_chunk_allowed_uses_shapes),
    "accepted_document_rights_shapes": dict(document_rights_shapes),
    "accepted_document_allowed_uses_shapes": dict(document_allowed_uses_shapes),
    "test_labels_or_metrics_read": False,
    "text_or_query_printed": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "FROZEN_POLICY_AND_RIGHTS_PREFLIGHT_COMPLETE",
  "frozen_configuration_sha256": "0794d5e87d832f15c4e08d53ec6fca8df1e25e2d343443a6fb7911b10353e239",
  "frozen_configuration_relevant_values": {
    "$.development_selection_rule.budget_violations": 0,
    "$.development_selection_rule.query_hit_rate_not_below_uncapped": true,
    "$.development_selection_rule.reranker_order_mismatches": 0,
    "$.selection_policy.document_cap_per_member_document_id": 5,
    "$.selection_policy.selection_order": "reranker_order",
    "$.selection_policy.source_cap_per_source_id": 8,
    "$.selection_policy.source_priority_order": "stored_as_separate_metadata"
  },
  "test_record_count": 1149,
  "unique_test_top10_chunk_id_count": 2910,
  "test_chunk_missing_or_unresolved_count": 0,
  "test_chunk_missing_member_document_ids_count": 0,
  "test_referenced_document_id_count": 16306,
  "matched_accepted_document_count": 16306,
  "unmatched_referenced_document_count": 0,
  "test_chunk_key_shapes": 

<div dir="rtl" align="right">

# Notebook 18 — Context Builder
## NB18-C82 — بررسی فرادادهٔ تو‌در‌توی حقوق استفاده

### هدف
بررسی ساختار و مقادیر دسته‌ای فیلدهای وضعیت، مجوز استفاده و مجوز انتشار در فرادادهٔ حقوق اسناد و Chunkهای ارجاع‌شده از Candidateهای Test.

### محدودیت‌ها
این سل متن سؤال یا شواهد را نمی‌خواند یا چاپ نمی‌کند؛ برچسب‌ها و معیارهای Test خوانده نمی‌شوند. هیچ فایلی نوشته نمی‌شود و API پولی فراخوانی نمی‌شود.

### خروجی مورد انتظار
شمارش وضعیت‌های حقوق و شکل فیلدهای تو‌در‌تو، بدون نمایش محتوای اسناد.

</div>

In [83]:
# NB18-C82 — Read-only nested rights metadata audit

from pathlib import Path
from collections import Counter
import json

root = Path(project_root).resolve()
accepted_documents_path = root / (
    "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl"
)

assert accepted_documents_path.is_file()
assert isinstance(chunk_record_by_id, dict)
assert isinstance(test_referenced_document_ids, set)

def shape_only(value):
    if isinstance(value, dict):
        return {
            "type": "dict",
            "keys": sorted(map(str, value.keys())),
        }
    if isinstance(value, list):
        child_types = sorted({type(item).__name__ for item in value})
        child_key_sets = sorted({
            ",".join(sorted(map(str, item.keys())))
            for item in value
            if isinstance(item, dict)
        })
        return {
            "type": "list",
            "length": len(value),
            "child_types": child_types,
            "dict_child_key_sets": child_key_sets[:20],
        }
    return {"type": type(value).__name__}

def primitive_summary(value):
    if isinstance(value, (str, int, float, bool)) or value is None:
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    return f"<{type(value).__name__}>"

def collect_rights_fields(value, counts):
    if isinstance(value, dict):
        for key, child in value.items():
            normalized_key = str(key).lower()
            if normalized_key in {
                "status",
                "allowed_uses",
                "inferred_release_permission",
            }:
                counts[normalized_key].update([primitive_summary(child)])
            collect_rights_fields(child, counts)
    elif isinstance(value, list):
        for child in value:
            collect_rights_fields(child, counts)

document_count = 0
document_rights_shapes = Counter()
document_rights_field_counts = {
    "status": Counter(),
    "allowed_uses": Counter(),
    "inferred_release_permission": Counter(),
}
matched_document_ids = set()

with accepted_documents_path.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        document = json.loads(line)
        document_id = document.get("document_id")
        if document_id not in test_referenced_document_ids:
            continue

        matched_document_ids.add(document_id)
        document_count += 1
        rights = document.get("rights")
        document_rights_shapes.update([json.dumps(
            shape_only(rights),
            ensure_ascii=False,
            sort_keys=True,
        )])
        collect_rights_fields(rights, document_rights_field_counts)

chunk_count = 0
chunk_rights_shapes = Counter()
chunk_rights_field_counts = {
    "status": Counter(),
    "allowed_uses": Counter(),
    "inferred_release_permission": Counter(),
}
member_rights_shapes = Counter()
member_rights_field_counts = {
    "status": Counter(),
    "allowed_uses": Counter(),
    "inferred_release_permission": Counter(),
}
missing_chunk_count = 0

for chunk_id in test_chunk_ids:
    chunk = chunk_record_by_id.get(chunk_id)
    if not isinstance(chunk, dict):
        missing_chunk_count += 1
        continue

    chunk_count += 1
    rights = chunk.get("rights")
    chunk_rights_shapes.update([json.dumps(
        shape_only(rights),
        ensure_ascii=False,
        sort_keys=True,
    )])
    collect_rights_fields(rights, chunk_rights_field_counts)

    if isinstance(rights, dict) and "member_rights" in rights:
        member_rights = rights["member_rights"]
        member_rights_shapes.update([json.dumps(
            shape_only(member_rights),
            ensure_ascii=False,
            sort_keys=True,
        )])
        collect_rights_fields(member_rights, member_rights_field_counts)

print(json.dumps({
    "status": "NESTED_RIGHTS_METADATA_AUDIT_COMPLETE",
    "test_referenced_document_count": len(test_referenced_document_ids),
    "matched_document_count": document_count,
    "unmatched_document_count": (
        len(test_referenced_document_ids - matched_document_ids)
    ),
    "test_top10_unique_chunk_count": len(test_chunk_ids),
    "resolved_chunk_count": chunk_count,
    "missing_chunk_count": missing_chunk_count,
    "document_rights_shapes": dict(document_rights_shapes),
    "document_rights_field_counts": {
        key: dict(counter)
        for key, counter in document_rights_field_counts.items()
    },
    "chunk_rights_shapes": dict(chunk_rights_shapes),
    "chunk_rights_field_counts": {
        key: dict(counter)
        for key, counter in chunk_rights_field_counts.items()
    },
    "member_rights_shapes": dict(member_rights_shapes),
    "member_rights_field_counts": {
        key: dict(counter)
        for key, counter in member_rights_field_counts.items()
    },
    "test_labels_or_metrics_read": False,
    "document_or_query_text_printed": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "NESTED_RIGHTS_METADATA_AUDIT_COMPLETE",
  "test_referenced_document_count": 16306,
  "matched_document_count": 16306,
  "unmatched_document_count": 0,
  "test_top10_unique_chunk_count": 2910,
  "resolved_chunk_count": 2910,
  "missing_chunk_count": 0,
  "document_rights_shapes": {
    "{\"keys\": [\"allowed_uses\", \"basis\", \"rights_evidence_id\", \"rights_holder\", \"status\"], \"type\": \"dict\"}": 16306
  },
  "document_rights_field_counts": {
    "status": {
      "null": 518,
      "\"EVIDENCE_REQUIRED\"": 15788
    },
    "allowed_uses": {
      "null": 16306
    },
    "inferred_release_permission": {}
  },
  "chunk_rights_shapes": {
    "{\"keys\": [\"inferred_release_permission\", \"member_rights\"], \"type\": \"dict\"}": 2910
  },
  "chunk_rights_field_counts": {
    "status": {
      "null": 969,
      "\"EVIDENCE_REQUIRED\"": 19376
    },
    "allowed_uses": {
      "null": 20345
    },
    "inferred_release_permission": {
      "false": 2910
    }
  },
  "

<div dir="rtl" align="right">

# Notebook 18 — Context Builder
## NB18-C83 — ساخت Contextهای Test در حافظه

### هدف
ساخت Context برای رکوردهای Test با استفاده از تنظیمات Frozen و ترتیب Reranker، همراه با اعمال سقف منبع، سقف سند و بودجهٔ توکن.

### مجوز و محدودیت‌ها
این اجرا با مجوز کاربر برای پردازش داخلی انجام می‌شود. وضعیت حقوق ثبت‌شده در اسناد و Chunkها بدون تغییر حفظ می‌شود؛ این مجوز به‌معنای تأیید انتشار عمومی نیست.

برچسب‌ها و معیارهای Test خوانده نمی‌شوند. فایلی نوشته نمی‌شود و API پولی فراخوانی نمی‌شود.

### خروجی مورد انتظار
۱۱۴۹ بستهٔ Test در حافظه، گزارش شمارش آیتم‌های انتخاب‌شده و حذف‌شده، و تأیید عدم نقض سقف‌ها و بودجه.

</div>

In [84]:
# NB18-C83 — Build Test context packets in memory under frozen policy

from pathlib import Path
from collections import Counter
import json

root = Path(project_root).resolve()
candidates_path = root / (
    "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl"
)
accepted_documents_path = root / (
    "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl"
)

assert candidates_path.is_file()
assert accepted_documents_path.is_file()
assert isinstance(nb18_frozen_configuration, dict)
assert isinstance(chunk_record_by_id, dict)
assert nb18_frozen_configuration_sha256 == (
    "0794d5e87d832f15c4e08d53ec6fca8df1e25e2d343443a6fb7911b10353e239"
)

policy = nb18_frozen_configuration.get("selection_policy")
assert isinstance(policy, dict)
assert policy.get("selection_order") == "reranker_order"
assert policy.get("source_priority_order") == "stored_as_separate_metadata"
assert policy.get("source_cap_per_source_id") == 8
assert policy.get("document_cap_per_member_document_id") == 5

def find_config_value(obj, accepted_keys):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key in accepted_keys:
                return value
        for value in obj.values():
            found = find_config_value(value, accepted_keys)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = find_config_value(value, accepted_keys)
            if found is not None:
                return found
    return None

evidence_token_budget = find_config_value(
    nb18_frozen_configuration,
    {"evidence_only_token_budget", "evidence_text_budget_tokens"},
)
assert evidence_token_budget == 7371, (
    f"Unexpected frozen evidence token budget: {evidence_token_budget!r}"
)

assert "nb18_tokenizer" in globals(), "Frozen tokenizer is not available."
tokenizer = nb18_tokenizer

# Read Test candidate rows and retain only the first ten reranker results.
test_records = []
test_candidate_ids = set()

with candidates_path.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)
        if record.get("split") != "test":
            continue

        candidate_id = record.get("candidate_id")
        assert isinstance(candidate_id, str) and candidate_id
        assert candidate_id not in test_candidate_ids, (
            f"Duplicate Test candidate_id: {candidate_id}"
        )
        test_candidate_ids.add(candidate_id)

        candidates = record.get("candidates")
        assert isinstance(candidates, list) and len(candidates) >= 10
        top_candidates = candidates[:10]
        assert [item.get("rank") for item in top_candidates] == list(range(1, 11))

        test_records.append((candidate_id, top_candidates))

assert len(test_records) == 1149, (
    f"Expected 1149 Test records; found {len(test_records)}."
)

# Identify the document IDs needed by these Test candidates.
required_document_ids = set()
for _, candidates in test_records:
    for candidate in candidates:
        chunk_id = candidate.get("chunk_id")
        chunk = chunk_record_by_id.get(chunk_id)
        assert isinstance(chunk, dict), f"Unresolved canonical chunk: {chunk_id}"

        member_ids = chunk.get("member_document_ids")
        assert isinstance(member_ids, list)
        required_document_ids.update(
            document_id
            for document_id in member_ids
            if isinstance(document_id, str)
        )

# Load only metadata fields for referenced documents into the lookup.
# The source rights metadata is copied as-is and is not interpreted here.
document_metadata_by_id = {}
document_metadata_fields = (
    "document_id",
    "source_type",
    "source_id",
    "title",
    "language",
    "locator",
    "attribution",
    "lineage",
    "content_origin",
    "rights",
)

with accepted_documents_path.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        document = json.loads(line)
        document_id = document.get("document_id")
        if document_id not in required_document_ids:
            continue

        document_metadata_by_id[document_id] = {
            key: document[key]
            for key in document_metadata_fields
            if key in document
        }

missing_document_ids = required_document_ids - set(document_metadata_by_id)
assert not missing_document_ids, (
    f"Unresolved accepted document IDs: {len(missing_document_ids)}"
)

def token_count(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

def make_evidence_item(candidate, chunk):
    text = chunk.get("content")
    assert isinstance(text, str)

    member_ids = chunk.get("member_document_ids", [])
    member_documents = [
        document_metadata_by_id[document_id]
        for document_id in member_ids
        if document_id in document_metadata_by_id
    ]

    return {
        "evidence_id": chunk["chunk_id"],
        "chunk_id": chunk["chunk_id"],
        "reranker_rank": candidate["rank"],
        "reranker_score": candidate.get("reranker_score"),
        "text": text,
        "source_token_count": token_count(text),
        "char_count": chunk.get("char_count"),
        "source": {
            "source_type": chunk.get("source_type"),
            "source_id": chunk.get("source_id"),
            "role": chunk.get("role"),
        },
        "locator": {
            "member_locators": chunk.get("member_locators"),
            "start": chunk.get("locator_start"),
            "end": chunk.get("locator_end"),
        },
        "member_document_ids": member_ids,
        "member_documents": member_documents,
        "attribution_context": chunk.get("attribution_context"),
        "rights": chunk.get("rights"),
        "parent_chunk_id": chunk.get("parent_chunk_id"),
        "lineage": chunk.get("lineage"),
    }

nb18_test_packets_in_memory = []
dropped_by_reason = Counter()
selected_item_count = 0
max_selected_evidence_tokens = 0
source_cap_violation_count = 0
document_cap_violation_count = 0
budget_violation_count = 0
empty_context_count = 0

for candidate_id, candidates in test_records:
    ranked_items = []
    source_priority_metadata = []

    for candidate in candidates:
        chunk = chunk_record_by_id[candidate["chunk_id"]]
        item = make_evidence_item(candidate, chunk)
        ranked_items.append(item)

        # Preserve source metadata separately without changing reranker order.
        source_priority_metadata.append({
            "reranker_rank": candidate["rank"],
            "chunk_id": candidate["chunk_id"],
            "source_type": chunk.get("source_type"),
            "source_id": chunk.get("source_id"),
        })

    selected_items = []
    source_counts = Counter()
    document_counts = Counter()
    selected_tokens = 0
    packet_dropped = Counter()

    # Selection follows the frozen reranker order.
    for item in ranked_items:
        source_id = item["source"].get("source_id")
        member_ids = item.get("member_document_ids", [])
        item_tokens = item["source_token_count"]

        if source_id is not None and source_counts[source_id] >= 8:
            packet_dropped["SOURCE_CAP"] += 1
            continue

        if any(document_counts[document_id] >= 5 for document_id in member_ids):
            packet_dropped["DOCUMENT_CAP"] += 1
            continue

        if selected_tokens + item_tokens > evidence_token_budget:
            packet_dropped["EVIDENCE_TEXT_BUDGET"] += 1
            continue

        selected_items.append(item)
        selected_tokens += item_tokens

        if source_id is not None:
            source_counts[source_id] += 1
        for document_id in member_ids:
            document_counts[document_id] += 1

    packet_source_violations = sum(
        count > 8 for count in source_counts.values()
    )
    packet_document_violations = sum(
        count > 5 for count in document_counts.values()
    )
    packet_budget_violation = selected_tokens > evidence_token_budget

    source_cap_violation_count += packet_source_violations
    document_cap_violation_count += packet_document_violations
    budget_violation_count += int(packet_budget_violation)

    if not selected_items:
        empty_context_count += 1

    selected_item_count += len(selected_items)
    max_selected_evidence_tokens = max(
        max_selected_evidence_tokens, selected_tokens
    )
    dropped_by_reason.update(packet_dropped)

    nb18_test_packets_in_memory.append({
        "candidate_id": candidate_id,
        "split": "test",
        "configuration_sha256": nb18_frozen_configuration_sha256,
        "selection_order": "reranker_order",
        "source_priority_order": "stored_as_separate_metadata",
        "source_priority_metadata": source_priority_metadata,
        "evidence_items": selected_items,
        "selected_evidence_tokens": selected_tokens,
        "dropped_by_reason": dict(packet_dropped),
        "operator_authorized_internal_processing": True,
        "rights_metadata_preserved_without_modification": True,
        "public_release_authorized": False,
    })

assert len(nb18_test_packets_in_memory) == 1149
assert source_cap_violation_count == 0
assert document_cap_violation_count == 0
assert budget_violation_count == 0

print(json.dumps({
    "status": "TEST_CONTEXT_PACKETS_BUILT_IN_MEMORY",
    "test_packet_count": len(nb18_test_packets_in_memory),
    "selected_item_count": selected_item_count,
    "dropped_item_count": sum(dropped_by_reason.values()),
    "dropped_by_reason": dict(dropped_by_reason),
    "empty_context_count": empty_context_count,
    "max_selected_evidence_tokens": max_selected_evidence_tokens,
    "source_cap_violation_count": source_cap_violation_count,
    "document_cap_violation_count": document_cap_violation_count,
    "budget_violation_count": budget_violation_count,
    "configuration_sha256": nb18_frozen_configuration_sha256,
    "test_labels_or_metrics_read": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

AssertionError: Unexpected frozen evidence token budget: None

In [86]:
# TEMP — Locate the frozen evidence token budget; read-only

from pathlib import Path
import json

root = Path(project_root).resolve()
config_path = root / (
    "artifacts/phase2/context_builder/context_builder_configuration_v1.json"
)
adr_path = root / "reports/phase2/context_builder_adr_v1.json"

def matching_values(obj, path="$"):
    found = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            child_path = f"{path}.{key}"
            key_lower = str(key).lower()
            if isinstance(value, (dict, list)):
                found.extend(matching_values(value, child_path))
            elif any(term in key_lower for term in ("token", "budget", "evidence")):
                found.append({
                    "path": child_path,
                    "type": type(value).__name__,
                    "value": value,
                })
    elif isinstance(obj, list):
        for index, value in enumerate(obj):
            found.extend(matching_values(value, f"{path}[{index}]"))
    return found

file_results = {}
for label, path in (("configuration", config_path), ("adr", adr_path)):
    if path.is_file():
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)
        file_results[label] = matching_values(payload)
    else:
        file_results[label] = {"exists": False}

runtime_results = {}
runtime_snapshot = list(globals().items())

for name, value in runtime_snapshot:
    name_lower = name.lower()
    if any(term in name_lower for term in ("token", "budget", "frozen")):
        if isinstance(value, (str, int, float, bool)) or value is None:
            shown_value = value
        elif isinstance(value, dict):
            shown_value = {
                "type": "dict",
                "keys": sorted(map(str, value.keys())),
            }
        else:
            shown_value = {"type": type(value).__name__}

        runtime_results[name] = shown_value

print(json.dumps({
    "status": "READ_ONLY_BUDGET_LOCATION_CHECK",
    "configuration_fields": file_results["configuration"],
    "adr_fields": file_results["adr"],
    "matching_runtime_variables": runtime_results,
    "test_candidate_records_read": False,
    "document_text_read": False,
    "files_written": 0,
    "paid_api_calls": 0,
}, ensure_ascii=False, indent=2))

{
  "status": "READ_ONLY_BUDGET_LOCATION_CHECK",
  "configuration_fields": [
    {
      "path": "$.development_selection_rule.budget_violations",
      "type": "int",
      "value": 0
    },
    {
      "path": "$.rights_gates.external_provider_allowed_requires_explicit_evidence",
      "type": "bool",
      "value": true
    },
    {
      "path": "$.token_budget.tokenizer.special_tokens_sha256",
      "type": "str",
      "value": "94f02d4e8b7a6569bef78a7ff2c3b40744a0fd54e603a9c5a5bdba4d05256a10"
    },
    {
      "path": "$.token_budget.tokens",
      "type": "int",
      "value": 7371
    }
  ],
  "adr_fields": [
    {
      "path": "$.decision.selected_policy_metrics.budget_violation_count",
      "type": "int",
      "value": 0
    },
    {
      "path": "$.decision.selected_policy_metrics.dropped_by_reason.evidence_text_budget",
      "type": "int",
      "value": 0
    },
    {
      "path": "$.decision.selection_rule.budget_violations",
      "type": "int",
      "value": 0
